# Taller · Ajedrez gráfico y agentes inteligentes
## Del agente aleatorio a la búsqueda adversaria
**Inteligencia Artificial · Enunciado para estudiantes**

**Modalidad propuesta:** parejas. **Duración:** dos semanas desde la asignación. El docente comunica la fecha exacta. **Producto principal:** este cuaderno completado, ejecutado y acompañado de los módulos propios, partidas PGN y resultados CSV.

### Desafío
Construyan una aplicación gráfica de ajedrez que permita humano contra humano, humano contra agente y agente contra agente. Implementen varios agentes, expliquen sus decisiones y compárenlos con evidencia reproducible.

Este cuaderno entrega un visor gráfico y un controlador básico de jugadas humanas. Los agentes, la búsqueda, la evaluación y el torneo se desarrollan por ustedes. Los bloques de pseudocódigo especifican contratos y decisiones, no soluciones en Python.

**Propósitos:** aplicar agentes racionales, modelado de estados, búsqueda de capítulos 3–4 y métodos adversarios del capítulo 5 de AIMA; distinguir corrección de reglas, calidad de juego y eficiencia.


## Cuaderno completado · Taller 3 de Inteligencia Artificial (UTP)

**Integrantes:** Santiago Guevara y Januar Mena (en pareja, la modalidad propuesta).

Este es el enunciado del docente **completado**. Conservamos todas sus celdas, tablas y pseudocódigo tal como vienen en el primer commit del repositorio. Debajo de cada sección agregamos nuestro código, pruebas, mediciones y explicaciones.

### Mapa del cuaderno

| Sección del enunciado | Qué agregamos |
|---|---|
| 1. Requisitos | Versiones reales del entorno comparadas con `requirements.txt` |
| 2. Análisis | PEAS, estado, clasificación del entorno, invariantes y política de tablas |
| 3. Base gráfica | Diseño de la interfaz `ChessApp` (tres modos, Deshacer, bucle cooperativo) |
| 4. Arquitectura | Módulos y su responsabilidad; `ChessEnvironment` |
| 5. Agentes | Evaluación E(s); A aleatorio y voraz; B minimax y alfa-beta; C profundización con plazo; D MCTS-UCT, mejora (ajuste de pesos) y registro de agentes |
| 6. Controlador | `GameController`, `ChessApp` y la aplicación con los seis agentes y Humano |
| 7. Integración | Torre (BFS, DFS, UCS, A\*, voraz), ajuste local de pesos, AND-OR, expectiminimax y Kriegspiel |
| 8. Pruebas | Las 7 pruebas pedidas, ejecutadas aquí con `assert`, posiciones propias y fallos corregidos |
| 9. Experimentos | Exp. 1 en vivo; torneo, ablación y MCTS desde los resultados guardados en `resultados/` |
| 11–12 | Autoevaluación con la rúbrica, respuestas de sustentación, conclusiones individuales y lista de entrega |

### Cómo está construido y cómo reproducirlo

El código vive en el paquete `chess_ai/`, con pruebas `pytest` en `tests/`. `tools/construir_cuaderno.py` inserta ese mismo código en las celdas (sin los imports internos), así que el cuaderno es autocontenido: se ejecuta de arriba abajo con un kernel nuevo (`tools/ejecutar_cuaderno.py`), desde la raíz del proyecto. Las tablas y figuras salen de ejecuciones reales: las rápidas se repiten aquí y las largas (torneo, ablación, benchmark de MCTS) se cargan de `resultados/`, donde las escribió `tools/correr_experimentos.py` con el mismo código. Cada decisión y cada error corregido están en `BITACORA.md`. El README explica la instalación.

### Fuentes y ayudas utilizadas (declaración)

- **python-chess** (reglas, jugadas legales, FEN, PGN y SVG). No decide jugadas: no usamos Stockfish ni ningún motor UCI.
- **ipywidgets** (interfaz), **matplotlib** (gráficas), **numpy** (ajuste de pesos), **nbformat/nbclient** (construir y ejecutar el cuaderno) y **pytest**.
- Russell y Norvig, *AIMA* 4.ª ed., caps. 2–5: pseudocódigo de minimax, alfa-beta, MCTS, AND-OR y expectiminimax. Kocsis y Szepesvári (2006) para la constante de UCT. La idea del ajuste estilo «Texel» es de conocimiento común en programación de ajedrez; la implementación es nuestra.
- **Asistente de IA:** usamos **Claude** (Anthropic), mediante **Claude Code**, para programar los módulos, escribir pruebas, correr experimentos y redactar estos apuntes, repartiendo el trabajo por módulo. Revisamos los resultados y preparamos guías de estudio para poder explicar, probar y modificar cualquier parte en la sustentación.
- Reutilizamos ideas y convenciones de nuestros talleres 1 y 2 (búsquedas no informadas, A\*, búsqueda local), citadas en la bitácora.

## 1. Requisitos de ejecución
Plataforma base: Jupyter Notebook o JupyterLab en computador. Colab puede requerir activar el gestor de widgets. No se exige compatibilidad con Pythonista.

Instalen manualmente en el entorno del kernel:

```bash
python -m pip install notebook python-chess ipywidgets matplotlib
```

`python-chess` se usa para reglas, movimientos legales, FEN, PGN y SVG. Los algoritmos de decisión deben ser propios. No se permiten Stockfish ni motores UCI como implementación de agentes. SimpleAI/AIMA-Python se pueden consultar y citar, pero el núcleo adversario del taller se implementa desde cero.

El tablero SVG es gráfico; la entrada base se realiza por selección de casillas en listas y botón. Deben ampliar esta base con selección de agentes y modos de partida. La interacción por clic directo en casillas es una mejora opcional.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Protocol
import importlib.util, random, time, csv
required = {"chess":"python-chess", "ipywidgets":"ipywidgets", "IPython":"ipython"}
missing = [package for module,package in required.items() if importlib.util.find_spec(module) is None]
READY = not missing
if missing:
    print("Faltan dependencias:", ", ".join(missing))
    print("Instálelas manualmente y reinicie el kernel. Las tareas de diseño pueden completarse mientras tanto.")
else:
    import chess, chess.svg, chess.pgn
    import ipywidgets as widgets
    from IPython.display import display, SVG, clear_output
    print("Entorno listo. Versión del módulo chess:",chess.__version__)


In [ ]:
# ===== Entorno de ejecucion: versiones reales frente a las fijadas en requirements.txt =====
# La reproducibilidad incluye las versiones de las bibliotecas, no solo las semillas.
# Esta celda NO instala nada: compara lo que tiene el kernel con lo que probamos.
import os as _os
import platform as _platform
import sys as _sys
from importlib import metadata as _metadata

assert READY, "Faltan dependencias: instalelas con requirements.txt y reinicie el kernel."

_fijadas = {}
if _os.path.exists("requirements.txt"):
    with open("requirements.txt", encoding="utf-8") as _fh:
        for _linea in _fh:
            _linea = _linea.split("#")[0].strip()
            if "==" in _linea:
                _paquete, _version = _linea.split("==")
                _fijadas[_paquete.strip()] = _version.strip()

print(f"Python {_platform.python_version()} ({_platform.system()} {_platform.release()})")
# Solo el nombre del entorno (no la ruta completa, que incluye el usuario de la maquina).
_en_venv = _sys.prefix != _sys.base_prefix
print(f"Entorno virtual: {_os.path.basename(_sys.prefix) if _en_venv else 'NO (Python del sistema)'}"
      " (el probado se llama ia-taller3)")
print(f"{'paquete':<20}{'instalada':<14}{'probada':<12}estado")
_distintas = 0
for _paquete, _probada in _fijadas.items():
    try:
        _instalada = _metadata.version(_paquete)
    except _metadata.PackageNotFoundError:
        _instalada = "(no instalada)"
    _ok = _instalada == _probada
    _distintas += not _ok
    print(f"{_paquete:<20}{_instalada:<14}{_probada:<12}{'igual' if _ok else 'DISTINTA'}")
print("Entorno verificado: Python 3.13.7, Windows 11.")
if _distintas:
    print(f"AVISO: {_distintas} version(es) distinta(s) de las probadas; los tiempos y algunos "
          "desempates podrian variar.")

# El cuaderno lee y escribe en resultados/: debe ejecutarse desde la raiz del proyecto.
assert _os.path.isdir("resultados"), "Ejecute el cuaderno desde la carpeta raiz del proyecto."
print("Directorio de trabajo correcto: existe resultados/ (salidas de las corridas).")


## 2. Análisis antes de programar
Completen un PEAS para su sistema. Definan si medirán la fuerza por puntos, precisión táctica o rendimiento bajo tiempo limitado.

| Componente | Su especificación |
|---|---|
| Desempeño | [Completar: legalidad, puntos, tiempo, reproducibilidad] |
| Entorno | [Completar: tablero, reglas, rival y reloj] |
| Actuadores | [Completar] |
| Sensores | [Completar] |

Describan el estado con suficiente detalle para enroque, captura al paso y repeticiones. Clasifiquen el entorno y expliquen por qué un rival desconocido no vuelve estocásticas las reglas.

**Invariantes:** todo movimiento aplicado debe ser legal; las búsquedas no modifican la posición real; el turno cambia una sola vez por movimiento; una partida terminal no pide otra jugada a un agente.

**Política de tablas del taller:** reclamar automáticamente tablas cuando `outcome(claim_draw=True)` lo permita. Es una convención experimental explícita. El ajedrez real permite decidir si reclamar algunas tablas. El límite de plies del torneo produce un estado “truncada”, separado de las tablas reglamentarias.


### Nuestro análisis (sección 2)

**Integrantes:** Santiago Guevara y Januar Mena.

| Componente | Nuestra especificación |
|---|---|
| **Desempeño** | (1) **Legalidad**: el 100 % de las jugadas aplicadas al estado real son legales; una jugada ilegal o una decisión `None` en un estado no terminal se registran como error, nunca se aplican. (2) **Puntos reglamentarios** bajo un presupuesto temporal fijo por decisión: victoria 1, tablas 0.5, derrota 0, solo sobre partidas terminadas; las truncadas se reportan aparte. (3) **Tiempo**: tiempo por jugada y número de excesos sobre el presupuesto. (4) **Reproducibilidad**: misma semilla y mismo presupuesto por simulaciones o profundidad ⇒ misma decisión. |
| **Entorno** | Tablero 8×8 con las reglas clásicas (python-chess las valida). Rival: humano u otro agente, desconocido para quien decide. Reloj: presupuesto cooperativo por decisión (0.2 s en el torneo, ajustado al equipo) y límite de 160 plies, que produce el estado «truncada». Política de tablas: reclamo automático cuando `outcome(claim_draw=True)` lo permite. |
| **Actuadores** | Elegir **una** jugada legal (en UCI, con la pieza de promoción cuando aplica) y devolverla en un `Decision`. El agente nunca toca el estado real: el controlador valida y aplica la jugada. |
| **Sensores** | Una copia del tablero **con su historia** (`board.copy(stack=True)`): colocación, turno, derechos de enroque, casilla de captura al paso, reloj de 50 jugadas y pila de jugadas; además, el presupuesto (`time_limit_s`) y la semilla. |

**¿Cómo mediremos la fuerza?** La medida principal es la de **puntos bajo un presupuesto temporal** (torneo, Experimento 2), porque compara agentes de naturaleza distinta (profundidad fija, profundización iterativa, MCTS) con el mismo recurso. Como medidas complementarias usamos la **precisión táctica** (acierto en posiciones con solución verificada, Experimento 4) y la **eficiencia** (nodos, hojas y podas a profundidad fija, Experimento 1). La profundidad fija sirve para estudiar alfa-beta; el presupuesto temporal, para comparar fuerza.

**Estado.** El FEN describe colocación, turno, enroques, casilla al paso y relojes, pero **no basta**: la repetición triple depende de las posiciones anteriores. Nuestro estado es el FEN más la historia de posiciones desde la última jugada irreversible. Por eso los agentes reciben `board.copy(stack=True)` y no un FEN aislado (lo comprobamos con una prueba: tras `Nf3 Nf6 Ng1 Ng8` ×2, la historia detecta la repetición y el FEN solo no).

**Clasificación del entorno (AIMA cap. 2).**

| Propiedad | Clasificación | Por qué |
|---|---|---|
| Observabilidad | Totalmente observable | El tablero y la historia son visibles para ambos jugadores. |
| Agentes | Multiagente competitivo | La utilidad de uno es la opuesta de la del otro (suma cero). |
| Determinismo | Determinista | Dado el estado y la jugada, el estado siguiente es único. |
| Episódico/secuencial | Secuencial | Cada jugada condiciona todas las siguientes. |
| Estático/dinámico | Semidinámico con reloj | El tablero no cambia mientras se piensa, pero el presupuesto sí se consume. |
| Discreto/continuo | Discreto | Número finito de estados, jugadas y turnos. |
| Conocido | Conocido | Las reglas (el modelo de transición) son conocidas por completo. |

**¿Por qué un rival desconocido no vuelve estocásticas las reglas?** En AIMA, «estocástico» se refiere al **modelo de transición**: si, dada una acción, el resultado tiene azar. En ajedrez, aplicar una jugada produce siempre la misma posición. Lo que no conocemos es **qué acción elegirá el otro agente**, y eso es incertidumbre estratégica, no azar del entorno. Por eso no se modela con probabilidades (expectiminimax) sino como adversario (minimax): se supone que el rival elige lo peor para nosotros. El azar real (dados) lo estudiamos aparte, en la actividad 7D, sin alterar las reglas del ajedrez.

**Cómo garantizamos los invariantes.**

| Invariante | Mecanismo | Prueba |
|---|---|---|
| Todo movimiento aplicado es legal | `ChessEnvironment.apply` rechaza lo que no esté en `legal_moves` y no cambia el estado | `test_environment_rejects_illegal_and_keeps_state` |
| Las búsquedas no modifican la posición real | Los agentes reciben una copia; dentro de la búsqueda, `push`/`pop` en `try/finally`; FEN y pila se verifican antes y después | Pruebas de «tablero intacto» de cada agente, también con plazo vencido |
| El turno cambia una sola vez por movimiento | El controlador aplica exactamente una jugada por turno y descarta decisiones de una `version` vieja del estado | `tests/test_controller.py` |
| Una partida terminal no pide otra jugada | `status()` es `terminal`/`truncated` ⇒ el controlador no consulta al agente; `apply` rechaza jugadas | `test_terminal_game_accepts_no_moves_and_truncation_is_not_a_draw` |

**Política de tablas.** En la partida real, reclamo automático (`outcome(claim_draw=True)`). Dentro de las búsquedas usamos una prueba terminal equivalente y más barata (`search_outcome`: mate, ahogado, material insuficiente, regla de 50 y repetición triple de la posición del nodo). Los reclamos «antes de mover» aparecen en el árbol como la repetición del hijo correspondiente (decisión D-08 de la bitácora). El límite de 160 plies produce **truncada**, con resultado oficial `*`, nunca tablas.

## 3. Base gráfica suministrada
La clase siguiente permite mover piezas, escoger promoción, deshacer y reiniciar. Muestra turno, historial SAN y posibles destinos de la casilla elegida. El núcleo valida reglas con `python-chess`.

**Prueba manual:** e2–e4, e7–e5, g1–f3; intente después mover una pieza que no corresponde al turno. Compruebe que el estado no cambia. Para una promoción, cargue una posición específica en el tablero y use “Actualizar vista”.


In [ ]:
if READY:
    class ChessView:
        def __init__(self):
            self.board=chess.Board()
            self.origin=widgets.Dropdown(options=chess.SQUARE_NAMES,value="e2",description="Origen")
            self.target=widgets.Dropdown(options=chess.SQUARE_NAMES,value="e4",description="Destino")
            self.promotion=widgets.Dropdown(options=[("Dama","q"),("Torre","r"),("Alfil","b"),("Caballo","n")],description="Promoción")
            self.move_button=widgets.Button(description="Mover",button_style="primary")
            self.undo_button=widgets.Button(description="Deshacer")
            self.reset_button=widgets.Button(description="Nueva partida")
            self.refresh_button=widgets.Button(description="Actualizar vista")
            self.output=widgets.Output()
            self.status=widgets.Label()
            self.move_button.on_click(self.play)
            self.undo_button.on_click(self.undo)
            self.reset_button.on_click(self.reset)
            self.refresh_button.on_click(lambda _:self.render())
            self.origin.observe(lambda _:self.render(),names="value")
            self.ui=widgets.VBox([widgets.HBox([self.origin,self.target]),self.promotion,
                                  widgets.HBox([self.move_button,self.undo_button]),
                                  widgets.HBox([self.reset_button,self.refresh_button]),self.status,self.output])
            self.render()
        def render(self,message=""):
            outcome=self.board.outcome(claim_draw=True)
            turn="blancas" if self.board.turn else "negras"
            self.status.value=message or (f"Resultado: {outcome.result()} · {outcome.termination.name}" if outcome else f"Turno: {turn}")
            sq=chess.parse_square(self.origin.value)
            arrows=[(m.from_square,m.to_square) for m in self.board.legal_moves if m.from_square==sq]
            last=self.board.peek() if self.board.move_stack else None
            with self.output:
                clear_output(wait=True)
                display(SVG(chess.svg.board(self.board,size=460,lastmove=last,arrows=arrows,
                                           check=self.board.king(self.board.turn) if self.board.is_check() else None)))
                replay=self.board.root(); history=[]
                for m in self.board.move_stack:
                    history.append(replay.san(m)); replay.push(m)
                print("Historial SAN:"," ".join(history) or "Sin jugadas")
        def play(self,_=None):
            if self.board.outcome(claim_draw=True):
                self.render("Partida terminada según la política de tablas del taller"); return
            src=chess.parse_square(self.origin.value); dst=chess.parse_square(self.target.value)
            piece=self.board.piece_at(src)
            promo=None
            if piece and piece.piece_type==chess.PAWN and chess.square_rank(dst) in (0,7):
                promo={"q":chess.QUEEN,"r":chess.ROOK,"b":chess.BISHOP,"n":chess.KNIGHT}[self.promotion.value]
            move=chess.Move(src,dst,promotion=promo)
            if move not in self.board.legal_moves:
                self.render("Movimiento ilegal. El tablero conserva su estado."); return
            self.board.push(move); self.render()
        def undo(self,_=None):
            if self.board.move_stack: self.board.pop()
            self.render()
        def reset(self,_=None):
            self.board=chess.Board(); self.render()
    view=ChessView()
    display(view.ui)


### Ampliaciones gráficas obligatorias
1. Selector independiente de agente para blancas y negras, incluyendo humano.
2. Botones “Jugar turno del agente”, “Iniciar/Pausar” y “Nueva partida”.
3. Profundidad, presupuesto temporal y semilla configurables; registro visible de la decisión.
4. Estado de la partida, jaque, último movimiento y resultados de búsqueda por turno.
5. Exportación de partida PGN y tabla CSV de métricas.

Mantengan la interfaz utilizable durante una búsqueda: un callback que ejecuta un torneo completo de forma síncrona bloqueará el kernel y el botón Pausar. Pueden empezar con un turno por botón y luego implementar una tarea cooperativa con cesiones periódicas y cancelación. El agente trabaja sobre una copia y el controlador aplica la jugada solo si sigue correspondiendo al estado vigente.

El botón Deshacer debe tener una política explícita: un ply o una pareja de turnos en modo humano contra agente. No alteren el estado mientras una búsqueda está activa sin cancelarla primero.


## Interfaz gráfica: `ChessApp`

Ampliamos la base suministrada `ChessView` en una clase nueva, `ChessApp` (`chess_ai/gui.py`).
No heredamos de `ChessView` para que el paquete funcione también fuera del cuaderno, pero
conservamos su forma de jugar: listas de origen, destino y promoción, botón **Mover**, destinos
posibles de la casilla elegida y tablero SVG con el último movimiento y el jaque resaltados.
Toda la lógica de turnos está en `GameController` (sección 6); la vista solo llama a sus métodos.

### Cómo usarla

```python
app = ChessApp({"Humano": None,
                "Aleatorio": lambda cfg: RandomAgent(),
                "Alfa-beta": lambda cfg: AlphaBetaAgent(depth=cfg.depth, ordering=True),
                "MCTS": lambda cfg: MCTSAgent(simulations=cfg.simulations)},
               white="Humano", black="Alfa-beta", max_plies=160)
app.display()
```

| Zona | Controles |
|---|---|
| Tablero | SVG con último movimiento, rey en jaque, destinos posibles de la casilla de origen (azul) y, opcionalmente, una flecha con la jugada elegida por el agente. «Ver desde negras» gira el tablero. |
| Jugada humana | Origen, Destino, Promoción y **Mover**. Debajo se listan los destinos legales de la casilla de origen. |
| Partida | **Jugar turno del agente**, **Iniciar/Pausar**, **Nueva partida**, **Deshacer**, **Actualizar vista**, campo FEN (vacío = posición inicial), **Exportar PGN**, **Exportar CSV**. |
| Jugadores y presupuesto | Selector de agente para blancas y para negras (incluye «Humano»), profundidad, plazo en segundos, semilla base, simulaciones MCTS (0 = usar el plazo), máximo de plies (0 = sin límite), pausa entre turnos del bucle y «El agente responde solo». |
| Estado | Modo, turno y de quién es, jaque, último movimiento, estado (en curso / terminal / truncada), resultado oficial, terminación, plies, búsqueda activa y bucle. |
| Historial y registro | Historial SAN numerado y una tabla con cada decisión: ply, color, agente, jugada, valor, nodos, hojas, podas, profundidad completada, tiempo, simulaciones, semilla y si excedió el plazo. |

Las exportaciones se guardan en `resultados/interfaz/<id de partida>.pgn|csv` y además se
muestran en el cuadro de texto inferior. El PGN lleva en los encabezados los agentes, la semilla
base, la política de semillas, el plazo, la tolerancia, la profundidad y las simulaciones. El CSV
tiene exactamente las columnas `CSV_FIELDS` del enunciado.

### Los tres modos

El modo sale de los dos selectores; no hay un selector de modo aparte.

- **Humano contra humano:** ambos selectores en «Humano». Cada jugada se hace con **Mover**.
- **Humano contra agente:** un selector en «Humano» y el otro en un agente. Si «El agente
  responde solo» está activo, tras cada jugada humana válida el agente responde automáticamente
  (con el mismo bucle cooperativo, que se detiene al volver el turno humano). Si está desactivado,
  el agente juega al pulsar **Jugar turno del agente**, útil para mostrar decisiones paso a paso.
  Si el agente lleva blancas, se pulsa **Jugar turno del agente** o **Iniciar** para la primera jugada.
- **Agente contra agente:** ambos selectores en agentes. **Jugar turno del agente** juega un
  solo ply; **Iniciar** juega seguido hasta pausar, terminar o llegar al límite de plies.

El turno se respeta siempre: en el turno de un agente, **Mover** se rechaza, y en un turno humano
**Jugar turno del agente** solo informa que le toca a un humano. Toda jugada humana pasa por
`ChessEnvironment.apply`, que valida la legalidad; una jugada ilegal no cambia el tablero.

### Política de Deshacer

- **Humano contra agente:** deshace hasta devolver el turno al humano. Si el agente ya
  respondió, se deshace la pareja de turnos (respuesta del agente y jugada humana); si el agente
  todavía no había respondido, solo la jugada humana. Así el humano puede corregir su jugada;
  deshacer un solo ply dejaría el turno al agente, que respondería lo mismo.
- **Humano contra humano y agente contra agente:** un ply.
- **Prohibido con una búsqueda activa o con el bucle en marcha:** hay que pausar primero. Esto
  cumple la regla de no alterar el estado mientras una búsqueda está activa sin cancelarla.

### El bucle cooperativo y sus límites

**Iniciar** lanza una tarea `asyncio` en el bucle de eventos del kernel (`run_async`). Cada
vuelta juega un turno, redibuja y cede el control con `await asyncio.sleep(pausa)`. En esa cesión
el kernel procesa los clics pendientes: si se pulsó **Pausar**, el bucle se detiene antes del
siguiente turno. También se detiene si la partida termina, si llega al máximo de plies (estado
**truncada**, resultado `*`), si le toca a un humano o si un agente devuelve algo inválido.

Limitaciones que declaramos:

1. **Un turno individual es síncrono.** Mientras un agente busca (aproximadamente el plazo
   configurado), el kernel está ocupado y no atiende clics; **Pausar** surte efecto al terminar
   ese turno. Con plazos de décimas de segundo la interfaz responde bien; con profundidad fija
   alta un turno puede tardar mucho más.
2. **La cancelación es cooperativa entre turnos**, no dentro de la búsqueda. Dentro de la
   búsqueda, el plazo también es cooperativo (`Deadline`): el agente pregunta y devuelve su
   mejor jugada completa.
3. **El controlador mide después de que el agente devuelve.** Si un agente excede el plazo, se
   registra `time_exceeded` y la jugada se aplica (política uniforme), pero no se puede
   interrumpir a un agente bloqueado. Para eso haría falta aislar al agente en otro proceso, que
   queda fuera del alcance mínimo (sección 6).

### Prueba manual del enunciado

1. Con ambos jugadores en «Humano»: Origen `e2`, Destino `e4`, **Mover**; luego `e7`→`e5` y
   `g1`→`f3`. El historial muestra `1. e4 e5 2. Nf3` y el turno es de negras.
2. **Mover pieza fuera de turno:** con turno de negras, Origen `d2`, Destino `d4`, **Mover**.
   Aparece «Movimiento ilegal… El tablero conserva su estado»; el tablero, el turno y el
   historial no cambian. La lista de destinos ya lo anticipaba: «Sin jugadas legales desde d2».
3. **Promoción cargando una posición:** escribimos en FEN `7k/P7/8/8/8/8/8/7K w - - 0 1` y
   pulsamos **Nueva partida**. Elegimos Origen `a7`, Destino `a8`, Promoción «Caballo» y
   **Mover**: aparece un caballo blanco en a8 (`a8=N`). **Actualizar vista** redibuja sin cambiar
   el estado.

Estas mismas comprobaciones están automatizadas en `tests/test_controller.py`
(`test_human_human_mode_plays_statement_manual_test`, `test_promotion_from_loaded_position`,
`test_gui_smoke_human_moves_and_agent_button`, `test_gui_promotion_and_new_game_from_fen`).

## 4. Arquitectura y contrato común
Separen reglas, decisión, controlador, vista y experimentos. Usen clases y `dataclasses` para configuración y métricas.

| Componente | Responsabilidad |
|---|---|
| `ChessEnvironment` | Estado real, validación y aplicación de movimientos |
| `Agent` | Elegir una jugada sobre una copia, sin tocar la vista |
| `Evaluator` | Evaluación consistente desde blancas o desde una perspectiva documentada |
| `SearchStats` | Nodos, hojas, podas, profundidad, tiempo y simulaciones |
| `GameController` | Alternancia, presupuesto, cancelación y terminación |
| `ChessView` | Controles, tablero y mensajes |
| `Tournament` | Emparejamientos, semillas, CSV y PGN |

El siguiente contrato está suministrado; no implementa agentes. `None` se devuelve únicamente cuando el estado ya es terminal según la política acordada. Un vencimiento de plazo debe devolver la mejor jugada legal disponible, no `None`.


In [ ]:
@dataclass
class SearchStats:
    nodes: int = 0
    leaves: int = 0
    cutoffs: int = 0
    completed_depth: int = 0
    elapsed_s: float = 0.0
    simulations: int = 0

@dataclass
class Decision:
    move: object | None
    score: float | None
    stats: SearchStats

class Agent(Protocol):
    name: str
    def choose(self, board, *, time_limit_s: float, seed: int) -> Decision:
        """Recibe copia con historia. Devuelve jugada legal y métricas."""
        ...

# TODO: escriban sus clases en las celdas de implementación siguientes.


### Nuestra arquitectura (sección 4)

El código vive en el paquete `chess_ai/`, con un módulo por responsabilidad y pruebas `pytest` en `tests/`. Las celdas siguientes contienen **el mismo código** que esos módulos (el cuaderno se genera con `tools/construir_cuaderno.py`), así que el cuaderno es autocontenido y se ejecuta de arriba abajo. Las clases `SearchStats`, `Decision` y `Agent` son las de la celda anterior del enunciado, sin cambios.

| Componente del enunciado | Dónde está | Responsabilidad |
|---|---|---|
| `ChessEnvironment` | `chess_ai/environment.py` | Estado real, legalidad, `version`, deshacer, PGN, estado `ongoing/terminal/truncated` |
| `Agent` (seis) | `agents_basic.py`, `search.py`, `mcts.py` | Elegir una jugada sobre una copia y devolver `Decision` con métricas |
| `Evaluator` | `evaluation.py` | E(s) desde blancas, con 4 características y pesos |
| `SearchStats` | contrato del enunciado | Nodos, hojas, podas, profundidad completada, tiempo y simulaciones |
| `GameController` | `controller.py` | Alternancia, presupuesto, versión vigente, cancelación cooperativa, terminación, registro |
| `ChessApp` (vista) | `gui.py` | Controles, tablero SVG y mensajes; amplía la `ChessView` suministrada |
| `Tournament` | `tournament.py` | Emparejamientos, aperturas, semillas, CSV y PGN |
| Mejora | `tuning.py` | Ajuste de los pesos de E por búsqueda local (train/test separados) |
| Integración caps. 3–5 | `rook_problem.py`, `uncertainty.py` | Torre con BFS/DFS/UCS/A\*/voraz, AND-OR, expectiminimax |

**Flujo de una decisión.** Vista → `GameController.request_turn()` → copia con historia → `agent.choose(copia, time_limit_s, seed)` → `Decision` → el controlador comprueba plazo, legalidad y versión → `ChessEnvironment.apply(jugada)` → registro de métricas → la vista se actualiza.

**Convenciones que comparten todos los agentes** (bitácora D-06 a D-10):

- **Perspectiva:** E(s) y las utilidades terminales se expresan siempre desde **blancas**. Cada agente las convierte con `perspective_sign(color)`; `Decision.score` se reporta desde el jugador que decide.
- **Terminales:** se resuelven **antes** de consultar E. Un mate a `ply` jugadas vale ±(100000 − ply), lo que prefiere mates cortos, y E está acotada a ±10000, así que cualquier mate domina a cualquier estimación.
- **Plazo:** el plazo es **cooperativo** (`Deadline`): la búsqueda pregunta y, si vence, lanza `SearchTimeout`, que el agente captura para devolver la mejor jugada completa.
- **Aleatoriedad:** siempre con `random.Random(seed)` local; nunca el generador global.

In [ ]:
# ===== chess_ai/environment.py =====
"""Entorno de ajedrez: estado real, politica de terminacion, utilidades y plazos.

Convenciones del taller (ver BITACORA D-06 .. D-10):

* PERSPECTIVA. Toda evaluacion E(s) y toda utilidad "absoluta" se expresan desde
  BLANCAS (positivo = bueno para blancas), en centipeones. Cada agente convierte a
  la perspectiva del jugador que decide con `perspective_sign(color)`.
  `Decision.score` se reporta SIEMPRE desde el jugador que decide en la raiz.
* TERMINALES. Se resuelven ANTES de consultar E. Victoria a distancia `ply` vale
  MATE_SCORE - ply (se prefieren mates mas cortos); tablas valen 0. Como
  |E| <= EVAL_BOUND < MATE_SCORE - MAX_MATE_PLY, cualquier mate domina a
  cualquier estimacion no terminal.
* POLITICA DE TABLAS. En la partida real se usa `board.outcome(claim_draw=True)`
  (reclamo automatico, convencion experimental del enunciado). Dentro de la
  busqueda se usa `search_outcome`, equivalente en el arbol pero mas barata
  (ver D-08): no evalua los reclamos "antes de mover", porque ese caso aparece
  como repeticion triple o regla de 50 en el hijo correspondiente.
* TRUNCADA. El limite de plies del torneo NO es un resultado reglamentario: el
  entorno lo reporta como estado "truncated", separado de las tablas.
"""
from __future__ import annotations

import time

import chess
import chess.pgn

MATE_SCORE = 100_000.0
MAX_MATE_PLY = 1_000
EVAL_BOUND = 10_000.0
DRAW_SCORE = 0.0


class SearchTimeout(Exception):
    """Se lanza dentro de una busqueda cuando vence el plazo cooperativo."""


class IllegalMoveError(ValueError):
    """Intento de aplicar al estado real una jugada que no es legal."""


class Deadline:
    """Plazo cooperativo. La busqueda pregunta; nadie la interrumpe desde fuera.

    `time_limit_s` <= 0 o None significa "sin plazo" (profundidad fija pura).
    """

    __slots__ = ("start", "limit", "end")

    def __init__(self, time_limit_s: float | None):
        self.start = time.perf_counter()
        self.limit = time_limit_s if time_limit_s and time_limit_s > 0 else None
        self.end = self.start + self.limit if self.limit is not None else float("inf")

    def expired(self) -> bool:
        return time.perf_counter() >= self.end

    def check(self) -> None:
        if time.perf_counter() >= self.end:
            raise SearchTimeout()

    def elapsed(self) -> float:
        return time.perf_counter() - self.start


def perspective_sign(color: chess.Color) -> int:
    """+1 si `color` es blancas, -1 si es negras (convierte valores desde blancas)."""
    return 1 if color == chess.WHITE else -1


def game_outcome(board: chess.Board) -> chess.Outcome | None:
    """Resultado de la PARTIDA REAL segun la politica del taller (reclamo automatico)."""
    return board.outcome(claim_draw=True)


def search_outcome(board: chess.Board) -> chess.Outcome | None:
    """Prueba terminal usada DENTRO de las busquedas (mas barata que claim_draw=True).

    Orden: mate/ahogado (sin jugadas legales), material insuficiente, regla de 50
    (halfmove_clock >= 100) y repeticion triple de la posicion actual.
    """
    if not any(board.generate_legal_moves()):
        if board.is_check():
            return chess.Outcome(chess.Termination.CHECKMATE, not board.turn)
        return chess.Outcome(chess.Termination.STALEMATE, None)
    if board.is_insufficient_material():
        return chess.Outcome(chess.Termination.INSUFFICIENT_MATERIAL, None)
    if board.halfmove_clock >= 100:
        return chess.Outcome(chess.Termination.FIFTY_MOVES, None)
    if board.is_repetition(3):
        return chess.Outcome(chess.Termination.THREEFOLD_REPETITION, None)
    return None


def terminal_value_white(outcome: chess.Outcome, ply: int = 0) -> float:
    """Utilidad exacta desde blancas de un estado terminal alcanzado a `ply` de la raiz."""
    if outcome.winner is None:
        return DRAW_SCORE
    value = MATE_SCORE - min(ply, MAX_MATE_PLY)
    return value if outcome.winner == chess.WHITE else -value


def is_mate_score(value: float) -> bool:
    return abs(value) >= MATE_SCORE - MAX_MATE_PLY


class ChessEnvironment:
    """Estado REAL de la partida. Unico lugar donde se aplican jugadas.

    Invariantes: toda jugada aplicada es legal; el turno cambia una sola vez por
    jugada; una partida terminal o truncada no acepta mas jugadas; `version`
    aumenta en cada cambio para que el controlador descarte decisiones obsoletas.
    """

    def __init__(self, fen: str | None = None, max_plies: int | None = None):
        self.start_fen = fen or chess.STARTING_FEN
        self.board = chess.Board(self.start_fen)
        self.max_plies = max_plies
        self.version = 0

    # --- consultas -------------------------------------------------------
    @property
    def ply_count(self) -> int:
        return len(self.board.move_stack)

    def outcome(self) -> chess.Outcome | None:
        return game_outcome(self.board)

    def is_truncated(self) -> bool:
        return (self.max_plies is not None and self.ply_count >= self.max_plies
                and self.outcome() is None)

    def status(self) -> str:
        """'ongoing' | 'terminal' | 'truncated'."""
        if self.outcome() is not None:
            return "terminal"
        if self.is_truncated():
            return "truncated"
        return "ongoing"

    def is_over(self) -> bool:
        return self.status() != "ongoing"

    def official_result(self) -> str:
        """'1-0', '0-1', '1/2-1/2' o '*' (en curso o truncada: sin resultado oficial)."""
        out = self.outcome()
        return out.result() if out is not None else "*"

    def termination(self) -> str:
        out = self.outcome()
        if out is not None:
            return out.termination.name.lower()
        return "truncated" if self.is_truncated() else "ongoing"

    def copy_for_agent(self) -> chess.Board:
        """Copia con la pila completa: repeticiones y reclamos siguen siendo detectables."""
        return self.board.copy(stack=True)

    def san_history(self) -> list[str]:
        replay = chess.Board(self.start_fen)
        out = []
        for mv in self.board.move_stack:
            out.append(replay.san(mv))
            replay.push(mv)
        return out

    # --- cambios -----------------------------------------------------------
    def apply(self, move: chess.Move) -> None:
        if self.is_over():
            raise IllegalMoveError("la partida ya termino (" + self.termination() + ")")
        if move is None or move not in self.board.legal_moves:
            raise IllegalMoveError(f"jugada ilegal en {self.board.fen()}: {move}")
        self.board.push(move)
        self.version += 1

    def undo(self, plies: int = 1) -> int:
        """Deshace hasta `plies` jugadas; devuelve cuantas deshizo."""
        done = 0
        while done < plies and self.board.move_stack:
            self.board.pop()
            done += 1
        if done:
            self.version += 1
        return done

    def reset(self, fen: str | None = None) -> None:
        if fen is not None:
            self.start_fen = fen
        self.board = chess.Board(self.start_fen)
        self.version += 1

    def to_pgn(self, headers: dict | None = None) -> chess.pgn.Game:
        game = chess.pgn.Game.from_board(self.board)
        if self.start_fen != chess.STARTING_FEN:
            game.headers["FEN"] = self.start_fen
            game.headers["SetUp"] = "1"
        game.headers["Result"] = self.official_result()
        game.headers["Termination"] = self.termination()
        for k, v in (headers or {}).items():
            game.headers[k] = str(v)
        return game


## 5. Agentes que deben construir
Se requieren **seis agentes**. Todos comparten el contrato y la política de terminación.

| Agente | Decisión | Qué permite estudiar |
|---|---|---|
| Aleatorio | Elegir uniformemente entre movimientos legales | Línea base y semillas |
| Voraz de un ply | Evaluar cada sucesor inmediato | Sesgo de corto plazo |
| Minimax | Alternar MAX/MIN hasta profundidad fija | Respuesta adversaria |
| Alfa-beta | Mismo árbol y evaluación, con poda | Ahorro sin cambiar el valor de raíz |
| Alfa-beta mejorado | Ordenación + profundización iterativa con plazo | Calidad bajo recursos limitados |
| MCTS-UCT | Selección, expansión, rollout y respaldo | Exploración frente a explotación |

En el último agente documenten una política de rollout y un límite. Un rollout truncado puede recibir una evaluación normalizada a la misma escala de recompensas, pero debe registrarse como truncado, no como tabla. Elijan la jugada final por una regla documentada, por ejemplo mayor número de visitas.

**Mejora adicional a escoger:** quietud; tabla de transposición con tipos de cota y profundidad; o evaluación cuyos pesos ajustan mediante búsqueda local. Al menos una debe implementarse y compararse mediante ablación.

### Evaluación común
Propongan $E(s)=w_mM(s)+w_pP(s)+w_kK(s)+w_aA(s)$, o una alternativa justificada. Definan cada característica, escala, signo y costo. Los valores terminales se resuelven antes de consultar $E$ y deben dominar a las estimaciones no terminales. Si bonifican mates más cortos, documenten el ajuste por distancia y su interacción con la caché.

Por ejemplo, pueden medir material y movilidad, añadir estructura de peones y seguridad del rey. No basta con asignar números a las piezas: expliquen qué decisiones pretende inducir cada término. La evaluación de negras debe favorecer a negras cuando ese color decide, de acuerdo con la perspectiva que hayan elegido.


### Nuestra evaluación común E(s) (sección 5)

Usamos la forma propuesta, $E(s)=w_mM(s)+w_pP(s)+w_kK(s)+w_aA(s)$, en centipeones y **siempre desde blancas**: cada característica es «valor de blancas − valor de negras», así que un número positivo es bueno para blancas en cualquier nodo y en cualquier partida (bitácora D-06 y D-11). El código está en la celda siguiente (`chess_ai/evaluation.py`); su docstring tiene la tabla completa.

| Característica | Definición | Escala típica | Costo | Qué decisión pretende inducir |
|---|---|---|---|---|
| $M$ material | P = 1, C = 3,2, A = 3,3, T = 5, D = 9 (en peones) | −39 … +39 | O(1) con bitboards | no regalar piezas; cambiar cuando se va ganando |
| $P$ peones | pasados (1 + avance/6) − doblados − aislados | ≈ −6 … 6 | O(peones), máscaras precalculadas | crear y empujar pasados; evitar peones débiles |
| $K$ rey | (escudo de peones − casillas de la zona del rey atacadas) × fase | ≈ −9 … 3 | 9 consultas de ataque | enrocar y no abrir el rey en el medio juego; en finales la fase → 0 y el término se apaga |
| $A$ actividad | casillas atacadas por C, A, T, D que no ocupan piezas propias | ≈ −40 … 40 | O(piezas) | desarrollar piezas y abrir líneas |

**Pesos iniciales** $E_0 = (100, 15, 10, 4)$ centipeones por unidad. Ningún término posicional pesa más que un peón por unidad, porque «no regalar material» debe dominar. Los pesos ajustados $E^*$ salen de la búsqueda local de la sección 7C.

**Terminales antes que E.** Ninguna búsqueda consulta $E$ en un estado terminal: primero se resuelve el resultado exacto (`search_outcome`). Una victoria a `ply` jugadas de la raíz vale $\pm(100\,000 - \text{ply})$, las tablas valen 0 y $E$ se recorta a ±10 000, así que **cualquier mate domina a cualquier estimación**. Restar el ply hace preferir el mate más corto. No usamos tabla de transposición; si se usara, ese valor de mate tendría que guardarse como «distancia desde el nodo» y convertirse de vuelta al leerlo (D-07).

**Perspectiva de negras.** Como $E$ está desde blancas, cuando deciden negras los agentes **minimizan** $E$ (equivale a maximizar $-E$). La celda que sigue al código lo comprueba en cuatro posiciones: al reflejar el tablero y cambiar los colores, $E$ cambia de signo exactamente.

In [ ]:
# ===== chess_ai/evaluation.py =====
"""Evaluacion comun E(s) = wm*M(s) + wp*P(s) + wk*K(s) + wa*A(s), desde BLANCAS.

Cada caracteristica es (valor de blancas) - (valor de negras), asi que el signo
es siempre "positivo = mejor para blancas" y la evaluacion es antisimetrica:
reflejar el tablero e intercambiar colores invierte el signo (prueba en tests).

| Car. | Definicion                                            | Escala tipica | Que decision pretende inducir                         | Costo |
|------|-------------------------------------------------------|---------------|-------------------------------------------------------|-------|
| M    | material en peones: P=1, N=3.2, B=3.3, R=5, Q=9       | -39 .. +39    | no regalar piezas; cambiar cuando se va ganando       | O(1) bitboards |
| P    | estructura de peones: pasados (con bono por avance)   | aprox -6 .. 6 | crear y empujar pasados; evitar doblados y aislados   | O(peones) |
|      | menos doblados menos aislados                         |               |                                                       |       |
| K    | seguridad del rey: escudo de peones menos casillas    | aprox -9 .. 3 | enrocar y no abrir el rey en el medio juego           | O(9) ataques |
|      | de la zona del rey atacadas por el rival, x fase      |               | (en finales la fase -> 0 y el termino se apaga)       |       |
| A    | actividad: casillas atacadas por N, B, R, Q que no    | aprox -40..40 | desarrollar piezas y abrir lineas                     | O(piezas) |
|      | estan ocupadas por piezas propias                     |               |                                                       |       |

Pesos por defecto (centipeones por unidad): wm=100, wp=15, wk=10, wa=4.
Los valores terminales NO pasan por aqui (ver environment.terminal_value_white);
el resultado se recorta a +-EVAL_BOUND para que un mate siempre domine.
"""
from __future__ import annotations

import chess


PIECE_VALUES = {chess.PAWN: 1.0, chess.KNIGHT: 3.2, chess.BISHOP: 3.3,
                chess.ROOK: 5.0, chess.QUEEN: 9.0, chess.KING: 0.0}
DEFAULT_WEIGHTS = (100.0, 15.0, 10.0, 4.0)
FEATURE_NAMES = ("M_material", "P_peones", "K_rey", "A_actividad")
# Material no-peon inicial de ambos bandos: 2*(2*3.2 + 2*3.3 + 2*5 + 9)
_FULL_PHASE = 2 * (2 * 3.2 + 2 * 3.3 + 2 * 5.0 + 9.0)


def _passed_mask(color: chess.Color, sq: int) -> int:
    """Casillas delante del peon en su columna y las vecinas: si no hay peon rival ahi, es pasado."""
    f, r = chess.square_file(sq), chess.square_rank(sq)
    ahead = 0
    ranks = range(r + 1, 8) if color == chess.WHITE else range(0, r)
    for rr in ranks:
        for ff in (f - 1, f, f + 1):
            if 0 <= ff < 8:
                ahead |= chess.BB_SQUARES[chess.square(ff, rr)]
    return ahead


# Precalculadas una vez (2 x 64 enteros): la version que las construia en cada
# llamada gastaba ~la mitad del costo de E (bitacora S-13 y E-12).
_PASSED_MASK = {c: [_passed_mask(c, sq) for sq in chess.SQUARES] for c in (chess.WHITE, chess.BLACK)}


def _material(board: chess.Board, color: chess.Color) -> float:
    return sum(PIECE_VALUES[pt] * chess.popcount(board.pieces_mask(pt, color))
               for pt in (chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN))


def _pawn_structure(board: chess.Board, color: chess.Color) -> float:
    own = board.pieces_mask(chess.PAWN, color)
    enemy = board.pieces_mask(chess.PAWN, not color)
    score = 0.0
    for f in range(8):
        n = chess.popcount(own & chess.BB_FILES[f])
        if n == 0:
            continue
        if n > 1:
            score -= n - 1                       # doblados
        neighbours = 0
        if f > 0:
            neighbours |= chess.BB_FILES[f - 1]
        if f < 7:
            neighbours |= chess.BB_FILES[f + 1]
        if not own & neighbours:
            score -= n                           # aislados
    masks = _PASSED_MASK[color]
    for sq in chess.scan_forward(own):
        if not enemy & masks[sq]:                # pasado: bono creciente con el avance
            r = chess.square_rank(sq)
            advance = r if color == chess.WHITE else 7 - r
            score += 1.0 + advance / 6.0
    return score


def _phase(board: chess.Board) -> float:
    non_pawn = sum(PIECE_VALUES[pt] * chess.popcount(board.pieces_mask(pt, c))
                   for c in (chess.WHITE, chess.BLACK)
                   for pt in (chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN))
    return min(1.0, non_pawn / _FULL_PHASE)


def _king_safety(board: chess.Board, color: chess.Color) -> float:
    king = board.king(color)
    if king is None:
        return 0.0
    f, r = chess.square_file(king), chess.square_rank(king)
    step = 1 if color == chess.WHITE else -1
    shield = 0
    own_pawns = board.pieces_mask(chess.PAWN, color)
    for dr in (step, 2 * step):
        rr = r + dr
        if 0 <= rr < 8:
            for ff in (f - 1, f, f + 1):
                if 0 <= ff < 8 and own_pawns & chess.BB_SQUARES[chess.square(ff, rr)]:
                    shield += 1
    shield = min(shield, 3)
    zone = chess.BB_KING_ATTACKS[king] | chess.BB_SQUARES[king]
    attacked = sum(1 for sq in chess.scan_forward(zone) if board.is_attacked_by(not color, sq))
    return float(shield - attacked)


def _activity(board: chess.Board, color: chess.Color) -> float:
    own_occ = board.occupied_co[color]
    total = 0
    for pt in (chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN):
        for sq in chess.scan_forward(board.pieces_mask(pt, color)):
            total += chess.popcount(board.attacks_mask(sq) & ~own_occ)
    return float(total)


def features(board: chess.Board) -> tuple[float, float, float, float]:
    """Vector (M, P, K, A) desde blancas. Base del ajuste de pesos (busqueda local)."""
    w, b = chess.WHITE, chess.BLACK
    phase = _phase(board)
    return (
        _material(board, w) - _material(board, b),
        _pawn_structure(board, w) - _pawn_structure(board, b),
        phase * (_king_safety(board, w) - _king_safety(board, b)),
        _activity(board, w) - _activity(board, b),
    )


def evaluate(board: chess.Board, weights=DEFAULT_WEIGHTS) -> float:
    """E(s) en centipeones desde blancas, recortada a +-EVAL_BOUND. No mira terminales."""
    value = sum(wi * fi for wi, fi in zip(weights, features(board)))
    return max(-EVAL_BOUND, min(EVAL_BOUND, value))


class Evaluator:
    """Evaluador con pesos fijos. Es invocable: `ev(board)` -> centipeones desde blancas."""

    def __init__(self, weights=DEFAULT_WEIGHTS, name: str = "E0"):
        if len(weights) != 4:
            raise ValueError("se esperan 4 pesos (wm, wp, wk, wa)")
        self.weights = tuple(float(x) for x in weights)
        self.name = name

    def __call__(self, board: chess.Board) -> float:
        return evaluate(board, self.weights)

    def features(self, board: chess.Board):
        return features(board)

    def __repr__(self) -> str:
        return f"Evaluator({self.name}, weights={self.weights})"


DEFAULT_EVALUATOR = Evaluator(DEFAULT_WEIGHTS, "E0")


In [ ]:
# ===== E(s) en algunas posiciones: caracteristicas, signo y dominio de los terminales =====
# (POSITIONS del enunciado se define mas adelante, en la seccion 8; aqui van FEN explicitos.)
_fens_e = {
    "inicial": chess.STARTING_FEN,
    "italiana": "r1bqk1nr/pppp1ppp/2n5/2b1p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4",
    "pasado_b": "8/8/1k6/8/8/6K1/1p6/8 w - - 0 1",
    "rey_abierto": "rnbq1rk1/ppp2p1p/3p1np1/8/3PP3/2N2N2/PPP2PPP/R1BQKB1R w KQ - 0 7",
}
print(f"{'posicion':<13}{'M':>7}{'P':>8}{'K':>8}{'A':>7}{'E0 (cp)':>10}{'E0 reflejada':>14}")
for _nombre, _fen in _fens_e.items():
    _b = chess.Board(_fen)
    _f = features(_b)
    _e, _e_ref = DEFAULT_EVALUATOR(_b), DEFAULT_EVALUATOR(_b.mirror())
    assert abs(_e + _e_ref) < 1e-9          # antisimetria: reflejar e intercambiar colores invierte el signo
    print(f"{_nombre:<13}{_f[0]:>7.1f}{_f[1]:>8.2f}{_f[2]:>8.2f}{_f[3]:>7.0f}{_e:>10.1f}{_e_ref:>14.1f}")

# Un mate siempre domina: la mayor estimacion posible frente al mate mas lejano representable.
_mate = terminal_value_white(chess.Board("7k/6Q1/5K2/8/8/8/8/8 b - - 0 1").outcome(), ply=MAX_MATE_PLY)
assert _mate > EVAL_BOUND
print(f"\nCota de E: +-{EVAL_BOUND:.0f} cp. Mate de blancas a {MAX_MATE_PLY} plies: {_mate:.0f} cp "
      f"(domina a cualquier E). Tablas: {DRAW_SCORE:.0f}.")


### Pseudocódigo orientador
```text
DECIDIR_MINIMAX(estado, profundidad):
    si estado es terminal: devolver su utilidad exacta
    si profundidad es cero: devolver E(estado)
    obtener acciones legales
    evaluar sucesores alternando MAX y MIN
    respaldar el máximo o mínimo según el turno
    devolver valor y primera acción de una variante óptima

DECIDIR_ALFABETA(estado, profundidad, alfa, beta):
    comprobar terminal antes de corte
    recorrer los sucesores en el orden elegido
    actualizar el mejor valor y la cota correspondiente
    detener el recorrido cuando las cotas justifican la poda
    registrar el evento de poda y devolver valor y acción

PROFUNDIZACION_ITERATIVA(estado, plazo):
    guardar una jugada legal de respaldo
    repetir con profundidades crecientes
        buscar sin exceder el plazo
        si la iteración termina: reemplazar la decisión guardada
        si no termina: conservar la última decisión completa
    devolver decisión y profundidad realmente completada

MCTS(estado, presupuesto):
    crear raíz con historial y jugador de referencia
    repetir hasta agotar presupuesto
        seleccionar usando UCT con perspectiva correcta
        expandir una acción todavía no explorada
        simular hasta terminal o límite documentado
        retropropagar visitas y recompensa de forma consistente
    escoger un hijo con la regla final definida
```

**Decisiones que deben resolver:** dónde verificar el plazo, qué cuenta como nodo, cómo desempatar, cómo restaurar un tablero tras una excepción y qué perspectiva conserva cada valor. Usar `push` debe acompañarse de `pop` en un bloque `finally` cuando corresponda.


In [ ]:
# IMPLEMENTACIÓN A: agente aleatorio, agente voraz y evaluación.
# Complete con su código. Verifique reproducibilidad y perspectiva de color.


In [ ]:
# ===== chess_ai/agents_basic.py =====
"""Agentes basicos (seccion 5 del enunciado): Aleatorio y Voraz de un ply.

Ambos cumplen el contrato `choose(board, *, time_limit_s, seed) -> Decision`:

* Reciben una COPIA del tablero con historia y la devuelven intacta (FEN y
  `move_stack`): el voraz hace push/pop dentro de `try/finally`.
* `Decision.move is None` solo si el estado ya es terminal segun
  `search_outcome` (mate, ahogado, material insuficiente, regla de 50 o
  repeticion triple). En ese caso `Decision.score` es la utilidad exacta del
  estado desde el jugador al turno (p. ej. -MATE_SCORE si le dieron mate).
* `Decision.score` se reporta SIEMPRE desde el jugador que decide en la raiz
  (BITACORA D-06): positivo = bueno para quien mueve.
* Todo lo aleatorio usa `random.Random(seed)` propio de la llamada: misma
  posicion + misma semilla -> misma jugada, sin depender del estado global.

Definiciones de metricas (SearchStats) en estos agentes:

| Campo           | Aleatorio                   | Voraz                                        |
|-----------------|-----------------------------|----------------------------------------------|
| nodes           | 0 (no genera sucesores)     | sucesores generados (push) = jugadas legales |
| leaves          | 0 (no evalua nada)          | sucesores evaluados (terminal exacto o E)    |
| cutoffs         | 0                           | 0 (no hay poda)                              |
| completed_depth | 0                           | 1 si evaluo todos los sucesores; 0 si el     |
|                 |                             | plazo corto la lista (devuelve el mejor visto)|
| elapsed_s       | tiempo de pared de choose   | tiempo de pared de choose                    |
| simulations     | 0                           | 0                                            |
"""
from __future__ import annotations

import random

import chess


# Tolerancia para considerar empatados dos valores de E (sumas de flotantes).
_BASIC_TIE_EPS = 1e-9

# ---------------------------------------------------------------------------
# Posicion donde el voraz FALLA (pregunta 1 de la sustentacion).
#
#   Negras: Rg8 (rey), Re8 (torre), peones b7 f7 g7 h7.
#   Blancas: Rg1 (rey), Db6 (dama), peones f2 g2 h2. Juegan blancas.
#
# Negras amenazan ...Te1# (mate del pasillo: el rey blanco esta encerrado por
# sus propios peones f2 g2 h2 y nada cubre la primera fila). El voraz mira un
# solo ply: ve que Dxb7 gana un peon (+100 cp de material) y no ve la respuesta
# ...Te1#. Un minimax de profundidad 2 si genera la respuesta de negras, ve que
# tras Dxb7 llega un mate (utilidad -MATE_SCORE + 2) y escoge una DEFENSA:
# cubrir e1 con la dama (Db4, la que eligio un minimax-2 de prueba con E0) o
# abrir una casilla de escape al rey (h3, g3, f3, ...).
# El caso lo verifica `tests/test_agents_basic.py::test_greedy_falls_into_back_rank_trap`;
# la parte de minimax la verifica la prueba del modulo de busqueda.
GREEDY_TRAP_FEN = "4r1k1/1p3ppp/1Q6/8/8/8/5PPP/6K1 w - - 0 1"
GREEDY_TRAP_GREEDY_MOVE = "b6b7"     # lo que juega el voraz: Dxb7??
GREEDY_TRAP_REFUTATION = "e8e1"      # respuesta de negras: Te1#


def _basic_sorted_moves(board: chess.Board) -> list:
    """Jugadas legales ordenadas por UCI.

    python-chess ya genera las jugadas en un orden determinista, pero ese orden
    es un detalle interno de la biblioteca. Ordenar por la cadena UCI fija el
    orden ANTES de muestrear o desempatar: la jugada que sale con una semilla
    depende solo de la posicion y de la semilla, no de la version del generador.
    """
    return sorted(board.legal_moves, key=lambda m: m.uci())


def _basic_terminal_decision(board: chess.Board, outcome, deadline: Deadline) -> Decision:
    """Decision para un estado ya terminal: sin jugada y con su utilidad exacta."""
    value = perspective_sign(board.turn) * terminal_value_white(outcome, 0)
    return Decision(move=None, score=value,
                    stats=SearchStats(elapsed_s=deadline.elapsed()))


class RandomAgent:
    """Agente Aleatorio: elige uniformemente entre las jugadas legales.

    Es la linea base del torneo. No evalua posiciones, por eso `score` es None
    en estados no terminales. Reproducible: `random.Random(seed)` sobre la lista
    de jugadas ordenada por UCI.
    """

    def __init__(self, name: str = "Aleatorio"):
        self.name = name

    def choose(self, board: chess.Board, *, time_limit_s: float = 0.0, seed: int = 0) -> Decision:
        deadline = Deadline(time_limit_s)
        outcome = search_outcome(board)
        if outcome is not None:
            return _basic_terminal_decision(board, outcome, deadline)
        rng = random.Random(seed)
        move = rng.choice(_basic_sorted_moves(board))
        return Decision(move=move, score=None,
                        stats=SearchStats(elapsed_s=deadline.elapsed()))

    def __repr__(self) -> str:
        return f"RandomAgent({self.name!r})"


class GreedyAgent:
    """Agente Voraz de un ply: evalua cada sucesor inmediato y toma el mejor.

    Para cada jugada legal (en orden UCI):
      1. push de la jugada;
      2. si el sucesor es terminal (`search_outcome`), su valor es la utilidad
         exacta a distancia 1: un mate inmediato vale MATE_SCORE - 1 y domina a
         cualquier E; tablas valen 0;
      3. si no, su valor es E(sucesor) desde blancas;
      4. pop (en `finally`, aunque algo falle).
    El valor desde blancas se convierte con `perspective_sign` del que decide,
    y se maximiza. Asi negras prefieren valores de E bajos (buenos para negras).

    Desempate: entre las jugadas cuyo valor esta a menos de 1e-9 del maximo se
    elige una con `random.Random(seed)` (lista en orden UCI). Es reproducible y
    evita el sesgo sistematico de "la primera jugada del generador" en
    posiciones tranquilas donde muchas jugadas valen lo mismo.

    Plazo: es cooperativo (D-10). Se revisa despues de cada sucesor; si vence,
    se devuelve la mejor jugada vista (siempre hay al menos una evaluada) con
    `completed_depth = 0`. Con plazo <= 0 no hay limite.

    `evaluator` es cualquier invocable `board -> centipeones desde blancas`
    (un `Evaluator` o una funcion), lo que permite usar el voraz con pesos
    ajustados o con solo material en el modulo de ajuste.
    """

    def __init__(self, evaluator=DEFAULT_EVALUATOR, name: str = "Voraz"):
        self.evaluator = evaluator
        self.name = name

    def choose(self, board: chess.Board, *, time_limit_s: float = 0.0, seed: int = 0) -> Decision:
        deadline = Deadline(time_limit_s)
        outcome = search_outcome(board)
        if outcome is not None:
            return _basic_terminal_decision(board, outcome, deadline)

        sign = perspective_sign(board.turn)
        stats = SearchStats()
        scored = []                       # (valor desde quien decide, jugada)
        moves = _basic_sorted_moves(board)
        for move in moves:
            board.push(move)
            try:
                stats.nodes += 1
                child = search_outcome(board)
                if child is not None:
                    value_white = terminal_value_white(child, 1)
                else:
                    value_white = self.evaluator(board)
                stats.leaves += 1
            finally:
                board.pop()
            scored.append((sign * value_white, move))
            if deadline.expired():
                break

        best = max(v for v, _ in scored)
        ties = [m for v, m in scored if v >= best - _BASIC_TIE_EPS]
        move = ties[0] if len(ties) == 1 else random.Random(seed).choice(ties)
        stats.completed_depth = 1 if len(scored) == len(moves) else 0
        stats.elapsed_s = deadline.elapsed()
        return Decision(move=move, score=best, stats=stats)

    def __repr__(self) -> str:
        return f"GreedyAgent({self.name!r}, {self.evaluator!r})"


### 5A. Agentes básicos: Aleatorio y Voraz de un ply

Implementamos los dos agentes básicos en `chess_ai/agents_basic.py`. Ambos cumplen el contrato
común `choose(board, *, time_limit_s, seed) -> Decision`, reciben una copia del tablero con su
historia y la devuelven intacta, y devuelven `move=None` **solo** cuando el estado ya es terminal
según `search_outcome` (en ese caso `score` es la utilidad exacta desde el jugador al turno: en la
posición `checkmate` del enunciado, negras reciben `-MATE_SCORE`).

**Aleatorio.** Elige uniformemente entre las jugadas legales con `random.Random(seed)`. Antes de
muestrear ordenamos las jugadas por su cadena UCI: así la jugada que sale depende solo de la
posición y de la semilla, y no del orden interno del generador de `python-chess`. No evalúa
posiciones, por eso su `score` es `None` y sus contadores de nodos y hojas son 0. Es la línea base
del torneo y la herramienta para estudiar el efecto de las semillas. Comprobamos que con 2000
semillas en la posición inicial aparecen las 20 jugadas y ninguna sale más del triple de veces que
la menos frecuente (prueba `test_random_agent_depends_on_seed_and_covers_all_moves`).

**Voraz de un ply.** Para cada jugada legal (en orden UCI) hacemos `push`, valoramos el sucesor y
hacemos `pop` dentro de un bloque `finally`. La valoración resuelve primero los terminales: un mate
inmediato vale `MATE_SCORE − 1` y domina a cualquier estimación; las tablas valen 0. Solo si el
sucesor no es terminal consultamos `E(s)`, que está desde blancas; el valor se multiplica por
`perspective_sign` del color que decide y se maximiza. Por eso, cuando juegan negras, el voraz
prefiere los valores de `E` más bajos, que son los buenos para negras (prueba con una posición
propia en la que negras ganan la dama con `Cxd5`). Los empates (valores a menos de `1e-9` del
máximo) se resuelven al azar con la semilla de la llamada: es reproducible y evita que en posiciones
tranquilas juegue siempre «la primera jugada de la lista». Métricas: `nodes` = sucesores generados,
`leaves` = sucesores valorados, `completed_depth = 1` si alcanzó a valorarlos todos. El plazo es
cooperativo: si vence, devuelve la mejor jugada vista hasta ese momento.

**Resultados.** En 10 partidas Voraz (E0) contra Aleatorio, cinco con cada color, semillas 0–9,
plazo de 0,2 s por decisión y límite de 160 plies, el voraz ganó 9 por mate y hubo 1 tablas por
repetición triple (duración media: 57,4 plies). El voraz tardó en promedio 11,1 ms por decisión
(máximo 87 ms) y generó en promedio 42 sucesores por decisión; el aleatorio, 0,13 ms. Repetir la
partida de la semilla 0 dio exactamente la misma partida (`1-0`, mate en el ply 81).

La partida en tablas es instructiva: negras (el voraz) tenían más de 34 peones de ventaja (dos
caballos, dos alfiles y una torre contra torre y peones), pero el voraz no sabe dar mate si el mate
está a más de un ply. Movía el rey de un lado a otro porque todas esas jugadas valían lo mismo para
`E`, y la política de reclamo automático del taller (`outcome(claim_draw=True)`) terminó la partida
en cuanto negras **podían** reclamar una repetición triple, aunque el voraz nunca habría elegido la
jugada que repite (para él valdría 0, mucho menos que su ventaja). Es el límite del horizonte de
un ply, no un error de las reglas.

**Dónde falla el voraz (pregunta 1 de la sustentación).** Dejamos documentada la posición
`GREEDY_TRAP_FEN = "4r1k1/1p3ppp/1Q6/8/8/8/5PPP/6K1 w - - 0 1"`. Negras amenazan `...Te1#` (mate
del pasillo: el rey blanco está encerrado por sus propios peones). El voraz juega `Dxb7` porque
gana un peón (valor `+430` cp desde blancas) y no ve la respuesta `...Te1#`. Un minimax de
profundidad 2 genera esa respuesta, ve que tras `Dxb7` llega un mate y elige una defensa: con E0 un
minimax-2 de comprobación escogió `Db4`, que cubre e1. La prueba
`test_greedy_falls_into_back_rank_trap` verifica que el voraz cae con cinco semillas y que `Te1` da
mate.

In [ ]:
# ===== Demostracion A: reproducibilidad por semilla y perspectiva de color =====
from collections import Counter as _Counter

_inicial = chess.Board()
_aleatorio, _voraz = RandomAgent(), GreedyAgent()

# 1) Aleatorio: misma semilla -> misma jugada; muchas semillas -> cubre las 20 jugadas.
_j1 = _aleatorio.choose(_inicial.copy(stack=True), time_limit_s=0, seed=42).move
_j2 = _aleatorio.choose(_inicial.copy(stack=True), time_limit_s=0, seed=42).move
assert _j1 == _j2
_cuenta = _Counter(_aleatorio.choose(_inicial, time_limit_s=0, seed=s).move.uci() for s in range(2000))
print(f"Aleatorio, semilla 42 dos veces: {_j1.uci()} y {_j2.uci()} (iguales).")
print(f"Aleatorio con 2000 semillas: {len(_cuenta)} jugadas distintas de 20; "
      f"frecuencia min/max = {min(_cuenta.values())}/{max(_cuenta.values())}.")

# 2) Voraz: un mate en uno domina a cualquier estimacion; tablero intacto.
_b = chess.Board("7k/8/5KQ1/8/8/8/8/8 w - - 0 1")
_antes = (_b.fen(), list(_b.move_stack))
_d = _voraz.choose(_b, time_limit_s=0, seed=0)
assert (_b.fen(), list(_b.move_stack)) == _antes
print(f"\nVoraz en mate en uno: {_b.san(_d.move)} con score {_d.score:.0f} "
      f"(nodos={_d.stats.nodes}, hojas={_d.stats.leaves}).")

# 3) Perspectiva: con negras al turno el voraz minimiza E (bueno para negras).
_b = chess.Board("4k3/p7/8/3Q4/8/2n5/7P/4K3 b - - 0 1")
_d = _voraz.choose(_b, time_limit_s=0, seed=0)
assert _d.move == chess.Move.from_uci("c3d5") and _d.score > 0
_b.push(_d.move)
print(f"Voraz con NEGRAS: {_b.peek().uci()} (Cxd5), score {_d.score:+.1f} cp desde negras; "
      f"E desde blancas tras la jugada = {DEFAULT_EVALUATOR(_b):+.1f} (negativa: buena para negras).")

# 4) Estado terminal: sin decision nueva y utilidad de derrota para negras.
_d = _voraz.choose(chess.Board("7k/6Q1/5K2/8/8/8/8/8 b - - 0 1"), time_limit_s=0, seed=0)
print(f"Voraz en la posicion de mate (juegan negras): move={_d.move}, score={_d.score:.0f}")

# 5) Donde falla el voraz (pregunta 1): GREEDY_TRAP_FEN.
_b = chess.Board(GREEDY_TRAP_FEN)
_d = _voraz.choose(_b, time_limit_s=0, seed=0)
_b2 = _b.copy(); _b2.push(_d.move); _b2.push_uci(GREEDY_TRAP_REFUTATION)
print(f"\nGREEDY_TRAP_FEN: el voraz juega {_b.san(_d.move)} ({_d.score:+.1f} cp) y tras "
      f"...Te1 la posicion es jaque mate: {_b2.is_checkmate()}")


In [ ]:
# IMPLEMENTACIÓN B: minimax y alfa-beta, con las mismas hojas y evaluación.
# Complete con su código. Primero compare a profundidad 1 y 2.


In [ ]:
# ===== chess_ai/search.py =====
"""Busqueda adversaria: minimax, alfa-beta y alfa-beta mejorado (AIMA cap. 5).

CONVENCION MAX/MIN (coherente con D-06)
    MAX = BLANCAS y MIN = NEGRAS, siempre, sobre valores desde blancas: la
    evaluacion E(s) y las utilidades terminales (`terminal_value_white`). Asi el
    codigo es el minimax de AIMA tal cual (dos ramas explicitas, no negamax) y un
    mismo numero significa lo mismo en cualquier nodo del arbol. El valor de la
    raiz se convierte al final a la perspectiva de quien decide con
    `perspective_sign(turno)`: eso es `Decision.score`. Cuando decide negras, la
    raiz es un nodo MIN y la jugada elegida minimiza E (= maximiza -E).

DEFINICIONES (identicas en minimax y alfa-beta, para que los numeros se comparen)
    * nodo  = cada llamada a la funcion recursiva sobre una posicion, incluida la
              raiz. Se cuenta al entrar, antes de mirar si es terminal.
    * hoja  = nodo donde se devuelve una utilidad terminal exacta (mate, ahogado,
              material insuficiente, regla de 50, repeticion triple segun
              `search_outcome`) o donde se aplica E porque la profundidad llego a 0.
              Toda hoja es nodo; un nodo interior no es hoja.
    * poda  = EVENTO de corte: en un nodo MAX el valor alcanzo `valor >= beta`
              (o en un nodo MIN `valor <= alfa`) y quedaban hermanos sin explorar.
              `SearchStats.cutoffs` cuenta eventos; ademas se cuenta aparte el
              numero de hijos omitidos (`pruned_children`). Si la condicion se
              cumple con el ultimo hijo no hay nada que podar y no se cuenta.
    * profundidad = plies que faltan por expandir; `ply` = distancia a la raiz
              (la usan los mates: ganar a distancia `ply` vale MATE_SCORE - ply).

ORDEN DE LAS COMPROBACIONES EN CADA NODO (pseudocodigo del enunciado)
    1. contar el nodo; 2. comprobar el plazo (solo si hay `Deadline`);
    3. terminal ANTES que corte de profundidad (D-07); 4. profundidad 0 -> E;
    5. expandir (orden natural de python-chess u ordenado), push/pop con
       try/finally para que el tablero quede intacto incluso si salta
       `SearchTimeout` u otra excepcion.

DESEMPATES
    En cada nodo se conserva el PRIMER hijo que alcanza el mejor valor (se
    reemplaza solo con `>` en MAX o `<` en MIN). Por eso minimax (orden natural)
    y alfa-beta ordenado pueden devolver jugadas distintas con el MISMO valor:
    el orden decide cual de las jugadas empatadas aparece primero.

PLAZO (solo el agente mejorado)
    Los agentes de profundidad fija ignoran `time_limit_s` (su objeto de estudio es
    el arbol completo; el controlador mide y registra el exceso, D-10). El
    mejorado crea un `Deadline` y lo consulta al ENTRAR a cada nodo: si vencio,
    lanza `SearchTimeout`, que deshace todos los push con los `finally` y llega a
    la profundizacion iterativa, que descarta la iteracion incompleta.
"""
from __future__ import annotations

import time
from dataclasses import dataclass, field
from typing import NamedTuple

import chess


_SEARCH_INF = float("inf")
_SEARCH_MAX_PLY = 128
# Valores para ordenar capturas MVV-LVA (victima mas valiosa, atacante menos valioso).
_SEARCH_ORDER_VALUES = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5,
                        chess.QUEEN: 9, chess.KING: 20}

# Pregunta 1 de la sustentacion: el voraz de un ply falla y minimax de profundidad 2
# encuentra la defensa. Blancas: Rg1, Dd1, Cd4, peones a2 b2 c2 f2 g2 h2. Negras: Rg8,
# Te8, peones a7 b7 c7 e5 f7 g7 h5. El peon e5 ATACA al caballo d4 y el peon h5 esta
# sin defensa. El voraz (= minimax de profundidad 1, una sola capa MAX) elige Dxh5:
# gana un peon (+1) y no mira la respuesta. Minimax de profundidad 2 ve que tras Dxh5
# negras juegan exd4 (el caballo vale 3.2) y prefiere poner el caballo a salvo (Cf5).
# Valores medidos con E0 (desde blancas): profundidad 1 -> Dxh5 = +786.7;
# profundidad 2 -> Dxh5 = +432.8 (tras exd4) frente a Cf5 = +673.4.
MINIMAX_DEFENSE_FEN = "4r1k1/ppp2pp1/8/4p2p/3N4/8/PPP2PPP/3Q2K1 w - - 0 1"
MINIMAX_DEFENSE_GREEDY_MOVE = "d1h5"                     # Dxh5?, deja el caballo colgado
# Retiradas que salvan el caballo (Cc6 y Ce6 no: las toman b7 y f7).
MINIMAX_DEFENSE_MOVES = ("d4f5", "d4b5", "d4b3", "d4f3", "d4e2")


@dataclass
class PruneEvent:
    """Una poda registrada (para trazarla a mano). Valores siempre desde blancas.

    * `node_type`: 'MAX' (juegan blancas, corta si valor >= beta) o 'MIN'
      (juegan negras, corta si valor <= alfa).
    * `path`: jugadas SAN desde la raiz hasta el nodo donde ocurrio la poda.
    * `alpha`, `beta`: ventana del nodo en el momento del corte.
    * `explored`: hijos ya explorados con el valor que devolvio cada uno.
    * `cause`: la jugada cuyo valor provoco el corte.
    * `pruned`: hermanos que no se exploraron.
    """
    ply: int
    depth: int
    node_type: str
    path: list
    fen: str
    alpha: float
    beta: float
    value: float
    cause: str
    explored: list = field(default_factory=list)
    pruned: list = field(default_factory=list)


def format_prune_event(ev: PruneEvent) -> str:
    """Texto legible de una poda: ruta, ventana, hijos explorados y omitidos."""
    ruta = " ".join(ev.path) if ev.path else "(raiz)"
    cond = "valor >= beta" if ev.node_type == "MAX" else "valor <= alfa"
    hijos = ", ".join(f"{san}={val:+.1f}" for san, val in ev.explored)
    return (f"[{ev.node_type} ply={ev.ply} prof={ev.depth}] ruta: {ruta}\n"
            f"  ventana alfa={ev.alpha:+.1f} beta={ev.beta:+.1f}; explorados: {hijos}\n"
            f"  corte por {ev.cause} ({cond}: {ev.value:+.1f}); "
            f"podados ({len(ev.pruned)}): {' '.join(ev.pruned)}")


class SearchContext:
    """Estado compartido por una busqueda: metricas, plazo, orden, PV y registro de podas.

    `pv[ply]` es la mejor linea encontrada desde el nodo del `ply` actual; `pv[0][0]`
    es la jugada de la raiz. `prev_pv` es la variante principal de la iteracion
    anterior (profundizacion iterativa) y se prueba primero en cada `ply`.
    """

    def __init__(self, evaluator=DEFAULT_EVALUATOR, deadline: Deadline | None = None,
                 ordering: bool = False, log_prunes: bool = False, max_log: int = 500,
                 quiescence: bool = False, root: chess.Board | None = None):
        self.evaluator = evaluator
        self.deadline = deadline
        self.ordering = ordering
        self.quiescence = quiescence
        self.stats = SearchStats()
        self.pruned_children = 0
        self.qnodes = 0
        self.sort_time = 0.0
        self.pv = [[] for _ in range(_SEARCH_MAX_PLY + 1)]
        self.prev_pv: list = []
        self.killers = [[] for _ in range(_SEARCH_MAX_PLY + 1)]
        self.prune_log = [] if log_prunes else None
        self.max_log = max_log
        self.root_board = root.copy(stack=False) if root is not None else None
        self.root_len = len(root.move_stack) if root is not None else 0


# ------------------------------------------------------------------ ordenacion
def _search_order_moves(board: chess.Board, ply: int, ctx: SearchContext) -> list:
    """Ordena las jugadas legales: PV previa > promociones > capturas MVV-LVA > killers > resto.

    Es un sort estable: entre jugadas del mismo grupo se conserva el orden de
    generacion de python-chess. El tiempo se acumula en `ctx.sort_time`.
    """
    t0 = time.perf_counter()
    pv_move = ctx.prev_pv[ply] if ply < len(ctx.prev_pv) else None
    killers = ctx.killers[ply] if ply <= _SEARCH_MAX_PLY else ()
    scored = []
    for move in board.legal_moves:
        if move == pv_move:
            s = 1_000_000
        elif move.promotion:
            s = 900_000 + _SEARCH_ORDER_VALUES[move.promotion]
        elif board.is_capture(move):
            victim = (chess.PAWN if board.is_en_passant(move)
                      else board.piece_type_at(move.to_square))
            attacker = board.piece_type_at(move.from_square)
            s = 500_000 + 100 * _SEARCH_ORDER_VALUES[victim] - _SEARCH_ORDER_VALUES[attacker]
        elif move in killers:
            s = 400_000 - killers.index(move)
        else:
            s = 0
        scored.append((s, move))
    scored.sort(key=lambda t: -t[0])
    ctx.sort_time += time.perf_counter() - t0
    return [m for _, m in scored]


def _search_store_killer(board: chess.Board, move: chess.Move, ply: int, ctx: SearchContext) -> None:
    """Jugada tranquila que produjo un corte: se recuerda (2 por ply) para probarla antes."""
    if ply > _SEARCH_MAX_PLY or move.promotion or board.is_capture(move):
        return
    ks = ctx.killers[ply]
    if move in ks:
        return
    ks.insert(0, move)
    del ks[2:]


def _search_path_san(board: chess.Board, ctx: SearchContext) -> list:
    replay = ctx.root_board.copy(stack=False)
    out = []
    for mv in board.move_stack[ctx.root_len:]:
        out.append(replay.san(mv))
        replay.push(mv)
    return out


def _search_log_cutoff(board, ctx, ply, depth, node_type, alpha, beta, value,
                       cause, explored, pruned) -> None:
    if ctx.prune_log is None or len(ctx.prune_log) >= ctx.max_log or ctx.root_board is None:
        return
    ctx.prune_log.append(PruneEvent(
        ply=ply, depth=depth, node_type=node_type, path=_search_path_san(board, ctx),
        fen=board.fen(), alpha=alpha, beta=beta, value=value, cause=board.san(cause),
        explored=[(board.san(m), v) for m, v in explored],
        pruned=[board.san(m) for m in pruned]))


# ------------------------------------------------------------------ minimax
def _search_minimax(board: chess.Board, depth: int, ply: int, ctx: SearchContext) -> float:
    """MINIMAX(estado, profundidad) con MAX = blancas y MIN = negras. Valor desde blancas."""
    ctx.stats.nodes += 1
    if ctx.deadline is not None:
        ctx.deadline.check()
    ctx.pv[ply] = []
    outcome = search_outcome(board)
    if outcome is not None:                      # terminal antes que corte
        ctx.stats.leaves += 1
        return terminal_value_white(outcome, ply)
    if depth <= 0:
        ctx.stats.leaves += 1
        return ctx.evaluator(board)
    moves = list(board.legal_moves)
    if board.turn == chess.WHITE:                # nodo MAX
        value = -_SEARCH_INF
        for move in moves:
            board.push(move)
            try:
                child = _search_minimax(board, depth - 1, ply + 1, ctx)
            finally:
                board.pop()
            if child > value:
                value = child
                ctx.pv[ply] = [move] + ctx.pv[ply + 1]
        return value
    value = _SEARCH_INF                          # nodo MIN
    for move in moves:
        board.push(move)
        try:
            child = _search_minimax(board, depth - 1, ply + 1, ctx)
        finally:
            board.pop()
        if child < value:
            value = child
            ctx.pv[ply] = [move] + ctx.pv[ply + 1]
    return value


# ------------------------------------------------------------------ alfa-beta
def _search_alphabeta(board: chess.Board, depth: int, ply: int, alpha: float, beta: float,
                      ctx: SearchContext) -> float:
    """ALFABETA(estado, profundidad, alfa, beta): mismo arbol y hojas que minimax, con poda.

    alfa = mejor valor ya garantizado para MAX en la ruta; beta = idem para MIN.
    Devuelve el valor exacto si cae dentro de (alfa, beta); si no, una cota
    (fail-soft) que basta para que el padre descarte esa rama. En la raiz la
    ventana es (-inf, +inf), asi que el valor de la raiz es EXACTO e igual al de
    minimax.
    """
    ctx.stats.nodes += 1
    if ctx.deadline is not None:
        ctx.deadline.check()
    ctx.pv[ply] = []
    outcome = search_outcome(board)
    if outcome is not None:
        ctx.stats.leaves += 1
        return terminal_value_white(outcome, ply)
    if depth <= 0:
        if ctx.quiescence:
            return _search_quiescence(board, ply, alpha, beta, ctx, entry=True)
        ctx.stats.leaves += 1
        return ctx.evaluator(board)
    moves = _search_order_moves(board, ply, ctx) if ctx.ordering else list(board.legal_moves)
    logging = ctx.prune_log is not None
    explored = [] if logging else None
    n = len(moves)
    if board.turn == chess.WHITE:                # nodo MAX
        value = -_SEARCH_INF
        for i, move in enumerate(moves):
            board.push(move)
            try:
                child = _search_alphabeta(board, depth - 1, ply + 1, alpha, beta, ctx)
            finally:
                board.pop()
            if logging:
                explored.append((move, child))
            if child > value:
                value = child
                ctx.pv[ply] = [move] + ctx.pv[ply + 1]
            if value >= beta:                    # MIN nunca dejara llegar aqui
                if i + 1 < n:
                    ctx.stats.cutoffs += 1
                    ctx.pruned_children += n - i - 1
                    if ctx.ordering:
                        _search_store_killer(board, move, ply, ctx)
                    if logging:
                        _search_log_cutoff(board, ctx, ply, depth, "MAX", alpha, beta, value,
                                           move, explored, moves[i + 1:])
                return value
            if value > alpha:
                alpha = value
        return value
    value = _SEARCH_INF                          # nodo MIN
    for i, move in enumerate(moves):
        board.push(move)
        try:
            child = _search_alphabeta(board, depth - 1, ply + 1, alpha, beta, ctx)
        finally:
            board.pop()
        if logging:
            explored.append((move, child))
        if child < value:
            value = child
            ctx.pv[ply] = [move] + ctx.pv[ply + 1]
        if value <= alpha:                       # MAX nunca dejara llegar aqui
            if i + 1 < n:
                ctx.stats.cutoffs += 1
                ctx.pruned_children += n - i - 1
                if ctx.ordering:
                    _search_store_killer(board, move, ply, ctx)
                if logging:
                    _search_log_cutoff(board, ctx, ply, depth, "MIN", alpha, beta, value,
                                       move, explored, moves[i + 1:])
            return value
        if value < beta:
            beta = value
    return value


def _search_quiescence(board: chess.Board, ply: int, alpha: float, beta: float,
                       ctx: SearchContext, entry: bool = False) -> float:
    """Busqueda de quietud (OPCIONAL, solo con quiescence=True en el mejorado).

    En la frontera no se aplica E de inmediato si hay capturas o promociones
    pendientes: el bando al turno puede "quedarse quieto" (stand pat = E) o
    seguir capturando. Cada llamada cuenta como nodo y como hoja (aplica E);
    `ctx.qnodes` cuenta aparte cuantos nodos son de quietud. ATENCION: con
    quietud el arbol ya no es el de profundidad fija, asi que sus valores no se
    comparan con minimax/alfa-beta sin quietud.
    """
    if not entry:                                # la entrada ya se conto en alfa-beta
        ctx.stats.nodes += 1
        ctx.qnodes += 1
        if ctx.deadline is not None:
            ctx.deadline.check()
        outcome = search_outcome(board)
        if outcome is not None:
            ctx.stats.leaves += 1
            return terminal_value_white(outcome, ply)
    ctx.pv[ply] = []
    ctx.stats.leaves += 1
    stand = ctx.evaluator(board)
    if ply >= _SEARCH_MAX_PLY - 1:
        return stand
    noisy = [m for m in _search_order_moves(board, ply, ctx)
             if m.promotion or board.is_capture(m)]
    if board.turn == chess.WHITE:
        value = stand
        if value >= beta:
            return value
        alpha = max(alpha, value)
        for move in noisy:
            board.push(move)
            try:
                child = _search_quiescence(board, ply + 1, alpha, beta, ctx)
            finally:
                board.pop()
            if child > value:
                value = child
            if value >= beta:
                return value
            alpha = max(alpha, value)
        return value
    value = stand
    if value <= alpha:
        return value
    beta = min(beta, value)
    for move in noisy:
        board.push(move)
        try:
            child = _search_quiescence(board, ply + 1, alpha, beta, ctx)
        finally:
            board.pop()
        if child < value:
            value = child
        if value <= alpha:
            return value
        beta = min(beta, value)
    return value


# ------------------------------------------------------------------ utilidades comunes
def _search_terminal_decision(board: chess.Board, start: float) -> Decision | None:
    """Si la PARTIDA ya termino (politica del taller, claim_draw=True) no hay decision nueva."""
    outcome = game_outcome(board)
    if outcome is None:
        return None
    stats = SearchStats(nodes=1, leaves=1, elapsed_s=time.perf_counter() - start)
    return Decision(None, perspective_sign(board.turn) * terminal_value_white(outcome, 0), stats)


def _search_fixed_depth(board: chess.Board, algorithm: str, depth: int, evaluator,
                        ordering: bool = False, log_prunes: bool = False,
                        max_log: int = 500) -> tuple:
    """Busqueda de profundidad fija desde la raiz. Devuelve (valor_blancas, jugada, ctx)."""
    if depth < 1:
        raise ValueError("la profundidad debe ser >= 1")
    if algorithm == "minimax":
        if ordering:
            raise ValueError("minimax se compara siempre en orden natural")
        ctx = SearchContext(evaluator)
        value = _search_minimax(board, depth, 0, ctx)
    elif algorithm in ("alphabeta", "alfabeta"):
        ctx = SearchContext(evaluator, ordering=ordering, log_prunes=log_prunes,
                            max_log=max_log, root=board if log_prunes else None)
        value = _search_alphabeta(board, depth, 0, -_SEARCH_INF, _SEARCH_INF, ctx)
    else:
        raise ValueError(f"algoritmo desconocido: {algorithm}")
    move = ctx.pv[0][0] if ctx.pv[0] else None
    ctx.stats.completed_depth = depth
    return value, move, ctx


class RootSearchResult(NamedTuple):
    """Resultado de `search_root`. `value` desde el jugador al turno en la raiz."""
    value: float | None
    move: chess.Move | None
    stats: SearchStats
    sort_time: float


def search_root(board: chess.Board, algorithm: str = "minimax", depth: int = 2,
                evaluator=DEFAULT_EVALUATOR, ordering: bool = False) -> RootSearchResult:
    """Utilidad del Experimento 1: minimax / alfa-beta natural / alfa-beta ordenado.

    `algorithm` en {"minimax", "alphabeta"}; `ordering` solo para alfa-beta.
    Mismo arbol, mismas hojas y misma E en los tres casos; deja el tablero intacto.
    `sort_time` es el tiempo gastado ordenando (incluido en `stats.elapsed_s`).
    """
    start = time.perf_counter()
    term = _search_terminal_decision(board, start)
    if term is not None:
        return RootSearchResult(term.score, None, term.stats, 0.0)
    value_white, move, ctx = _search_fixed_depth(board, algorithm, depth, evaluator, ordering)
    ctx.stats.elapsed_s = time.perf_counter() - start
    return RootSearchResult(perspective_sign(board.turn) * value_white, move, ctx.stats,
                            ctx.sort_time)


# ------------------------------------------------------------------ agentes
class MinimaxAgent:
    """Minimax MAX/MIN de profundidad fija (AIMA 5.2). Ignora `time_limit_s` (ver modulo)."""

    def __init__(self, depth: int = 2, evaluator=DEFAULT_EVALUATOR, name: str = "Minimax"):
        self.depth = depth
        self.evaluator = evaluator
        self.name = name
        self.last_pv: list = []

    def choose(self, board: chess.Board, *, time_limit_s: float = 0.0, seed: int = 0) -> Decision:
        start = time.perf_counter()
        term = _search_terminal_decision(board, start)
        if term is not None:
            return term
        value_white, move, ctx = _search_fixed_depth(board, "minimax", self.depth, self.evaluator)
        self.last_pv = list(ctx.pv[0])
        ctx.stats.elapsed_s = time.perf_counter() - start
        return Decision(move, perspective_sign(board.turn) * value_white, ctx.stats)


class AlphaBetaAgent:
    """Alfa-beta de profundidad fija (AIMA 5.3): mismo valor que minimax, menos nodos.

    * `ordering=False`: orden natural de python-chess (el mismo que usa minimax).
    * `ordering=True`: "alfa-beta ordenado" del Experimento 1 (promociones,
      capturas MVV-LVA y killers). El arbol y las hojas son los mismos; cambia
      solo el orden en que se visitan los hijos, por eso el valor no cambia.
    * `log_prunes=True`: guarda en `last_prune_log` cada poda (`PruneEvent`)
      para trazarla a mano; `format_prune_event` la imprime.
    """

    def __init__(self, depth: int = 2, evaluator=DEFAULT_EVALUATOR, ordering: bool = False,
                 name: str = "Alfa-beta", log_prunes: bool = False, max_log: int = 500):
        self.depth = depth
        self.evaluator = evaluator
        self.ordering = ordering
        self.name = name
        self.log_prunes = log_prunes
        self.max_log = max_log
        self.last_prune_log: list = []
        self.last_pruned_children = 0
        self.last_sort_time_s = 0.0
        self.last_pv: list = []

    def choose(self, board: chess.Board, *, time_limit_s: float = 0.0, seed: int = 0) -> Decision:
        start = time.perf_counter()
        term = _search_terminal_decision(board, start)
        if term is not None:
            return term
        value_white, move, ctx = _search_fixed_depth(board, "alphabeta", self.depth,
                                                     self.evaluator, self.ordering,
                                                     self.log_prunes, self.max_log)
        self.last_prune_log = ctx.prune_log or []
        self.last_pruned_children = ctx.pruned_children
        self.last_sort_time_s = ctx.sort_time
        self.last_pv = list(ctx.pv[0])
        ctx.stats.elapsed_s = time.perf_counter() - start
        return Decision(move, perspective_sign(board.turn) * value_white, ctx.stats)


class AlphaBetaImprovedAgent:
    """Alfa-beta con ordenacion + profundizacion iterativa con plazo (`time_limit_s`).

    PROFUNDIZACION_ITERATIVA (pseudocodigo del enunciado):
      1. si la partida ya termino -> Decision(None, utilidad);
      2. jugada de RESPALDO = primera jugada legal segun la ordenacion (capturas
         MVV-LVA primero), con score None y completed_depth 0;
      3. para d = 1, 2, ..., max_depth: alfa-beta ordenado a profundidad d con la
         variante principal (PV) de la iteracion anterior probada primero en cada
         ply, mas promociones, capturas MVV-LVA y killers;
         - si la iteracion TERMINA: reemplaza (jugada, valor) y completed_depth = d;
         - si salta `SearchTimeout`: se descarta la iteracion parcial y se
           conserva la ultima completa (o el respaldo si ninguna termino);
         - si el valor es de mate, se para: mas profundidad no lo cambia.
    El plazo se comprueba al entrar a cada nodo y antes de empezar cada iteracion.
    `stats.nodes/leaves/cutoffs` suman TODAS las iteraciones (incluida la
    abortada): es el trabajo real gastado. `last_iterations` detalla cada una y
    `last_sort_time_s` el tiempo de ordenar. Sin plazo (time_limit_s <= 0) llega
    hasta `max_depth`.
    """

    def __init__(self, max_depth: int = 8, evaluator=DEFAULT_EVALUATOR,
                 name: str = "Alfa-beta mejorado", quiescence: bool = False):
        self.max_depth = max_depth
        self.evaluator = evaluator
        self.name = name
        self.quiescence = quiescence
        self.last_iterations: list = []
        self.last_sort_time_s = 0.0
        self.last_pv: list = []

    def choose(self, board: chess.Board, *, time_limit_s: float = 0.2, seed: int = 0) -> Decision:
        start = time.perf_counter()
        term = _search_terminal_decision(board, start)
        if term is not None:
            return term
        deadline = Deadline(time_limit_s)
        ctx = SearchContext(self.evaluator, deadline=deadline, ordering=True,
                            quiescence=self.quiescence)
        best_move = _search_order_moves(board, 0, ctx)[0]          # respaldo legal
        best_value_white = None
        completed = 0
        self.last_iterations = []
        for depth in range(1, self.max_depth + 1):
            if deadline.expired():
                break
            before = (ctx.stats.nodes, ctx.stats.leaves, ctx.stats.cutoffs, deadline.elapsed())
            try:
                value = _search_alphabeta(board, depth, 0, -_SEARCH_INF, _SEARCH_INF, ctx)
            except SearchTimeout:
                self.last_iterations.append(self._search_iteration_row(depth, False, ctx, before,
                                                                       deadline, None, None))
                break
            best_move, best_value_white, completed = ctx.pv[0][0], value, depth
            ctx.prev_pv = list(ctx.pv[0])
            self.last_iterations.append(self._search_iteration_row(depth, True, ctx, before,
                                                                   deadline, value, best_move))
            if is_mate_score(value):
                break
        self.last_pv = list(ctx.prev_pv)
        self.last_sort_time_s = ctx.sort_time
        ctx.stats.completed_depth = completed
        ctx.stats.elapsed_s = time.perf_counter() - start
        score = (None if best_value_white is None
                 else perspective_sign(board.turn) * best_value_white)
        return Decision(best_move, score, ctx.stats)

    @staticmethod
    def _search_iteration_row(depth, completed, ctx, before, deadline, value, move) -> dict:
        return {"depth": depth, "completed": completed,
                "nodes": ctx.stats.nodes - before[0], "leaves": ctx.stats.leaves - before[1],
                "cutoffs": ctx.stats.cutoffs - before[2],
                "elapsed_s": deadline.elapsed() - before[3],
                "value_white": value, "move": move.uci() if move is not None else None}


### Búsqueda adversaria: minimax, alfa-beta y alfa-beta mejorado (implementaciones B y C)

El código está en `chess_ai/search.py`. Implementamos los tres agentes de búsqueda sobre el mismo núcleo y con las mismas convenciones que el resto del taller (bitácora D-06 a D-10 y S-01 a S-18).

**Perspectiva.** Implementamos minimax con MAX = **blancas** y MIN = **negras**, con dos ramas explícitas como en el pseudocódigo (no negamax). Todos los valores del árbol están desde blancas: la evaluación E(s) y las utilidades terminales `±(100000 − ply)`. Solo al final convertimos el valor de la raíz a la perspectiva de quien decide (`Decision.score`). Cuando juegan negras, la raíz es un nodo MIN y el agente minimiza E, que es lo mismo que maximizar −E: por eso «la evaluación de negras favorece a negras».

**Qué contamos.** Usamos las mismas definiciones en minimax y en alfa-beta, para que los números se puedan comparar:

| Término | Definición |
|---|---|
| Nodo | cada llamada recursiva sobre una posición, incluida la raíz (se cuenta al entrar) |
| Hoja | nodo donde se devuelve una utilidad terminal exacta o se aplica E porque la profundidad llegó a 0 |
| Poda (`cutoffs`) | **evento** de corte: en MAX `valor ≥ β`, en MIN `valor ≤ α`, con al menos un hermano sin explorar; aparte contamos los hijos omitidos |

**Orden de cada nodo.** (1) contar el nodo; (2) consultar el plazo, si lo hay; (3) comprobar si es terminal **antes** del corte de profundidad; (4) si la profundidad es 0, aplicar E; (5) expandir con `push` y deshacer con `pop` dentro de `finally`, de modo que el tablero recibido queda con el mismo FEN y la misma pila de jugadas aunque salte una excepción o venza el plazo.

#### Minimax y alfa-beta
`MinimaxAgent(depth)` recorre el árbol completo hasta la profundidad fija. `AlphaBetaAgent(depth, ordering)` recorre **el mismo árbol, con las mismas hojas y la misma E**, pero mantiene α (lo mejor que MAX ya tiene asegurado en la ruta) y β (lo mejor para MIN) y deja de explorar los hermanos restantes cuando la rama ya no puede influir en la raíz. En la raíz la ventana es (−∞, +∞), por eso su valor es exacto: en las pruebas, minimax, alfa-beta natural y alfa-beta ordenado dan **exactamente el mismo número** (comparado con `==`) en 13 posiciones a profundidades 1, 2 y 3.

**¿Por qué pueden elegir jugadas distintas con el mismo valor?** En cada nodo conservamos el *primer* hijo que alcanza el mejor valor. Si dos jugadas empatan, el orden de recorrido decide cuál aparece primero; alfa-beta ordenado recorre en otro orden y puede devolver la otra. Además, alfa-beta devuelve solo cotas para las ramas podadas, así que nunca «ve» el valor exacto de una alternativa que no puede superar a la que ya tiene. Las pruebas comprueban que, cuando las jugadas difieren, cada una alcanza exactamente el valor de la raíz.

**Voraz frente a minimax (pregunta 1).** En `MINIMAX_DEFENSE_FEN = 4r1k1/ppp2pp1/8/4p2p/3N4/8/PPP2PPP/3Q2K1 w` el peón e5 ataca al caballo d4 y el peón h5 está sin defensa. Un voraz de un ply (que es minimax de profundidad 1) elige **Dxh5** (+786.7): gana un peón y no mira la respuesta. Minimax de profundidad 2 calcula que tras Dxh5 negras juegan exd4 (+432.8) y elige **Cf5** (+673.4), que salva el caballo.

#### Una poda real, trazada a mano (pregunta 2)
Registro de `AlphaBetaAgent(depth=2, log_prunes=True)` en orden natural sobre `MINIMAX_DEFENSE_FEN`. Los valores están desde blancas (MAX). La raíz es MAX; sus hijos son nodos MIN de profundidad 1 cuyas respuestas se evalúan con E.

1. La raíz explora sus primeras jugadas en el orden de python-chess: Ce6 vale +330.2 (lo toma la torre) → α = 330.2; Cc6 vale +398.2 (bxc6) → α = 398.2; **Cf5 vale +673.4** → **α = 673.4**. Hasta aquí no hay podas: en la raíz β = +∞.
2. Siguiente hijo de la raíz: **Cf3**, un nodo MIN que recibe α = 673.4, β = +∞. Su primera respuesta, **Rh8**, vale **+669.4**. El registro dice:
   ```
   [MIN ply=1 prof=1] ruta: Nf3
     ventana alfa=+673.4 beta=+inf; explorados: Kh8=+669.4
     corte por Kh8 (valor <= alfa: +669.4); podados (21): Kf8 Kh7 Rf8 Rd8 Rc8 ... a5
   ```
   (El registro usa la notación SAN inglesa de python-chess: K = rey, Q = dama, R = torre, N = caballo; en el texto usamos la española: R, D, T, C.)
3. **Por qué es válida:** el nodo es MIN, así que su valor final será el mínimo de sus hijos, y ya tiene uno de +669.4: valor(Cf3) ≤ 669.4. En la raíz, MAX ya tiene asegurado +673.4 con Cf5. Como 669.4 ≤ 673.4, MAX nunca preferirá Cf3, sin importar lo que valgan las 21 respuestas que no miramos: si alguna vale menos, Cf3 empeora; si vale más, MIN no la elegiría. El valor de la raíz no cambia. De hecho, el valor exacto de Cf3 (calculado aparte con minimax) es +650.7 (tras Td8), todavía menor que 673.4.
4. Otro corte del mismo registro, tras **Dxh5**: el nodo MIN explora Rf8 +785.4, Tf8 +794.7, Td8 +778.7, … y la novena respuesta, **exd4**, vale **+432.8 ≤ α = 673.4** → se podan las 11 jugadas restantes. Este corte es justo la «defensa» que el voraz no ve.

Resultado: la misma jugada (Cf5) y el mismo valor (+673.375) que minimax, con **190 nodos frente a 757**, 29 podas y 567 hijos omitidos.

#### Alfa-beta mejorado: ordenación y profundización iterativa
`AlphaBetaImprovedAgent(max_depth, time_limit_s)` combina:

- **Ordenación de jugadas** (`_search_order_moves`): primero la jugada de la variante principal de la iteración anterior en ese ply; después promociones, capturas por MVV-LVA (víctima más valiosa con el atacante menos valioso), *killers* (dos jugadas tranquilas por ply que ya causaron un corte) y el resto en el orden de generación. Medimos aparte el tiempo de ordenar: entre 5 % y 11 % del total a profundidad 3, a cambio de 32–62 % menos nodos que el alfa-beta natural.
- **Profundización iterativa con plazo**: guardamos primero una jugada legal de respaldo; luego buscamos a profundidad 1, 2, 3… El plazo (`Deadline`) se consulta al **entrar a cada nodo** y antes de empezar cada iteración; si vence, se lanza `SearchTimeout`, los `finally` deshacen todas las jugadas y la iteración incompleta se descarta.

**¿Qué devuelve el programa si vence el plazo durante una iteración? (pregunta 5)** La jugada y el valor de la **última iteración completa**, con `completed_depth` igual a esa profundidad. Si ni siquiera la profundidad 1 terminó, devuelve la jugada de respaldo (la primera según la ordenación), con `score=None` y `completed_depth=0`. Nunca devuelve `None` en una posición no terminal. Ejemplo medido desde la posición inicial con 0.2 s: las iteraciones 1, 2 y 3 terminan (21, 68 y 543 nodos); la 4 se interrumpe tras 596 nodos; el agente devuelve e3 con el valor de la profundidad 3 (+62) y `completed_depth=3`. Los nodos reportados suman todas las iteraciones, incluida la interrumpida, porque es el trabajo realmente gastado.

**Rendimiento.** E cuesta ~165–205 µs por llamada y casi todos los nodos son hojas, así que buscamos a unos 4 500–6 000 nodos/s en medio juego. Con 0.2 s por jugada, minimax y alfa-beta alcanzan profundidad 2, y el mejorado completa 2 (medio juego) o 3 (inicial y finales). Las tablas completas están en la bitácora S-12.

**Quietud (opcional).** Dejamos un parámetro `quiescence=False`. Si se activa, las hojas siguen explorando capturas y promociones antes de aplicar E. No es nuestra mejora oficial (la oficial es el ajuste de pesos) y, si se usa en una ablación, el árbol ya no es el de profundidad fija.

In [ ]:
# ===== Demostracion B: minimax = alfa-beta en valor (profundidades 1 y 2), con menos nodos =====
_fens_b = {
    "inicial": chess.STARTING_FEN,
    "italiana": "r1bqk1nr/pppp1ppp/2n5/2b1p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4",
    "defensa_q1": MINIMAX_DEFENSE_FEN,
    "negras_jaque": "r3k3/ppp2ppp/8/1B6/8/8/PPP2PPP/4K2R b K - 0 1",
    "final_torres": "8/5k2/8/3R4/8/2K5/5r2/8 w - - 0 1",
}
print(f"{'posicion':<14}{'d':>2}  {'minimax':>10} {'nodos':>6} | {'alfa-beta':>10} {'nodos':>6} "
      f"{'podas':>6} | {'AB ordenado':>11} {'nodos':>6}  jugadas (MM / AB / ABo)")
for _nombre, _fen in _fens_b.items():
    for _prof in (1, 2):
        _b = chess.Board(_fen)
        _mm = search_root(_b, "minimax", _prof)
        _ab = search_root(_b, "alphabeta", _prof)
        _abo = search_root(_b, "alphabeta", _prof, ordering=True)
        assert _mm.value == _ab.value == _abo.value          # igualdad EXACTA del valor de la raiz
        assert _ab.stats.nodes <= _mm.stats.nodes
        print(f"{_nombre:<14}{_prof:>2}  {_mm.value:>10.1f} {_mm.stats.nodes:>6} | {_ab.value:>10.1f} "
              f"{_ab.stats.nodes:>6} {_ab.stats.cutoffs:>6} | {_abo.value:>11.1f} {_abo.stats.nodes:>6}  "
              f"{_b.san(_mm.move)} / {_b.san(_ab.move)} / {_b.san(_abo.move)}")
print("Mismo valor en las 10 comparaciones (assert ==). Si las jugadas difieren, es un empate de valor.")

# Pregunta 1: el voraz (= minimax de profundidad 1) gana un peon; minimax-2 ve la respuesta exd4.
_b = chess.Board(MINIMAX_DEFENSE_FEN)
_d1 = MinimaxAgent(depth=1).choose(_b, time_limit_s=0, seed=0)
_d2 = MinimaxAgent(depth=2).choose(_b, time_limit_s=0, seed=0)
print(f"\nMINIMAX_DEFENSE_FEN: profundidad 1 -> {_b.san(_d1.move)} ({_d1.score:+.1f}); "
      f"profundidad 2 -> {_b.san(_d2.move)} ({_d2.score:+.1f}).")

# Pregunta 2: registro de podas reales (trazado a mano en el texto de arriba).
_ab_log = AlphaBetaAgent(depth=2, log_prunes=True)
_d = _ab_log.choose(chess.Board(MINIMAX_DEFENSE_FEN), time_limit_s=0, seed=0)
print(f"\nAlfa-beta natural d=2: {chess.Board(MINIMAX_DEFENSE_FEN).san(_d.move)} ({_d.score:+.3f}), "
      f"{_d.stats.nodes} nodos, {_d.stats.cutoffs} podas, {_ab_log.last_pruned_children} hijos omitidos. "
      "Primeras dos podas del registro:")
for _ev in _ab_log.last_prune_log[:2]:
    print(format_prune_event(_ev))


In [ ]:
# IMPLEMENTACIÓN C: ordenación y profundización iterativa con presupuesto.
# Complete con su código. Preserve la última iteración terminada.


### Implementación C: dónde está y qué muestra la celda siguiente

La ordenación y la profundización iterativa ya están en el código de la implementación B (`AlphaBetaImprovedAgent` y `_search_order_moves`, en `chess_ai/search.py`), porque comparten el núcleo de alfa-beta. El diseño está explicado arriba, en «Alfa-beta mejorado». Aquí lo ejecutamos con plazo y mostramos cada iteración:

- **Respaldo:** antes de iterar se guarda una jugada legal (la primera según la ordenación). Si el plazo vence antes de que termine la profundidad 1, el agente devuelve esa jugada con `score=None` y `completed_depth=0`. Nunca devuelve `None` en una posición no terminal.
- **Última iteración completa:** el plazo se consulta al entrar a cada nodo. Si vence, `SearchTimeout` sube por la recursión y los `finally` deshacen cada `push`; la iteración parcial se **descarta** (fila «NO» de la tabla) y se devuelve la jugada y el valor de la última iteración terminada. `completed_depth` es esa profundidad, no la que se alcanzó a empezar.
- **Contadores:** los nodos del `SearchStats` suman todas las iteraciones, incluida la abortada, porque es el trabajo realmente gastado.

Los tiempos y la profundidad alcanzada dependen de la máquina y de su carga en el momento de ejecutar. Por eso la celda solo **afirma** invariantes: la jugada es legal, el tablero queda intacto, `completed_depth` coincide con las iteraciones terminadas y, sin plazo, el valor es el de alfa-beta a esa profundidad.

In [ ]:
# ===== Demostracion C: profundizacion iterativa con plazo (AlphaBetaImprovedAgent) =====
def _tabla_iteraciones(agente):
    print(f"   {'prof':>4} {'termino':>8} {'nodos':>7} {'hojas':>7} {'podas':>6} {'tiempo s':>9} "
          f"{'valor (blancas)':>16}  jugada")
    for _it in agente.last_iterations:
        _v = "-" if _it["value_white"] is None else f"{_it['value_white']:+.1f}"
        print(f"   {_it['depth']:>4} {('si' if _it['completed'] else 'NO'):>8} {_it['nodes']:>7} "
              f"{_it['leaves']:>7} {_it['cutoffs']:>6} {_it['elapsed_s']:>9.4f} {_v:>16}  "
              f"{_it['move'] or '(descartada)'}")


_mej = AlphaBetaImprovedAgent(max_depth=10)
for _titulo, _fen, _plazo in (("inicial, plazo 0.2 s", chess.STARTING_FEN, 0.2),
                              ("medio juego (italiana), plazo 0.2 s",
                               "r1bqk1nr/pppp1ppp/2n5/2b1p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4", 0.2),
                              ("inicial, plazo agotado de entrada (1e-9 s)", chess.STARTING_FEN, 1e-9)):
    _b = chess.Board(_fen)
    _antes = (_b.fen(), list(_b.move_stack))
    _d = _mej.choose(_b, time_limit_s=_plazo, seed=0)
    assert (_b.fen(), list(_b.move_stack)) == _antes         # tablero intacto aunque se interrumpa
    assert _d.move in _b.legal_moves                         # nunca None en un estado no terminal
    _hechas = [it for it in _mej.last_iterations if it["completed"]]
    assert _d.stats.completed_depth == len(_hechas)
    print(f"\n{_titulo}: devuelve {_b.san(_d.move)}, score={_d.score if _d.score is None else round(_d.score, 1)}, "
          f"completed_depth={_d.stats.completed_depth}, nodos totales={_d.stats.nodes}, "
          f"tiempo={_d.stats.elapsed_s:.3f} s")
    if _mej.last_iterations:
        _tabla_iteraciones(_mej)
    else:
        print("   ninguna iteracion empezo: se devuelve la jugada de RESPALDO (primera segun la ordenacion)")

# Sin plazo y con max_depth fija, el valor coincide con alfa-beta (= minimax) a esa profundidad.
_b = chess.Board("8/5k2/8/3R4/8/2K5/5r2/8 w - - 0 1")
_d = AlphaBetaImprovedAgent(max_depth=3).choose(_b, time_limit_s=0, seed=0)
assert _d.score == search_root(_b, "alphabeta", 3).value
print(f"\nFinal de torres sin plazo, max_depth=3: {_b.san(_d.move)} {_d.score:+.1f} = valor de alfa-beta d=3.")


In [ ]:
# IMPLEMENTACIÓN D: MCTS-UCT y mejora adicional elegida.
# Complete con su código. Documente recompensas y simulaciones truncadas.


In [ ]:
# ===== chess_ai/mcts.py =====
"""MCTS-UCT (AIMA 5.4): seleccion, expansion, simulacion (rollout) y retropropagacion.

Convenciones (ver apuntes/bitacora_M.md, entradas M-01 .. M-10):

* RECOMPENSA en [0, 1]: victoria 1, tablas 0.5, derrota 0. Un rollout que llega al
  limite de plies sin terminar se puntua con sigmoide(E / ROLLOUT_SCALE_CP), con E la
  evaluacion comun desde blancas; se cuenta como TRUNCADO, nunca como tablas.
* PERSPECTIVA. Cada nodo guarda `wins` (W) desde el jugador que HIZO la jugada que
  lleva a ese nodo (`node.mover`). Asi, al elegir entre los hijos de un nodo, UCT
  maximiza W/N para el jugador que esta AL TURNO en el padre: blancas maximizan para
  blancas en sus nodos y negras para negras en los suyos (pregunta 6).
* PRESUPUESTO. `simulations=N` fija el numero de simulaciones (reproducible exacto con
  la misma semilla). `simulations=None` usa el plazo `time_limit_s` del contrato:
  el numero de simulaciones depende entonces de la maquina y de su carga.
* JUGADA FINAL: hijo de la raiz con mas visitas; empate -> mayor valor medio;
  empate -> menor UCI (orden lexicografico, determinista).
* `Decision.score`: valor medio q del hijo elegido (probabilidad estimada de ganar
  del jugador raiz) llevado a centipeones con el logit inverso de la sigmoide:
  ROLLOUT_SCALE_CP * ln(q / (1 - q)), recortado a +-EVAL_BOUND. q tambien queda en
  `last_info["win_probability"]`. Si el hijo elegido es terminal se reporta su valor
  exacto (MATE_SCORE - 1 si gana el jugador raiz), igual que los agentes de minimax.
"""
from __future__ import annotations

import csv
import math
import random
import statistics
import time

import chess


# Escala de la sigmoide que lleva centipeones a [0, 1] (rollouts truncados) y del logit
# inverso que lleva el valor medio de vuelta a centipeones (Decision.score).
# Con 400 cp: +1 peon -> 0.56, +pieza menor -> 0.69, +dama -> 0.91 (< 1 = victoria real).
ROLLOUT_SCALE_CP = 400.0
MCTS_DEFAULT_SIMULATIONS = 200      # si no hay ni simulaciones ni plazo
MCTS_ROLLOUT_POLICIES = ("uniform", "captures")


def _mcts_sigmoid(cp: float, scale: float = ROLLOUT_SCALE_CP) -> float:
    """Centipeones desde blancas -> recompensa de blancas en (0, 1)."""
    x = max(-60.0, min(60.0, cp / scale))
    return 1.0 / (1.0 + math.exp(-x))


def _mcts_logit_cp(q: float, scale: float = ROLLOUT_SCALE_CP) -> float:
    """Inversa de la sigmoide: probabilidad en [0, 1] -> centipeones, recortada."""
    eps = 1e-6
    q = min(1.0 - eps, max(eps, q))
    return max(-EVAL_BOUND, min(EVAL_BOUND, scale * math.log(q / (1.0 - q))))


def _mcts_terminal_reward_white(outcome: chess.Outcome) -> float:
    """Recompensa exacta de un terminal desde blancas: 1, 0.5 o 0."""
    if outcome.winner is None:
        return 0.5
    return 1.0 if outcome.winner == chess.WHITE else 0.0


class _MCTSNode:
    """Nodo del arbol. `wins` se mide desde `mover` (quien jugo `move` para llegar aqui).

    `terminal` es la recompensa exacta desde blancas si el estado es terminal (o None).
    `untried` guarda las jugadas aun no expandidas (orden UCI barajado con la semilla;
    se expanden sacandolas del final de la lista).
    """

    __slots__ = ("move", "parent", "children", "untried", "visits", "wins", "mover",
                 "terminal", "depth")

    def __init__(self, move, parent, mover, depth):
        self.move = move
        self.parent = parent
        self.children = []
        self.untried = []
        self.visits = 0
        self.wins = 0.0
        self.mover = mover
        self.terminal = None
        self.depth = depth

    def mean(self) -> float:
        return self.wins / self.visits if self.visits else 0.0


class MCTSAgent:
    """Agente MCTS-UCT con recompensas en [0, 1] y perspectiva por nodo.

    Parametros:
        simulations: simulaciones por jugada (None -> usar `time_limit_s`).
        exploration: constante c de UCT. sqrt(2) es el valor de UCB1 para
            recompensas en [0, 1] (Kocsis y Szepesvari, 2006); ver M-03.
        rollout_limit: plies maximos del rollout antes de truncar (0 = evaluar la hoja).
        rollout_policy: "uniform" (uniforme entre legales) o "captures" (con
            probabilidad `capture_bias` elige al azar entre capturas si las hay).
        evaluator: E(s) desde blancas para puntuar rollouts truncados.
        restore: "push_pop" (por defecto, mas rapido; ver M-08) o "copy"
            (cada simulacion trabaja sobre `board.copy(stack=True)`).
    """

    def __init__(self, simulations: int | None = None, exploration: float = math.sqrt(2),
                 rollout_limit: int = 4, rollout_policy: str = "captures",
                 evaluator=DEFAULT_EVALUATOR, name: str = "MCTS",
                 capture_bias: float = 0.75, restore: str = "push_pop"):
        if rollout_policy not in MCTS_ROLLOUT_POLICIES:
            raise ValueError(f"rollout_policy debe ser una de {MCTS_ROLLOUT_POLICIES}")
        if restore not in ("push_pop", "copy"):
            raise ValueError("restore debe ser 'push_pop' o 'copy'")
        if simulations is not None and simulations < 1:
            raise ValueError("simulations debe ser >= 1 o None")
        self.simulations = simulations
        self.exploration = float(exploration)
        self.rollout_limit = int(rollout_limit)
        self.rollout_policy = rollout_policy
        self.capture_bias = float(capture_bias)
        self.evaluator = evaluator
        self.name = name
        self.restore = restore
        self.last_info: dict = {}
        self.last_root = None

    # ------------------------------------------------------------ perspectiva
    def _reward_for(self, mover: chess.Color, reward_white: float) -> float:
        """Recompensa desde el jugador `mover`. Punto unico de la perspectiva (prueba P6)."""
        return reward_white if mover == chess.WHITE else 1.0 - reward_white

    # ------------------------------------------------------------ piezas del ciclo
    def _new_node(self, board: chess.Board, move, parent, rng: random.Random) -> _MCTSNode:
        node = _MCTSNode(move, parent, not board.turn,
                         0 if parent is None else parent.depth + 1)
        outcome = search_outcome(board)
        if outcome is not None:
            node.terminal = _mcts_terminal_reward_white(outcome)
        else:
            moves = sorted(board.legal_moves, key=lambda m: m.uci())
            rng.shuffle(moves)
            node.untried = moves
        return node

    def _select_child(self, node: _MCTSNode) -> _MCTSNode:
        """UCT: argmax W/N + c*sqrt(ln N_padre / N_hijo); empate -> primer hijo expandido."""
        log_n = math.log(node.visits)
        c = self.exploration
        best, best_val = None, -math.inf
        for child in node.children:
            val = child.wins / child.visits + c * math.sqrt(log_n / child.visits)
            if val > best_val:
                best, best_val = child, val
        return best

    def _rollout(self, board: chess.Board, rng: random.Random, pushed: list) -> tuple[float, str]:
        """Juega hasta terminal o `rollout_limit` plies. Devuelve (recompensa blancas, tipo).

        Cada push se anota en `pushed[0]` para que `choose` deshaga todo en su finally.
        """
        for _ in range(self.rollout_limit):
            moves = list(board.legal_moves)
            if not moves:
                break
            choice = None
            if self.rollout_policy == "captures" and rng.random() < self.capture_bias:
                captures = [m for m in moves if board.is_capture(m)]
                if captures:
                    choice = rng.choice(captures)
            if choice is None:
                choice = rng.choice(moves)
            board.push(choice)
            pushed[0] += 1
            outcome = search_outcome(board)
            if outcome is not None:
                return _mcts_terminal_reward_white(outcome), "terminal"
        outcome = search_outcome(board)      # sin jugadas legales o rollout_limit == 0
        if outcome is not None:
            return _mcts_terminal_reward_white(outcome), "terminal"
        return _mcts_sigmoid(self.evaluator(board)), "truncated"

    def _simulate(self, root: _MCTSNode, board: chess.Board, rng: random.Random,
                  counters: dict) -> None:
        """Una simulacion completa. Deja `board` exactamente como lo recibio."""
        pushed = [0]
        try:
            node = root
            # 1. seleccion: bajar mientras el nodo este totalmente expandido y no sea terminal
            while node.terminal is None and not node.untried and node.children:
                node = self._select_child(node)
                board.push(node.move)
                pushed[0] += 1
            # 2. expansion de UNA accion no explorada
            if node.terminal is None and node.untried:
                move = node.untried.pop()
                board.push(move)
                pushed[0] += 1
                child = self._new_node(board, move, node, rng)
                node.children.append(child)
                node = child
                counters["nodes"] += 1
                counters["max_depth"] = max(counters["max_depth"], child.depth)
            # 3. simulacion
            if node.terminal is not None:
                reward_white, kind = node.terminal, "terminal"
            else:
                reward_white, kind = self._rollout(board, rng, pushed)
            counters[kind] += 1
            # 4. retropropagacion: cada nodo suma desde SU jugador (quien movio para llegar)
            while node is not None:
                node.visits += 1
                node.wins += self._reward_for(node.mover, reward_white)
                node = node.parent
        finally:
            for _ in range(pushed[0]):
                board.pop()

    # ------------------------------------------------------------ contrato
    def choose(self, board: chess.Board, *, time_limit_s: float = 0.0, seed: int = 0) -> Decision:
        """Decide una jugada. Restaura `board` (FEN y pila) incluso si hay excepcion."""
        deadline = Deadline(time_limit_s)
        stats = SearchStats()
        root_color = board.turn
        rng = random.Random(seed)
        outcome = search_outcome(board)
        if outcome is not None:
            stats.elapsed_s = deadline.elapsed()
            self.last_info = {"terminal": True}
            self.last_root = None
            return Decision(None, perspective_sign(root_color) * terminal_value_white(outcome, 0),
                            stats)

        root = self._new_node(board, None, None, rng)
        counters = {"nodes": 1, "max_depth": 0, "terminal": 0, "truncated": 0}
        if self.simulations is not None:
            budget, mode = self.simulations, "simulations"
        elif deadline.limit is not None:
            budget, mode = None, "time"
        else:
            budget, mode = MCTS_DEFAULT_SIMULATIONS, "default_simulations"
        work = board if self.restore == "push_pop" else None
        done = 0
        while True:
            if budget is not None and done >= budget:
                break
            if budget is None and deadline.expired():
                break
            if work is None:
                self._simulate(root, board.copy(stack=True), rng, counters)
            else:
                self._simulate(root, work, rng, counters)
            done += 1

        if root.children:
            best = max(root.children,
                       key=lambda ch: (ch.visits, ch.mean(), _mcts_uci_desc_key(ch.move)))
            move, q = best.move, best.mean()
            if best.terminal is not None:
                # el hijo elegido ES terminal: valor exacto con la escala comun (mate a 1 ply)
                r = self._reward_for(root_color, best.terminal)
                score = 0.0 if r == 0.5 else (MATE_SCORE - 1 if r == 1.0 else -(MATE_SCORE - 1))
            else:
                score = _mcts_logit_cp(q)
        else:
            # vencio el plazo antes de la primera simulacion: primera jugada del orden
            # barajado (legal) y la evaluacion estatica como mejor estimacion disponible.
            move, q = root.untried[-1], None
            score = perspective_sign(root_color) * self.evaluator(board)

        stats.simulations = done
        stats.nodes = counters["nodes"]
        stats.leaves = counters["terminal"] + counters["truncated"]
        stats.completed_depth = counters["max_depth"]
        stats.elapsed_s = deadline.elapsed()
        ordered = sorted(root.children, key=lambda ch: (-ch.visits, ch.move.uci()))
        self.last_root = root
        self.last_info = {
            "terminal": False,
            "budget_mode": mode,
            "simulations": done,
            "truncated_rollouts": counters["truncated"],
            "terminal_rollouts": counters["terminal"],
            "root_visits": {ch.move.uci(): ch.visits for ch in ordered},
            "root_values": {ch.move.uci(): round(ch.mean(), 4) for ch in ordered},
            "win_probability": q,
            "sims_per_s": done / stats.elapsed_s if stats.elapsed_s > 0 else float("nan"),
        }
        return Decision(move, score, stats)


def _mcts_uci_desc_key(move: chess.Move) -> tuple:
    """Clave para que, a igualdad de visitas y valor, `max` prefiera el MENOR UCI."""
    return tuple(-ord(ch) for ch in move.uci())


# ====================================================================== Experimento 4
# Tacticas verificadas por reglas (ver `mcts_verify_tactics`). `kind` define como se
# calcula el conjunto correcto: mate_in_1 = jugadas que dan mate; win_queen = capturas
# de la dama rival; avoid_mate_in_1 = jugadas tras las cuales el rival NO tiene mate en uno;
# mate_in_2 = jugadas (que no son mate) tras las cuales TODA respuesta permite mate en uno.
MCTS_TACTICS = [
    {"id": "T1_mate1_w_queen", "kind": "mate_in_1",
     "fen": "7k/8/5KQ1/8/8/8/8/8 w - - 0 1", "solutions": {"g6g7"}},
    {"id": "T2_mate1_w_backrank", "kind": "mate_in_1",
     "fen": "6k1/5ppp/8/8/8/8/5PPP/R5K1 w - - 0 1", "solutions": {"a1a8"}},
    {"id": "T3_mate1_b_backrank", "kind": "mate_in_1",
     "fen": "r5k1/5ppp/8/8/8/8/5PPP/6K1 b - - 0 1", "solutions": {"a8a1"}},
    {"id": "T4_mate1_w_scholar", "kind": "mate_in_1",
     "fen": "r1bqkb1r/pppp1ppp/2n2n2/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR w KQkq - 4 4",
     "solutions": {"h5f7"}},
    {"id": "T5_queen_w", "kind": "win_queen",
     "fen": "r3k2r/ppp2ppp/2n5/3q4/8/2N5/PPP2PPP/R3K2R w KQkq - 0 1", "solutions": {"c3d5"}},
    {"id": "T6_queen_b", "kind": "win_queen",
     "fen": "r3k2r/ppp2ppp/2n5/8/3Q4/8/PPP2PPP/R3K2R b KQkq - 0 1", "solutions": {"c6d4"}},
    {"id": "T7_avoid_b_scholar", "kind": "avoid_mate_in_1",
     "fen": "r1bqkbnr/pppp1ppp/2n5/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR b KQkq - 3 3",
     "solutions": {"d7d5", "d8e7", "d8f6", "d8g5", "d8h4", "e8e7", "g7g6", "g8h6"}},
    {"id": "T8_avoid_w_qh4", "kind": "avoid_mate_in_1",
     "fen": "rnb1k1nr/pppp1ppp/8/2b1p3/4P2q/2N5/PPPP1PPP/R1BQKBNR w KQkq - 4 4",
     "solutions": {"d1e2", "d1f3", "d1g4", "d1h5", "d2d4", "e1e2", "g1h3", "g2g3"}},
    {"id": "T9_mate2_w_rooks", "kind": "mate_in_2",
     "fen": "2r3k1/1p3ppp/p7/8/8/3R4/P2R1PPP/6K1 w - - 0 1", "solutions": {"d3d8"}},
    {"id": "T10_mate2_b_rooks", "kind": "mate_in_2",
     "fen": "6k1/p2r1ppp/3r4/8/8/P7/1P3PPP/2R3K1 b - - 0 1", "solutions": {"d6d1"}},
]


def _mcts_has_mate_in_one(board: chess.Board) -> bool:
    for m in board.legal_moves:
        board.push(m)
        try:
            if board.is_checkmate():
                return True
        finally:
            board.pop()
    return False


def _mcts_forces_mate_next(board: chess.Board) -> bool:
    """Tras la jugada candidata (rival al turno, aun no es mate): TODA respuesta legal
    permite un mate en uno. Asi se define 'mate en dos' sin contar los mates en uno."""
    if board.is_checkmate():
        return False
    replies = list(board.legal_moves)
    if not replies:
        return False
    for r in replies:
        board.push(r)
        try:
            if not _mcts_has_mate_in_one(board):
                return False
        finally:
            board.pop()
    return True


def mcts_tactic_solutions(fen: str, kind: str) -> set:
    """Calcula con las REGLAS (no con el agente) el conjunto de jugadas correctas."""
    board = chess.Board(fen)
    out = set()
    for m in list(board.legal_moves):
        if kind == "win_queen":
            target = board.piece_at(m.to_square)
            if target is not None and target.piece_type == chess.QUEEN:
                out.add(m.uci())
            continue
        board.push(m)
        try:
            if kind == "mate_in_1" and board.is_checkmate():
                out.add(m.uci())
            elif kind == "avoid_mate_in_1" and not _mcts_has_mate_in_one(board):
                out.add(m.uci())
            elif kind == "mate_in_2" and _mcts_forces_mate_next(board):
                out.add(m.uci())
        finally:
            board.pop()
    return out


def mcts_verify_tactics(tactics=None) -> list[str]:
    """Devuelve la lista de discrepancias entre `solutions` y las reglas (vacia = todo bien)."""
    problems = []
    for t in tactics or MCTS_TACTICS:
        board = chess.Board(t["fen"])
        if not board.is_valid():
            problems.append(f"{t['id']}: FEN invalido")
            continue
        rules = mcts_tactic_solutions(t["fen"], t["kind"])
        if rules != set(t["solutions"]):
            problems.append(f"{t['id']}: reglas={sorted(rules)} declaradas={sorted(t['solutions'])}")
    return problems


def mcts_tactics_benchmark(positions_with_solutions=None, budgets=(50, 200, 800),
                           seeds=range(5), agent_factory=None, label: str = "",
                           **agent_kwargs) -> list[dict]:
    """Experimento 4: cada tactica x presupuesto (simulaciones fijas) x semilla -> una fila.

    `agent_factory(budget)` permite cambiar el agente; por defecto
    `MCTSAgent(simulations=budget, **agent_kwargs)`.
    """
    rows = []
    for t in positions_with_solutions or MCTS_TACTICS:
        for budget in budgets:
            agent = (agent_factory(budget) if agent_factory is not None
                     else MCTSAgent(simulations=budget, **agent_kwargs))
            for seed in seeds:
                board = chess.Board(t["fen"])
                t0 = time.perf_counter()
                d = agent.choose(board, time_limit_s=0.0, seed=seed)
                wall = time.perf_counter() - t0
                info = agent.last_info
                uci = d.move.uci() if d.move is not None else ""
                visits = info.get("root_visits", {})
                rows.append({
                    "config": label, "tactic": t["id"], "kind": t["kind"], "budget": budget, "seed": seed,
                    "move": uci, "correct": int(uci in t["solutions"]),
                    "elapsed_s": round(wall, 5), "simulations": d.stats.simulations,
                    "nodes": d.stats.nodes, "leaves": d.stats.leaves,
                    "max_depth": d.stats.completed_depth,
                    "best_visits": visits.get(uci, 0),
                    "visit_share": round(visits.get(uci, 0) / max(1, d.stats.simulations), 4),
                    "truncated_rollouts": info.get("truncated_rollouts", 0),
                    "terminal_rollouts": info.get("terminal_rollouts", 0),
                    "win_probability": round(info.get("win_probability") or 0.0, 4),
                    "score_cp": round(d.score, 1) if d.score is not None else "",
                    "n_solutions": len(t["solutions"]),
                    "n_legal": chess.Board(t["fen"]).legal_moves.count(),
                })
    return rows


def mcts_benchmark_summary(rows) -> list[dict]:
    """Por configuracion y presupuesto: tasa de acierto, tiempo medio y su desviacion."""
    out = []
    for config, budget in sorted({(r.get("config", ""), r["budget"]) for r in rows}):
        sel = [r for r in rows if r.get("config", "") == config and r["budget"] == budget]
        times = [r["elapsed_s"] for r in sel]
        out.append({
            "config": config, "budget": budget, "trials": len(sel),
            "hit_rate": sum(r["correct"] for r in sel) / len(sel),
            "time_mean_s": statistics.mean(times),
            "time_std_s": statistics.stdev(times) if len(times) > 1 else 0.0,
        })
    return out


def mcts_non_monotonic_cases(rows) -> list[tuple]:
    """(config, tactica, semilla, aciertos por presupuesto) donde un presupuesto mayor
    FALLA despues de que uno menor acerto: la curva media no garantiza mejora por semilla."""
    budgets = sorted({r["budget"] for r in rows})
    table = {}
    for r in rows:
        table.setdefault((r.get("config", ""), r["tactic"], r["seed"]), {})[r["budget"]] = r["correct"]
    cases = []
    for key, hits in sorted(table.items()):
        seq = [hits.get(b) for b in budgets]
        if any(a == 1 and b == 0 for a, b in zip(seq, seq[1:])):
            cases.append((key[0], key[1], key[2], seq))
    return cases


def mcts_rows_to_csv(rows, path: str) -> None:
    """Escribe las filas del benchmark en CSV (UTF-8, encabezado = claves de la fila)."""
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


### MCTS-UCT: diseño y Experimento 4

Implementamos MCTS-UCT en `chess_ai/mcts.py` (`MCTSAgent`). Cada simulación recorre los cuatro
pasos de AIMA: **selección** con UCT entre hijos de nodos totalmente expandidos, **expansión** de una
sola jugada no explorada, **simulación** (rollout) y **retropropagación**.

**Perspectiva.** Cada nodo acumula su recompensa W desde el jugador que hizo la jugada que lleva a él.
Así, cuando UCT elige entre los hijos de un nodo, maximiza `W/N + c·sqrt(ln N_padre / N_hijo)` para el
jugador que está al turno en ese nodo: blancas maximizan para blancas y negras para negras. La
recompensa se calcula una vez desde blancas y el método `_reward_for` la convierte para cada nodo
(`r` o `1 − r`). Lo comprobamos con una variante defectuosa que guarda todo desde blancas: con negras al
turno, en un mate en uno (Ta1#) y en una dama colgada, el agente correcto acierta y el defectuoso falla
con las mismas semillas. También observamos que la variante «todo desde el jugador raíz» (suponer que el
rival colabora) eligió las mismas jugadas que el agente correcto con ≤ 800 simulaciones: casi ningún hijo
de la raíz llega a tener todas sus respuestas expandidas, así que los valores de los nietos aún no deciden.
Por eso un acierto en el benchmark no basta para validar la perspectiva.

**Recompensa.** Terminal (con `search_outcome`): victoria 1, tablas 0.5, derrota 0. Si el rollout
llega al límite sin terminar, la recompensa es `1/(1 + e^(−E/400))`, con E la evaluación común desde
blancas: un peón de ventaja vale 0.56, una dama 0.905, y nunca 1, que se reserva para la victoria real.
Esos rollouts se cuentan como **truncados**, separados de las tablas (`last_info["truncated_rollouts"]`).

**Política y límite de rollout.** Por defecto, 4 plies con sesgo a capturas: con probabilidad 0.75 se
elige al azar entre las capturas disponibles; si no, una jugada legal uniforme. La elegimos midiendo:
el rollout uniforme de 8 plies acertaba menos y costaba el doble. También ofrecemos `rollout_limit=0`,
que puntúa la hoja directamente con la sigmoide de E.

**Regla final y puntaje.** Se juega el hijo de la raíz con más visitas; si hay empate, el de mayor valor
medio y, después, el de menor UCI. `Decision.score` convierte el valor medio q del hijo elegido (la
probabilidad estimada de ganar del jugador raíz) a centipeones con la inversa de la sigmoide,
`400·ln(q/(1−q))`, para usar la misma escala que minimax. Si el hijo elegido es un mate inmediato se
reporta `MATE_SCORE − 1`.

**Presupuesto y reproducibilidad.** Con `simulations=N` y la misma semilla, el árbol, las visitas y la
jugada son idénticos. Con `simulations=None` se usa el plazo del contrato, revisado antes de cada
simulación. Ese modo depende de la máquina y de su carga: con 0.2 s y la misma semilla hicimos entre 339
y 510 simulaciones en la posición inicial (10 corridas) y entre 221 y 460 en una posición de medio juego.
El tablero se restaura contando los `push` de cada simulación y deshaciéndolos en un `finally`. Esta
opción fue un poco más rápida que copiar el tablero en cada simulación (3220 frente a 3114
simulaciones/s en la posición inicial; 1752 frente a 1737 en medio juego), aunque la diferencia queda
dentro del ruido.

#### Rendimiento (simulaciones por segundo, 1000 simulaciones, media de 3 semillas)

| Configuración | Posición inicial | Medio juego (14 plies) |
|---|---:|---:|
| capturas, L4 (por defecto) | 3220 | 1752 |
| sin rollout, L0 | 6345 | 3497 |
| uniforme, L4 | 2764 | 1705 |
| uniforme, L8 | 1420 | 1423 |

Con el plazo del torneo (0.2 s) caben unas 380–415 simulaciones en promedio con la configuración por
defecto (rango observado 221–510). Las mediciones se hicieron con otros procesos del taller en
ejecución, así que el tiempo tiene ruido apreciable.

#### Experimento 4: tácticas × presupuesto × semilla

Diez tácticas (`MCTS_TACTICS`, con blancas o negras al turno), cuyo conjunto de jugadas correctas se
recalcula con las reglas de python-chess: 4 mates en uno, 2 damas colgadas, 2 posiciones de «evitar mate
en uno» (8 de 28 jugadas son correctas en cada una) y 2 mates en dos con sacrificio de torre.
Presupuestos de 50, 200, 800 y 2000 simulaciones; semillas 0 a 4; tres configuraciones. Las 600 filas
están en `resultados/mcts_tacticas.csv`.

| Configuración | Simulaciones | Tasa de acierto | Tiempo medio (s) | Desv. estándar (s) |
|---|---:|---:|---:|---:|
| capturas, L4 | 50 | 0.70 | 0.021 | 0.009 |
| capturas, L4 | 200 | 0.80 | 0.088 | 0.037 |
| capturas, L4 | 800 | 0.80 | 0.351 | 0.119 |
| capturas, L4 | 2000 | 0.80 | 0.840 | 0.304 |
| sin rollout, L0 | 50 | 0.80 | 0.017 | 0.024 |
| sin rollout, L0 | 200 | 0.80 | 0.053 | 0.023 |
| sin rollout, L0 | 800 | 0.80 | 0.242 | 0.127 |
| sin rollout, L0 | 2000 | 0.80 | 0.536 | 0.154 |
| uniforme, L8 | 50 | 0.66 | 0.054 | 0.039 |
| uniforme, L8 | 200 | 0.76 | 0.199 | 0.103 |
| uniforme, L8 | 800 | 0.72 | 0.796 | 0.369 |
| uniforme, L8 | 2000 | 0.80 | 1.841 | 0.807 |

Aciertos por táctica (de 5 semillas) con 50 / 200 / 800 / 2000 simulaciones:

| Táctica | capturas, L4 | sin rollout, L0 | uniforme, L8 |
|---|---|---|---|
| T1–T3 mates en uno (blancas y negras) | 5/5/5/5 | 5/5/5/5 | 5/5/5/5 |
| T4 mate en uno (Dxf7#) | 2/5/5/5 | 5/5/5/5 | 5/5/5/5 |
| T5 dama colgada (blancas) | 4/5/5/5 | 5/5/5/5 | 4/5/5/5 |
| T6 dama colgada (negras) | 1/5/5/5 | 5/5/5/5 | 5/5/5/5 |
| T7 evitar mate (negras) | 4/5/5/5 | 5/5/5/5 | 2/4/2/5 |
| T8 evitar mate (blancas) | 2/5/5/5 | 5/5/5/5 | 2/4/4/5 |
| T9 mate en dos (blancas) | 3/0/0/0 | 0/0/0/0 | 0/0/0/0 |
| T10 mate en dos (negras) | 4/0/0/0 | 0/0/0/0 | 0/0/0/0 |

**Lectura.**
- Las tácticas de un ply (mates en uno y damas colgadas) y las defensas contra un mate en uno se
  resuelven desde 200 simulaciones en las tres configuraciones, salvo el rollout uniforme de 8 plies, cuyo
  ruido hace fallar las defensas incluso con 800 simulaciones.
- En estas tácticas el rollout no mejora la evaluación directa de la hoja: L0 acierta igual o más con
  todos los presupuestos y es más rápido. Lo atribuimos a que el rollout aleatorio añade varianza y E ya
  mide el material.
- Ninguna configuración resuelve los mates en dos con 200 simulaciones o más. Tras el sacrificio de
  torre, el rival tiene una única respuesta y luego hay unas 20 jugadas posibles. UCT promedia esas ramas:
  con 2000 simulaciones el nodo posterior a la respuesta tenía 40 visitas repartidas entre 20 hijos y un
  valor medio de 0.52, por debajo del de jugadas tranquilas (0.755). Minimax tomaría el mínimo y vería el
  mate a profundidad 3.
- **Una curva media creciente no implica mejora monótona por semilla, y nuestros datos lo muestran.**
  Con capturas y L4, la tasa media sube de 0.70 a 0.80 y luego se mantiene. Sin embargo, 7 pares
  (táctica, semilla) acertaron con 50 simulaciones y fallaron con todos los presupuestos mayores (T9 y T10,
  secuencia 1→0→0→0). Además, los aciertos con 50 simulaciones son un artefacto del desempate: los 25
  hijos de la raíz tenían exactamente 2 visitas y ganó el de mayor valor medio, porque el rollout con
  sesgo a capturas encontraba Txd8# por casualidad. Con el rollout uniforme L8, T7 dio las secuencias
  0→1→0→1 (semilla 0) y 1→1→0→1 (semilla 1), y T8 dio 1→1→0→1 (semilla 4). En esa configuración la media
  baja de 0.76 (200) a 0.72 (800) antes de subir a 0.80.

In [ ]:
# ===== Demostracion D: MCTS-UCT con simulaciones fijas (reproducible) =====
_tacticas = {t["id"]: t for t in MCTS_TACTICS}
for _id in ("T3_mate1_b_backrank", "T6_queen_b"):
    _t = _tacticas[_id]
    _b = chess.Board(_t["fen"])
    _antes = (_b.fen(), list(_b.move_stack))
    _ag = MCTSAgent(simulations=300)
    _d = _ag.choose(_b, time_limit_s=0, seed=0)
    _ag_otro = MCTSAgent(simulations=300)
    _d_otra = _ag_otro.choose(_b, time_limit_s=0, seed=0)
    assert (_b.fen(), list(_b.move_stack)) == _antes
    # misma semilla y mismas simulaciones -> misma jugada y mismas visitas en la raiz
    assert _d.move == _d_otra.move and _ag.last_info["root_visits"] == _ag_otro.last_info["root_visits"]
    _info = _ag.last_info
    _top = list(_info["root_visits"].items())[:4]
    print(f"{_id} ({'negras' if _b.turn == chess.BLACK else 'blancas'} al turno): juega {_b.san(_d.move)} "
          f"{'(correcta)' if _d.move.uci() in _t['solutions'] else '(INCORRECTA)'}; "
          f"q = {_info['win_probability']:.3f} desde quien decide; score = {_d.score:+.0f} cp")
    print(f"   simulaciones={_d.stats.simulations}, nodos={_d.stats.nodes}, rollouts truncados="
          f"{_info['truncated_rollouts']}, terminales={_info['terminal_rollouts']}, "
          f"profundidad max={_d.stats.completed_depth}")
    print("   visitas de la raiz (top 4): " + ", ".join(
        f"{_b.san(chess.Move.from_uci(u))}={v} (valor {_info['root_values'][u]:.2f})" for u, v in _top))


### Mejora adicional elegida: pesos de E ajustados por búsqueda local

De las tres mejoras que propone el enunciado (quietud; tabla de transposición; evaluación con pesos ajustados por búsqueda local) elegimos la **tercera**. Así, la misma implementación cuenta para la mejora con ablación y para la actividad 7C. Lo que hay en cada lugar del cuaderno:

| Qué | Dónde |
|---|---|
| Código del ajuste: datos de autojuego, objetivo, hill climbing y recocido con presupuesto | celda siguiente (`chess_ai/tuning.py`) |
| Pesos iniciales/finales, entrenamiento frente a prueba, ablación por peso y sobreajuste | sección 7C (resultados guardados en `resultados/pesos_ajustados.json`) |
| Ablación en partidas: el mismo alfa-beta con $E_0$ y con $E^*$, todo lo demás fijo | sección 9, Experimento 3 |

La quietud quedó como una **opción apagada** del alfa-beta mejorado (`quiescence=False`). No la presentamos como mejora porque cambia el árbol y sus valores ya no se comparan con minimax de profundidad fija (bitácora S-11).

Después del ajuste va el **registro de agentes** (`chess_ai/registry.py`): una sola tabla `AGENT_FACTORIES` con los seis agentes, que usan tanto la interfaz como el torneo. Así, lo que se juega en la interfaz es exactamente lo que se mide en el torneo. Minimax y alfa-beta usan la profundidad configurada; alfa-beta (el «natural», sin ordenar) se distingue del mejorado justamente por la ordenación y la profundización con plazo; MCTS usa simulaciones fijas si se indican y, si no, el plazo.

In [ ]:
# ===== chess_ai/tuning.py =====
"""Mejora adicional: pesos de E(s) ajustados por busqueda local (apartado 7C).

Resumen del metodo (detalles y justificacion en apuntes/07c_ajuste.md y en la
bitacora T-01 .. T-xx):

1. DATOS. Posiciones muestreadas de partidas de autojuego entre jugadores
   epsilon-voraces: en cada ply el jugador juega al azar con probabilidad eps
   (Aleatorio) y si no juega el Voraz de un ply con SOLO material. Cada bando de
   cada partida sortea su eps en `eps_range`. Entrenamiento y prueba usan
   generadores de semillas distintos (`train_seed`, `test_seed`), se separan
   POR PARTIDA (nunca una misma partida en ambos lados) y se eliminan los
   duplicados por FEN (colocacion, turno, enroques y al paso): una posicion de
   prueba que tambien aparezca en entrenamiento se descarta.
   Se descartan: partidas truncadas (sin resultado oficial), posiciones
   terminales, posiciones en jaque y posiciones cuya ultima jugada fue una
   captura (intercambio a medias: E estatica mide mal el material).
2. ETIQUETA (estilo Texel). y = resultado final de la partida desde blancas:
   1 (ganan blancas), 0.5 (tablas), 0 (ganan negras). No viene de ningun motor.
   Es ruidosa: una posicion buena puede perderse por una jugada aleatoria
   posterior. El ruido se promedia con muchas posiciones; no se elimina.
3. OBJETIVO. J(w) = media[(sigmoide(E_w(s)/K) - y)^2] sobre ENTRENAMIENTO, con
   E_w = w . features(s) recortada a +-EVAL_BOUND (igual que `evaluate`).
   K (centipeones) se calibra UNA vez, con los pesos iniciales y solo con
   entrenamiento (seccion dorada sobre log K); despues queda fija.
4. ESCALA. wm = 100 queda FIJO: fija la unidad (centipeones). Si wm fuera
   libre, multiplicar todos los pesos y K por la misma constante daria el
   mismo J: la escala no estaria identificada.
5. BUSQUEDA LOCAL con presupuesto EXPLICITO de evaluaciones de J (cuenta todo:
   la evaluacion inicial y la calibracion de T0 del recocido):
   * Hill climbing por coordenadas: vecinos w_i +- paso_i de cada peso libre;
     se mueve al mejor vecino si mejora ESTRICTAMENTE; si ninguno mejora, los
     pasos se reducen a la mitad. Es el HC del Taller 2 (TSP) con otra vecindad.
   * Recocido simulado con semilla: perturba un peso al azar, acepta empeorar
     con prob. exp(-delta/T) (Metropolis), enfriamiento geometrico; T0 se
     calibra como en el Taller 2: T0 = -mediana(deltas>0)/ln(p0).
   Regla de eleccion fijada ANTES de mirar prueba: el resultado final es el de
   menor J de ENTRENAMIENTO entre HC y SA (empate -> HC).
6. REPORTE. Pesos inicial/final, J de entrenamiento y de prueba antes/despues,
   historial (CSV para graficar), ablacion por peso (devolver UN peso del
   ajustado al inicial) y una comparacion pareada en prueba agrupada por
   partida. Prueba solo se mira DESPUES de terminar la busqueda.

Salidas de `run_tuning`: resultados/pesos_ajustados.json (pesos y
configuracion), resultados/tuning_historial.csv, resultados/tuning_posiciones.csv
(posiciones de entrenamiento y prueba con su etiqueta) y
resultados/tuning_ablacion.csv. `load_tuned_evaluator()` devuelve
`Evaluator(pesos, "E*")` para la ablacion a nivel de partidas.
"""
from __future__ import annotations

import csv
import json
import math
import os
import platform
import random
import time
from dataclasses import asdict, dataclass

import chess
import numpy as np


_TUNE_WEIGHT_KEYS = ("wm", "wp", "wk", "wa")
_TUNE_ROUND = 3            # decimales de los pesos candidatos (lo guardado = lo evaluado)
_TUNE_IMPROVE_EPS = 1e-12  # mejora estricta minima en J


@dataclass
class TuningConfig:
    """Configuracion completa de una corrida (se guarda junto a los pesos)."""
    # --- datos ---
    train_games: int = 600
    test_games: int = 300
    train_seed: int = 31
    test_seed: int = 7919
    eps_range: tuple = (0.0, 0.5)
    max_plies: int = 200
    min_ply: int = 8          # no se muestrean las primeras jugadas (apertura casi fija)
    stride: int = 3           # una posicion cada `stride` plies (menos correlacion)
    # --- objetivo ---
    k_evals: int = 30         # evaluaciones de J usadas para calibrar K
    k_bounds: tuple = (20.0, 5000.0)
    # --- busqueda local ---
    initial_weights: tuple = DEFAULT_WEIGHTS
    free: tuple = (1, 2, 3)   # indices libres: wp, wk, wa (wm fijo = escala)
    bounds: tuple = ((100.0, 100.0), (0.0, 100.0), (0.0, 100.0), (0.0, 30.0))
    steps: tuple = (0.0, 8.0, 8.0, 2.0)
    min_step_ratio: float = 1.0 / 64.0
    max_evals: int = 300      # presupuesto de evaluaciones de J POR metodo
    sa_seed: int = 2026
    sa_p0: float = 0.5
    sa_calib: int = 20
    sa_t_final_ratio: float = 1e-3


@dataclass(frozen=True)
class TuningSample:
    """Una posicion etiquetada. `label` es el resultado final desde blancas."""
    fen: str
    label: float
    game_id: str
    ply: int


# ---------------------------------------------------------------- datos
def _tune_material_white(board: chess.Board) -> float:
    """Solo material (centipeones, desde blancas): evaluador de los jugadores de autojuego.

    Se usa SOLO material para generar partidas: asi los datos no presuponen los
    pesos posicionales (wp, wk, wa) que luego se ajustan.
    """
    total = 0.0
    for pt in (chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN):
        total += PIECE_VALUES[pt] * (chess.popcount(board.pieces_mask(pt, chess.WHITE))
                                     - chess.popcount(board.pieces_mask(pt, chess.BLACK)))
    return 100.0 * total


def _tune_position_key(board: chess.Board) -> str:
    """Clave de duplicado: colocacion, turno, enroques y al paso (sin relojes)."""
    return " ".join(board.fen().split()[:4])


def _tune_play_game(game_seed: int, cfg: TuningConfig):
    """Juega una partida eps-voraz contra eps-voraz.

    Devuelve (resultado, terminacion, posiciones) donde posiciones es una lista
    de (ply, fen, clave) muestreadas segun min_ply/stride y el filtro de quietud.
    resultado es None si la partida se trunca en `max_plies`.
    """
    rng = random.Random(game_seed)
    lo, hi = cfg.eps_range
    eps = {chess.WHITE: rng.uniform(lo, hi), chess.BLACK: rng.uniform(lo, hi)}
    greedy = GreedyAgent(_tune_material_white, name="VorazMaterial")
    rand = RandomAgent()
    board = chess.Board()
    sampled = []
    last_was_capture = False
    while True:
        outcome = game_outcome(board)
        if outcome is not None:
            return outcome.result(), outcome.termination.name.lower(), sampled
        ply = len(board.move_stack)
        if ply >= cfg.max_plies:
            return None, "truncated", sampled
        if (ply >= cfg.min_ply and (ply - cfg.min_ply) % cfg.stride == 0
                and not last_was_capture and not board.is_check()
                and search_outcome(board) is None):
            sampled.append((ply, board.fen(), _tune_position_key(board)))
        agent = rand if rng.random() < eps[board.turn] else greedy
        move = agent.choose(board, time_limit_s=0.0, seed=rng.randrange(2 ** 31)).move
        last_was_capture = board.is_capture(move)
        board.push(move)


def _tune_collect(n_games: int, base_seed: int, split: str, cfg: TuningConfig, exclude):
    """Juega `n_games` partidas y devuelve (muestras, info). `exclude`: claves prohibidas."""
    master = random.Random(base_seed)
    samples, seen = [], set()
    info = {"games": n_games, "truncated_games": 0, "results": {}, "terminations": {},
            "positions_sampled": 0, "duplicates_dropped": 0, "excluded_by_other_split": 0}
    for g in range(n_games):
        game_seed = master.randrange(2 ** 31)
        result, termination, positions = _tune_play_game(game_seed, cfg)
        info["terminations"][termination] = info["terminations"].get(termination, 0) + 1
        if result is None:
            info["truncated_games"] += 1
            continue
        info["results"][result] = info["results"].get(result, 0) + 1
        label = {"1-0": 1.0, "0-1": 0.0}.get(result, 0.5)
        game_id = f"{split}-{g:04d}"
        for ply, fen, key in positions:
            info["positions_sampled"] += 1
            if exclude is not None and key in exclude:
                info["excluded_by_other_split"] += 1
                continue
            if key in seen:
                info["duplicates_dropped"] += 1
                continue
            seen.add(key)
            samples.append(TuningSample(fen, label, game_id, ply))
    info["positions_kept"] = len(samples)
    return samples, info


def generate_tuning_data(cfg: TuningConfig | None = None) -> dict:
    """Posiciones de entrenamiento y prueba SEPARADAS (por partida y por FEN).

    Devuelve {"train": [TuningSample], "test": [TuningSample], "info": {...}}.
    Reproducible: depende solo de la configuracion (semillas incluidas).
    """
    cfg = cfg or TuningConfig()
    train, info_train = _tune_collect(cfg.train_games, cfg.train_seed, "train", cfg, None)
    train_keys = {_tune_position_key(chess.Board(s.fen)) for s in train}
    test, info_test = _tune_collect(cfg.test_games, cfg.test_seed, "test", cfg, train_keys)
    return {"train": train, "test": test, "info": {"train": info_train, "test": info_test}}


def tuning_feature_matrix(samples) -> tuple:
    """(F, y): matriz n x 4 de features (M, P, K, A) y vector de etiquetas."""
    feats = np.array([features(chess.Board(s.fen)) for s in samples], dtype=float).reshape(-1, 4)
    labels = np.array([s.label for s in samples], dtype=float)
    return feats, labels


# ---------------------------------------------------------------- objetivo
def tuning_objective(weights, feats, labels, k: float) -> float:
    """J = media[(sigmoide(E_w/K) - y)^2]. E_w se recorta a +-EVAL_BOUND como en `evaluate`."""
    e = np.clip(feats @ np.asarray(weights, dtype=float), -EVAL_BOUND, EVAL_BOUND)
    p = 1.0 / (1.0 + np.exp(-e / k))
    return float(np.mean((p - labels) ** 2))


def fit_scale_k(feats, labels, weights, bounds=(20.0, 5000.0), evals: int = 30):
    """Calibra K por seccion dorada sobre log K (solo datos de entrenamiento).

    Devuelve (K, evaluaciones usadas). J(K) es unimodal en la practica para
    esta familia; si no lo fuera, la seccion dorada da un minimo local.
    """
    a, b = math.log(bounds[0]), math.log(bounds[1])
    inv_phi = (math.sqrt(5.0) - 1.0) / 2.0
    c, d = b - inv_phi * (b - a), a + inv_phi * (b - a)
    fc = tuning_objective(weights, feats, labels, math.exp(c))
    fd = tuning_objective(weights, feats, labels, math.exp(d))
    used = 2
    while used < evals:
        if fc < fd:
            b, d, fd = d, c, fc
            c = b - inv_phi * (b - a)
            fc = tuning_objective(weights, feats, labels, math.exp(c))
        else:
            a, c, fc = c, d, fd
            d = a + inv_phi * (b - a)
            fd = tuning_objective(weights, feats, labels, math.exp(d))
        used += 1
    return math.exp(c if fc < fd else d), used


# ---------------------------------------------------------------- busqueda local
def _tune_clip(value: float, lo: float, hi: float) -> float:
    return round(min(hi, max(lo, value)), _TUNE_ROUND)


def _tune_row(method, n_eval, cand, f_cand, f_cur, f_best, accepted, control):
    row = {"method": method, "eval": n_eval, "candidate_obj": f_cand,
           "current_obj": f_cur, "best_obj": f_best, "accepted": int(bool(accepted)),
           "control": control}
    row.update({k: v for k, v in zip(_TUNE_WEIGHT_KEYS, cand)})
    return row


def hill_climb_weights(objective_fn, initial, *, free, bounds, steps, max_evals: int,
                       min_step_ratio: float = 1.0 / 64.0, shrink: float = 0.5) -> dict:
    """Hill climbing por coordenadas con reduccion de paso y presupuesto explicito.

    Vecindad: para cada indice libre i, w_i + paso_i y w_i - paso_i (recortado a
    `bounds`; un vecino igual al actual no se evalua). Se evaluan los vecinos y
    se acepta el MEJOR si mejora estrictamente J (ascenso mas empinado). Si
    ninguno mejora, todos los pasos se multiplican por `shrink`; se para cuando
    todos son menores que paso_inicial * min_step_ratio (optimo local a esa
    resolucion) o cuando se agota el presupuesto. Cada llamada a
    `objective_fn` cuenta 1, incluida la del punto inicial. Nunca empeora J.
    """
    if max_evals < 1:
        raise ValueError("max_evals debe ser >= 1")
    current = [float(x) for x in initial]
    f_cur = objective_fn(tuple(current))
    evals = 1
    history = [_tune_row("HC", evals, current, f_cur, f_cur, f_cur, True, "inicio")]
    step = [float(s) for s in steps]
    min_step = [s * min_step_ratio for s in steps]
    stop = "presupuesto"
    while evals < max_evals:
        best_cand, best_f, best_row = None, f_cur, None
        control = "paso=" + "/".join(f"{step[j]:g}" for j in free)
        for i in free:
            for sgn in (1.0, -1.0):
                if evals >= max_evals:
                    break
                cand = list(current)
                cand[i] = _tune_clip(current[i] + sgn * step[i], *bounds[i])
                if cand[i] == current[i]:
                    continue
                f = objective_fn(tuple(cand))
                evals += 1
                if f < best_f - _TUNE_IMPROVE_EPS:
                    best_cand, best_f = cand, f
                    best_row = len(history)
                # best_obj de la fila = mejor J conocido hasta esta evaluacion
                history.append(_tune_row("HC", evals, cand, f, f_cur, best_f, False, control))
        if best_cand is not None:
            current, f_cur = best_cand, best_f
            history[best_row]["accepted"] = 1
        else:
            if evals >= max_evals:
                break
            step = [s * shrink for s in step]
            if all(step[i] < min_step[i] for i in free):
                stop = "optimo_local"
                break
    return {"method": "HC", "best_weights": tuple(current), "best_obj": f_cur,
            "initial_obj": history[0]["candidate_obj"], "evals": evals,
            "stop_reason": stop, "final_steps": tuple(step), "history": history}


def anneal_weights(objective_fn, initial, *, free, bounds, steps, max_evals: int, seed: int,
                   p0: float = 0.5, calib_samples: int = 20, t_final_ratio: float = 1e-3) -> dict:
    """Recocido simulado con semilla y presupuesto explicito (Metropolis).

    Propuesta: un indice libre al azar, w_i + U(-paso_i, paso_i), recortado.
    Calibracion (cuenta en el presupuesto): `calib_samples` vecinos del punto
    inicial; T0 = -mediana(deltas > 0)/ln(p0) (si no hay deltas positivos se
    usa la mediana de |delta| y, en ultimo caso, 1e-6). Enfriamiento
    geometrico tal que T llega a T0 * t_final_ratio al agotar el presupuesto.
    Se devuelve la MEJOR solucion vista (nunca peor que la inicial).
    """
    if max_evals < 1:
        raise ValueError("max_evals debe ser >= 1")
    rng = random.Random(seed)
    current = [float(x) for x in initial]
    f_cur = objective_fn(tuple(current))
    evals = 1
    best, f_best = list(current), f_cur
    history = [_tune_row("SA", evals, current, f_cur, f_cur, f_best, True, "inicio")]

    def propose(point):
        i = rng.choice(free)
        cand = list(point)
        cand[i] = _tune_clip(point[i] + rng.uniform(-steps[i], steps[i]), *bounds[i])
        return cand

    deltas = []
    for _ in range(min(calib_samples, max_evals - evals)):
        cand = propose(current)
        f = objective_fn(tuple(cand))
        evals += 1
        deltas.append(f - f_cur)
        if f < f_best - _TUNE_IMPROVE_EPS:
            best, f_best = cand, f
        history.append(_tune_row("SA", evals, cand, f, f_cur, f_best, False, "calibracion"))
    pos = sorted(d for d in deltas if d > 0) or sorted(abs(d) for d in deltas if d != 0)
    scale = pos[len(pos) // 2] if pos else 1e-6
    t0 = -scale / math.log(p0)
    remaining = max(1, max_evals - evals)
    alpha = t_final_ratio ** (1.0 / remaining)
    temp = t0
    while evals < max_evals:
        cand = propose(current)
        f = objective_fn(tuple(cand))
        evals += 1
        delta = f - f_cur
        accepted = delta <= 0 or rng.random() < math.exp(-delta / temp)
        if accepted:
            current, f_cur = cand, f
        if f < f_best - _TUNE_IMPROVE_EPS:
            best, f_best = cand, f
        history.append(_tune_row("SA", evals, cand, f, f_cur, f_best, accepted, f"T={temp:.3e}"))
        temp *= alpha
    return {"method": "SA", "best_weights": tuple(best), "best_obj": f_best,
            "initial_obj": history[0]["candidate_obj"], "evals": evals,
            "stop_reason": "presupuesto", "t0": t0, "seed": seed, "history": history}


# ---------------------------------------------------------------- reporte
def _tune_paired_by_game(samples, feats, labels, k, w_old, w_new) -> dict:
    """Diferencia pareada del error cuadratico (nuevo - viejo) promediada por PARTIDA.

    Las posiciones de una misma partida estan correlacionadas (misma etiqueta),
    por eso la unidad independiente es la partida, no la posicion. Negativo =
    los pesos nuevos predicen mejor. Se reporta media, error estandar entre
    partidas y fraccion de partidas donde mejora.
    """
    def sq_err(w):
        e = np.clip(feats @ np.asarray(w, dtype=float), -EVAL_BOUND, EVAL_BOUND)
        return (1.0 / (1.0 + np.exp(-e / k)) - labels) ** 2
    diff = sq_err(w_new) - sq_err(w_old)
    per_game = {}
    for s, d in zip(samples, diff):
        per_game.setdefault(s.game_id, []).append(float(d))
    means = np.array([np.mean(v) for v in per_game.values()], dtype=float)
    n = len(means)
    se = float(np.std(means, ddof=1) / math.sqrt(n)) if n > 1 else float("nan")
    return {"games": n, "mean_diff": float(np.mean(means)) if n else float("nan"),
            "se": se, "frac_games_improved": float(np.mean(means < 0)) if n else float("nan")}


def run_tuning(cfg: TuningConfig | None = None, out_dir: str | None = None,
               write: bool = True, verbose: bool = False) -> dict:
    """Corrida completa: datos -> K -> HC y SA (solo entrenamiento) -> reporte en prueba.

    Si `write`, guarda en `out_dir` (por defecto resultados/ del proyecto):
    pesos_ajustados.json, tuning_historial.csv, tuning_posiciones.csv y
    tuning_ablacion.csv. Devuelve el mismo diccionario que se guarda en el JSON
    (mas "history" y "data").
    """
    cfg = cfg or TuningConfig()
    t_start, cpu_start = time.perf_counter(), time.process_time()
    data = generate_tuning_data(cfg)
    t_data, cpu_data = time.perf_counter() - t_start, time.process_time() - cpu_start
    if not data["train"] or not data["test"]:
        raise ValueError("no hay posiciones suficientes: aumente train_games/test_games")
    f_tr, y_tr = tuning_feature_matrix(data["train"])
    f_te, y_te = tuning_feature_matrix(data["test"])
    w0 = tuple(float(x) for x in cfg.initial_weights)

    # --- solo entrenamiento a partir de aqui hasta elegir los pesos ---
    k, k_used = fit_scale_k(f_tr, y_tr, w0, cfg.k_bounds, cfg.k_evals)

    def obj_train(w):
        return tuning_objective(w, f_tr, y_tr, k)

    t_search = time.perf_counter()
    hc = hill_climb_weights(obj_train, w0, free=cfg.free, bounds=cfg.bounds, steps=cfg.steps,
                            max_evals=cfg.max_evals, min_step_ratio=cfg.min_step_ratio)
    sa = anneal_weights(obj_train, w0, free=cfg.free, bounds=cfg.bounds, steps=cfg.steps,
                        max_evals=cfg.max_evals, seed=cfg.sa_seed, p0=cfg.sa_p0,
                        calib_samples=cfg.sa_calib, t_final_ratio=cfg.sa_t_final_ratio)
    t_search = time.perf_counter() - t_search
    chosen = hc if hc["best_obj"] <= sa["best_obj"] else sa
    w_star = tuple(chosen["best_weights"])

    # --- recien ahora se mira prueba (los pesos ya estan elegidos) ---
    def obj_test(w):
        return tuning_objective(w, f_te, y_te, k)

    const = float(np.mean(y_tr))
    material_only = (w0[0], 0.0, 0.0, 0.0)
    ablation = []
    for i in cfg.free:
        w = list(w_star)
        w[i] = w0[i]
        ablation.append({"weight": _TUNE_WEIGHT_KEYS[i], "feature": FEATURE_NAMES[i],
                         "value_tuned": w_star[i], "value_initial": w0[i],
                         "train_obj": obj_train(w), "test_obj": obj_test(w)})
    report = {
        "weights": list(w_star),
        "weight_names": list(_TUNE_WEIGHT_KEYS),
        "feature_names": list(FEATURE_NAMES),
        "initial_weights": list(w0),
        "chosen_method": chosen["method"],
        "selection_rule": "menor J de entrenamiento entre HC y SA (empate -> HC); prueba no participa",
        "K": k,
        "k_calibration_evals": k_used,
        "objective": {
            "train_initial": obj_train(w0), "train_tuned": obj_train(w_star),
            "test_initial": obj_test(w0), "test_tuned": obj_test(w_star),
            "train_material_only": obj_train(material_only),
            "test_material_only": obj_test(material_only),
            "train_constant": float(np.mean((const - y_tr) ** 2)),
            "test_constant": float(np.mean((const - y_te) ** 2)),
            "constant_prediction": const,
        },
        "methods": {
            m["method"]: {"best_weights": list(m["best_weights"]), "train_obj": m["best_obj"],
                          "test_obj": obj_test(m["best_weights"]), "evals": m["evals"],
                          "stop_reason": m["stop_reason"],
                          **({"t0": m["t0"], "seed": m["seed"]} if m["method"] == "SA" else {}),
                          **({"final_steps": list(m["final_steps"])} if m["method"] == "HC" else {})}
            for m in (hc, sa)},
        "ablation": ablation,
        "paired_test_by_game": _tune_paired_by_game(data["test"], f_te, y_te, k, w0, w_star),
        "dataset": {"train_positions": len(data["train"]), "test_positions": len(data["test"]),
                    "train_label_mean": float(np.mean(y_tr)), "test_label_mean": float(np.mean(y_te)),
                    **data["info"]},
        "config": {k2: (list(v) if isinstance(v, tuple) else v) for k2, v in asdict(cfg).items()},
        "versions": {"python": platform.python_version(), "chess": chess.__version__,
                     "numpy": np.__version__},
        # pared (incluye esperas o suspension del equipo) y CPU del proceso (trabajo real)
        "timing_s": {"data": t_data, "search": t_search, "total": time.perf_counter() - t_start,
                     "cpu_data": cpu_data, "cpu_total": time.process_time() - cpu_start},
    }
    if verbose:
        print("K =", round(k, 2), "| pesos iniciales", w0, "-> ajustados", w_star,
              "(" + chosen["method"] + ")")
        print("J entrenamiento", report["objective"]["train_initial"], "->",
              report["objective"]["train_tuned"], "| J prueba",
              report["objective"]["test_initial"], "->", report["objective"]["test_tuned"])
    if write:
        out_dir = out_dir or os.path.dirname(TUNED_WEIGHTS_PATH)
        os.makedirs(out_dir, exist_ok=True)
        save_tuned_weights(report, os.path.join(out_dir, "pesos_ajustados.json"))
        _tune_write_csv(os.path.join(out_dir, "tuning_historial.csv"), hc["history"] + sa["history"])
        _tune_write_csv(os.path.join(out_dir, "tuning_ablacion.csv"), ablation)
        _tune_write_csv(os.path.join(out_dir, "tuning_posiciones.csv"),
                        [{"split": split, "game_id": s.game_id, "ply": s.ply, "label": s.label,
                          "fen": s.fen}
                         for split in ("train", "test") for s in data[split]])
    report["history"] = hc["history"] + sa["history"]
    report["data"] = data
    return report


def _tune_write_csv(path: str, rows) -> None:
    rows = list(rows)
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


# ---------------------------------------------------------------- guardar / cargar
def _tune_project_root() -> str:
    """Raiz del proyecto: junto al paquete si hay __file__; si no (cuaderno), el cwd."""
    here = globals().get("__file__")
    if here:
        return os.path.dirname(os.path.dirname(os.path.abspath(here)))
    return os.getcwd()


TUNED_WEIGHTS_PATH = os.path.join(_tune_project_root(), "resultados", "pesos_ajustados.json")


def save_tuned_weights(report: dict, path: str | None = None) -> str:
    """Guarda pesos + configuracion + metricas en JSON (sin historial ni datos)."""
    path = path or TUNED_WEIGHTS_PATH
    payload = {k: v for k, v in report.items() if k not in ("history", "data")}
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)
    return path


def load_tuned_weights(path: str | None = None, missing_ok: bool = False):
    """Lee los pesos ajustados (tupla de 4). Con `missing_ok`, None si no existe el archivo."""
    path = path or TUNED_WEIGHTS_PATH
    if not os.path.exists(path):
        if missing_ok:
            return None
        raise FileNotFoundError(f"no existe {path}: ejecute run_tuning() primero")
    with open(path, encoding="utf-8") as fh:
        payload = json.load(fh)
    weights = tuple(float(x) for x in payload["weights"])
    if len(weights) != 4:
        raise ValueError("el archivo de pesos debe tener 4 pesos (wm, wp, wk, wa)")
    return weights


def load_tuned_evaluator(path: str | None = None, name: str = "E*") -> Evaluator:
    """`Evaluator(pesos ajustados, "E*")` para la ablacion E0 vs E* en partidas."""
    return Evaluator(load_tuned_weights(path), name)


# Pesos producidos por la ultima corrida real guardada (None si aun no se ha corrido).
# Leer un JSON pequeno es la unica accion a nivel de modulo; el ajuste no se ejecuta aqui.
TUNED_WEIGHTS = load_tuned_weights(missing_ok=True)


In [ ]:
# ===== chess_ai/registry.py =====
"""Registro unico de los seis agentes: lo usan la interfaz, el torneo y los experimentos.

Cada fabrica recibe una configuracion con atributos `depth`, `time_limit_s`, `seed`
y `simulations` (por ejemplo `controller.AgentConfig`) y devuelve un agente NUEVO,
asi ninguna partida comparte estado interno (killers, arbol de MCTS) con otra.

* Minimax y Alfa-beta usan la profundidad configurada (profundidad fija: ignoran
  el plazo; el controlador registra los excesos, D-10).
* Alfa-beta es el NATURAL (sin ordenar): la ordenacion es justamente lo que agrega
  el agente mejorado, y asi la comparacion entre ambos la aisla.
* Alfa-beta mejorado usa el plazo (profundizacion iterativa).
* MCTS usa simulaciones fijas si `simulations` no es None (reproducible); si es
  None, el plazo.
"""
from __future__ import annotations


AGENT_NAMES = ("Aleatorio", "Voraz", "Minimax", "Alfa-beta", "Alfa-beta mejorado", "MCTS")

AGENT_FACTORIES = {
    "Aleatorio": lambda cfg: RandomAgent(),
    "Voraz": lambda cfg: GreedyAgent(),
    "Minimax": lambda cfg: MinimaxAgent(depth=cfg.depth),
    "Alfa-beta": lambda cfg: AlphaBetaAgent(depth=cfg.depth, ordering=False),
    "Alfa-beta mejorado": lambda cfg: AlphaBetaImprovedAgent(),
    "MCTS": lambda cfg: MCTSAgent(simulations=cfg.simulations),
}


def make_agent(label: str, config):
    """Crea un agente del registro; error claro si la etiqueta no existe."""
    if label not in AGENT_FACTORIES:
        raise KeyError(f"agente desconocido: {label!r}; opciones: {', '.join(AGENT_NAMES)}")
    return AGENT_FACTORIES[label](config)


## 6. Integración del controlador
```text
AL_SOLICITAR_TURNO:
    si hay búsqueda activa o partida terminada: informar y salir
    identificar agente según el color al turno
    si es humano: esperar una jugada por la interfaz
    si es agente:
        capturar identificador de versión del estado real
        copiar el tablero conservando la historia
        solicitar Decision al agente
        comprobar plazo, legalidad y que la versión siga vigente
        aplicar exactamente una jugada al tablero real
        registrar métricas y actualizar vista
```

El historial debe sobrevivir a las copias: `board.copy(stack=True)` conserva la pila. Una clave de búsqueda basada solo en la colocación de piezas no detecta por sí misma todas las condiciones de tablas.

Si adoptan un límite temporal estricto de torneo, el controlador debe poder detectar su incumplimiento. Una medición después de devolver no interrumpe un agente bloqueado: documenten la diferencia entre verificación cooperativa y aislamiento en proceso. Para el alcance mínimo basta presupuesto cooperativo y registro del exceso.


In [ ]:
# IMPLEMENTACIÓN E: controlador, selector de agentes y actualización de la vista.
# Reutilice ChessView o adapte su arquitectura conservando la validación legal.


In [ ]:
# ===== chess_ai/controller.py =====
"""Controlador de partida (seccion 6 del enunciado): alternancia, presupuesto,
cancelacion cooperativa, terminacion, registro de metricas y exportacion.

No crea widgets: la vista (`chess_ai.gui.ChessApp`) solo llama a estos metodos y
se suscribe a `on_update`. Asi la logica se prueba con pytest sin Jupyter.

AL_SOLICITAR_TURNO (pseudocodigo del enunciado) -> `GameController.request_turn`:

    si hay busqueda activa o partida terminada: informar y salir    -> "busy" / "over"
    identificar agente segun el color al turno
    si es humano: esperar una jugada por la interfaz                -> "human"
    si es agente:
        capturar `env.version`                                       (version vigente)
        copiar el tablero conservando la historia                    (copy_for_agent)
        solicitar Decision al agente                                 (seed = seed_base + ply)
        comprobar plazo, legalidad y que la version siga vigente     -> "error" / "stale"
        aplicar exactamente una jugada al tablero real               (env.apply)
        registrar metricas y actualizar vista                        (MoveRecord + on_update)

Politicas (ver apuntes/bitacora_G.md):

* SEMILLA POR JUGADA: `seed = config.seed + ply_count` (ply_count antes de mover).
  La misma semilla base reproduce la partida; cada jugada recibe una semilla
  distinta y conocida.
* PLAZO: cooperativo. El controlador mide el tiempo real de `choose` y marca
  `time_exceeded` si supera `time_limit_s + tolerance_s`. La jugada SE APLICA de
  todos modos (politica uniforme para todos los agentes); el exceso queda en el
  registro. Medir despues de devolver no interrumpe un agente bloqueado: el
  aislamiento en otro proceso queda fuera del alcance minimo (seccion 6).
* JUGADA INVALIDA: si un agente devuelve `None` en un estado no terminal, una
  jugada ilegal o lanza una excepcion, el turno se registra como ERROR, el estado
  real NO cambia, el bucle automatico se detiene y no se inventa ningun resultado.
* DECISION OBSOLETA: si `env.version` cambio mientras el agente pensaba, la
  decision se descarta sin aplicarla (estado "stale").
* DESHACER: humano contra agente -> deshace hasta devolver el turno al humano (una
  pareja de turnos si el agente ya respondio); otros modos -> un ply. Prohibido con
  busqueda activa o con el bucle automatico en marcha (hay que pausar primero).
"""
from __future__ import annotations

import asyncio
import csv
import io
import time
from dataclasses import dataclass, field
from typing import Callable, Optional

import chess
import chess.pgn


# Columnas del CSV de decisiones, exactamente las del enunciado (seccion 9).
CSV_FIELDS = ["game_id", "opening_id", "seed", "white_agent", "black_agent", "ply",
              "move_uci", "nodes", "leaves", "cutoffs", "completed_depth",
              "elapsed_s", "simulations", "time_exceeded", "termination", "official_result"]

HUMAN_LABEL = "Humano"          # etiqueta reservada: en el diccionario de fabricas vale None
DEFAULT_TIME_TOLERANCE_S = 0.05  # margen sobre el presupuesto antes de marcar exceso


@dataclass
class AgentConfig:
    """Configuracion que la interfaz entrega a cada fabrica de agentes.

    `depth`: profundidad para agentes de profundidad fija (minimax, alfa-beta).
    `time_limit_s`: presupuesto por decision (<= 0: sin plazo).
    `seed`: semilla BASE; cada jugada recibe `seed + ply` (ver `seed_for_ply`).
    `simulations`: simulaciones fijas de MCTS (None: usar el plazo temporal).
    """
    depth: int = 2
    time_limit_s: float = 0.2
    seed: int = 0
    simulations: Optional[int] = None

    def seed_for_ply(self, ply: int) -> int:
        return int(self.seed) + int(ply)


AgentFactory = Callable[[AgentConfig], object]


def build_agent(factories: dict, label: str, config: AgentConfig):
    """Construye el agente `label` con su fabrica; `None` significa humano."""
    if label not in factories:
        raise KeyError(f"agente desconocido: {label!r}")
    factory = factories[label]
    return None if factory is None else factory(config)


def agent_label(agent) -> str:
    """Nombre a mostrar y registrar: `agent.name` o 'Humano' si es None."""
    if agent is None:
        return HUMAN_LABEL
    return str(getattr(agent, "name", type(agent).__name__))


@dataclass
class MoveRecord:
    """Una jugada aplicada al estado real (agente o humano).

    Las primeras columnas son exactamente CSV_FIELDS. `elapsed_s` es el tiempo
    REAL medido por el controlador alrededor de `choose` (incluye la copia);
    `agent_elapsed_s` es el que reporta el propio agente en SearchStats.
    `termination` y `official_result` son el estado de la partida justo despues
    de esta jugada; al exportar se sustituyen por el estado final (ver export_csv).
    """
    game_id: str
    opening_id: str
    seed: int
    white_agent: str
    black_agent: str
    ply: int                     # 1, 2, 3, ... (numero de la jugada en la partida)
    move_uci: str
    nodes: int = 0
    leaves: int = 0
    cutoffs: int = 0
    completed_depth: int = 0
    elapsed_s: float = 0.0
    simulations: int = 0
    time_exceeded: bool = False
    termination: str = "ongoing"
    official_result: str = "*"
    # --- extra (no van al CSV del enunciado) ---
    color: str = "white"
    agent: str = HUMAN_LABEL
    san: str = ""
    score: Optional[float] = None
    agent_elapsed_s: float = 0.0
    fen_before: str = ""

    def csv_row(self, final_termination: str | None = None,
                final_result: str | None = None) -> dict:
        row = {name: getattr(self, name) for name in CSV_FIELDS}
        if final_termination is not None:
            row["termination"] = final_termination
        if final_result is not None:
            row["official_result"] = final_result
        row["elapsed_s"] = round(float(row["elapsed_s"]), 6)
        row["time_exceeded"] = int(bool(row["time_exceeded"]))
        return row


@dataclass
class TurnReport:
    """Respuesta de cada operacion del controlador (tambien se envia a on_update).

    status: 'applied' | 'human' | 'busy' | 'over' | 'stale' | 'error' | 'undone'
            | 'reset' | 'agents' | 'rejected'
    """
    status: str
    message: str
    record: Optional[MoveRecord] = None
    decision: Optional[Decision] = None


class GameController:
    """Coordina el entorno real con los dos jugadores (agente o humano = None)."""

    def __init__(self, env: ChessEnvironment | None = None, white=None, black=None,
                 config: AgentConfig | None = None, *, game_id: str = "gui-001",
                 opening_id: str = "", tolerance_s: float = DEFAULT_TIME_TOLERANCE_S,
                 on_update: Callable[[TurnReport], None] | None = None):
        self.env = env if env is not None else ChessEnvironment()
        self.white = white
        self.black = black
        self.config = config if config is not None else AgentConfig()
        self.game_id = game_id
        self.opening_id = opening_id
        self.tolerance_s = tolerance_s
        self.on_update = on_update
        self.records: list[MoveRecord] = []
        self.errors: list[str] = []
        self.last_report: TurnReport | None = None
        self.searching = False        # True solo mientras un agente ejecuta choose()
        self.running = False          # True mientras el bucle cooperativo esta activo
        self._pause_requested = False
        self._games_started = 1

    # ------------------------------------------------------------- consultas
    def agent_for(self, color: chess.Color):
        return self.white if color == chess.WHITE else self.black

    def current_agent(self):
        return self.agent_for(self.env.board.turn)

    def is_human_turn(self) -> bool:
        return self.current_agent() is None

    def mode(self) -> str:
        humans = (self.white is None) + (self.black is None)
        return {2: "humano-humano", 1: "humano-agente", 0: "agente-agente"}[humans]

    def busy(self) -> bool:
        return self.searching

    # ---------------------------------------------------------- notificacion
    def _ctl_emit(self, report: TurnReport) -> TurnReport:
        self.last_report = report
        if report.status == "error":
            self.errors.append(report.message)
        if self.on_update is not None:
            self.on_update(report)
        return report

    def _ctl_make_record(self, move: chess.Move, color: chess.Color, fen_before: str,
                         san: str, agent, seed: int, decision: Decision | None,
                         elapsed: float, exceeded: bool) -> MoveRecord:
        stats = decision.stats if decision is not None and decision.stats is not None else SearchStats()
        return MoveRecord(
            game_id=self.game_id, opening_id=self.opening_id, seed=seed,
            white_agent=agent_label(self.white), black_agent=agent_label(self.black),
            ply=self.env.ply_count, move_uci=move.uci(),
            nodes=int(stats.nodes), leaves=int(stats.leaves), cutoffs=int(stats.cutoffs),
            completed_depth=int(stats.completed_depth), elapsed_s=float(elapsed),
            simulations=int(stats.simulations), time_exceeded=bool(exceeded),
            termination=self.env.termination(), official_result=self.env.official_result(),
            color="white" if color == chess.WHITE else "black", agent=agent_label(agent),
            san=san, score=None if decision is None else decision.score,
            agent_elapsed_s=float(stats.elapsed_s), fen_before=fen_before)

    # ------------------------------------------------------- turno de agente
    def request_turn(self) -> TurnReport:
        """AL_SOLICITAR_TURNO: aplica como maximo UNA jugada de agente."""
        if self.searching:
            return self._ctl_emit(TurnReport("busy", "Hay una busqueda activa; espere o pause."))
        if self.env.is_over():
            return self._ctl_emit(TurnReport(
                "over", f"Partida terminada ({self.env.termination()}, "
                        f"resultado {self.env.official_result()}); no se pide jugada."))
        color = self.env.board.turn
        agent = self.agent_for(color)
        if agent is None:
            side = "blancas" if color == chess.WHITE else "negras"
            return self._ctl_emit(TurnReport("human", f"Turno humano ({side}): use Mover."))

        version = self.env.version
        board = self.env.copy_for_agent()
        fen_before = board.fen()
        seed = self.config.seed_for_ply(self.env.ply_count)
        name = agent_label(agent)
        self.searching = True
        start = time.perf_counter()
        try:
            decision = agent.choose(board, time_limit_s=self.config.time_limit_s, seed=seed)
        except Exception as exc:  # el agente fallo: no se toca el estado real
            return self._ctl_emit(TurnReport(
                "error", f"{name} lanzo {type(exc).__name__}: {exc}. Estado sin cambios."))
        finally:
            self.searching = False
        elapsed = time.perf_counter() - start
        limit = self.config.time_limit_s
        exceeded = bool(limit and limit > 0 and elapsed > limit + self.tolerance_s)

        if self.env.version != version:
            return self._ctl_emit(TurnReport(
                "stale", f"Decision de {name} descartada: el estado cambio mientras pensaba.",
                decision=decision))
        move = None if decision is None else decision.move
        if isinstance(move, str):
            try:
                move = chess.Move.from_uci(move)
            except ValueError:
                move = None
        if move is None:
            return self._ctl_emit(TurnReport(
                "error", f"{name} devolvio None en un estado no terminal. Estado sin cambios.",
                decision=decision))
        if move not in self.env.board.legal_moves:
            return self._ctl_emit(TurnReport(
                "error", f"{name} devolvio la jugada ilegal {move}. Estado sin cambios.",
                decision=decision))
        san = self.env.board.san(move)
        try:
            self.env.apply(move)
        except IllegalMoveError as exc:  # defensa redundante: apply valida de nuevo
            return self._ctl_emit(TurnReport("error", str(exc), decision=decision))
        record = self._ctl_make_record(move, color, fen_before, san, agent, seed,
                                       decision, elapsed, exceeded)
        self.records.append(record)
        msg = f"{name} jugo {san} ({move.uci()}) en {elapsed:.3f} s"
        if exceeded:
            msg += f" — EXCEDIO el presupuesto de {limit:g} s"
        return self._ctl_emit(TurnReport("applied", msg, record=record, decision=decision))

    # -------------------------------------------------------- jugada humana
    def human_move(self, move) -> TurnReport:
        """Valida y aplica una jugada humana. Toda jugada pasa por env.apply."""
        if self.searching:
            return self._ctl_emit(TurnReport("busy", "Hay una busqueda activa; espere."))
        if self.env.is_over():
            return self._ctl_emit(TurnReport(
                "over", f"Partida terminada ({self.env.termination()}); no se aceptan jugadas."))
        if not self.is_human_turn():
            return self._ctl_emit(TurnReport(
                "rejected", f"Le toca al agente {agent_label(self.current_agent())}; "
                            "use 'Jugar turno del agente'."))
        if isinstance(move, str):
            try:
                move = chess.Move.from_uci(move)
            except ValueError:
                return self._ctl_emit(TurnReport("rejected", f"Texto de jugada invalido: {move!r}."))
        if move is None or move not in self.env.board.legal_moves:
            return self._ctl_emit(TurnReport(
                "rejected", f"Movimiento ilegal {move}. El tablero conserva su estado."))
        color = self.env.board.turn
        fen_before = self.env.board.fen()
        san = self.env.board.san(move)
        self.env.apply(move)
        record = self._ctl_make_record(move, color, fen_before, san, None,
                                       self.config.seed_for_ply(self.env.ply_count - 1),
                                       None, 0.0, False)
        self.records.append(record)
        return self._ctl_emit(TurnReport("applied", f"Humano jugo {san} ({move.uci()}).",
                                         record=record))

    # -------------------------------------------------------------- deshacer
    def undo_plies_needed(self) -> int:
        """Cuantos plies deshace la politica en el estado actual."""
        stack = self.env.board.move_stack
        if not stack:
            return 0
        if self.mode() != "humano-agente":
            return 1
        # Al deshacer k plies le toca mover a quien hizo la jugada stack[-k].
        # Se deshace 1; si el que queda al turno es el agente, se deshace otro.
        mover_last = not self.env.board.turn          # quien hizo la ultima jugada
        if self.agent_for(mover_last) is not None and len(stack) >= 2:
            return 2
        return 1

    def undo(self) -> TurnReport:
        if self.searching:
            return self._ctl_emit(TurnReport("busy", "No se deshace con una busqueda activa."))
        if self.running:
            return self._ctl_emit(TurnReport(
                "busy", "El bucle automatico esta en marcha: pause primero y luego deshaga."))
        n = self.undo_plies_needed()
        if n == 0:
            return self._ctl_emit(TurnReport("rejected", "No hay jugadas para deshacer."))
        done = self.env.undo(n)
        self.records = [r for r in self.records if r.ply <= self.env.ply_count]
        return self._ctl_emit(TurnReport(
            "undone", f"Deshecho(s) {done} ply ({self.mode()}). Turno de "
                      f"{'blancas' if self.env.board.turn else 'negras'}."))

    # -------------------------------------------- nueva partida / agentes
    def new_game(self, fen: str | None = None, *, game_id: str | None = None,
                 opening_id: str | None = None) -> TurnReport:
        if self.searching:
            return self._ctl_emit(TurnReport("busy", "Hay una busqueda activa; espere."))
        if fen is not None:
            try:
                board = chess.Board(fen)
            except ValueError as exc:
                return self._ctl_emit(TurnReport("rejected", f"FEN invalido: {exc}."))
            if not board.is_valid():
                return self._ctl_emit(TurnReport("rejected", f"Posicion invalida: {board.status()!r}."))
        self._pause_requested = True   # un bucle activo termina en su proxima cesion
        self.env.reset(fen if fen is not None else chess.STARTING_FEN)
        self._games_started += 1
        self.game_id = game_id or f"gui-{self._games_started:03d}"
        if opening_id is not None:
            self.opening_id = opening_id
        self.records = []
        self.errors = []
        return self._ctl_emit(TurnReport("reset", f"Nueva partida {self.game_id}."))

    def set_agents(self, white=None, black=None, config: AgentConfig | None = None) -> TurnReport:
        """Cambia jugadores (None = humano). Permitido entre turnos, no durante choose()."""
        if self.searching:
            return self._ctl_emit(TurnReport("busy", "No se cambian agentes durante una busqueda."))
        self.white, self.black = white, black
        if config is not None:
            self.config = config
        return self._ctl_emit(TurnReport(
            "agents", f"Blancas: {agent_label(white)} · Negras: {agent_label(black)} ({self.mode()})."))

    # ------------------------------------------------- bucle cooperativo
    def pause(self) -> None:
        """Pide detener el bucle; se atiende en la siguiente cesion (entre turnos)."""
        self._pause_requested = True

    async def run_async(self, *, delay_s: float = 0.0, max_turns: int | None = None) -> str:
        """Juega turnos de agente seguidos cediendo el control entre turnos.

        Cada turno es SINCRONO (bloquea mientras el agente busca, ~presupuesto);
        entre turnos `await asyncio.sleep(delay_s)` devuelve el control al bucle de
        eventos del kernel, que asi procesa el clic de Pausar. Devuelve el motivo
        de parada: 'paused' | 'terminal' | 'truncated' | 'human' | 'error' |
        'stale' | 'busy' | 'max_turns'.
        """
        if self.running or self.searching:
            return "busy"
        self.running = True
        self._pause_requested = False
        turns = 0
        try:
            while True:
                if self._pause_requested:
                    return "paused"
                if self.env.is_over():
                    return self.env.status()
                if self.is_human_turn():
                    return "human"
                if max_turns is not None and turns >= max_turns:
                    return "max_turns"
                report = self.request_turn()
                if report.status != "applied":
                    return report.status
                turns += 1
                await asyncio.sleep(delay_s)   # cesion: aqui se atienden los clics
        finally:
            self.running = False

    def play_until_stop(self, max_turns: int | None = None) -> str:
        """Version sincronica (sin bucle de eventos): solo para pruebas y scripts."""
        turns = 0
        while not self.env.is_over() and not self.is_human_turn():
            if max_turns is not None and turns >= max_turns:
                return "max_turns"
            report = self.request_turn()
            if report.status != "applied":
                return report.status
            turns += 1
        return self.env.status() if self.env.is_over() else "human"

    # ---------------------------------------------------------- exportacion
    def pgn_headers(self) -> dict:
        cfg = self.config
        return {
            "Event": "Taller 3 IA UTP - interfaz grafica",
            "Site": "Jupyter",
            "Date": time.strftime("%Y.%m.%d"),
            "Round": self.game_id,
            "White": agent_label(self.white),
            "Black": agent_label(self.black),
            "GameId": self.game_id,
            "Opening": self.opening_id or "-",
            "SeedBase": cfg.seed,
            "SeedPolicy": "seed_base + ply",
            "TimeBudgetS": cfg.time_limit_s,
            "TimeToleranceS": self.tolerance_s,
            "Depth": cfg.depth,
            "Simulations": cfg.simulations if cfg.simulations is not None else "-",
            "Mode": self.mode(),
        }

    def export_pgn(self, path: str | None = None) -> str:
        text = str(self.env.to_pgn(self.pgn_headers())) + "\n"
        if path:
            with open(path, "w", encoding="utf-8", newline="\n") as fh:
                fh.write(text)
        return text

    def export_csv(self, path: str | None = None, *, final: bool = True) -> str:
        """CSV con EXACTAMENTE las columnas CSV_FIELDS.

        Con `final=True` las columnas de partida `termination` y `official_result`
        llevan el estado actual de la partida en todas las filas (como game_id o
        white_agent); con `final=False`, el estado justo despues de cada jugada.
        """
        term = self.env.termination() if final else None
        res = self.env.official_result() if final else None
        buf = io.StringIO()
        writer = csv.DictWriter(buf, fieldnames=CSV_FIELDS, lineterminator="\n")
        writer.writeheader()
        for rec in self.records:
            writer.writerow(rec.csv_row(term, res))
        text = buf.getvalue()
        if path:
            with open(path, "w", encoding="utf-8", newline="") as fh:
                fh.write(text)
        return text


## Integración del controlador: `GameController`

`GameController` (`chess_ai/controller.py`) coordina el estado real (`ChessEnvironment`) con los
dos jugadores. Cada jugador es un agente que cumple el contrato `Agent` o `None` (humano). No
crea widgets: la interfaz se suscribe con `on_update` y recibe un `TurnReport` después de cada
operación. Por eso toda la lógica de turnos se prueba con `pytest` sin Jupyter.

### AL_SOLICITAR_TURNO en código

`request_turn()` sigue el pseudocódigo de la sección 6 paso a paso:

| Pseudocódigo | Implementación | Resultado |
|---|---|---|
| si hay búsqueda activa o partida terminada: informar y salir | `searching` o `env.is_over()` | `busy` / `over` (nunca se pide jugada a un agente en un estado terminal) |
| identificar agente según el color al turno | `agent_for(board.turn)` | |
| si es humano: esperar | agente `None` | `human` |
| capturar identificador de versión | `version = env.version` | |
| copiar el tablero conservando la historia | `env.copy_for_agent()` (`copy(stack=True)`) | |
| solicitar Decision | `agent.choose(copia, time_limit_s=plazo, seed=semilla_base + ply)` | |
| comprobar plazo | tiempo real medido > plazo + 0,05 s → `time_exceeded` | se registra, la jugada se aplica |
| comprobar que la versión siga vigente | `env.version != version` | `stale`: decisión descartada |
| comprobar legalidad | `None`, jugada ilegal o excepción | `error`: estado sin cambios, sin resultado inventado |
| aplicar exactamente una jugada | `env.apply(move)` | `applied` |
| registrar métricas y actualizar vista | `MoveRecord` + `on_update` | |

Cada llamada aplica **como máximo una** jugada. Las jugadas humanas entran por `human_move()`,
que rechaza jugadas ilegales, jugadas en el turno de un agente y jugadas en partidas terminadas;
las válidas pasan también por `env.apply`.

### Por qué se compara `version`

El agente trabaja sobre una copia. Entre el momento en que se copia el tablero y el momento en
que vuelve la decisión, el estado real podría cambiar: el usuario deshace, empieza una partida
nueva o hay otra jugada. Si el controlador aplicara la decisión sin comprobarlo, aplicaría una
jugada pensada para una posición que ya no existe: podría ser ilegal o, peor, legal pero sin
sentido (y además el turno avanzaría dos veces).

`ChessEnvironment.version` aumenta en cada `apply`, `undo` y `reset`. Compararla es más sólido
que comparar el FEN: el FEN no distingue «deshice y volví a jugar lo mismo» y no contiene la
historia (repeticiones, regla de 50) con la que el agente decidió. Hoy cada turno es síncrono y
de un solo hilo, así que el caso solo aparece si algo modifica el tablero durante `choose`; la
prueba `test_stale_decision_is_discarded` lo provoca con un agente que mueve el tablero real
mientras «piensa», y comprueba que su jugada se descarta. La comprobación deja el controlador
listo para ejecutar agentes en un hilo o en otro proceso sin cambiar la lógica.

### Políticas

- **Semilla por jugada:** `semilla = semilla_base + ply` (ply antes de mover). La misma semilla
  base reproduce la partida; cada jugada tiene una semilla conocida (columna `seed` del CSV).
- **Plazo:** cooperativo. Se mide el tiempo real de `choose`; si supera el plazo más una
  tolerancia de 0,05 s se marca `time_exceeded` y la jugada se aplica igual, para todos los
  agentes por igual. Plazo 0 = sin plazo. Medir después no interrumpe a un agente bloqueado;
  el aislamiento en proceso queda fuera del alcance mínimo.
- **Respuesta inválida:** `None` en estado no terminal, jugada ilegal o excepción → error
  registrado en `controller.errors`, estado sin cambios y bucle detenido.
- **Deshacer:** humano contra agente vuelve al turno del humano (pareja de turnos si el agente
  ya respondió); otros modos, un ply; prohibido con búsqueda activa o bucle en marcha.
- **Truncada:** el límite de plies lo aplica el entorno (`max_plies`): el estado pasa a
  `truncated`, el controlador ya no pide jugadas y el PGN/CSV llevan resultado `*`, no tablas.

### Bucle cooperativo

`run_async()` juega turnos seguidos y entre uno y otro ejecuta `await asyncio.sleep(pausa)`,
que devuelve el control al bucle de eventos del kernel para que atienda el botón **Pausar**.
Se detiene por pausa, partida terminada, truncamiento, turno humano o error, y devuelve el
motivo. Un turno sigue siendo síncrono: la cancelación es entre turnos, no dentro de la búsqueda.
`play_until_stop()` es la versión síncrona para pruebas y scripts sin bucle de eventos.

### Exportación

- `export_pgn(ruta)`: `env.to_pgn` con encabezados de agentes, semilla base y política,
  plazo, tolerancia, profundidad, simulaciones y modo. Una partida truncada lleva `Result "*"`.
- `export_csv(ruta)`: exactamente las columnas `CSV_FIELDS` del enunciado; una fila por jugada
  (agente o humano). `termination` y `official_result` llevan el estado de la partida al exportar.

In [ ]:
# ===== chess_ai/gui.py =====
"""Interfaz grafica (ipywidgets) del taller: `ChessApp`.

Amplia la base suministrada `ChessView` (celda 5 del enunciado) SIN heredar de
ella: el cuaderno del profesor ya define `ChessView` y el paquete debe funcionar
tambien fuera del cuaderno. Se conserva su idea (listas origen/destino/promocion,
boton Mover, destinos posibles, SVG con ultimo movimiento y jaque) y se agregan
las ampliaciones obligatorias: selector de agente por color (incluido «Humano»),
«Jugar turno del agente», «Iniciar/Pausar», «Nueva partida», «Deshacer»,
profundidad, plazo, semilla y simulaciones, estado de la partida, historial SAN,
registro visible de cada decision y exportacion PGN/CSV.

Toda la logica de turnos vive en `chess_ai.controller.GameController`; esta clase
solo traduce clics en llamadas al controlador y redibuja en `on_update`.
No se crea ningun widget al importar el modulo, solo al instanciar `ChessApp`.

Uso en el cuaderno:

    app = ChessApp({"Humano": None, "Aleatorio": lambda cfg: RandomAgent(),
                    "Alfa-beta": lambda cfg: AlphaBetaAgent(depth=cfg.depth)},
                   white="Humano", black="Alfa-beta")
    app.display()
"""
from __future__ import annotations

import asyncio
import html
import os

import chess
import chess.svg
import ipywidgets as widgets


_GUI_STATUS_ES = {"ongoing": "en curso", "terminal": "terminal", "truncated": "truncada"}
_GUI_REPORT_COLORS = {"applied": "#1b5e20", "error": "#b71c1c", "rejected": "#b71c1c",
                      "stale": "#e65100", "busy": "#e65100", "over": "#4a148c"}
_GUI_PROMOTIONS = {"q": chess.QUEEN, "r": chess.ROOK, "b": chess.BISHOP, "n": chess.KNIGHT}


class ChessApp:
    """Aplicacion de ajedrez con los tres modos: humano-humano, humano-agente y
    agente-agente.

    `factories`: {etiqueta: fabrica(AgentConfig) -> Agent}; la etiqueta «Humano»
    se agrega siempre con valor None. Las fabricas se vuelven a llamar cuando
    cambia el selector o la configuracion (profundidad, plazo, semilla, sims).
    """

    def __init__(self, factories: dict | None = None, *, white: str = HUMAN_LABEL,
                 black: str = HUMAN_LABEL, config: AgentConfig | None = None,
                 fen: str | None = None, max_plies: int | None = None,
                 export_dir: str = os.path.join("resultados", "interfaz"),
                 board_size: int = 440, auto_reply: bool = True, delay_s: float = 0.3):
        self.factories = {HUMAN_LABEL: None}
        for label, factory in (factories or {}).items():
            if label != HUMAN_LABEL:
                self.factories[label] = factory
        for label in (white, black):
            if label not in self.factories:
                raise KeyError(f"agente desconocido: {label!r}")
        cfg = config if config is not None else AgentConfig()
        self.env = ChessEnvironment(fen, max_plies=max_plies)
        self.controller = GameController(self.env, None, None, cfg,
                                         on_update=self._gui_on_update)
        self.export_dir = export_dir
        self.board_size = board_size
        self._task = None
        self._loop_active = False   # True desde Iniciar hasta que la tarea termina
        self._agents_key = None
        self._message = ""
        self._message_color = "#333333"
        self._gui_build(white, black, cfg, max_plies, auto_reply, delay_s)
        self._gui_sync_agents()
        self._message = "Listo. Elija agentes o juegue con Mover."
        self.render()

    # ------------------------------------------------------------ widgets
    def _gui_build(self, white, black, cfg, max_plies, auto_reply, delay_s):
        W = widgets
        wide = {"description_width": "110px"}
        lay = W.Layout(width="260px")
        labels = list(self.factories)
        self.white_select = W.Dropdown(options=labels, value=white, description="Blancas",
                                       style=wide, layout=lay)
        self.black_select = W.Dropdown(options=labels, value=black, description="Negras",
                                       style=wide, layout=lay)
        self.depth = W.BoundedIntText(value=cfg.depth, min=1, max=10, description="Profundidad",
                                      style=wide, layout=lay)
        self.time_limit = W.BoundedFloatText(value=cfg.time_limit_s, min=0.0, max=600.0, step=0.1,
                                             description="Plazo (s)", style=wide, layout=lay)
        self.seed = W.IntText(value=cfg.seed, description="Semilla base", style=wide, layout=lay)
        self.simulations = W.BoundedIntText(value=cfg.simulations or 0, min=0, max=10_000_000,
                                            description="Sims MCTS (0=plazo)", style=wide, layout=lay)
        self.max_plies = W.BoundedIntText(value=max_plies or 0, min=0, max=10_000,
                                          description="Máx. plies (0=∞)", style=wide, layout=lay)
        self.delay = W.BoundedFloatText(value=delay_s, min=0.0, max=10.0, step=0.1,
                                        description="Pausa bucle (s)", style=wide, layout=lay)
        self.auto_reply = W.Checkbox(value=auto_reply, description="El agente responde solo",
                                     indent=False)
        self.show_arrow = W.Checkbox(value=True, description="Flecha de la decisión", indent=False)
        self.flip = W.Checkbox(value=False, description="Ver desde negras", indent=False)

        self.origin = W.Dropdown(options=chess.SQUARE_NAMES, value="e2", description="Origen",
                                 layout=W.Layout(width="150px"))
        self.target = W.Dropdown(options=chess.SQUARE_NAMES, value="e4", description="Destino",
                                 layout=W.Layout(width="150px"))
        self.promotion = W.Dropdown(options=[("Dama", "q"), ("Torre", "r"), ("Alfil", "b"),
                                             ("Caballo", "n")], description="Promoción",
                                    layout=W.Layout(width="200px"))
        self.move_button = W.Button(description="Mover", button_style="primary")
        self.destinations = W.HTML()

        self.agent_button = W.Button(description="Jugar turno del agente", button_style="info",
                                     layout=W.Layout(width="190px"))
        self.run_button = W.Button(description="Iniciar", button_style="success")
        self.new_button = W.Button(description="Nueva partida")
        self.undo_button = W.Button(description="Deshacer")
        self.refresh_button = W.Button(description="Actualizar vista")
        self.pgn_button = W.Button(description="Exportar PGN")
        self.csv_button = W.Button(description="Exportar CSV")
        self.fen_text = W.Text(value="", placeholder="vacío = posición inicial",
                               description="FEN", layout=W.Layout(width="440px"))

        self.board_html = W.HTML(layout=W.Layout(width=f"{self.board_size + 10}px"))
        self.status_html = W.HTML()
        self.message_html = W.HTML()
        self.history_html = W.HTML()
        self.log_html = W.HTML()
        self.export_area = W.Textarea(value="", layout=W.Layout(width="100%", height="120px"),
                                      placeholder="Aquí aparece el PGN o CSV exportado")

        self.move_button.on_click(self.on_move)
        self.agent_button.on_click(self.on_agent_turn)
        self.run_button.on_click(self.on_toggle_run)
        self.new_button.on_click(self.on_new_game)
        self.undo_button.on_click(self.on_undo)
        self.refresh_button.on_click(lambda _: self.render())
        self.pgn_button.on_click(self.on_export_pgn)
        self.csv_button.on_click(self.on_export_csv)
        self.origin.observe(self._gui_on_origin, names="value")
        self.white_select.observe(self._gui_on_selector, names="value")
        self.black_select.observe(self._gui_on_selector, names="value")
        self.max_plies.observe(self._gui_on_max_plies, names="value")
        for w in (self.show_arrow, self.flip):
            w.observe(lambda _: self.render(), names="value")

        players = W.VBox([W.HTML("<b>Jugadores y presupuesto</b>"), self.white_select,
                          self.black_select, self.depth, self.time_limit, self.seed,
                          self.simulations, self.max_plies, self.delay, self.auto_reply,
                          self.show_arrow, self.flip])
        human = W.VBox([W.HTML("<b>Jugada humana</b>"),
                        W.HBox([self.origin, self.target]),
                        W.HBox([self.promotion, self.move_button]), self.destinations])
        actions = W.VBox([W.HTML("<b>Partida</b>"),
                          W.HBox([self.agent_button, self.run_button]),
                          W.HBox([self.new_button, self.undo_button, self.refresh_button]),
                          self.fen_text,
                          W.HBox([self.pgn_button, self.csv_button])])
        right = W.VBox([self.status_html, self.message_html, human, actions, players])
        self.ui = W.VBox([W.HBox([self.board_html, right]),
                          W.HTML("<b>Historial SAN</b>"), self.history_html,
                          W.HTML("<b>Registro de decisiones</b> (más reciente arriba)"),
                          self.log_html, self.export_area])

    def display(self):
        from IPython.display import display as _ipy_display
        _ipy_display(self.ui)

    def _ipython_display_(self):
        self.display()

    # ---------------------------------------------------- configuracion
    def current_config(self) -> AgentConfig:
        sims = int(self.simulations.value)
        return AgentConfig(depth=int(self.depth.value), time_limit_s=float(self.time_limit.value),
                           seed=int(self.seed.value), simulations=sims if sims > 0 else None)

    def _gui_sync_agents(self) -> bool:
        """(Re)construye los agentes si cambiaron selectores o configuracion."""
        cfg = self.current_config()
        key = (self.white_select.value, self.black_select.value, cfg.depth, cfg.time_limit_s,
               cfg.seed, cfg.simulations)
        if key == self._agents_key:
            return True
        if self.controller.searching:
            return False
        try:
            white = build_agent(self.factories, self.white_select.value, cfg)
            black = build_agent(self.factories, self.black_select.value, cfg)
        except Exception as exc:
            self._gui_set_message(f"No se pudo crear el agente: {type(exc).__name__}: {exc}",
                                  "#b71c1c")
            return False
        self._agents_key = key
        self.controller.set_agents(white, black, cfg)
        return True

    def _gui_on_selector(self, _change):
        self._gui_sync_agents()

    def _gui_on_max_plies(self, change):
        self.env.max_plies = int(change["new"]) or None
        self.render()

    def _gui_on_origin(self, _change):
        dests = self._gui_destinations()
        if dests and chess.parse_square(self.target.value) not in dests:
            self.target.value = chess.square_name(dests[0])
        self.render()

    # ---------------------------------------------------------- acciones
    def on_move(self, _=None) -> TurnReport:
        """Jugada humana con las listas (como la base). Siempre pasa por env.apply."""
        board = self.env.board
        src = chess.parse_square(self.origin.value)
        dst = chess.parse_square(self.target.value)
        piece = board.piece_at(src)
        promo = None
        if piece and piece.piece_type == chess.PAWN and chess.square_rank(dst) in (0, 7):
            promo = _GUI_PROMOTIONS[self.promotion.value]
        self._gui_sync_agents()
        report = self.controller.human_move(chess.Move(src, dst, promotion=promo))
        if (report.status == "applied" and self.auto_reply.value and not self.env.is_over()
                and not self.controller.is_human_turn()):
            self._gui_start_loop()    # respuesta automatica: juega hasta el turno humano
        return report

    def on_agent_turn(self, _=None) -> TurnReport | None:
        if self.controller.running:
            self._gui_set_message("El bucle automático está en marcha; use Pausar.", "#e65100")
            return None
        if not self._gui_sync_agents():
            return None
        return self.controller.request_turn()

    def on_toggle_run(self, _=None):
        if self.controller.running or self._loop_active:
            self.controller.pause()
            self._gui_set_message("Pausa solicitada: se atiende al terminar el turno en curso.",
                                  "#e65100")
            return
        self._gui_start_loop(require_loop=True)

    def on_new_game(self, _=None) -> TurnReport:
        fen = self.fen_text.value.strip() or None
        self.env.max_plies = int(self.max_plies.value) or None
        self._gui_sync_agents()
        return self.controller.new_game(fen)

    def on_undo(self, _=None) -> TurnReport:
        return self.controller.undo()

    def on_export_pgn(self, _=None) -> str:
        path = self._gui_export_path("pgn")
        text = self.controller.export_pgn(path)
        self.export_area.value = text
        self._gui_set_message(f"PGN guardado en {path}", "#1b5e20")
        return path

    def on_export_csv(self, _=None) -> str:
        path = self._gui_export_path("csv")
        text = self.controller.export_csv(path)
        self.export_area.value = text
        self._gui_set_message(f"CSV ({len(self.controller.records)} filas) guardado en {path}",
                              "#1b5e20")
        return path

    def _gui_export_path(self, ext: str) -> str:
        os.makedirs(self.export_dir, exist_ok=True)
        return os.path.join(self.export_dir, f"{self.controller.game_id}.{ext}")

    # ------------------------------------------------ bucle cooperativo
    def _gui_start_loop(self, require_loop: bool = False):
        if not self._gui_sync_agents():
            return
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            loop = None
        if loop is None:
            # Fuera de un kernel no hay bucle de eventos al que ceder.
            if require_loop:
                self._gui_set_message("No hay bucle de eventos activo (fuera de Jupyter): "
                                      "use 'Jugar turno del agente'.", "#e65100")
            else:
                self.controller.play_until_stop()
            return
        self._loop_active = True
        self.run_button.description = "Pausar"
        self.run_button.button_style = "warning"
        self._task = loop.create_task(self._gui_run_task(float(self.delay.value)))

    async def _gui_run_task(self, delay_s: float) -> str:
        reason = "error"
        try:
            reason = await self.controller.run_async(delay_s=delay_s)
        except Exception as exc:   # en una tarea los errores serian silenciosos
            self._message = f"Error en el bucle: {type(exc).__name__}: {exc}"
            self._message_color = "#b71c1c"
        else:
            reasons = {"paused": "pausado", "terminal": "partida terminada",
                       "truncated": "límite de plies (truncada)", "human": "turno humano",
                       "error": "error del agente", "stale": "decisión obsoleta",
                       "busy": "ya había un bucle activo"}
            self._message += f" · Bucle detenido: {reasons.get(reason, reason)}."
        finally:
            # OJO: aqui la tarea aun no esta 'done()'; por eso se usa una bandera.
            self._loop_active = False
            self.render()
        return reason

    # ------------------------------------------------------- vista
    def _gui_on_update(self, report: TurnReport):
        self._message = report.message
        self._message_color = _GUI_REPORT_COLORS.get(report.status, "#333333")
        self.render()

    def _gui_set_message(self, text: str, color: str = "#333333"):
        self._message, self._message_color = text, color
        self.render()

    def _gui_destinations(self) -> list[int]:
        src = chess.parse_square(self.origin.value)
        seen, out = set(), []
        for m in self.env.board.legal_moves:
            if m.from_square == src and m.to_square not in seen:
                seen.add(m.to_square)
                out.append(m.to_square)
        return sorted(out)

    def render(self):
        ctl, env, board = self.controller, self.env, self.env.board
        running = ctl.running or self._loop_active
        self.run_button.description = "Pausar" if running else "Iniciar"
        self.run_button.button_style = "warning" if running else "success"

        # --- tablero
        src = chess.parse_square(self.origin.value)
        dests = self._gui_destinations()
        fill = {sq: "#7fb3d5aa" for sq in dests}
        if dests:
            fill[src] = "#f4d03faa"
        arrows = []
        last = board.peek() if board.move_stack else None
        last_rec = ctl.records[-1] if ctl.records else None
        if (self.show_arrow.value and last is not None and last_rec is not None
                and last_rec.agent != HUMAN_LABEL and last_rec.move_uci == last.uci()):
            arrows.append(chess.svg.Arrow(last.from_square, last.to_square, color="#1565c0cc"))
        check = board.king(board.turn) if board.is_check() else None
        self.board_html.value = chess.svg.board(
            board, size=self.board_size, lastmove=last, check=check, arrows=arrows, fill=fill,
            orientation=chess.BLACK if self.flip.value else chess.WHITE)

        # --- destinos posibles
        if dests:
            names = ", ".join(chess.square_name(s) for s in dests)
            self.destinations.value = f"Destinos de <b>{self.origin.value}</b>: {names}"
        else:
            self.destinations.value = (f"Sin jugadas legales desde <b>{self.origin.value}</b> "
                                       "(casilla vacía, pieza del rival o bloqueada).")

        # --- estado
        turn = "blancas" if board.turn == chess.WHITE else "negras"
        status = env.status()
        limit = f"/{env.max_plies}" if env.max_plies else ""
        last_txt = "—"
        if last_rec is not None:
            last_txt = f"{last_rec.san} ({last_rec.move_uci}) por {html.escape(last_rec.agent)}"
        check_txt = " · <b style='color:#b71c1c'>¡Jaque!</b>" if board.is_check() else ""
        self.status_html.value = (
            f"<div style='line-height:1.5'>"
            f"<b>Modo:</b> {ctl.mode()} · <b>Partida:</b> {html.escape(ctl.game_id)}<br>"
            f"<b>Turno:</b> {turn} ({html.escape(agent_label(ctl.current_agent()))}){check_txt}<br>"
            f"<b>Último movimiento:</b> {last_txt}<br>"
            f"<b>Estado:</b> {_GUI_STATUS_ES.get(status, status)} · "
            f"<b>Resultado:</b> {env.official_result()} · <b>Terminación:</b> {env.termination()}"
            f" · <b>Plies:</b> {env.ply_count}{limit}<br>"
            f"<b>Búsqueda activa:</b> {'sí' if ctl.searching else 'no'} · "
            f"<b>Bucle:</b> {'en marcha' if running else 'detenido'}</div>")
        self.message_html.value = (f"<div style='color:{self._message_color}'>"
                                   f"{html.escape(self._message)}</div>")

        # --- historial SAN
        san = chess.Board(env.start_fen).variation_san(board.move_stack) if board.move_stack else ""
        self.history_html.value = f"<code>{html.escape(san) or 'Sin jugadas'}</code>"

        # --- registro de decisiones
        head = ("<tr><th>Ply</th><th>Color</th><th>Agente</th><th>Jugada</th><th>Valor</th>"
                "<th>Nodos</th><th>Hojas</th><th>Podas</th><th>Prof.</th><th>Tiempo (s)</th>"
                "<th>Sims</th><th>Semilla</th><th>Exceso</th></tr>")
        rows = []
        for r in reversed(ctl.records[-20:]):
            score = "—" if r.score is None else f"{r.score:.1f}"
            exceeded = "<b style='color:#b71c1c'>sí</b>" if r.time_exceeded else "no"
            rows.append(
                f"<tr><td>{r.ply}</td><td>{'B' if r.color == 'white' else 'N'}</td>"
                f"<td>{html.escape(r.agent)}</td><td>{r.san} ({r.move_uci})</td><td>{score}</td>"
                f"<td>{r.nodes}</td><td>{r.leaves}</td><td>{r.cutoffs}</td>"
                f"<td>{r.completed_depth}</td><td>{r.elapsed_s:.3f}</td><td>{r.simulations}</td>"
                f"<td>{r.seed}</td><td>{exceeded}</td></tr>")
        style = ("<style>.chessapp-log td,.chessapp-log th{padding:2px 6px;"
                 "border-bottom:1px solid #ddd;text-align:right}</style>")
        self.log_html.value = (style + "<table class='chessapp-log'>" + head + "".join(rows)
                               + "</table>") if rows else "<i>Sin decisiones todavía.</i>"


### La aplicación: seis agentes y humano en una sola interfaz

La celda siguiente crea `ChessApp(AGENT_FACTORIES)` con los seis agentes del registro y «Humano» en los dos selectores, y la muestra. Así se abren los tres modos: humano contra humano, humano contra agente y agente contra agente (el uso está explicado en la sección 3, «Interfaz gráfica: `ChessApp`»).

Al ejecutar el cuaderno completo de forma automática no hay nadie que haga clic, así que la misma celda también **juega dos turnos por programa** (con los mismos métodos que llaman los botones). Con eso queda en la salida un tablero que ya tiene jugadas y su registro de decisiones. Para jugar de verdad hay que usar los controles en un Jupyter abierto con el kernel del taller: los widgets solo responden con un kernel en marcha.

In [ ]:
# ===== Aplicacion grafica: selector con los seis agentes y Humano para cada color =====
# Cambie los selectores Blancas/Negras para elegir el modo (humano-humano, humano-agente o
# agente-agente). "Nueva partida" reinicia; "Iniciar/Pausar" juega seguido entre agentes.
app = ChessApp(AGENT_FACTORIES, white=HUMAN_LABEL, black="Alfa-beta mejorado",
               config=AgentConfig(depth=2, time_limit_s=0.2, seed=0, simulations=None),
               max_plies=160, auto_reply=False)
print("Agentes en el selector:", ", ".join(app.white_select.options))

# Dos turnos jugados por programa con los mismos metodos que llaman los botones, para que la
# salida guardada muestre un tablero con jugadas: Mover (humano e2-e4) y "Jugar turno del agente".
app.origin.value, app.target.value = "e2", "e4"
_r_humano = app.on_move()
_r_agente = app.on_agent_turn()
_rec = app.controller.records[-1]
print(f"Humano: {_r_humano.status}; agente: {_r_agente.status}. Historial SAN: "
      f"{' '.join(app.env.san_history())}")
print(f"Decision de {_rec.black_agent}: {_rec.move_uci}, nodos={_rec.nodes}, podas={_rec.cutoffs}, "
      f"profundidad completada={_rec.completed_depth}, tiempo={_rec.elapsed_s:.3f} s, "
      f"excedio plazo={_rec.time_exceeded}")
assert app.controller.mode() == "humano-agente" and len(app.env.board.move_stack) == 2
app.display()


## 7. Todo lo visto: actividades de integración
Estas actividades son obligatorias, además de los seis agentes. No se exige forzar BFS, UCS o A* como jugadores completos.

### A. Búsqueda no informada en un subproblema cooperativo
Construyan un microproblema con una torre en un tablero vacío, sin reyes ni adversario: origen a1, destino h8, acciones de torre y costo unitario por movimiento. Resuelvan con BFS y DFS, y con UCS asignando a cada movimiento el número de casillas recorridas. Definan estados, acciones y costos; distingan longitud de ruta y costo.

Analicen por qué encontrar una secuencia a un objetivo bajo control de todos los movimientos no garantiza mate contra un rival. La actividad usa reglas simplificadas explícitas, no posiciones legales completas de ajedrez.

### B. Búsqueda informada y ordenación
Resuelvan el microproblema anterior con A* y voraz. Para costo unitario pueden usar $h=0$ en destino, $h=1$ si comparte fila o columna y $h=2$ en otro caso. Para costo por distancia estudien la distancia Manhattan. Justifiquen admisibilidad para A* respecto al costo adoptado y comparen expansiones. Después expliquen la analogía y las diferencias con ordenar jugadas para alfa-beta: ordenar no altera por sí mismo el conjunto de respuestas rivales.

### C. Búsqueda local para ajustar la evaluación
Definan un vector de pesos y una función objetivo sobre posiciones de entrenamiento. Realicen una búsqueda local pequeña de pesos con presupuesto explícito. Separen entrenamiento y prueba; no elijan pesos usando resultados de prueba. Reporten pesos iniciales/finales y una ablación. Si esta es su mejora adicional, la implementación cuenta para ambos apartados.

### D. Incertidumbre y garantías
1. Representen un problema táctico de mate forzado como OR para el atacante y AND para el defensor. Expliquen por qué deben cubrir todas las defensas.
2. En un árbol pequeño con azar, calculen expectiminimax y contrasten con minimax. Definan probabilidades y escala de utilidad.
3. Describan cómo cambiarían percepción y estado interno para Kriegspiel. Den un ejemplo de información que su agente de ajedrez actual usa y que allí no podría consultar.

**No cambien las reglas del ajedrez clásico para introducir dados.** El ejemplo estocástico se realiza por separado.


In [ ]:
# INTEGRACIÓN: microproblema de torre, búsquedas y ajuste local de pesos.
# Agregue tantas celdas como necesite e incluya gráficas de las comparaciones.


In [ ]:
# ===== chess_ai/rook_problem.py =====
"""Seccion 7, apartados A y B: microproblema de la torre (AIMA caps. 3 y 3.5).

REGLAS SIMPLIFICADAS EXPLICITAS (no son posiciones legales de ajedrez):
tablero 8x8, una sola torre blanca, sin reyes, sin rival y sin turnos. Opcionalmente
hay casillas bloqueadas (obstaculos fijos que la torre no puede ocupar ni saltar).
No se usa python-chess: el movimiento se programa desde cero.

Formulacion AIMA
----------------
* Estado: la casilla de la torre, tupla de enteros (columna, fila) en 0..7
  (a1 = (0, 0), h8 = (7, 7)). Inmutable y hashable, como (fila, columna) en el
  Taller 1 (D-08 de aquel taller).
* Acciones: ("N"|"E"|"S"|"O", k): avanzar k >= 1 casillas en una direccion, sin
  salir del tablero ni atravesar u ocupar un obstaculo. ORDEN FIJO: direcciones
  N, E, S, O (el mismo del Taller 1) y, dentro de cada direccion, k creciente
  (o decreciente si `long_moves_first=True`). El orden NO cambia el costo optimo
  de UCS/A*, pero SI la ruta que devuelve DFS (y los desempates).
* Modelo de transicion: RESULT(s, (d, k)) = s + k * delta(d).
* Prueba objetivo: la torre esta en la casilla destino (h8 por omision).
* Costo de paso, dos modelos:
    - "unit":     c(s, a, s') = 1           (numero de movimientos)
    - "distance": c(s, a, s') = k           (casillas recorridas)
  LONGITUD de una ruta = numero de movimientos; COSTO = suma de c segun el modelo.
  Con costo unitario ambas coinciden; con costo por distancia no.

Heuristicas (seccion 7.B)
-------------------------
* `h_rook_lines` (la de la consigna): 0 en el destino, 1 si comparte fila o
  columna, 2 en otro caso. Admisible y consistente para costo UNITARIO (cota
  inferior del numero de movimientos de torre, aun con obstaculos).
* `h_manhattan`: |dc| + |df|. Admisible y consistente para costo por DISTANCIA
  (una torre avanza a lo sumo una unidad de Manhattan por casilla recorrida).
  Con costo UNITARIO NO es admisible: h(a1) = 14 y el costo real es 2.

Metricas (mismas convenciones que el Taller 1, D-19/D-20): prueba de objetivo al
EXTRAER; un nodo es "expandido" si se generan sus sucesores (el objetivo extraido
no se expande); "generado" = sucesor construido (la raiz no cuenta, los repetidos
descartados si); frontera maxima incluye entradas obsoletas de la cola de prioridad.
"""
from __future__ import annotations

import heapq
from collections import deque
from dataclasses import dataclass, field

ROOK_BOARD_SIZE = 8
ROOK_DIRECTIONS = {"N": (0, 1), "E": (1, 0), "S": (0, -1), "O": (-1, 0)}
ROOK_DEFAULT_ORDER = ("N", "E", "S", "O")
ROOK_COST_MODELS = ("unit", "distance")
_ROOK_FILES = "abcdefgh"


def rook_square(name: str) -> tuple[int, int]:
    """'a1' -> (0, 0); 'h8' -> (7, 7)."""
    name = name.strip().lower()
    col = _ROOK_FILES.index(name[0])
    row = int(name[1:]) - 1
    if not (0 <= col < ROOK_BOARD_SIZE and 0 <= row < ROOK_BOARD_SIZE):
        raise ValueError(f"casilla fuera del tablero: {name}")
    return (col, row)


def rook_square_name(sq: tuple[int, int]) -> str:
    """(0, 0) -> 'a1'."""
    return _ROOK_FILES[sq[0]] + str(sq[1] + 1)


@dataclass(frozen=True)
class RookProblem:
    """Problema de busqueda de la torre (formulacion AIMA de 5 componentes)."""

    start: tuple[int, int] = (0, 0)
    goal: tuple[int, int] = (7, 7)
    obstacles: frozenset = frozenset()
    cost_model: str = "unit"
    order: tuple = ROOK_DEFAULT_ORDER
    long_moves_first: bool = False

    def __post_init__(self):
        if self.cost_model not in ROOK_COST_MODELS:
            raise ValueError(f"modelo de costo desconocido: {self.cost_model}")
        if self.start in self.obstacles or self.goal in self.obstacles:
            raise ValueError("origen o destino sobre un obstaculo")
        if sorted(self.order) != sorted(ROOK_DIRECTIONS):
            raise ValueError("el orden debe ser una permutacion de N, E, S, O")

    def actions(self, state):
        """Acciones aplicables en `state`, en el orden documentado."""
        acts = []
        for d in self.order:
            dc, dr = ROOK_DIRECTIONS[d]
            ks = []
            c, r = state
            k = 0
            while True:
                c, r = c + dc, r + dr
                k += 1
                if not (0 <= c < ROOK_BOARD_SIZE and 0 <= r < ROOK_BOARD_SIZE):
                    break
                if (c, r) in self.obstacles:   # no puede ocupar ni saltar
                    break
                ks.append(k)
            if self.long_moves_first:
                ks.reverse()
            acts.extend((d, kk) for kk in ks)
        return acts

    def result(self, state, action):
        d, k = action
        dc, dr = ROOK_DIRECTIONS[d]
        return (state[0] + k * dc, state[1] + k * dr)

    def is_goal(self, state) -> bool:
        return state == self.goal

    def step_cost(self, state, action, next_state) -> int:
        return 1 if self.cost_model == "unit" else action[1]

    def with_cost(self, cost_model: str) -> "RookProblem":
        return RookProblem(self.start, self.goal, self.obstacles, cost_model,
                           self.order, self.long_moves_first)

    def is_valid_path(self, path) -> bool:
        """Cada paso es un movimiento de torre legal en las reglas simplificadas."""
        if not path or path[0] != self.start or path[-1] != self.goal:
            return False
        for a, b in zip(path, path[1:]):
            if a == b or (a[0] != b[0] and a[1] != b[1]):
                return False
            dc = (b[0] > a[0]) - (b[0] < a[0])
            dr = (b[1] > a[1]) - (b[1] < a[1])
            c, r = a
            while (c, r) != b:
                c, r = c + dc, r + dr
                if not (0 <= c < ROOK_BOARD_SIZE and 0 <= r < ROOK_BOARD_SIZE):
                    return False
                if (c, r) in self.obstacles:
                    return False
        return True


# --- heuristicas ------------------------------------------------------------
def h_rook_lines(state, goal) -> int:
    """Heuristica de la consigna: 0 en destino, 1 si comparte fila o columna, 2 si no."""
    if state == goal:
        return 0
    if state[0] == goal[0] or state[1] == goal[1]:
        return 1
    return 2


def h_manhattan(state, goal) -> int:
    """Distancia Manhattan |dc| + |df|."""
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])


def h_zero(state, goal) -> int:
    return 0


# --- resultados ---------------------------------------------------------------
def rook_path_costs(path) -> tuple[int, int]:
    """(costo unitario = numero de movimientos, costo por distancia = casillas recorridas)."""
    unit = max(len(path) - 1, 0)
    dist = sum(abs(a[0] - b[0]) + abs(a[1] - b[1]) for a, b in zip(path, path[1:]))
    return unit, dist


@dataclass
class RookSearchResult:
    algorithm: str
    cost_model: str
    path: list | None
    expanded: int = 0
    generated: int = 0
    max_frontier: int = 0
    expansion_order: list = field(default_factory=list)

    @property
    def found(self) -> bool:
        return self.path is not None

    @property
    def length(self) -> int:
        """Longitud = numero de movimientos (-1 si no hay ruta)."""
        return len(self.path) - 1 if self.path else -1

    @property
    def cost_unit(self) -> float:
        return rook_path_costs(self.path)[0] if self.path else float("inf")

    @property
    def cost_distance(self) -> float:
        return rook_path_costs(self.path)[1] if self.path else float("inf")

    @property
    def cost(self) -> float:
        """Costo segun el modelo con el que se busco."""
        return self.cost_unit if self.cost_model == "unit" else self.cost_distance

    def route_text(self) -> str:
        return " -> ".join(rook_square_name(s) for s in self.path) if self.path else "sin ruta"

    def row(self) -> dict:
        return {"algoritmo": self.algorithm, "modelo": self.cost_model,
                "ruta": self.route_text(), "longitud": self.length,
                "costo_unitario": self.cost_unit, "costo_distancia": self.cost_distance,
                "expandidos": self.expanded, "generados": self.generated,
                "frontera_max": self.max_frontier}


def _rook_rebuild(parents, state):
    path = [state]
    while parents[state] is not None:
        state = parents[state]
        path.append(state)
    path.reverse()
    return path


# --- busqueda no informada: BFS y DFS (nucleo comun, frontera inyectada) -------
def _rook_graph_search(problem: RookProblem, lifo: bool, name: str) -> RookSearchResult:
    """BFS (FIFO) o DFS (LIFO) en grafo, marcando al generar (Taller 1, D-17/D-19).

    En DFS los hijos se apilan en orden inverso para que el primero en salir sea el
    de la primera accion (Taller 1, D-18).
    """
    res = RookSearchResult(name, problem.cost_model, None)
    parents = {problem.start: None}
    frontier = deque([problem.start])
    res.max_frontier = 1
    while frontier:
        state = frontier.pop() if lifo else frontier.popleft()
        if problem.is_goal(state):
            res.path = _rook_rebuild(parents, state)
            return res
        res.expanded += 1
        res.expansion_order.append(state)
        children = []
        for action in problem.actions(state):
            child = problem.result(state, action)
            res.generated += 1
            if child not in parents:
                parents[child] = state
                children.append(child)
        frontier.extend(reversed(children) if lifo else children)
        res.max_frontier = max(res.max_frontier, len(frontier))
    return res


def rook_bfs(problem: RookProblem) -> RookSearchResult:
    """Busqueda en anchura: optima en LONGITUD (numero de movimientos)."""
    return _rook_graph_search(problem, lifo=False, name="BFS")


def rook_dfs(problem: RookProblem) -> RookSearchResult:
    """Busqueda en profundidad con control de repetidos: completa aqui (espacio
    finito, 64 estados) pero no optima; su ruta depende del orden de acciones."""
    return _rook_graph_search(problem, lifo=True, name="DFS")


# --- primero-el-mejor: UCS, A* y voraz --------------------------------------
def _rook_best_first(problem: RookProblem, kind: str, h, name: str) -> RookSearchResult:
    """BEST-FIRST-SEARCH de AIMA con eliminacion perezosa (Taller 1 D-21, Taller 2 A-04).

    Prioridades: UCS (g, orden); A* (g+h, h, orden); voraz (h, orden). `best_g`
    guarda el menor g conocido; una entrada con g mayor es obsoleta y se descarta
    al extraerla. Un hijo entra a la frontera solo si mejora `best_g`.
    """
    res = RookSearchResult(name, problem.cost_model, None)
    goal = problem.goal
    counter = 0

    def priority(g, s):
        hv = h(s, goal) if h is not None else 0
        if kind == "ucs":
            return (g, counter)
        if kind == "astar":
            return (g + hv, hv, counter)
        return (hv, counter)            # voraz

    best_g = {problem.start: 0}
    parents = {problem.start: None}
    frontier = [(priority(0, problem.start), 0, problem.start)]
    res.max_frontier = 1
    while frontier:
        _, g, state = heapq.heappop(frontier)
        if g > best_g[state]:
            continue                    # entrada obsoleta
        if problem.is_goal(state):
            res.path = _rook_rebuild(parents, state)
            return res
        res.expanded += 1
        res.expansion_order.append(state)
        for action in problem.actions(state):
            child = problem.result(state, action)
            res.generated += 1
            g2 = g + problem.step_cost(state, action, child)
            if child not in best_g or g2 < best_g[child]:
                best_g[child] = g2
                parents[child] = state
                counter += 1
                heapq.heappush(frontier, (priority(g2, child), g2, child))
        res.max_frontier = max(res.max_frontier, len(frontier))
    return res


def rook_ucs(problem: RookProblem) -> RookSearchResult:
    return _rook_best_first(problem, "ucs", None, "UCS")


def rook_astar(problem: RookProblem, h=None) -> RookSearchResult:
    """A* con la heuristica `h`; por omision la adecuada al modelo de costo."""
    h = h or rook_default_heuristic(problem.cost_model)
    return _rook_best_first(problem, "astar", h, f"A*({h.__name__})")


def rook_greedy(problem: RookProblem, h=None) -> RookSearchResult:
    """Voraz primero-el-mejor: ordena solo por h; no es optimo."""
    h = h or rook_default_heuristic(problem.cost_model)
    return _rook_best_first(problem, "greedy", h, f"Voraz({h.__name__})")


def rook_default_heuristic(cost_model: str):
    """Heuristica admisible adoptada para cada modelo de costo."""
    return h_rook_lines if cost_model == "unit" else h_manhattan


# --- verificacion exhaustiva de admisibilidad y consistencia ------------------
def rook_true_costs(problem: RookProblem) -> dict:
    """h*(s): costo optimo real de s al destino, para TODAS las casillas alcanzables.

    Dijkstra hacia atras desde el destino. Un movimiento de torre es reversible y
    cuesta lo mismo en ambos sentidos (mismas casillas recorridas), asi que el grafo
    inverso coincide con el directo. Es independiente de los buscadores evaluados
    (misma idea que en el Taller 1, D-07: no validar con lo que se valida).
    """
    dist = {problem.goal: 0}
    heap = [(0, problem.goal)]
    while heap:
        d, s = heapq.heappop(heap)
        if d > dist[s]:
            continue
        for a in problem.actions(s):
            t = problem.result(s, a)
            nd = d + problem.step_cost(s, a, t)
            if t not in dist or nd < dist[t]:
                dist[t] = nd
                heapq.heappush(heap, (nd, t))
    return dist


def rook_check_heuristic(problem: RookProblem, h) -> dict:
    """Comprueba admisibilidad (h <= h*) y consistencia (h(s) <= c + h(s')) en todo el espacio."""
    hstar = rook_true_costs(problem)
    goal = problem.goal
    not_admissible = [(s, h(s, goal), v) for s, v in sorted(hstar.items()) if h(s, goal) > v]
    inconsistent = []
    for s in hstar:
        for a in problem.actions(s):
            t = problem.result(s, a)
            if h(s, goal) > problem.step_cost(s, a, t) + h(t, goal):
                inconsistent.append((s, t))
    return {"states": len(hstar), "not_admissible": not_admissible,
            "inconsistent": len(inconsistent), "admissible": not not_admissible,
            "consistent": not inconsistent, "max_overestimate":
            max((hv - v for _, hv, v in not_admissible), default=0)}


# --- experimentos -------------------------------------------------------------
def rook_run_all(problem: RookProblem) -> list[RookSearchResult]:
    """BFS y DFS (no usan costo), UCS, A* y voraz con el modelo de `problem`."""
    out = [rook_bfs(problem), rook_dfs(problem), rook_ucs(problem),
           rook_astar(problem), rook_greedy(problem)]
    for r in out:
        r.cost_model = problem.cost_model
    return out


def rook_scenarios() -> dict:
    """Escenarios del informe: el de la consigna y variantes (otros extremos, obstaculos)."""
    return {
        "a1->h8": RookProblem(),
        "d4->h8": RookProblem(start=rook_square("d4")),
        "a1->h8 obstaculos": RookProblem(obstacles=ROOK_OBSTACLES_LENGTH_VS_COST),
    }


# Nueve obstaculos donde la ruta de MENOS MOVIMIENTOS no es la de MENOS CASILLAS:
# UCS unitario da 5 movimientos / 16 casillas; UCS por distancia, 7 movimientos /
# 14 casillas. Hallada con un muestreo aleatorio (random.Random(7), 4-14 obstaculos)
# y fijada aqui; las pruebas verifican la propiedad, no la busqueda.
ROOK_OBSTACLES_LENGTH_VS_COST = frozenset(
    rook_square(s) for s in ("a2", "b4", "c2", "d3", "d8", "e3", "f3", "g2", "h5"))


def rook_board_text(problem: RookProblem, path=None) -> str:
    """Tablero en texto: T origen, X destino, # obstaculo, * casilla de giro de la ruta."""
    turns = set(path[1:-1]) if path else set()
    lines = []
    for r in range(ROOK_BOARD_SIZE - 1, -1, -1):
        cells = []
        for c in range(ROOK_BOARD_SIZE):
            s = (c, r)
            ch = ("T" if s == problem.start else "X" if s == problem.goal else
                  "#" if s in problem.obstacles else "*" if s in turns else ".")
            cells.append(ch)
        lines.append(f"{r + 1} " + " ".join(cells))
    lines.append("  " + " ".join(_ROOK_FILES))
    return "\n".join(lines)


def rook_results_table(problem: RookProblem) -> list[dict]:
    """Filas con ambos modelos de costo para un escenario."""
    rows = []
    for model in ROOK_COST_MODELS:
        for r in rook_run_all(problem.with_cost(model)):
            rows.append(r.row())
    return rows


def rook_dfs_order_demo(problem: RookProblem | None = None) -> list[dict]:
    """La ruta de DFS depende del orden de las acciones (el costo optimo no)."""
    base = problem or RookProblem()
    variants = [("N,E,S,O; k creciente", ROOK_DEFAULT_ORDER, False),
                ("N,E,S,O; k decreciente", ROOK_DEFAULT_ORDER, True),
                ("E,N,O,S; k creciente", ("E", "N", "O", "S"), False),
                ("S,O,N,E; k creciente", ("S", "O", "N", "E"), False)]
    rows = []
    for label, order, longf in variants:
        p = RookProblem(base.start, base.goal, base.obstacles, base.cost_model, order, longf)
        r = rook_dfs(p)
        row = r.row()
        row["orden"] = label
        row["ucs_unit_cost"] = rook_ucs(p.with_cost("unit")).cost
        rows.append(row)
    return rows


def rook_unit_manhattan_counterexample() -> dict:
    """Manhattan con costo UNITARIO: sobreestima y A* deja de ser optimo.

    Busca, en un orden fijo y reproducible, un escenario con obstaculos donde
    A*(Manhattan) devuelve mas movimientos que BFS. Devuelve el primero encontrado.
    """
    unit = RookProblem()
    check = rook_check_heuristic(unit, h_manhattan)
    for prob in _rook_counterexample_candidates():
        opt = rook_bfs(prob)
        bad = rook_astar(prob, h_manhattan)
        if opt.found and bad.found and bad.length > opt.length:
            return {"problem": prob, "optimal": opt, "manhattan": bad,
                    "a1_h": h_manhattan(unit.start, unit.goal),
                    "a1_true": rook_true_costs(unit)[unit.start],
                    "violations_empty_board": len(check["not_admissible"])}
    return {"problem": None, "a1_h": h_manhattan(unit.start, unit.goal),
            "a1_true": rook_true_costs(unit)[unit.start],
            "violations_empty_board": len(check["not_admissible"])}


def _rook_counterexample_candidates():
    """Candidatos deterministas: muros horizontales con un hueco, de origen a1 a h8."""
    for row in range(1, 7):
        for gap in range(8):
            wall = frozenset((c, row) for c in range(8) if c != gap)
            yield RookProblem(obstacles=wall)


def plot_rook_expansions(table_by_scenario: dict, path: str | None = None):
    """Barras de nodos expandidos por algoritmo, escenario y modelo de costo.

    `table_by_scenario`: {nombre: filas de `rook_results_table`}. Devuelve la figura;
    si `path` se da, la guarda en PNG.
    """
    import matplotlib.pyplot as plt

    names = list(table_by_scenario)
    fig, axes = plt.subplots(1, len(ROOK_COST_MODELS), figsize=(12, 4.2), sharey=True)
    colors = ["#4C72B0", "#55A868", "#C44E52"]
    for ax, model in zip(axes, ROOK_COST_MODELS):
        algos = [r["algoritmo"] for r in table_by_scenario[names[0]] if r["modelo"] == model]
        width = 0.8 / len(names)
        for i, name in enumerate(names):
            rows = [r for r in table_by_scenario[name] if r["modelo"] == model]
            xs = [j + (i - (len(names) - 1) / 2) * width for j in range(len(algos))]
            bars = ax.bar(xs, [r["expandidos"] for r in rows], width,
                          label=name, color=colors[i % len(colors)])
            ax.bar_label(bars, fontsize=7)
        ax.set_xticks(range(len(algos)))
        ax.set_xticklabels(algos, rotation=20, fontsize=8)
        ax.set_title("costo unitario" if model == "unit" else "costo = casillas recorridas")
        ax.grid(axis="y", alpha=0.3)
    axes[0].set_ylabel("nodos expandidos")
    axes[0].legend(fontsize=8)
    fig.suptitle("Microproblema de la torre: nodos expandidos por algoritmo")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130)
    return fig


In [ ]:
# ===== 7.A y 7.B: microproblema de la torre (BFS, DFS, UCS, A*, voraz) =====
import matplotlib.pyplot as plt


def _imprimir_filas(filas, extra=()):
    cols = ("algoritmo", "modelo", "longitud", "costo_unitario", "costo_distancia", "expandidos",
            "generados", "frontera_max") + tuple(extra)
    print("  " + " | ".join(f"{c[:12]:>12}" for c in cols) + " | ruta")
    for f in filas:
        print("  " + " | ".join(f"{str(f[c])[:12]:>12}" for c in cols) + f" | {f['ruta']}")


_tablas_torre = {}
for _nombre, _prob in rook_scenarios().items():
    _tablas_torre[_nombre] = rook_results_table(_prob)
    _ucs_u = rook_ucs(_prob.with_cost("unit"))
    print(f"\n=== {_nombre} (BFS y DFS ignoran el costo; UCS, A* y voraz usan el modelo de la fila) ===")
    print(rook_board_text(_prob, _ucs_u.path))
    _imprimir_filas(_tablas_torre[_nombre])

# Longitud frente a costo con obstaculos: menos movimientos != menos casillas.
_obs = rook_scenarios()["a1->h8 obstaculos"]
_ru, _rd = rook_ucs(_obs.with_cost("unit")), rook_ucs(_obs.with_cost("distance"))
assert _ru.length < _rd.length and _rd.cost_distance < _ru.cost_distance
print(f"\nCon obstaculos: UCS unitario {_ru.length} movimientos / {_ru.cost_distance} casillas; "
      f"UCS por distancia {_rd.length} movimientos / {_rd.cost_distance} casillas.")

print("\nDFS depende del orden de las acciones (el optimo no):")
for _f in rook_dfs_order_demo():
    print(f"  {_f['orden']:<24} longitud {_f['longitud']:>2}, expandidos {_f['expandidos']:>2}, "
          f"optimo (UCS unitario) {_f['ucs_unit_cost']}")

print("\n7.B Verificacion exhaustiva de las heuristicas contra h* (Dijkstra hacia atras):")
for _etq, _h, _modelo in (("consigna", h_rook_lines, "unit"), ("Manhattan", h_manhattan, "distance"),
                          ("Manhattan", h_manhattan, "unit")):
    for _esc in ("a1->h8", "a1->h8 obstaculos"):
        _c = rook_check_heuristic(rook_scenarios()[_esc].with_cost(_modelo), _h)
        print(f"  {_etq:<10} costo {_modelo:<9} {_esc:<18} estados={_c['states']:>2}  "
              f"h>h*: {len(_c['not_admissible']):>2}  exceso max: {_c['max_overestimate']:>2}  "
              f"inconsistentes: {_c['inconsistent']:>3}")

_ce = rook_unit_manhattan_counterexample()
print(f"\nContraejemplo (Manhattan con costo unitario): BFS {_ce['optimal'].route_text()} "
      f"({_ce['optimal'].length} mov.) frente a A*(Manhattan) {_ce['manhattan'].route_text()} "
      f"({_ce['manhattan'].length} mov.); h(a1)={_ce['a1_h']} y h*(a1)={_ce['a1_true']}.")
print(rook_board_text(_ce["problem"], _ce["manhattan"].path))

_fig = plot_rook_expansions(_tablas_torre)      # misma figura que resultados/torre_expansiones.png
plt.show()


### 7.A · Microproblema de la torre: búsqueda no informada

**Reglas simplificadas explícitas.** Tablero 8×8 vacío, una sola torre, sin reyes, sin rival y
sin turnos. Opcionalmente hay casillas bloqueadas que la torre no puede ocupar ni saltar. No es
una posición legal de ajedrez, y por eso no usamos python-chess: implementamos el movimiento
desde cero en `chess_ai/rook_problem.py`.

**Formulación (AIMA, cap. 3).**

| Componente | Definición |
|---|---|
| Estado | Casilla de la torre, tupla `(columna, fila)` en 0..7 (a1 = (0,0), h8 = (7,7)) |
| Estado inicial | a1 (también probamos d4) |
| Acciones | `(d, k)`: avanzar k ≥ 1 casillas en la dirección d ∈ {N, E, S, O} sin salir del tablero ni atravesar obstáculos. Orden fijo: N, E, S, O y k creciente |
| Transición | RESULT(s, (d, k)) = s + k·δ(d) |
| Prueba objetivo | la torre está en h8 |
| Costo unitario | c(s, a, s') = 1: cuenta **movimientos** |
| Costo por distancia | c(s, a, s') = k: cuenta **casillas recorridas** |

**Longitud y costo no son lo mismo.** La *longitud* de una ruta es su número de movimientos. El
*costo* es la suma de los c según el modelo que adoptemos. Con costo unitario coinciden; con costo
por distancia no. a1→a8→h8 tiene longitud 2, costo unitario 2 y costo por distancia 14. BFS
optimiza la longitud y UCS optimiza el costo que le demos.

Como en el Taller 1, BFS y DFS comparten un núcleo con la frontera inyectada (FIFO o LIFO),
marcan los estados al generarlos y prueban el objetivo al extraerlos. En DFS los hijos se apilan
invertidos, para que salga primero el de la primera acción (D-17, D-18 y D-19 de aquel taller). UCS
usa un montículo con `best_g` y entradas obsoletas, igual que en el Taller 1 (D-21).

**Resultados reales, a1 → h8 en tablero vacío.** BFS y DFS no usan el costo, así que su ruta es
la misma en los dos modelos.

| Algoritmo | Costo con el que busca | Ruta | Longitud | Costo unit. | Costo dist. | Expandidos | Generados | Frontera máx. |
|---|---|---|---:|---:|---:|---:|---:|---:|
| BFS | — | a1→a8→h8 | 2 | 2 | 14 | 63 | 882 | 56 |
| DFS | — | a1→a2→b2→b3→c3→…→g7→g8→h8 | 14 | 14 | 14 | 14 | 196 | 50 |
| UCS | unitario | a1→a8→h8 | 2 | 2 | 14 | 63 | 882 | 56 |
| UCS | distancia | a1→a8→h8 | 2 | 2 | 14 | 63 | 882 | 43 |

**d4 → h8 (otro origen).**

| Algoritmo | Costo | Ruta | Longitud | Costo unit. | Costo dist. | Expandidos | Generados | Frontera máx. |
|---|---|---|---:|---:|---:|---:|---:|---:|
| BFS | — | d4→d8→h8 | 2 | 2 | 8 | 39 | 546 | 52 |
| DFS | — | d4→d5→e5→e6→f6→f7→g7→g8→h8 | 8 | 8 | 8 | 8 | 112 | 44 |
| UCS | unitario | d4→d8→h8 | 2 | 2 | 8 | 39 | 546 | 52 |
| UCS | distancia | d4→d8→h8 | 2 | 2 | 8 | 63 | 882 | 46 |

**a1 → h8 con 9 obstáculos** (a2, b4, c2, d3, d8, e3, f3, g2, h5). Este es el caso donde
longitud y costo se separan:

```
8 . . . # . . . X        ruta de UCS unitario        ruta de UCS por distancia
7 . . . . . . . .        a1→h1→h3→g3→g8→h8           a1→b1→b3→c3→c4→e4→e8→h8
6 . . . . . . . .        5 movimientos, 16 casillas  7 movimientos, 14 casillas
5 . . . . . . . #
4 . # . . . . . .
3 . . . # # # . .
2 # . # . . . # .
1 T . . . . . . .
  a b c d e f g h
```

| Algoritmo | Costo | Ruta | Longitud | Costo unit. | Costo dist. | Expandidos | Generados | Frontera máx. |
|---|---|---|---:|---:|---:|---:|---:|---:|
| BFS | — | a1→h1→h3→g3→g8→h8 | 5 | 5 | 16 | 54 | 451 | 26 |
| DFS | — | a1→b1→b3→c3→c4→d4→…→g8→h8 | 13 | 13 | 14 | 14 | 119 | 33 |
| UCS | unitario | a1→h1→h3→g3→g8→h8 | 5 | 5 | 16 | 54 | 451 | 26 |
| UCS | distancia | a1→b1→b3→c3→c4→e4→e8→h8 | 7 | 7 | 14 | 54 | 451 | 25 |

Con estas tablas observamos lo siguiente:

- **BFS es óptimo en longitud, no en costo por distancia:** con obstáculos, su ruta de 5
  movimientos recorre 16 casillas, y la más barata recorre 14 con 7 movimientos. Es el mismo
  fenómeno «menos pasos ≠ menor costo» que en el Taller 1 construimos con barro (D-39). Aquí
  aparece sin cambiar los costos de las aristas, solo cambiando lo que medimos.
- **UCS con costo unitario es BFS:** expande los mismos nodos en el mismo orden en los tres
  escenarios (lo comprueba la prueba). Ya lo habíamos predicho y comprobado en el Taller 1 (D-22).
- **DFS no es óptimo y su ruta depende del orden de las acciones.** Con control de repetidos es
  completo, porque solo hay 64 estados, pero devuelve una escalera de 14 movimientos.

| Orden de acciones en DFS | Ruta | Longitud | Expandidos |
|---|---|---:|---:|
| N, E, S, O; k creciente (base) | a1→a2→b2→b3→…→g8→h8 | 14 | 14 |
| N, E, S, O; k decreciente | a1→a8→h8 | 2 | 2 |
| E, N, O, S; k creciente | a1→b1→b2→c2→…→h7→h8 | 14 | 14 |
| S, O, N, E; k creciente | a1→a2→b2→b3→…→g8→h8 | 14 | 14 |

  El óptimo sigue siendo 2 en los cuatro casos. Ordenar distinto cambia **qué** encuentra DFS,
  no lo que existe. La misma idea vuelve a aparecer en 7.B con la ordenación de jugadas en
  alfa-beta.
- **Más costo no significa más trabajo para BFS:** en d4→h8, UCS por distancia expande 63 nodos
  y BFS 39. El costo óptimo es 8 y casi todo el tablero está a distancia ≤ 8, así que UCS tiene
  que vaciar casi toda la cola antes de sacar h8.

#### Por qué una secuencia al objetivo no garantiza mate contra un rival (pregunta 8)

En el microproblema **controlamos todos los movimientos**. El entorno es determinista, de un solo
agente y estático, así que la solución es una **secuencia fija** de acciones: si a1→a8→h8 lleva a
h8, lleva siempre. Con un rival deja de servir lo siguiente:

1. **La solución como secuencia.** Entre cada par de jugadas nuestras, el rival elige su jugada.
   La solución tiene que ser un **plan contingente**: «si responde x, juego y». Eso es la búsqueda
   AND-OR (7.D) o, cuando no podemos llegar a terminales, minimax.
2. **La prueba objetivo «llegar a una casilla».** El mate no es una casilla. Es una propiedad de
   **todas** las respuestas del rival: ninguna jugada legal le saca del jaque. Una ruta que llega
   bien a la meta contra una respuesta puede fallar contra otra. En la posición
   `8/4K3/7k/p7/8/8/6Q1/8`, la secuencia 1.Kf6 Kh7 2.Qg7# es correcta contra Kh7, pero no cubre
   Kh5 ni a4, y nuestro verificador la rechaza.
3. **El costo como suma de pasos.** Lo que se optimiza deja de ser un costo acumulado y pasa a
   ser el **valor minimax**. Una ruta corta no sirve si el rival la puede refutar.
4. **El modelo de transición.** Con un rival, RESULT(s, a) ya no determina el siguiente estado en
   el que volvemos a decidir. Las casillas también pueden quedar bloqueadas o atacadas por las
   piezas del rival, y la torre podría ser capturada.

Lo que **sí** se conserva: la formulación (estados, acciones, prueba terminal), el control de
estados repetidos (en ajedrez con la historia, no con un FEN aislado) y la idea de ordenar para
explorar menos.

### 7.B · Búsqueda informada (A* y voraz) y su relación con la ordenación de jugadas

Implementamos UCS, A* y voraz primero-el-mejor sobre un solo núcleo (`_rook_best_first`), con
`best_g` y entradas obsoletas. Solo cambia la prioridad:

| Algoritmo | Prioridad del montículo |
|---|---|
| UCS | (g, orden de inserción) |
| A* | (g + h, h, orden): a igual f, menor h, como en el A* del Taller 2 (A-04) |
| Voraz | (h, orden) |

**Heurísticas y el costo para el que sirven.**

| Heurística | Definición | Costo unitario | Costo por distancia |
|---|---|---|---|
| `h_rook_lines` (consigna) | 0 en h8; 1 si comparte fila o columna; 2 si no | admisible y consistente | admisible y consistente, pero débil |
| `h_manhattan` | \|Δcol\| + \|Δfila\| | **no admisible** (sobreestima) | admisible y consistente |

*Por qué la de la consigna es admisible con costo unitario.* Un movimiento de torre cambia **una
sola** coordenada. Si la torre no está en h8 hace falta al menos 1 movimiento. Si no comparte
fila ni columna con h8, las dos coordenadas son distintas y hacen falta al menos 2. Entonces
h ≤ h\* en todas las casillas. Con obstáculos sigue valiendo, porque los obstáculos solo quitan
movimientos y nunca abaratan el costo real. *Consistencia:* el único caso en que podría fallar
h(s) ≤ 1 + h(s') sería h(s) = 2 y h(s') = 0, es decir, llegar a h8 en un movimiento desde una
casilla que no comparte fila ni columna con h8, y eso es imposible.

*Por qué Manhattan es admisible con costo por distancia.* Un movimiento de k casillas cambia la
distancia Manhattan en a lo sumo k (desigualdad triangular), así que h(s) ≤ k + h(s'). Es decir,
es **consistente** y, como h(h8) = 0, también admisible. Es el mismo argumento que usamos en el
Taller 2 para el MST (A-08): una cota consistente no puede bajar más de lo que cuesta el paso.

*Por qué Manhattan **no** sirve con costo unitario.* En a1, h = 14 y el costo real es 2 (dos
movimientos). Lo verificamos de forma exhaustiva contra h\*, que calculamos con Dijkstra hacia
atrás desde h8. El oráculo es independiente de los buscadores (D-07 del Taller 1).

| Verificación exhaustiva (todas las casillas alcanzables) | Estados | Casillas con h > h\* | Exceso máx. | Transiciones inconsistentes |
|---|---:|---:|---:|---:|
| consigna, costo unitario, tablero vacío | 64 | 0 | 0 | 0 |
| consigna, costo unitario, con obstáculos | 55 | 0 | 0 | 0 |
| Manhattan, costo por distancia, tablero vacío | 64 | 0 | 0 | 0 |
| Manhattan, costo por distancia, con obstáculos | 55 | 0 | 0 | 0 |
| **Manhattan, costo unitario, tablero vacío** | 64 | **60** | **12** | **336** |
| **Manhattan, costo unitario, con obstáculos** | 55 | **51** | **9** | **147** |

**Que sobreestime no basta: hay que ver que cambie la respuesta.** Pusimos un muro en la fila 3
con hueco en b3 (`rook_unit_manhattan_counterexample`):

```
8 . . . . . . . X     BFS / A*(consigna):  a1→b1→b8→h8       3 movimientos (óptimo)
7 . . . . . . . .     A*(Manhattan):       a1→a2→b2→b8→h8    4 movimientos
6 . . . . . . . .
5 . . . . . . . .     en la ruta de A*(Manhattan): h(a1)=14 frente a h*=3;
4 . . . . . . . .     h(a2)=13 frente a 3; h(b2)=12 frente a 2; h(b8)=6 frente a 1
3 # . # # # # # #
2 . . . . . . . .
1 T . . . . . . .
  a b c d e f g h
```

a2 y b1 empatan en f = 1 + 13 = 14, y el orden de las acciones (N antes que E) saca primero a2.
Desde a2, b2 tiene g = 2 y h = 12, así que f = 14 con menor h que b1, y sale antes. Desde b2, b8
tiene f = 3 + 6 = 9, y h8 sale con f = 4. b1, con f = 14, no se vuelve a mirar. La sobreestimación
hace que un paso de más (g + 1) pese menos que la caída de h, y A* se comporta casi como el voraz.
A* con la h de la consigna, en el mismo tablero, devuelve los 3 movimientos.

**Resultados reales: A* y voraz con la heurística adecuada a cada costo.**

| Escenario | Costo | Algoritmo | Ruta | Longitud | Costo unit. | Costo dist. | Expandidos | Generados | Frontera máx. |
|---|---|---|---|---:|---:|---:|---:|---:|---:|
| a1→h8 | unitario | A*(consigna) | a1→a8→h8 | 2 | 2 | 14 | 2 | 28 | 20 |
| a1→h8 | unitario | Voraz(consigna) | a1→a8→h8 | 2 | 2 | 14 | 2 | 28 | 20 |
| a1→h8 | distancia | A*(Manhattan) | a1→a8→h8 | 2 | 2 | 14 | 2 | 28 | 20 |
| a1→h8 | distancia | Voraz(Manhattan) | a1→a8→h8 | 2 | 2 | 14 | 2 | 28 | 20 |
| d4→h8 | unitario | A*(consigna) | d4→d8→h8 | 2 | 2 | 8 | 2 | 28 | 20 |
| d4→h8 | distancia | A*(Manhattan) | d4→d8→h8 | 2 | 2 | 8 | 2 | 28 | 20 |
| obstáculos | unitario | A*(consigna) | a1→h1→h3→g3→g8→h8 | 5 | 5 | 16 | 18 | 108 | 13 |
| obstáculos | unitario | Voraz(consigna) | a1→h1→h3→g3→g8→h8 | 5 | 5 | 16 | 13 | 93 | 16 |
| obstáculos | distancia | A*(Manhattan) | a1→b1→b3→c3→c7→h7→h8 | 6 | 6 | 14 | 19 | 127 | 24 |
| obstáculos | distancia | Voraz(Manhattan) | a1→h1→h4→g4→g8→h8 | 5 | 5 | **16** | 6 | 47 | 18 |

![Nodos expandidos](resultados/torre_expansiones.png)

Lectura de la tabla y de la figura:

- **A* expande mucho menos que UCS y da el mismo costo:** en el tablero vacío, 2 nodos frente a
  63 con los dos modelos de costo. Con obstáculos, 18 frente a 54 (unitario) y 19 frente a 54
  (distancia). Con obstáculos y costo por distancia, A* devuelve una ruta distinta de la de UCS
  (6 movimientos frente a 7), pero del mismo costo, 14: hay empates. Las pruebas comprueban que
  A* nunca expande más que UCS.
- **El voraz es el que menos expande, pero no es óptimo:** con obstáculos y costo por distancia
  expande 6 nodos y devuelve 16 casillas cuando el óptimo es 14. En el tablero vacío acierta
  porque la heurística ya lleva directo a la meta.
- **Una heurística admisible más informada expande menos.** Con costo por distancia también es
  admisible la h de la consigna, pero es mucho más débil. En a1→h8 expanden 63 nodos con h = 0,
  60 con la de la consigna y 2 con Manhattan; con obstáculos, 54, 51 y 19. Manhattan **domina** a
  la de la consigna (es ≥ en cada casilla). Es el mismo patrón que vimos en el Taller 2, donde el
  A* con MST expandía entre 2.5 y 300 veces menos que con h = 0, con el mismo costo final.

#### Analogía y diferencias con ordenar jugadas en alfa-beta

**Lo que se parece.** En los dos casos usamos conocimiento del dominio para decidir **qué mirar
primero**. A* saca de la frontera el nodo de menor g + h. Alfa-beta mejorado (`_search_order_moves`)
prueba primero la jugada de la variante principal de la iteración anterior, luego las
promociones, las capturas MVV-LVA y las *killer moves*. En los dos,
una buena guía reduce el trabajo sin cambiar la respuesta, siempre que se respeten las
condiciones de cada algoritmo: el costo óptimo en A*, el valor minimax en alfa-beta. En el
AND-OR de 7.D también ordenamos los jaques primero, por la misma razón.

**Lo que es distinto.**

1. **En A*, la heurística decide qué nodos se expanden.** Los nodos con g + h > C\* no se
   expanden nunca, y por eso la corrección **depende** de que h sea admisible: con Manhattan y
   costo unitario, A* devolvió 4 movimientos en lugar de 3. **En alfa-beta, el orden nunca
   compromete el resultado:** cualquier orden da el mismo valor en la raíz. Un mal orden solo
   poda menos (en el mejor caso se exploran del orden de b^(m/2) nodos; en el peor, b^m).
2. **Ordenar no altera el conjunto de respuestas del rival.** En un nodo MIN, las respuestas
   legales del rival son las que dan las reglas, en el orden que sea. Alfa-beta solo deja de
   mirar una rama cuando las cotas α y β **demuestran** que no puede cambiar la decisión. Es una
   poda exacta, no una estimación. En cambio, A* «descarta» lo que h estima caro, y si h miente
   el descarte es un error.
3. **Alcance de la ordenación.** A* ordena una frontera **global**, con nodos de cualquier
   profundidad. Alfa-beta ordena **localmente** a los hermanos dentro de un recorrido en
   profundidad.
4. **Qué se puede cambiar.** Ordenar en alfa-beta puede cambiar **qué jugada** se devuelve entre
   varias de igual valor (pregunta 3 de la sustentación), pero no el valor. En A*, el desempate
   entre nodos de igual f cambia la ruta, pero no el costo. En el escenario con obstáculos, A* y
   UCS devolvieron rutas distintas del mismo costo, 14.

In [ ]:
# ===== 7.C: resultados del ajuste de pesos (corrida guardada) y su verificacion =====
# La corrida completa (run_tuning(TuningConfig()), ~80 s) produjo los archivos de resultados/.
# Aqui se LEEN, se comprueban contra los datos guardados y se grafican; no se reajusta nada.
import csv
import json
import os

import matplotlib.pyplot as plt
import numpy as np

with open(os.path.join("resultados", "pesos_ajustados.json"), encoding="utf-8") as _fh:
    _rep = json.load(_fh)
_obj = _rep["objective"]
print(f"Metodo elegido: {_rep['chosen_method']} ({_rep['selection_rule']}); K = {_rep['K']:.1f} cp")
print(f"{'':<22}{'wm':>8}{'wp':>8}{'wk':>9}{'wa':>7}{'J entren.':>12}{'J prueba':>11}")
print(f"{'iniciales E0':<22}" + "".join(f"{w:>8.3g}" for w in _rep["initial_weights"])
      + f"{_obj['train_initial']:>13.6f}{_obj['test_initial']:>11.6f}")
print(f"{'ajustados E*':<22}" + "".join(f"{w:>8.4g}" for w in _rep["weights"])
      + f"{_obj['train_tuned']:>13.6f}{_obj['test_tuned']:>11.6f}")
print(f"{'solo material (ref.)':<22}{'':>32}{_obj['train_material_only']:>13.6f}{_obj['test_material_only']:>11.6f}")
for _m, _info in _rep["methods"].items():
    print(f"  {_m}: {_info['evals']} evaluaciones de J ({_info['stop_reason']}), "
          f"J entren. = {_info['train_obj']:.8f}, pesos {_info['best_weights']}")

# Entrenamiento y prueba separados: ninguna partida ni posicion compartida.
_pos = {"train": {}, "test": {}}
with open(os.path.join("resultados", "tuning_posiciones.csv"), encoding="utf-8") as _fh:
    for _fila in csv.DictReader(_fh):
        _pos[_fila["split"]][" ".join(_fila["fen"].split()[:4])] = (_fila["game_id"], float(_fila["label"]))
_juegos = {s: {g for g, _ in v.values()} for s, v in _pos.items()}
assert not set(_pos["train"]) & set(_pos["test"]) and not _juegos["train"] & _juegos["test"]
print(f"\nPosiciones: {len(_pos['train'])} de entrenamiento ({len(_juegos['train'])} partidas) y "
      f"{len(_pos['test'])} de prueba ({len(_juegos['test'])} partidas); sin FEN ni partida en comun.")

# Recalcular J de prueba con los pesos guardados y la E del cuaderno: debe dar lo mismo.
_F = np.array([features(chess.Board(f + " 0 1")) for f in _pos["test"]], dtype=float)
_y = np.array([lab for _, lab in _pos["test"].values()], dtype=float)
_j0 = tuning_objective(_rep["initial_weights"], _F, _y, _rep["K"])
_j1 = tuning_objective(_rep["weights"], _F, _y, _rep["K"])
assert abs(_j0 - _obj["test_initial"]) < 1e-9 and abs(_j1 - _obj["test_tuned"]) < 1e-9
print(f"J de prueba recalculado aqui: E0 = {_j0:.6f}, E* = {_j1:.6f} (coincide con el JSON).")

print("\nAblacion a nivel de objetivo (devolver UN peso a su valor inicial):")
with open(os.path.join("resultados", "tuning_ablacion.csv"), encoding="utf-8") as _fh:
    for _fila in csv.DictReader(_fh):
        print(f"  {_fila['weight']}: {float(_fila['value_tuned']):g} -> {float(_fila['value_initial']):g}"
              f"   J entren. {float(_fila['train_obj']):.6f}   J prueba {float(_fila['test_obj']):.6f}")
_pt = _rep["paired_test_by_game"]
print(f"Comparacion pareada por partida en prueba: diferencia media {_pt['mean_diff']:+.5f} "
      f"(error estandar {_pt['se']:.5f}, {_pt['games']} partidas); E* mejor en "
      f"{100 * _pt['frac_games_improved']:.2f} % de ellas.")

# Historial de la busqueda local: mejor J de entrenamiento frente a evaluaciones gastadas.
_hist = {}
with open(os.path.join("resultados", "tuning_historial.csv"), encoding="utf-8") as _fh:
    for _fila in csv.DictReader(_fh):
        _hist.setdefault(_fila["method"], []).append((int(_fila["eval"]), float(_fila["best_obj"]),
                                                      float(_fila["candidate_obj"])))
_fig, _ax = plt.subplots(figsize=(8, 3.6))
for _m, _serie in _hist.items():
    _ax.plot([e for e, _, _ in _serie], [c for _, _, c in _serie], ".", ms=3, alpha=0.35,
             label=f"{_m}: J del candidato")
    _ax.plot([e for e, _, _ in _serie], [b for _, b, _ in _serie], "-", lw=2, label=f"{_m}: mejor J")
_ax.axhline(_obj["train_initial"], color="gray", ls="--", lw=1, label="J entren. con E0")
_ax.set_xlabel("evaluaciones de J (presupuesto 300 por metodo)")
_ax.set_ylabel("J de entrenamiento")
_ax.set_ylim(_obj["train_tuned"] - 0.0003, _obj["train_initial"] + 0.0006)
_ax.set_title("Busqueda local de pesos: hill climbing frente a recocido simulado")
_ax.legend(fontsize=7, ncol=2)
_ax.grid(alpha=0.3)
plt.show()


### 7C. Búsqueda local para ajustar la evaluación (y mejora adicional elegida)

Elegimos como mejora adicional la **evaluación con pesos ajustados por búsqueda local**, así que
esta implementación (`chess_ai/tuning.py`) cuenta para el apartado 7C y para la mejora con
ablación. La ablación en partidas (alfa-beta con E0 contra alfa-beta con E\*) está en los
experimentos; aquí reportamos el ajuste y la ablación a nivel de la función objetivo.

**Vector de pesos.** $w = (w_m, w_p, w_k, w_a)$ de $E(s)=w_mM(s)+w_pP(s)+w_kK(s)+w_aA(s)$.
Mantenemos $w_m = 100$ fijo: fija la unidad (centipeones). Si $w_m$ también fuera libre,
multiplicar todos los pesos y la escala $K$ por la misma constante daría la misma función
objetivo, y el problema no tendría una solución única. Ajustamos $w_p, w_k, w_a$ dentro de cotas
fijadas de antemano: $w_p, w_k \in [0, 100]$ y $w_a \in [0, 30]$ (ningún término posicional vale
más de un peón por unidad).

**Datos de entrenamiento y prueba.** No usamos ningún motor. Generamos partidas de autojuego
entre jugadores ε-voraces: en cada ply el jugador juega al azar con probabilidad ε (nuestro agente
Aleatorio) y, si no, juega nuestro Voraz de un ply con **solo material**. Cada bando de cada
partida sortea su ε en [0; 0,5], lo que produce desequilibrios y resultados decisivos. Usamos solo
material para que los datos no presupongan los pesos posicionales que vamos a ajustar.

- Entrenamiento: 600 partidas con el generador de semillas 31; prueba: 300 partidas con el
  generador 7919. Una partida va completa a un solo lado.
- Muestreamos una posición cada 3 plies desde el ply 8 y descartamos las partidas truncadas en
  200 plies (no tienen resultado oficial), las posiciones terminales, las posiciones en jaque y
  las que siguen a una captura (cambio de piezas a medias).
- Eliminamos duplicados por FEN (colocación, turno, enroques, al paso) y quitamos de prueba toda
  posición que también esté en entrenamiento.
- Resultado: **13 019 posiciones de entrenamiento** (473 partidas terminadas: 92 ganadas por
  blancas, 102 por negras, 279 tablas; 127 truncadas descartadas) y **6690 de prueba** (246
  partidas: 52, 53 y 141; 54 truncadas). Las posiciones con su etiqueta quedan en
  `resultados/tuning_posiciones.csv`.

**Etiqueta y función objetivo (estilo Texel).** La etiqueta $y$ es el resultado final de la
partida desde blancas (1, 0,5 o 0). Minimizamos sobre entrenamiento

$$J(w) = \frac{1}{n}\sum_i \left(\sigma\!\left(\frac{E_w(s_i)}{K}\right) - y_i\right)^2,$$

con $E_w$ recortada a ±10 000 igual que en `evaluate`. $K$ se calibra una sola vez con los pesos
iniciales y solo con entrenamiento (sección dorada sobre $\log K$, 30 evaluaciones): **K = 1789,5
cp**. Es una escala grande: una dama de ventaja (+900) predice solo un 62 % de puntos, lo que
muestra lo ruidosa que es la etiqueta en juego débil (una posición ganada puede perderse por una
jugada al azar posterior).

**Búsqueda local con presupuesto explícito.** Conectamos con el Taller 2: allí el estado era una
ruta del TSP y el vecino invertía un tramo (2-opt); aquí el estado es el vector de pesos y el
vecino cambia una coordenada.

- *Hill climbing por coordenadas:* evaluamos los vecinos $w_i \pm \text{paso}_i$ (pasos iniciales
  8, 8 y 2) y nos movemos al mejor si mejora estrictamente $J$; si ninguno mejora, reducimos los
  pasos a la mitad, hasta 1/64 del paso inicial.
- *Recocido simulado* (semilla 2026): perturba un peso al azar, acepta empeorar con probabilidad
  $e^{-\Delta/T}$, enfriamiento geométrico; $T_0 = -\text{mediana}(\Delta>0)/\ln 0{,}5$ calibrado
  con 20 vecinos.
- Presupuesto: **300 evaluaciones de J por método**, contando la evaluación inicial y la
  calibración de $T_0$. Regla fijada de antemano: el resultado final es el de menor $J$ de
  **entrenamiento**; los datos de prueba solo se miran después.

El hill climbing llegó a un óptimo local en 135 evaluaciones (22 movimientos aceptados); el
recocido usó las 300 y terminó en $J$ = 0,0444185, igual al del HC hasta el octavo decimal y apenas
peor después. Ganó el HC. El historial de ambos, evaluación por evaluación, está en
`resultados/tuning_historial.csv`.

| | $w_m$ | $w_p$ | $w_k$ | $w_a$ | $J$ entrenamiento | $J$ prueba |
|---|---:|---:|---:|---:|---:|---:|
| Pesos iniciales E0 | 100 | 15 | 10 | 4 | 0,045206 | 0,050634 |
| **Pesos ajustados E\*** (HC) | 100 | **100** | **53,375** | **0** | **0,044418** | **0,049990** |
| Recocido simulado | 100 | 100 | 53,681 | 0 | 0,044418 | 0,049991 |
| Solo material (referencia) | 100 | 0 | 0 | 0 | 0,045297 | 0,051034 |
| Predicción constante 0,487 (referencia) | — | — | — | — | 0,060761 | 0,071802 |

La mejora es de 1,7 % en entrenamiento y de 1,3 % en prueba. $w_p$ quedó en su cota superior: la
restricción está activa y el óptimo sin cotas estaría más arriba.

**Ablación a nivel de objetivo** (devolvemos UN peso del valor ajustado al inicial; el resto queda
ajustado):

| Peso devuelto | Ajustado → inicial | $J$ entrenamiento | $J$ prueba |
|---|---|---:|---:|
| ninguno (E\*) | — | 0,044418 | 0,049990 |
| $w_p$ | 100 → 15 | 0,045018 | 0,050788 |
| $w_k$ | 53,375 → 10 | 0,044435 | 0,049955 |
| $w_a$ | 0 → 4 | 0,044662 | 0,049944 |

En entrenamiento, devolver cualquier peso empeora $J$, como debe ser en un óptimo local. En
prueba solo $w_p$ se sostiene: devolver $w_k$ o $w_a$ a su valor inicial **mejora** levemente la
prueba. Es decir, la mejora que generaliza viene de los peones pasados (en juego débil nadie los
frena y deciden partidas), mientras que los cambios de $w_k$ y $w_a$ se ajustaron a peculiaridades
del entrenamiento. $w_a$ baja a 0 aunque la actividad tiene correlación 0,40 con el resultado,
porque está casi alineada con el material (más piezas implican más casillas atacadas) y, una vez
contado el material, aporta poco.

**¿Cómo sabemos si hubo sobreajuste? (pregunta 9).**
1. Separamos entrenamiento y prueba por partida y por FEN, y elegimos los pesos y el método solo con
   entrenamiento. La prueba nunca intervino en la elección.
2. La mejora aparece también en prueba (−1,3 %), así que no es un efecto exclusivo del
   entrenamiento. Pero es pequeña: en la comparación pareada por partida (la partida es la unidad
   independiente, porque sus posiciones comparten etiqueta) la diferencia media del error es
   −0,00028 con error estándar 0,00056 en 240 partidas, y E\* predice mejor en el 48,75 % de ellas.
   **No es estadísticamente distinguible de cero.**
3. La ablación en prueba muestra un sobreajuste leve de $w_k$ y $w_a$. Calculamos después la
   variante «solo $w_p$ ajustado» (100, 100, 10, 4), que da $J$ de prueba 0,049890, mejor que E\*.
   **No la adoptamos**: elegirla mirando prueba sería justamente usar prueba para ajustar. Para
   hacerlo bien habría que separar una validación dentro de entrenamiento y decidir la regla de
   antemano.
4. Con solo 3 parámetros libres y 13 019 posiciones, el riesgo de memorizar es bajo. El riesgo
   mayor es el **cambio de distribución**: los pesos predicen resultados de partidas entre
   jugadores ε-voraces, no de partidas de alfa-beta. Por eso la evidencia que decide si E\* es
   mejor es la ablación en partidas con presupuesto fijo (experimento 3), no esta tabla.

**Reproducibilidad.** `run_tuning(TuningConfig())` repite toda la corrida (78 s en nuestro
equipo, 76,7 s de CPU; la búsqueda local tarda 0,1 s porque las características se calculan una
vez). Una segunda corrida dio un archivo de posiciones idéntico byte a byte y los mismos pesos y
valores de $J$. Los pesos, la configuración completa, las semillas y las versiones (Python 3.13.7,
python-chess 1.11.2, numpy 2.5.4) quedan en `resultados/pesos_ajustados.json`;
`load_tuned_evaluator()` devuelve `Evaluator(pesos, "E*")` para la ablación en partidas.

In [ ]:
# ===== chess_ai/uncertainty.py =====
"""Seccion 7, apartado D: incertidumbre y garantias (AIMA 4.3, 4.4 y 5.5).

D1. Busqueda AND-OR (AIMA, fig. 4.11) sobre un mate forzado REAL de ajedrez:
    nodos OR = jugadas del atacante (basta UNA que funcione); nodos AND = TODAS
    las respuestas legales del defensor (el plan debe funcionar contra cada una).
    El resultado es un PLAN CONDICIONAL, no una secuencia:
        {jugada_atacante: {respuesta_1: subplan_1, respuesta_2: subplan_2, ...}}
    Una hoja de mate es {jugada: {}} y solo se acepta si la jugada da jaque mate.
D2. Arbol pequeno con nodos de azar, SEPARADO del ajedrez (no se cambian reglas
    del ajedrez con dados): expectiminimax frente a minimax de peor caso.
D3. Kriegspiel: estado de creencia (conjunto de tableros posibles) y su
    crecimiento tras jugadas no observadas del rival.
"""
from __future__ import annotations

from dataclasses import dataclass

import chess


# =============================================================================
# D1. AND-OR para mates forzados
# =============================================================================


@dataclass
class AndOrStats:
    """Contadores de la busqueda AND-OR (para comparar posiciones y ordenes)."""

    or_nodes: int = 0
    and_nodes: int = 0
    failures_by_draw: int = 0


def _unc_key(board: chess.Board) -> str:
    """Clave de estado para el control de ciclos del camino (sin relojes de jugadas)."""
    return board.epd()


def _unc_attacker_order(board: chess.Board):
    """Orden de las jugadas OR: jaques primero, luego capturas, luego el resto.

    Solo cambia CUANTO se explora antes de hallar un plan, no si existe; es la misma
    idea que ordenar jugadas en alfa-beta (seccion 7.B).
    """
    moves = list(board.legal_moves)
    return sorted(moves, key=lambda m: (not board.gives_check(m), not board.is_capture(m),
                                        m.uci()))


def _unc_or_search(board, n_left, path, stats):
    stats.or_nodes += 1
    if n_left <= 0:
        return None
    key = _unc_key(board)
    if key in path:                       # ciclo en el camino actual: fracaso (AIMA)
        return None
    path = path | {key}
    for move in _unc_attacker_order(board):
        board.push(move)
        try:
            if board.is_checkmate():
                return {move: {}}
            if n_left > 1 and search_outcome(board) is None:
                sub = _unc_and_search(board, n_left - 1, path, stats)
                if sub is not None:
                    return {move: sub}
            elif search_outcome(board) is not None:
                stats.failures_by_draw += 1   # ahogado/tablas: no es exito
        finally:
            board.pop()
    return None


def _unc_and_search(board, n_left, path, stats):
    """Nodo AND: el plan debe existir para CADA respuesta legal del defensor."""
    stats.and_nodes += 1
    replies = list(board.legal_moves)
    if not replies:
        # AIMA devuelve el plan vacio si no hay estados que cubrir; en ajedrez eso
        # seria aceptar un AHOGADO como exito. Aqui solo el mate (ya detectado en el
        # nodo OR) es exito; sin respuestas y sin mate = fracaso.
        return None
    plan = {}
    for reply in replies:
        board.push(reply)
        try:
            if search_outcome(board) is not None:   # tablas tras la defensa
                stats.failures_by_draw += 1
                return None
            sub = _unc_or_search(board, n_left, path, stats)
        finally:
            board.pop()
        if sub is None:
            return None                   # UNA defensa sin respuesta tumba el plan
        plan[reply] = sub
    return plan


def and_or_mate_search(board: chess.Board, max_attacker_moves: int,
                       stats: AndOrStats | None = None, shortest: bool = True):
    """Busca un plan de mate forzado en a lo sumo `max_attacker_moves` jugadas del bando al turno.

    Devuelve el plan condicional {jugada: {respuesta: subplan}} o None (fracaso).
    Con `shortest=True` profundiza n = 1, 2, ..., max (como IDDFS en el Taller 1):
    el primer plan hallado es el del mate forzado MAS CORTO en jugadas del atacante.
    Con `shortest=False` acepta el primer plan de longitud <= max que encuentre.
    Trabaja sobre una copia: el tablero recibido (FEN y pila) queda intacto.
    """
    stats = stats if stats is not None else AndOrStats()
    work = board.copy(stack=True)
    if search_outcome(work) is not None:
        return None
    limits = range(1, max_attacker_moves + 1) if shortest else (max_attacker_moves,)
    for n in limits:
        plan = _unc_or_search(work, n, frozenset(), stats)
        if plan is not None:
            return plan
    return None


def verify_mate_plan(board: chess.Board, plan) -> dict:
    """Verificacion INDEPENDIENTE del buscador: recorre el plan con las reglas.

    Exige: en cada nodo OR una sola jugada legal; si su subplan es {} la jugada da
    jaque mate; si no, las claves del subplan son EXACTAMENTE todas las respuestas
    legales del defensor. Devuelve {'ok', 'leaves', 'max_attacker_moves', 'error'}.
    """
    info = {"ok": True, "leaves": 0, "max_attacker_moves": 0, "error": ""}

    def walk(b, p, depth):
        if not isinstance(p, dict) or len(p) != 1:
            raise ValueError("un nodo OR debe tener exactamente una jugada")
        (move, sub), = p.items()
        if move not in b.legal_moves:
            raise ValueError(f"jugada ilegal {move.uci()} en {b.fen()}")
        b.push(move)
        try:
            if not sub:
                if not b.is_checkmate():
                    raise ValueError(f"hoja sin mate tras {move.uci()}: {b.fen()}")
                info["leaves"] += 1
                info["max_attacker_moves"] = max(info["max_attacker_moves"], depth)
                return
            legal = set(b.legal_moves)
            if set(sub) != legal:
                missing = sorted(m.uci() for m in legal - set(sub))
                raise ValueError(f"defensas sin cubrir: {missing}")
            for reply, subplan in sub.items():
                b.push(reply)
                try:
                    walk(b, subplan, depth + 1)
                finally:
                    b.pop()
        finally:
            b.pop()

    try:
        walk(board.copy(stack=True), plan, 1)
    except ValueError as exc:
        info["ok"] = False
        info["error"] = str(exc)
    return info


def format_mate_plan(board: chess.Board, plan, indent: str = "") -> str:
    """Plan en SAN, legible: cada nivel AND lista todas las defensas."""
    lines = []
    b = board.copy(stack=True)

    def number(bb):
        return f"{bb.fullmove_number}." if bb.turn == chess.WHITE else f"{bb.fullmove_number}..."

    def rec(bb, p, ind):
        (move, sub), = p.items()
        text = f"{ind}{number(bb)} {bb.san(move)}"
        bb.push(move)
        if not sub:
            lines.append(text + "   (mate)")
        else:
            lines.append(text + f"   -> {len(sub)} defensa(s):")
            for reply, subplan in sorted(sub.items(), key=lambda kv: bb.san(kv[0])):
                lines.append(f"{ind}    si {number(bb)} {bb.san(reply)}:")
                bb.push(reply)
                rec(bb, subplan, ind + "        ")
                bb.pop()
        bb.pop()

    if plan is None:
        return indent + "sin plan (fracaso)"
    rec(b, plan, indent)
    return "\n".join(lines)


def count_plan_leaves(plan) -> int:
    (move, sub), = plan.items()
    if not sub:
        return 1
    return sum(count_plan_leaves(p) for p in sub.values())


# Posiciones de D1. Las "mate en 2" se verifican en las pruebas: existe plan de 2
# y NO existe plan de 1.
MATE_POSITIONS = {
    # La de la seccion 8 del enunciado: Qg7# (y no hay otra forma de fallar).
    "mate_en_1": "7k/8/5KQ1/8/8/8/8/8 w - - 0 1",
    # Mate del pasillo (ultima fila) con torres dobladas: 1.Rd8+ Rxd8 2.Rxd8#.
    # Patron clasico; la unica defensa legal es capturar (AND con 1 rama).
    "mate_en_2_pasillo": "2r3k1/5ppp/8/8/8/8/3R1PPP/3R2K1 w - - 0 1",
    # Rey y dama contra rey y peon: jugada TRANQUILA 1.Kf6 y tres defensas, cada
    # una con un mate DISTINTO (Kh5 Qg5#, Kh7 Qg7#, a4 Qg6#). La hallamos con un
    # generador aleatorio de posiciones y la certificamos con verify_mate_plan.
    "mate_en_2_dama": "8/4K3/7k/p7/8/8/6Q1/8 w - - 0 1",
}


# =============================================================================
# D2. Arbol con azar: expectiminimax frente a minimax
# =============================================================================


@dataclass
class ChanceNode:
    """Nodo de un arbol de juego con azar.

    kind: 'max' | 'min' | 'chance' | 'leaf'. `children`: lista de (etiqueta, prob, nodo);
    prob solo se usa en nodos de azar (debe sumar 1). `utility` solo en hojas.
    """

    kind: str
    children: tuple = ()
    utility: float = 0.0


def _unc_leaf(u):
    return ChanceNode("leaf", (), float(u))


def _unc_min(*vals):
    return ChanceNode("min", tuple((f"r{i + 1}", None, _unc_leaf(v)) for i, v in enumerate(vals)))


def expectiminimax(node: ChanceNode) -> float:
    """Valor EXPECTIMINIMAX (AIMA 5.5): MAX, MIN y esperanza en nodos de azar."""
    if node.kind == "leaf":
        return node.utility
    vals = [(p, expectiminimax(ch)) for _, p, ch in node.children]
    if node.kind == "max":
        return max(v for _, v in vals)
    if node.kind == "min":
        return min(v for _, v in vals)
    total = sum(p for p, _ in vals)
    if abs(total - 1.0) > 1e-9:
        raise ValueError(f"probabilidades que no suman 1: {total}")
    return sum(p * v for p, v in vals)


def worst_case_minimax(node: ChanceNode) -> float:
    """Minimax sobre el MISMO arbol, tratando el azar como un adversario (peor caso).

    Es lo que haria un agente minimax que ignora las probabilidades: supone que el
    "dado" elige siempre el resultado peor para MAX.
    """
    if node.kind == "leaf":
        return node.utility
    vals = [worst_case_minimax(ch) for _, _, ch in node.children]
    return max(vals) if node.kind == "max" else min(vals)


def chance_root_decision(root: ChanceNode, value_fn) -> tuple[str, dict]:
    """Accion elegida en la raiz MAX y valor de cada accion segun `value_fn`."""
    values = {label: value_fn(ch) for label, _, ch in root.children}
    best = max(values, key=lambda k: values[k])    # empate: la primera listada
    return best, values


def transform_utilities(node: ChanceNode, fn) -> ChanceNode:
    """Copia del arbol con utilidades transformadas por `fn` (misma estructura)."""
    if node.kind == "leaf":
        return _unc_leaf(fn(node.utility))
    return ChanceNode(node.kind, tuple((lab, p, transform_utilities(ch, fn))
                                       for lab, p, ch in node.children))


def chance_tree_leaves(node: ChanceNode) -> list:
    if node.kind == "leaf":
        return [node.utility]
    return [u for _, _, ch in node.children for u in chance_tree_leaves(ch)]


def chance_tree_safe_vs_risky() -> ChanceNode:
    """MAX elige 'segura' o 'arriesgada'; luego azar (moneda cargada); luego MIN; hojas.

    Escala de utilidad: puntos para MAX en [-10, 10] (+ bueno para MAX).
    Probabilidades: resultado favorable 0.75, desfavorable 0.25.
    """
    return ChanceNode("max", (
        ("segura", None, ChanceNode("chance", (
            ("favorable", 0.75, _unc_min(3, 4)),
            ("desfavorable", 0.25, _unc_min(2, 5))))),
        ("arriesgada", None, ChanceNode("chance", (
            ("favorable", 0.75, _unc_min(6, 8)),
            ("desfavorable", 0.25, _unc_min(-4, 0))))),
    ))


def chance_tree_scale() -> ChanceNode:
    """Arbol para mostrar que expectiminimax depende de la ESCALA (AIMA 5.5, fig. 5.12).

    Utilidades en {1, ..., 9}; probabilidades 0.9 / 0.1. Con la transformacion
    monotona u -> 4**u cambia la decision de expectiminimax y no la de minimax.
    """
    return ChanceNode("max", (
        ("a1", None, ChanceNode("chance", (
            ("p=0.9", 0.9, _unc_min(2, 5)),
            ("p=0.1", 0.1, _unc_min(3, 6))))),
        ("a2", None, ChanceNode("chance", (
            ("p=0.9", 0.9, _unc_min(1, 7)),
            ("p=0.1", 0.1, _unc_min(4, 9))))),
    ))


def chance_scale_transform(u: float) -> float:
    """Transformacion monotona creciente NO lineal usada en D2: u -> 4**u."""
    return 4.0 ** u


# =============================================================================
# D3. Kriegspiel: estado de creencia
# =============================================================================


def _unc_referee_percept(board: chess.Board, move: chess.Move) -> tuple:
    """Lo que el arbitro anuncia de una jugada del rival: (captura en casilla, da jaque).

    Simplificacion documentada: el arbitro de Kriegspiel anuncia capturas (con la
    casilla) y jaques; no anuncia que pieza se movio ni desde donde.
    """
    capture = chess.square_name(move.to_square) if board.is_capture(move) else None
    return (capture, board.gives_check(move))


def belief_after_hidden_move(belief: dict, percept=(None, False)) -> dict:
    """Actualiza el estado de creencia tras UNA jugada no observada del rival.

    `belief`: {epd: tablero}. Cada mundo se expande con todas las jugadas legales del
    rival cuyo anuncio del arbitro coincide con `percept`; se fusionan los tableros
    iguales.
    """
    out = {}
    for b in belief.values():
        for mv in b.legal_moves:
            if _unc_referee_percept(b, mv) != percept:
                continue
            nb = b.copy(stack=False)
            nb.push(mv)
            out.setdefault(nb.epd(), nb)
    return out


def belief_after_own_try(belief: dict, move_uci: str) -> tuple[dict, dict]:
    """Nuestro intento de jugada: el arbitro dice 'legal' o 'ilegal'.

    Devuelve (mundos donde seria legal, mundos donde seria ilegal): el anuncio parte
    el estado de creencia. En los mundos legales la jugada se aplica.
    """
    legal, illegal = {}, {}
    for key, b in belief.items():
        mv = chess.Move.from_uci(move_uci)
        if mv in b.legal_moves:
            nb = b.copy(stack=False)
            nb.push(mv)
            legal[nb.epd()] = nb
        else:
            illegal[key] = b
    return legal, illegal


def kriegspiel_belief_demo(fen: str = chess.STARTING_FEN,
                           own_moves=("e2e4", "g1f3")) -> list[dict]:
    """Crecimiento del estado de creencia de blancas en Kriegspiel.

    Blancas conocen sus jugadas; las de negras no se ven (percepcion del arbitro:
    "sin captura, sin jaque"). Devuelve una fila por paso con el tamano del estado
    de creencia.
    """
    board = chess.Board(fen)
    belief = {board.epd(): board}
    rows = [{"paso": "inicio", "mundos_posibles": len(belief)}]
    for uci in own_moves:
        belief, illegal = belief_after_own_try(belief, uci)
        rows.append({"paso": f"blancas juegan {uci} (conocida)", "mundos_posibles": len(belief),
                     "mundos_donde_era_ilegal": len(illegal)})
        belief = belief_after_hidden_move(belief)
        rows.append({"paso": "negras responden (no observada)", "mundos_posibles": len(belief)})
    return rows


In [ ]:
# ===== 7.D: plan AND-OR en SAN, expectiminimax frente a minimax y estado de creencia de Kriegspiel =====
print("D1. Mate forzado como busqueda AND-OR (plan condicional verificado con las reglas)")
for _nombre, _fen in MATE_POSITIONS.items():
    _b = chess.Board(_fen)
    _st = AndOrStats()
    _plan = and_or_mate_search(_b, 2, _st)
    _ver = verify_mate_plan(_b, _plan) if _plan else {"ok": False, "leaves": 0}
    _sin_n1 = and_or_mate_search(_b, 1) is None
    assert _plan is not None and _ver["ok"]
    print(f"\n{_nombre}  {_fen}")
    print(f"  plan verificado: {_ver['ok']}; jugadas del atacante: {_ver['max_attacker_moves']}; "
          f"hojas (mates): {_ver['leaves']}; nodos OR/AND: {_st.or_nodes}/{_st.and_nodes}; "
          f"sin plan de 1 jugada: {_sin_n1}")
    print(format_mate_plan(_b, _plan, indent="  "))

# Un plan que no cubre todas las defensas se rechaza.
_b = chess.Board(MATE_POSITIONS["mate_en_2_dama"])
_lineal = {chess.Move.from_uci("e7f6"): {chess.Move.from_uci("h6h7"): {chess.Move.from_uci("g2g7"): {}}}}
print("\nSecuencia lineal 1.Kf6 Kh7 2.Qg7# como si fuera plan:", verify_mate_plan(_b, _lineal))

print("\nD2. Arbol con azar (separado del ajedrez): expectiminimax frente a minimax")
for _titulo, _arbol in (("segura / arriesgada (p = 0.75 / 0.25, utilidad en [-10, 10])",
                         chance_tree_safe_vs_risky()),
                        ("escala original (p = 0.9 / 0.1)", chance_tree_scale()),
                        ("misma escala transformada u -> 4^u",
                         transform_utilities(chance_tree_scale(), chance_scale_transform))):
    _ex, _vex = chance_root_decision(_arbol, expectiminimax)
    _mm, _vmm = chance_root_decision(_arbol, worst_case_minimax)
    print(f"  {_titulo}:")
    print(f"     expectiminimax {{{', '.join(f'{k}: {v:g}' for k, v in _vex.items())}}} -> {_ex}")
    print(f"     minimax (azar = adversario) {{{', '.join(f'{k}: {v:g}' for k, v in _vmm.items())}}} -> {_mm}")

print("\nD3. Kriegspiel: crecimiento del estado de creencia de blancas (arbitro: sin captura, sin jaque)")
for _fila in kriegspiel_belief_demo():
    print(f"  {_fila['paso']:<38} mundos posibles: {_fila['mundos_posibles']}")
_b = chess.Board()
_b.push_uci("e2e4")
_creencia = belief_after_hidden_move({_b.epd(): _b})
_legal, _ilegal = belief_after_own_try(_creencia, "e4e5")
print(f"  Tras 1.e4 y la respuesta oculta, intentar e4-e5: 'ilegal' en {len(_ilegal)} mundo(s) y "
      f"'legal' en {len(_legal)}; un intento fallido deja un solo tablero posible.")


### 7.D · Incertidumbre y garantías

#### D1. Mate forzado como búsqueda AND-OR

En la torre (7.A), la solución era una secuencia porque nosotros movíamos siempre. En una
táctica de mate, el defensor elige entre jugada y jugada. Por eso lo modelamos como AND-OR (AIMA
4.3, `AND-OR-SEARCH`) sobre posiciones **reales** de python-chess:

- **Nodo OR (atacante):** basta **una** jugada que funcione. Probamos las jugadas legales (jaques
  primero, luego capturas, luego el resto) hasta que una tenga plan.
- **Nodo AND (defensor):** el plan tiene que funcionar contra **todas** las respuestas legales. Si
  una sola respuesta no tiene plan, la jugada que llevó a ese nodo no sirve.
- **Éxito:** solo cuando la jugada del atacante da jaque mate. A diferencia del AND-SEARCH literal
  de AIMA, un nodo AND **sin respuestas y sin mate** (ahogado) o una posición de tablas cuentan
  como **fracaso**. Si no lo hiciéramos, el ahogado contaría como «objetivo alcanzado».
- **Profundidad:** `and_or_mate_search(board, n)` profundiza con 1, 2, …, n jugadas del atacante,
  como la profundización iterativa del Taller 1. Así el plan devuelto es el del **mate más
  corto**. Sin profundizar, en el mate en 1 del enunciado y con límite 3, la búsqueda devolvía un
  plan válido pero de 3 jugadas (1.Qe8+ Kh7 2.Qd7+ … 3.Qg7#/Qh3#). Ver ERROR CORREGIDO I-13.

El resultado es un **plan condicional** `{jugada: {respuesta: subplan}}`. Lo comprobamos con
`verify_mate_plan`, que no usa el buscador: recorre el plan con las reglas, exige que cada nivel
AND tenga **exactamente** las respuestas legales y que cada hoja sea jaque mate.

| Posición | FEN | Plan con n=1 | Plan con n=2 | Nodos OR / AND (n=2) | Hojas (mates) |
|---|---|---|---|---:|---:|
| Mate en 1 (sección 8 del enunciado) | `7k/8/5KQ1/8/8/8/8/8 w` | sí | (el de 1) | 1 / 0 | 1 |
| Mate del pasillo | `2r3k1/5ppp/8/8/8/8/3R1PPP/3R2K1 w` | no | sí | 3 / 1 | 1 |
| Rey y dama contra rey y peón | `8/4K3/7k/p7/8/8/6Q1/8 w` | no | sí | 18 / 14 | 3 |

Los nodos cuentan las dos iteraciones (n=1 y n=2). Planes obtenidos (SAN):

```
Mate del pasillo                         Rey y dama contra rey y peón
1. Rd8+   -> 1 defensa(s):               1. Kf6   -> 3 defensa(s):
    si 1... Rxd8:                            si 1... Kh5:
        2. Rxd8#   (mate)                        2. Qg5#   (mate)
                                             si 1... Kh7:
                                                 2. Qg7#   (mate)
                                             si 1... a4:
                                                 2. Qg6#   (mate)
```

**Por qué hay que cubrir todas las defensas.** El segundo plan lo muestra: cada defensa necesita
**un mate distinto**. Probamos dos «planes» incompletos con el verificador:

| Plan propuesto | Veredicto del verificador |
|---|---|
| Secuencia lineal 1.Kf6 Kh7 2.Qg7# | rechazado: «defensas sin cubrir: a5a4, h6h5» |
| 1.Kf6 y 2.Qg7 contra cualquier respuesta | rechazado: tras 1…Kh5, 2.Qg7 no es mate |

Una secuencia que gana contra la respuesta que **nosotros** imaginamos no es una garantía. Si el
rival tiene una sola defensa sin respuesta, el mate no está forzado, y un rival racional elegirá
precisamente esa defensa. Por eso el nodo AND es un «para todo» y el nodo OR un «existe». Es
la misma estructura que minimax con utilidades de solo dos valores (mate / no mate), pero aquí
la salida es el plan completo y no solo un valor.

Sobre la posición del pasillo: la primera versión, con las torres en d1 y e1, **no** era mate en
2, y lo descubrió nuestra propia búsqueda (I-14). La del rey y la dama la hallamos con un
generador aleatorio de posiciones. Que sea mate en 2 lo certifican las pruebas (plan válido con
n=2 y ningún plan con n=1).

#### D2. Un árbol pequeño con azar: expectiminimax frente a minimax

Este ejemplo está **separado del ajedrez**: no cambiamos ninguna regla del ajedrez para meter
dados. Estructura: MAX elige una acción → un nodo de azar (una moneda cargada) → MIN elige una
de dos respuestas → hoja. **Escala de utilidad:** puntos para MAX en [−10, 10] (positivo es bueno
para MAX). **Probabilidades:** resultado favorable 0.75 y desfavorable 0.25. *Minimax* trata el
nodo de azar como si fuera un adversario (peor caso). *Expectiminimax* calcula la esperanza
(AIMA 5.5).

| Acción | Favorable (p=0.75): MIN{…} | Desfavorable (p=0.25): MIN{…} | Expectiminimax | Minimax (peor caso) |
|---|---|---|---:|---:|
| segura | MIN{3, 4} = 3 | MIN{2, 5} = 2 | 0.75·3 + 0.25·2 = **2.75** | min(3, 2) = **2** |
| arriesgada | MIN{6, 8} = 6 | MIN{−4, 0} = −4 | 0.75·6 + 0.25·(−4) = **3.5** | min(6, −4) = **−4** |
| **Decisión** | | | **arriesgada** | **segura** |

Expectiminimax maximiza el resultado **promedio**: acepta un 25 % de perder 4 puntos porque en
promedio gana más. Minimax maximiza el **peor caso**: garantiza al menos 2 contra cualquier
resultado. Ninguno de los dos «se equivoca»: responden preguntas distintas. Si el azar es azar
de verdad, el promedio es lo racional. Si el «azar» es un rival, solo sirve la garantía (D1).

**La escala importa con azar y no importa sin él.** En un segundo árbol (probabilidades 0.9 y 0.1;
utilidades de 1 a 9) aplicamos la transformación monótona no lineal u → 4^u, que conserva el
orden de todas las hojas:

| Acción | Hojas (MIN de cada rama) | Expectiminimax, u original | Expectiminimax, 4^u | Minimax, u | Minimax, 4^u |
|---|---|---:|---:|---:|---:|
| a1 | 0.9 → MIN{2,5}=2; 0.1 → MIN{3,6}=3 | **2.1** | 20.8 | **2** | **16** |
| a2 | 0.9 → MIN{1,7}=1; 0.1 → MIN{4,9}=4 | 1.3 | **29.2** | 1 | 4 |
| **Decisión** | | **a1** | **a2** | **a1** | **a1** |

Minimax solo compara valores, así que una transformación que conserva el orden no cambia su
decisión. Expectiminimax **promedia** valores, y una transformación convexa como 4^u premia la
rama con la hoja alta poco probable. Por eso, con azar, la utilidad tiene que ser una escala con
significado (una utilidad esperada) y no solo un orden de preferencias (AIMA 5.5, fig. 5.12).

#### D3. Kriegspiel: percepción y estado interno

En Kriegspiel cada jugador ve **solo sus propias piezas**. Un árbitro, que ve los dos tableros,
anuncia: si un intento de jugada es «legal» o «ilegal» (si es ilegal, se intenta otra), si hubo
una **captura** y en qué casilla, y si hay **jaque** (y en qué dirección).

**Cómo cambiaría nuestro agente.**

| | Nuestros agentes actuales | Agente de Kriegspiel |
|---|---|---|
| Percepción | el `chess.Board` completo: todas las piezas, turno, enroques, al paso e historia | nuestras piezas + los anuncios del árbitro |
| Estado interno | el tablero exacto (estado físico) | un **estado de creencia**: el conjunto de tableros compatibles con todo lo percibido |
| Actualización | `push` de la jugada | predicción (expandir cada mundo con todas las jugadas rivales posibles) + filtrado (quedarse con los mundos compatibles con el anuncio) |
| Acción | una jugada legal | un **intento**: la respuesta «ilegal» también es información |
| Decisión | minimax / alfa-beta / MCTS sobre un estado | hay que planear sobre conjuntos de estados (AND-OR sobre estados de creencia, o muestrear mundos), con objetivos como «mate con alta probabilidad» |

**Información que nuestros agentes usan y que allí no podrían consultar.** En cada nodo MIN,
alfa-beta llama a `board.legal_moves` para **las jugadas del rival**, y eso exige saber dónde
está cada pieza rival. La evaluación común también lo usa: el término K mira
`board.is_attacked_by(not color, sq)` sobre la zona de nuestro rey. El material M cuenta las
piezas rivales con `board.pieces_mask(pt, color)`, y el término P decide si un peón es pasado
mirando `board.pieces_mask(chess.PAWN, not color)`, es decir, los peones del rival. En Kriegspiel nada de eso es observable: el agente no
sabe si la dama rival está en d8 o ya en h4 atacando f2. Lo mismo pasa con los *rollouts* de
MCTS, que simulan jugadas legales del rival sobre un tablero exacto.

**Cómo crece el estado de creencia** (`kriegspiel_belief_demo`, medido desde la posición inicial,
cuando el árbitro anuncia «sin captura, sin jaque» tras cada jugada negra):

| Paso | Tableros posibles |
|---|---:|
| Inicio | 1 |
| Blancas juegan 1.e4 (la conocen) | 1 |
| Negras responden sin que lo veamos | 20 |
| Blancas juegan 2.Nf3 | 20 |
| Negras responden sin que lo veamos | 266 |

Con solo dos jugadas negras no observadas ya hay 266 tableros posibles, y el crecimiento es
aproximadamente multiplicativo, como el árbol de juego. Los anuncios del árbitro sirven para
recortarlo: si tras 1.e4 y la respuesta oculta blancas **intentan** e4–e5, el árbitro dice
«ilegal» en exactamente 1 de los 20 mundos (cuando negras jugaron …e5) y «legal» en los otros 19.
Un intento fallido es una medición: deja el estado de creencia en un solo tablero.

## 8. Pruebas de corrección
El paquete de posiciones siguiente es un punto de partida. Las expectativas se deben verificar mediante reglas y análisis, sin asumir que siempre exista una única mejor jugada. Añadan al menos dos posiciones propias con justificación.

| Caso | Criterio |
|---|---|
| Inicial | 20 movimientos legales |
| Enroque | Ambos enroques blancos disponibles en la posición dada |
| Promoción | Cuatro promociones legales de a7 a a8 |
| Captura al paso | e5d6 legal y captura al paso |
| Mate en uno | El agente debe elegir alguna jugada que deje mate |
| Mate terminal | Sin decisión nueva, utilidad de derrota para negras |
| Ahogado | Tablas sin estar en jaque |
| Repetición | Reclamo detectable desde una historia construida |

A igual profundidad y evaluación, minimax y alfa-beta deben coincidir en valor. Las jugadas pueden diferir por empate. Una búsqueda debe dejar intactos FEN y pila de movimientos del tablero recibido. Si se interrumpe, ese invariante sigue aplicando.


In [ ]:
POSITIONS = {
    "initial": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "castling": "r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1",
    "promotion": "7k/P7/8/8/8/8/8/7K w - - 0 1",
    "en_passant": "7k/8/8/3pP3/8/8/8/7K w - d6 0 2",
    "mate_in_one": "7k/8/5KQ1/8/8/8/8/8 w - - 0 1",
    "checkmate": "7k/6Q1/5K2/8/8/8/8/8 b - - 0 1",
    "stalemate": "7k/5K2/6Q1/8/8/8/8/8 b - - 0 1",
}
if READY:
    for name,fen in POSITIONS.items():
        b=chess.Board(fen)
        assert b.is_valid(), (name,b.status())
        print(name,"movimientos legales:",b.legal_moves.count())
    assert chess.Board(POSITIONS["initial"]).legal_moves.count()==20
    assert chess.Move.from_uci("e1g1") in chess.Board(POSITIONS["castling"]).legal_moves
    assert chess.Move.from_uci("e1c1") in chess.Board(POSITIONS["castling"]).legal_moves
    assert chess.Board(POSITIONS["checkmate"]).is_checkmate()
    assert chess.Board(POSITIONS["stalemate"]).is_stalemate()
    repetition=chess.Board()
    for uci in ["g1f3","g8f6","f3g1","f6g8"]*2:
        repetition.push_uci(uci)
    assert repetition.can_claim_threefold_repetition()
    print("Reglas de las posiciones base verificadas.")


In [ ]:
# PRUEBAS DE SUS AGENTES:
# 1. Minimax frente a alfa-beta a igual profundidad/evaluación.
# 2. Decisiones legales de todos los agentes en posiciones no terminales.
# 3. Victoria inmediata y evasión de jaque.
# 4. Tablero e historia intactos tras búsqueda normal e interrumpida.
# 5. Misma semilla/presupuesto por simulaciones produce resultado reproducible.
# 6. Profundización devuelve jugada legal al agotarse el presupuesto.
# 7. MCTS elige para el rival sin maximizar accidentalmente al jugador raíz.
# Implemente y muestre los resultados, incluidos fallos que haya corregido.


In [ ]:
# ===== PRUEBAS DE NUESTROS AGENTES (las 7 de la celda anterior), ejecutadas aqui con asserts =====
# Las pruebas de comportamiento usan time_limit_s=0 (sin plazo) o simulaciones fijas, para que
# el resultado NO dependa de la velocidad de la maquina; las de plazo solo afirman invariantes.
import time as _time

_INTERMEDIAS = {
    "piezas_menores": "r3k3/ppp2ppp/2n5/3p4/3P4/2N5/PPP2PPP/R3K3 w Qq - 0 1",
    "captura_caballo": "4k3/pp3ppp/2n5/3p4/3N4/8/PP3PPP/4K3 w - - 0 1",
    "negras_en_jaque": "r3k3/ppp2ppp/8/1B6/8/8/PPP2PPP/4K2R b K - 0 1",
    "final_torres": "8/5k2/8/3R4/8/2K5/5r2/8 w - - 0 1",
    "final_peones": "8/8/4k3/3p4/3P4/4K3/8/8 w - - 0 1",
}
_NO_TERMINALES = {k: POSITIONS[k] for k in ("initial", "castling", "promotion", "en_passant",
                                            "mate_in_one")}
_EVASION = "k3r3/8/8/7Q/8/8/8/4K3 w - - 0 1"      # blancas en jaque por la torre e8


def _seis_agentes():
    """Los seis agentes, configurados para pruebas reproducibles (sin plazo de reloj)."""
    return [RandomAgent(), GreedyAgent(), MinimaxAgent(depth=2), AlphaBetaAgent(depth=2),
            AlphaBetaImprovedAgent(max_depth=2), MCTSAgent(simulations=200)]


def _foto(board):
    return board.fen(), list(board.move_stack)


def _con_historia():
    board = chess.Board()
    for uci in ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4", "g8f6"]:
        board.push_uci(uci)
    return board


def _valor_tras(board, move, depth):
    """Valor minimax desde blancas de jugar `move` y buscar depth-1 plies (mate medido desde la raiz)."""
    ctx = SearchContext(DEFAULT_EVALUATOR)
    board.push(move)
    try:
        return _search_minimax(board, depth - 1, 1, ctx)
    finally:
        board.pop()


_resultados_pruebas = []


def _registrar(numero, nombre, casos, detalle):
    _resultados_pruebas.append((numero, nombre, casos, detalle))


# --- 1. Minimax frente a alfa-beta (natural y ordenado) a igual profundidad y evaluacion
_casos, _difieren, _ahorro = 0, 0, []
for _nombre, _fen in {**_NO_TERMINALES, **_INTERMEDIAS}.items():
    for _prof in ((1, 2, 3) if _nombre in ("promotion", "en_passant", "mate_in_one", "final_torres",
                                            "final_peones") else (1, 2)):
        _b = chess.Board(_fen)
        _mm = search_root(_b, "minimax", _prof)
        for _orden in (False, True):
            _ab = search_root(_b, "alphabeta", _prof, ordering=_orden)
            assert _ab.value == _mm.value, (_nombre, _prof, _orden)       # igualdad exacta
            assert _ab.stats.nodes <= _mm.stats.nodes
            if _ab.move != _mm.move:        # empate: la jugada de alfa-beta alcanza el mismo valor
                _difieren += 1
                assert _valor_tras(_b, _ab.move, _prof) == _valor_tras(_b, _mm.move, _prof)
            _ahorro.append(1 - _ab.stats.nodes / _mm.stats.nodes)
            _casos += 1
_registrar(1, "minimax = alfa-beta en valor", _casos,
           f"valor identico (==) en {_casos} comparaciones (prof. 1-3); jugadas distintas por empate: "
           f"{_difieren}; ahorro medio de nodos {100 * sum(_ahorro) / len(_ahorro):.0f} %")

# --- 2. Decisiones legales de los seis agentes en posiciones no terminales (y terminales sin jugada)
_casos = 0
for _nombre, _fen in {**_NO_TERMINALES, "piezas_menores": _INTERMEDIAS["piezas_menores"],
                      "negras_en_jaque": _INTERMEDIAS["negras_en_jaque"]}.items():
    for _ag in _seis_agentes():
        _b = chess.Board(_fen)
        _d = _ag.choose(_b, time_limit_s=0, seed=7)
        assert _d.move in _b.legal_moves, (_ag.name, _nombre)
        _casos += 1
for _nombre, _esperado in (("checkmate", -MATE_SCORE), ("stalemate", 0.0)):
    for _ag in _seis_agentes():
        _d = _ag.choose(chess.Board(POSITIONS[_nombre]), time_limit_s=0, seed=0)
        assert _d.move is None and _d.score == _esperado, (_ag.name, _nombre, _d.score)
        _casos += 1
_registrar(2, "jugadas legales de los 6 agentes", _casos,
           "jugada legal en 7 posiciones x 6 agentes; en mate y ahogado: move=None con utilidad "
           "-100000 (derrota de negras) y 0")

# --- 3. Victoria inmediata y evasion de jaque (tambien con negras al turno)
_casos = 0
for _ag in _seis_agentes()[1:]:                          # el aleatorio no busca: no se le exige
    for _b in (chess.Board(POSITIONS["mate_in_one"]), chess.Board(POSITIONS["mate_in_one"]).mirror()):
        _d = _ag.choose(_b, time_limit_s=0, seed=0)
        _b.push(_d.move)
        assert _b.is_checkmate(), _ag.name
        _casos += 1
for _ag in _seis_agentes():
    _b = chess.Board(_EVASION)
    _d = _ag.choose(_b, time_limit_s=0, seed=0)
    _b.push(_d.move)
    assert not _b.was_into_check()                       # evasion legal
    if _ag.name not in ("Aleatorio", "MCTS"):
        assert _d.move == chess.Move.from_uci("h5e8"), _ag.name   # la mejor: capturar la torre
    _casos += 1
_registrar(3, "victoria inmediata y evasion", _casos,
           "5 agentes que evaluan dan mate en uno con blancas y con negras (tablero reflejado); "
           "todos evaden el jaque y los de busqueda capturan la torre que lo da (Dxe8+)")

# --- 4. Tablero e historia intactos tras busqueda normal e interrumpida
_casos = 0
for _ag in _seis_agentes():
    _b = _con_historia()
    _antes = _foto(_b)
    _ag.choose(_b, time_limit_s=0, seed=1)
    assert _foto(_b) == _antes, _ag.name
    _casos += 1
_b = _con_historia(); _antes = _foto(_b)
for _fn in (lambda ctx: _search_minimax(_b, 4, 0, ctx),
            lambda ctx: _search_alphabeta(_b, 4, 0, -float("inf"), float("inf"), ctx)):
    _ctx = SearchContext(DEFAULT_EVALUATOR, deadline=Deadline(0.02))
    try:
        _fn(_ctx)
        raise AssertionError("se esperaba SearchTimeout")
    except SearchTimeout:
        pass
    assert _ctx.stats.nodes > 1 and _foto(_b) == _antes   # cortada a mitad del arbol
    _casos += 1
for _ag, _plazo in ((AlphaBetaImprovedAgent(max_depth=10), 0.05), (MCTSAgent(simulations=None), 0.03),
                    (GreedyAgent(), 1e-9)):
    _d = _ag.choose(_b, time_limit_s=_plazo, seed=0)
    assert _foto(_b) == _antes and _d.move in _b.legal_moves, _ag.name
    _casos += 1
_n = {"n": 0}


def _evaluador_roto(board):
    _n["n"] += 1
    if _n["n"] > 50:
        raise RuntimeError("fallo simulado de la evaluacion")
    return DEFAULT_EVALUATOR(board)


for _ag in (AlphaBetaAgent(depth=3, evaluator=_evaluador_roto), MCTSAgent(simulations=200, evaluator=_evaluador_roto)):
    _n["n"] = 0
    try:
        _ag.choose(_b, time_limit_s=0, seed=0)
        raise AssertionError("se esperaba RuntimeError")
    except RuntimeError:
        pass
    assert _foto(_b) == _antes, _ag.name
    _casos += 1
_registrar(4, "tablero e historia intactos", _casos,
           "FEN y pila iguales tras buscar (6 agentes), tras SearchTimeout a mitad del arbol "
           "(minimax y alfa-beta), con plazo vencido (mejorado, MCTS, voraz) y si E lanza una excepcion")

# --- 5. Reproducibilidad: misma semilla / mismas simulaciones -> mismo resultado
_casos = 0
for _ag_cls in (RandomAgent, GreedyAgent):
    for _fen in (POSITIONS["initial"], _INTERMEDIAS["piezas_menores"]):
        _m = {_ag_cls().choose(chess.Board(_fen), time_limit_s=0, seed=11).move for _ in range(3)}
        assert len(_m) == 1
        _casos += 1
_a1, _a2, _a3 = MCTSAgent(simulations=150), MCTSAgent(simulations=150), MCTSAgent(simulations=150)
_d1 = _a1.choose(chess.Board(), time_limit_s=0, seed=5)
_d2 = _a2.choose(chess.Board(), time_limit_s=0, seed=5)
_a3.choose(chess.Board(), time_limit_s=0, seed=6)
assert _d1.move == _d2.move and _a1.last_info["root_visits"] == _a2.last_info["root_visits"]
assert _a1.last_info["root_visits"] != _a3.last_info["root_visits"]     # otra semilla, otro arbol
_casos += 2
_partidas = []
for _ in range(2):        # partida completa (20 plies) con el controlador: misma semilla base
    _cfg = AgentConfig(depth=1, time_limit_s=0, seed=3, simulations=30)
    _ctl = GameController(ChessEnvironment(max_plies=20), make_agent("Voraz", _cfg),
                          make_agent("MCTS", _cfg), _cfg)
    _ctl.play_until_stop()
    _partidas.append([r.move_uci for r in _ctl.records])
assert _partidas[0] == _partidas[1] and len(_partidas[0]) == 20
_casos += 1
_registrar(5, "reproducibilidad por semilla", _casos,
           "aleatorio y voraz: misma jugada con la misma semilla; MCTS con 150 simulaciones: mismas "
           "visitas en la raiz (otra semilla, otras visitas); partida Voraz-MCTS de 20 plies identica")

# --- 6. Profundizacion iterativa: jugada legal al agotarse el presupuesto
_casos = 0
for _fen in (POSITIONS["initial"], _INTERMEDIAS["piezas_menores"], _INTERMEDIAS["negras_en_jaque"]):
    _b = chess.Board(_fen)
    _d = AlphaBetaImprovedAgent().choose(_b, time_limit_s=1e-9, seed=0)
    assert _d.move in _b.legal_moves and _d.stats.completed_depth == 0 and _d.score is None
    _ag = AlphaBetaImprovedAgent(max_depth=10)
    _d = _ag.choose(_b, time_limit_s=0.1, seed=0)
    _hechas = [it for it in _ag.last_iterations if it["completed"]]
    assert _d.move in _b.legal_moves and _d.stats.completed_depth == len(_hechas) >= 1
    assert _d.move.uci() == _hechas[-1]["move"]          # la de la ULTIMA iteracion completa
    assert _d.score == search_root(_b, "alphabeta", _d.stats.completed_depth).value
    assert sum(it["nodes"] for it in _ag.last_iterations) == _d.stats.nodes
    _casos += 2
_registrar(6, "profundizacion con plazo agotado", _casos,
           "plazo 1e-9 s: jugada de respaldo legal, score None, completed_depth 0; plazo 0.1 s: "
           "jugada y valor de la ultima iteracion completa (= alfa-beta a esa profundidad)")


# --- 7. MCTS elige para el rival sin maximizar accidentalmente al jugador raiz
class _MCTSPerspectivaBlancas(MCTSAgent):
    """Variante con el ERROR de la pregunta 6: todo W desde blancas (negras maximizan para blancas)."""

    def _reward_for(self, mover, reward_white):
        return reward_white


_tac = {t["id"]: t for t in MCTS_TACTICS}
_casos = 0
for _id in ("T3_mate1_b_backrank", "T6_queen_b"):              # negras al turno
    for _semilla in range(3):
        _bien = MCTSAgent(simulations=300).choose(chess.Board(_tac[_id]["fen"]), time_limit_s=0,
                                                  seed=_semilla)
        _mal = _MCTSPerspectivaBlancas(simulations=300).choose(chess.Board(_tac[_id]["fen"]),
                                                              time_limit_s=0, seed=_semilla)
        assert _bien.move.uci() in _tac[_id]["solutions"] and _mal.move.uci() not in _tac[_id]["solutions"]
        _casos += 1
_ag = MCTSAgent(simulations=300)
_ag.choose(chess.Board(_tac["T3_mate1_b_backrank"]["fen"]), time_limit_s=0, seed=0)
_pila = [(_ag.last_root, chess.Board(_tac["T3_mate1_b_backrank"]["fen"]))]
_nodos_revisados = 0
while _pila:                                       # cada hijo "pertenece" a quien mueve en el padre
    _nodo, _bb = _pila.pop()
    for _hijo in _nodo.children:
        assert _hijo.mover == _bb.turn
        _nb = _bb.copy(stack=False); _nb.push(_hijo.move)
        _pila.append((_hijo, _nb)); _nodos_revisados += 1
_registrar(7, "perspectiva de MCTS", _casos + 1,
           f"negras al turno (T3 mate Ta1#, T6 dama colgada), 3 semillas: el agente acierta y la variante "
           f"'todo desde blancas' falla siempre; {_nodos_revisados} nodos del arbol con dueno correcto")

# --- Posiciones propias (seccion 8): reglas especiales y comportamiento de los agentes
_PROPIAS = {
    "ep_clavada": "8/8/8/K2pP2r/8/8/8/7k w - d6 0 2",
    "enroque_atacado": "4k3/8/8/8/8/8/6b1/R3K2R w KQ - 0 1",
    "mate_en_dos": "k7/8/2K5/8/8/8/8/7R w - - 0 1",
    "pieza_envenenada": "6k1/pp3ppp/4p3/3n4/8/8/PP3PPP/3Q2K1 w - - 0 1",
}
for _fen in _PROPIAS.values():
    assert chess.Board(_fen).is_valid()
_b = chess.Board(_PROPIAS["ep_clavada"])
assert _b.has_legal_en_passant() is False and chess.Move.from_uci("e5d6") not in _b.legal_moves
assert chess.Move.from_uci("e5e6") in _b.legal_moves
_b = chess.Board(_PROPIAS["enroque_atacado"])
assert chess.Move.from_uci("e1g1") not in _b.legal_moves and chess.Move.from_uci("e1c1") in _b.legal_moves
_b = chess.Board(_PROPIAS["mate_en_dos"])
assert search_root(_b, "minimax", 2).value < MATE_SCORE - 1000
assert search_root(_b, "alphabeta", 3).value == MATE_SCORE - 3 == search_root(_b, "minimax", 3).value
_b = chess.Board(_PROPIAS["pieza_envenenada"])
assert GreedyAgent().choose(_b, time_limit_s=0, seed=0).move == chess.Move.from_uci("d1d5")
assert MinimaxAgent(depth=2).choose(_b, time_limit_s=0, seed=0).move != chess.Move.from_uci("d1d5")
_registrar("8", "posiciones propias", 4,
           "captura al paso ilegal por clavada; enroque corto prohibido por casilla atacada; mate en dos "
           "(prof. 3 = 100000-3, prof. 2 no lo ve); pieza envenenada (el voraz la toma, minimax-2 no)")

print(f"{'#':>2}  {'prueba':<34}{'casos':>6}  resultado  detalle")
for _num, _nom, _cas, _det in _resultados_pruebas:
    print(f"{_num:>2}  {_nom:<34}{_cas:>6}  PASA       {_det}")
print(f"\nTodas las afirmaciones se verificaron con assert ({len(_resultados_pruebas)} grupos, "
      f"{sum(r[2] for r in _resultados_pruebas)} casos).")


### Pruebas de corrección: qué verifica la celda anterior y qué fallos corregimos

La celda anterior ejecuta, **dentro del cuaderno**, las siete pruebas que pide el enunciado sobre los seis agentes. Cada afirmación es un `assert`: si una falla, la ejecución del cuaderno se detiene ahí. La tabla final resume casos y resultado.

| # | Prueba del enunciado | Cómo la comprobamos |
|---|---|---|
| 1 | Minimax frente a alfa-beta | 12 posiciones (5 del enunciado y 5 intermedias: captura, jaque, final de torres, final de peones y piezas menores), profundidades 1 y 2 (y 3 en las baratas). Alfa-beta natural y ordenado dan **exactamente** el mismo valor que minimax (`==`, no aproximado). Cuando la jugada difiere, verificamos que ambas alcanzan el mismo valor: es un empate y no un error (pregunta 3). |
| 2 | Decisiones legales | Los seis agentes en 7 posiciones no terminales devuelven una jugada de `legal_moves`. En `checkmate` y `stalemate` todos devuelven `move=None` con la utilidad exacta: −100 000 (derrota de negras) y 0. |
| 3 | Victoria inmediata y evasión | Mate en uno con blancas y con el tablero reflejado (negras al turno). En la evasión propia `k3r3/8/8/7Q/8/8/8/4K3 w` todos salen del jaque y los agentes de búsqueda capturan la torre que lo da. |
| 4 | Tablero e historia intactos | Mismo FEN y misma pila de jugadas tras buscar; tras un `SearchTimeout` a mitad del árbol; con el plazo vencido; y aunque la evaluación lance una excepción (los `pop` están en `finally`). |
| 5 | Reproducibilidad | Misma semilla, misma jugada (aleatorio, voraz). MCTS con simulaciones fijas da las mismas visitas en la raíz. Una partida completa Voraz–MCTS por el controlador se repite jugada a jugada. |
| 6 | Profundización con plazo agotado | Con plazo de 1e-9 s devuelve la jugada de respaldo (legal, `score=None`, `completed_depth=0`). Con 0,1 s devuelve la jugada y el valor de la última iteración completa. |
| 7 | Perspectiva de MCTS | Con negras al turno, el agente correcto acierta y una variante con el error típico (guardar todo desde blancas) falla, con las mismas semillas. Además recorremos el árbol y verificamos que cada nodo «pertenece» al jugador que movió para llegar a él. |

**Posiciones propias (sección 8).** Agregamos cuatro, cada una con su justificación:

| Posición | FEN | Qué comprueba y por qué la elegimos |
|---|---|---|
| Captura al paso ilegal | `8/8/8/K2pP2r/8/8/8/7k w - d6 0 2` | La captura al paso exd6 quitaría **dos** peones de la quinta fila y dejaría al rey de a5 en jaque por la torre de h5. El FEN marca d6 como casilla al paso, pero la jugada no es legal. Una regla especial que cualquier implementación propia de reglas suele olvidar. |
| Enroque prohibido | `4k3/8/8/8/8/8/6b1/R3K2R w KQ - 0 1` | Hay derechos de enroque, pero el alfil de g2 ataca f1: O-O es ilegal (el rey pasaría por una casilla atacada) y O-O-O sí es legal. Distingue «tener derecho» de «poder enrocar ahora». |
| Mate en dos | `k7/8/2K5/8/8/8/8/7R w - - 0 1` | Con profundidad 2 minimax no ve el mate; con profundidad 3 da exactamente 100 000 − 3. Muestra el horizonte y la preferencia por el mate más corto. |
| Pieza envenenada | `6k1/pp3ppp/4p3/3n4/8/8/PP3PPP/3Q2K1 w - - 0 1` | El caballo de d5 parece colgado, pero lo defiende e6. El voraz juega Dxd5 y pierde la dama; minimax de profundidad 2 no la toma. |

Además, la suite `pytest` del repositorio (`tests/`) tiene **258 pruebas** que cubren los mismos invariantes con más casos (todas pasan; ver el README para ejecutarla).

#### Fallos que corregimos (de la bitácora)

Estos fallos ocurrieron de verdad durante el desarrollo; cada uno está en `BITACORA.md` con la marca ERROR CORREGIDO.

| Entrada | Qué pasó | Cómo se corrigió y cómo lo comprobamos |
|---|---|---|
| G-E1 | Tras pausar, el bucle se detenía, pero el botón seguía diciendo «Pausar». | El `render()` final se ejecuta dentro de la propia tarea, que aún no está `done()`. Usamos una bandera explícita `_loop_active`. La prueba `test_gui_start_pause_inside_event_loop` pasa. |
| S-16 | La posición de perspectiva daba 0 (tablas) a toda profundidad. | Tras Cxa4 quedaba material insuficiente, que es terminal y se resuelve **antes** que E. Añadimos peones y ahora Cxa4 vale +336,5 desde negras. |
| S-17 | Una prueba exigía `score > 500` y a veces fallaba según la carga de la máquina. | El valor es el de la posición, no la ganancia de la jugada (umbral corregido), y las pruebas de comportamiento pasaron a usar `time_limit_s=0`. Verificado con 3 corridas seguidas y 4 en paralelo. |
| S-18 | La quietud contaba dos veces el nodo frontera. | Lo encontramos revisando el código contra la definición de nodo antes de ejecutarlo. Parámetro `entry=True`: solo las llamadas recursivas cuentan nodo. |
| M-E1 | El rollout uniforme de 8 plies empeoraba la táctica (0,90 frente a 1,00 de evaluar la hoja). | Rollout con sesgo a capturas y límite 4, elegido midiendo (benchmark repetido). |
| M-E2 | La prueba de perspectiva de MCTS pasaba o fallaba por suerte. | Con 8 defensas correctas de 28, un agente equivocado acierta a veces. Usamos tácticas de solución única (T3, T6), donde la variante errónea falla con todas las semillas, y añadimos un recorrido estructural del árbol (prueba 7). |
| M-E3 | MCTS reportaba un mate en uno como +5526 cp en vez de 99 999. | La inversa de la sigmoide no llega a una victoria. Si el hijo elegido es terminal se reporta su valor exacto; el benchmark se repitió completo. |
| I-13 | La búsqueda AND-OR devolvía un plan de mate en 3 en una posición con mate en 1. | Profundización 1, 2, …, n: el primer plan hallado es el más corto (`test_shortest_plan_is_preferred`). |
| I-14 | La posición de «mate del pasillo» copiada de memoria no era mate en 2. | Lo detectó el propio buscador AND-OR. Doblamos las torres (d1 y d2), y el verificador independiente certifica el plan. |

## 9. Protocolo experimental y torneo
### Experimento 1: ahorro de búsqueda
Usen cinco posiciones intermedias válidas con distinto carácter: captura, jaque, defensa, posición tranquila y final. Comparen minimax, alfa-beta natural y alfa-beta ordenado a profundidades 1, 2 y 3 con evaluación fija. Mantengan el mismo árbol y condiciones de corte. Muestren valores y métricas; separen el tiempo de ordenar del tiempo total si lo instrumentan.

### Experimento 2: fuerza de juego bajo presupuesto
Torneo de los seis agentes, todos contra todos. Cada par juega al menos **cuatro partidas**, dos con cada color, desde posiciones iniciales/aperturas equilibradas y con semillas distintas. Son 60 partidas como mínimo. Compartan las mismas aperturas entre emparejamientos e intercambien colores. Usen, por ejemplo, un presupuesto de 0.2 segundos por decisión ajustado a su hardware y un límite de 160 plies. Si un algoritmo de profundidad fija excede el presupuesto, registren el exceso y apliquen una política uniforme documentada.

Puntuación reglamentaria: victoria 1, tablas 0.5 y derrota 0. Un límite artificial produce “truncada”: reporten su proporción por separado y dejen su resultado oficial sin asignar. Pueden presentar una clasificación provisional de partidas terminadas con la cobertura de cada agente; no den tablas ni victorias por evaluación estática como si fueran resultados oficiales.

### Experimento 3: ablación
Comparen una mejora con su versión base manteniendo los demás factores fijos. Por ejemplo, alfa-beta con/sin ordenación o evaluación con pesos iniciales/ajustados. Si comparan quietud con ausencia de quietud a igual profundidad, aclaren que el árbol ya no es el mismo.

### Experimento 4: MCTS
Repitan un conjunto de tácticas con al menos tres presupuestos de simulación y cinco semillas. Reporten tasa de acierto táctico, media y dispersión temporal. Una curva de mejora promedio no implica mejora monótona por semilla.

**Reporte mínimo:** nodos, hojas, podas, profundidad completada, tiempo por jugada, simulaciones, resultados, excesos y truncamientos. Para comparar fuerza use presupuesto temporal; para estudiar alfa-beta use profundidad fija. Para reproducibilidad exacta de MCTS use simulaciones fijas, porque un plazo temporal depende de la máquina.


In [ ]:
# Esquema orientador de CSV. Escriba archivos solo cuando ejecute sus experimentos.
CSV_FIELDS = ["game_id","opening_id","seed","white_agent","black_agent","ply",
              "move_uci","nodes","leaves","cutoffs","completed_depth",
              "elapsed_s","simulations","time_exceeded","termination","official_result"]
# TODO: torneo, exportación y gráficos. Preserve configuración y versiones usadas.


In [ ]:
# ===== chess_ai/tournament.py =====
"""Torneo todos contra todos (Experimento 2) y motor de partidas para la ablacion.

PROTOCOLO (justificacion completa en apuntes/09_protocolo.md y bitacora X-):

* APERTURAS. `OPENINGS` son lineas teoricas conocidas y equilibradas (5-10 plies),
  verificadas legales jugada a jugada (`verify_openings`). Cada partida empieza en
  la posicion INICIAL y el controlador aplica la apertura con `env.apply`, asi el
  PGN y la historia (repeticiones) quedan completos. TODOS los emparejamientos
  usan las MISMAS aperturas (`TournamentConfig.opening_ids`).
* COLORES. Cada par de agentes juega cada apertura dos veces, intercambiando
  colores: 15 pares x 8 aperturas x 2 colores = 240 partidas con la configuracion
  por defecto (el minimo del enunciado es 60).
* SEMILLAS. La partida numero k (1, 2, ...) tiene semilla base
  `seed_base + seed_stride * k`; el controlador entrega a cada jugada
  `semilla base + ply` (G-04). Como ninguna partida pasa de ~180 plies y el paso
  es 1000, ninguna jugada de dos partidas distintas comparte semilla.
* PRESUPUESTO. `time_limit_s` por decision (0.2 s por defecto, X-03). Minimax y
  Alfa-beta son de profundidad fija (ignoran el plazo); Alfa-beta mejorado y MCTS
  lo usan. Politica UNIFORME de excesos (G-05): el controlador mide el tiempo real
  de `choose`; si supera `time_limit_s + tolerance_s` la jugada SE APLICA igual y
  se marca `time_exceeded = 1` en el CSV. Se reporta la proporcion por agente.
* LIMITE. `max_plies` plies jugados por los AGENTES (despues de la apertura). Al
  llegar al limite sin final reglamentario la partida es TRUNCADA: resultado
  oficial `*`, fuera de la puntuacion. NUNCA se adjudica por evaluacion estatica;
  el balance de material final se guarda solo como dato descriptivo.
* PUNTOS. Victoria 1, tablas 0.5, derrota 0, solo con partidas terminadas.

Ejecucion en paralelo: `workers > 1` usa `multiprocessing` (procesos). Solo
funciona importando el paquete (por ejemplo desde `tools/correr_experimentos.py`);
en el cuaderno el codigo vive en `__main__` y se juega en serie.
"""
from __future__ import annotations

import csv
import datetime
import json
import math
import os
import platform
import random
import statistics
import sys
import time
from dataclasses import asdict, dataclass
from itertools import combinations
from typing import Optional

import chess
import chess.pgn


# --------------------------------------------------------------------- aperturas
# Lineas teoricas principales (ECO), todas terminan con BLANCAS al turno.
OPENINGS = [
    {"id": "A1_italiana", "name": "Italiana (Giuoco Piano)", "eco": "C50",
     "moves": "e2e4 e7e5 g1f3 b8c6 f1c4 f8c5"},
    {"id": "A2_gambito_dama_rehusado", "name": "Gambito de dama rehusado", "eco": "D35",
     "moves": "d2d4 d7d5 c2c4 e7e6 b1c3 g8f6"},
    {"id": "A3_siciliana_najdorf", "name": "Siciliana Najdorf", "eco": "B90",
     "moves": "e2e4 c7c5 g1f3 d7d6 d2d4 c5d4 f3d4 g8f6 b1c3 a7a6"},
    {"id": "A4_inglesa", "name": "Inglesa (cuatro caballos invertida)", "eco": "A28",
     "moves": "c2c4 e7e5 b1c3 g8f6 g1f3 b8c6"},
    {"id": "A5_espanola", "name": "Espanola (Ruy Lopez)", "eco": "C78",
     "moves": "e2e4 e7e5 g1f3 b8c6 f1b5 a7a6 b5a4 g8f6"},
    {"id": "A6_francesa", "name": "Francesa clasica", "eco": "C11",
     "moves": "e2e4 e7e6 d2d4 d7d5 b1c3 g8f6"},
    {"id": "A7_caro_kann", "name": "Caro-Kann clasica", "eco": "B18",
     "moves": "e2e4 c7c6 d2d4 d7d5 b1c3 d5e4 c3e4 c8f5"},
    {"id": "A8_eslava", "name": "Eslava", "eco": "D11",
     "moves": "d2d4 d7d5 c2c4 c7c6 g1f3 g8f6"},
    {"id": "A9_india_rey", "name": "India de rey", "eco": "E90",
     "moves": "d2d4 g8f6 c2c4 g7g6 b1c3 f8g7 e2e4 d7d6"},
    {"id": "A10_nimzoindia", "name": "Nimzoindia", "eco": "E20",
     "moves": "d2d4 g8f6 c2c4 e7e6 b1c3 f8b4"},
    {"id": "A11_escocesa", "name": "Escocesa", "eco": "C45",
     "moves": "e2e4 e7e5 g1f3 b8c6 d2d4 e5d4 f3d4 g8f6"},
    {"id": "A12_londres", "name": "Sistema Londres", "eco": "D02",
     "moves": "d2d4 d7d5 g1f3 g8f6 c1f4 e7e6"},
    {"id": "A13_india_dama", "name": "India de dama", "eco": "E12",
     "moves": "d2d4 g8f6 c2c4 e7e6 g1f3 b7b6"},
    {"id": "A14_escandinava", "name": "Escandinava", "eco": "B01",
     "moves": "e2e4 d7d5 e4d5 d8d5 b1c3 d5a5"},
    {"id": "A15_pirc", "name": "Pirc", "eco": "B07",
     "moves": "e2e4 d7d6 d2d4 g8f6 b1c3 g7g6"},
    {"id": "A16_inglesa_simetrica", "name": "Inglesa simetrica", "eco": "A30",
     "moves": "c2c4 c7c5 g1f3 g8f6 b1c3 b8c6"},
]
OPENINGS_BY_ID = {o["id"]: o for o in OPENINGS}
# Las ocho del torneo (A1-A8): abiertas (e4 e5), cerradas (d4 d5), asimetricas
# (Siciliana, Francesa, Caro-Kann) y de flanco (Inglesa). 15 pares x 8 x 2 = 240 partidas.
TOURNAMENT_OPENING_IDS = tuple(o["id"] for o in OPENINGS[:8])

TOURNAMENT_BUDGET_S = 0.2
TOURNAMENT_MAX_PLIES = 160
TOURNAMENT_DEPTH = 2

# Columnas del CSV por partida (una fila por partida).
GAME_FIELDS = ["game_id", "opening_id", "seed", "white_agent", "black_agent",
               "opening_plies", "plies", "agent_plies", "termination", "official_result",
               "truncated", "white_points", "black_points", "winner_agent",
               "duration_s", "white_moves", "white_time_s", "white_time_max_s",
               "white_exceeded", "white_depth_mean", "white_sims_mean",
               "black_moves", "black_time_s", "black_time_max_s", "black_exceeded",
               "black_depth_mean", "black_sims_mean", "material_white_final", "status"]


def opening_moves(opening) -> list:
    """Lista de `chess.Move` de una apertura (dict de OPENINGS o id)."""
    op = OPENINGS_BY_ID[opening] if isinstance(opening, str) else opening
    return [chess.Move.from_uci(u) for u in op["moves"].split()]


def opening_board(opening) -> chess.Board:
    """Tablero (con historia) tras jugar la apertura desde la posicion inicial."""
    board = chess.Board()
    for move in opening_moves(opening):
        if move not in board.legal_moves:
            raise ValueError(f"apertura {opening}: jugada ilegal {move.uci()}")
        board.push(move)
    return board


def verify_openings(openings=None) -> list:
    """Comprueba legalidad, validez y que la partida siga abierta; devuelve filas con E0."""
    rows = []
    for op in openings or OPENINGS:
        board = opening_board(op)
        assert board.is_valid(), (op["id"], board.status())
        assert board.outcome(claim_draw=True) is None, op["id"]
        rows.append({"opening_id": op["id"], "name": op["name"], "eco": op["eco"],
                     "plies": len(board.move_stack), "turn": "blancas" if board.turn else "negras",
                     "fen": board.fen(), "E0_cp": round(DEFAULT_EVALUATOR(board), 1),
                     "legal_moves": board.legal_moves.count()})
    return rows


# --------------------------------------------------------------------- agentes
# Agentes adicionales para la ablacion (Experimento 3). Se construyen por etiqueta
# dentro de cada proceso, asi la especificacion de una partida es solo texto.
ABLATION_E0_LABEL = "AB mejorado E0"
ABLATION_ESTAR_LABEL = "AB mejorado E*"
TOURNAMENT_EXTRA_FACTORIES = {
    ABLATION_E0_LABEL: lambda cfg: AlphaBetaImprovedAgent(evaluator=DEFAULT_EVALUATOR,
                                                          name=ABLATION_E0_LABEL),
    ABLATION_ESTAR_LABEL: lambda cfg: AlphaBetaImprovedAgent(evaluator=load_tuned_evaluator(),
                                                             name=ABLATION_ESTAR_LABEL),
}


def tournament_make_agent(label: str, cfg: AgentConfig):
    """Agente nuevo por etiqueta: registro de los seis o agentes extra de la ablacion."""
    if label in AGENT_FACTORIES:
        return AGENT_FACTORIES[label](cfg)
    if label in TOURNAMENT_EXTRA_FACTORIES:
        return TOURNAMENT_EXTRA_FACTORIES[label](cfg)
    raise KeyError(f"agente desconocido: {label!r}")


def describe_agent(label: str, cfg: AgentConfig) -> dict:
    """Parametros efectivos de un agente (van a torneo_config.json)."""
    agent = tournament_make_agent(label, cfg)
    out = {"label": label, "class": type(agent).__name__}
    for attr in ("depth", "max_depth", "ordering", "quiescence", "simulations", "exploration",
                 "rollout_limit", "rollout_policy", "capture_bias"):
        if hasattr(agent, attr):
            value = getattr(agent, attr)
            out[attr] = round(value, 6) if isinstance(value, float) else value
    ev = getattr(agent, "evaluator", None)
    if ev is not None:
        out["evaluator"] = {"name": getattr(ev, "name", str(ev)),
                            "weights": list(getattr(ev, "weights", ()))}
    if isinstance(agent, MCTSAgent):
        out["budget"] = ("simulaciones fijas" if agent.simulations is not None
                         else "plazo temporal (time_limit_s)")
    elif hasattr(agent, "max_depth"):
        out["budget"] = "plazo temporal (profundizacion iterativa)"
    elif hasattr(agent, "depth"):
        out["budget"] = "profundidad fija (ignora el plazo; exceso registrado)"
    else:
        out["budget"] = "un ply / sin busqueda"
    return out


# --------------------------------------------------------------------- configuracion
@dataclass
class TournamentConfig:
    """Todo lo que define el torneo. Se guarda completo en el JSON de configuracion."""
    agents: tuple = AGENT_NAMES
    opening_ids: tuple = TOURNAMENT_OPENING_IDS
    time_limit_s: float = TOURNAMENT_BUDGET_S
    max_plies: int = TOURNAMENT_MAX_PLIES          # plies de los agentes, sin contar la apertura
    depth: int = TOURNAMENT_DEPTH                  # Minimax y Alfa-beta
    mcts_simulations: Optional[int] = None         # None: MCTS usa el plazo
    tolerance_s: float = DEFAULT_TIME_TOLERANCE_S
    seed_base: int = 0
    seed_stride: int = 1000
    event: str = "Taller 3 IA UTP - torneo (Experimento 2)"
    game_prefix: str = "T"

    def agent_config(self, seed: int) -> AgentConfig:
        return AgentConfig(depth=self.depth, time_limit_s=self.time_limit_s, seed=seed,
                           simulations=self.mcts_simulations)


def round_robin_schedule(cfg: TournamentConfig) -> list:
    """Todos contra todos: por par y por apertura, una partida con cada color.

    Devuelve dicts {game_id, index, opening_id, seed, white, black}. El orden es
    determinista (pares en el orden de `cfg.agents`, luego aperturas, luego colores).
    """
    games = []
    for a, b in combinations(cfg.agents, 2):
        for op in cfg.opening_ids:
            for white, black in ((a, b), (b, a)):
                games.append({"white": white, "black": black, "opening_id": op})
    return _tour_number(games, cfg)


def paired_schedule(cfg: TournamentConfig, pairs) -> list:
    """Como `round_robin_schedule` pero solo para los pares dados (ablacion)."""
    games = []
    for a, b in pairs:
        for op in cfg.opening_ids:
            for white, black in ((a, b), (b, a)):
                games.append({"white": white, "black": black, "opening_id": op})
    return _tour_number(games, cfg)


def _tour_number(games: list, cfg: TournamentConfig) -> list:
    width = max(3, len(str(len(games))))
    for k, g in enumerate(games, start=1):
        g["index"] = k
        g["game_id"] = f"{cfg.game_prefix}{k:0{width}d}"
        g["seed"] = cfg.seed_base + cfg.seed_stride * k
    return games


# --------------------------------------------------------------------- una partida
def _tour_material_white(board: chess.Board) -> float:
    total = 0.0
    for pt, val in PIECE_VALUES.items():
        total += val * (chess.popcount(board.pieces_mask(pt, chess.WHITE))
                        - chess.popcount(board.pieces_mask(pt, chess.BLACK)))
    return round(total, 1)


def _tour_side_stats(records, color: str) -> dict:
    sel = [r for r in records if r.color == color]
    times = [r.elapsed_s for r in sel]
    return {"moves": len(sel), "time_s": round(sum(times), 4),
            "time_max_s": round(max(times), 4) if times else 0.0,
            "exceeded": sum(1 for r in sel if r.time_exceeded),
            "depth_mean": round(statistics.mean(r.completed_depth for r in sel), 3) if sel else 0.0,
            "sims_mean": round(statistics.mean(r.simulations for r in sel), 1) if sel else 0.0}


def play_game(spec: dict, cfg: TournamentConfig, workers: int = 1) -> dict:
    """Juega UNA partida con `GameController` y devuelve sus filas y su PGN.

    Resultado: {"game": fila de GAME_FIELDS, "decisions": filas con CSV_FIELDS,
    "pgn": texto PGN}. El estado de la partida (terminal/truncada) lo decide el
    entorno; si un agente devolviera algo invalido la partida queda con estado
    'error' y sin resultado (no ocurre en las corridas guardadas).
    """
    t0 = time.perf_counter()
    op = OPENINGS_BY_ID[spec["opening_id"]]
    moves = opening_moves(op)
    env = ChessEnvironment(max_plies=len(moves) + cfg.max_plies)
    for move in moves:
        env.apply(move)
    agent_cfg = cfg.agent_config(spec["seed"])
    white = tournament_make_agent(spec["white"], agent_cfg)
    black = tournament_make_agent(spec["black"], agent_cfg)
    ctl = GameController(env, white, black, agent_cfg, game_id=spec["game_id"],
                         opening_id=op["id"], tolerance_s=cfg.tolerance_s)
    status = ctl.play_until_stop()
    duration = time.perf_counter() - t0

    termination = env.termination() if status in ("terminal", "truncated") else "error"
    result = env.official_result() if status == "terminal" else "*"
    truncated = int(status == "truncated")
    wp = {"1-0": 1.0, "0-1": 0.0, "1/2-1/2": 0.5}.get(result)
    winner = ""
    if result == "1-0":
        winner = spec["white"]
    elif result == "0-1":
        winner = spec["black"]
    ws, bs = _tour_side_stats(ctl.records, "white"), _tour_side_stats(ctl.records, "black")
    game = {
        "game_id": spec["game_id"], "opening_id": op["id"], "seed": spec["seed"],
        "white_agent": spec["white"], "black_agent": spec["black"],
        "opening_plies": len(moves), "plies": env.ply_count,
        "agent_plies": env.ply_count - len(moves), "termination": termination,
        "official_result": result, "truncated": truncated,
        "white_points": "" if wp is None else wp, "black_points": "" if wp is None else 1.0 - wp,
        "winner_agent": winner, "duration_s": round(duration, 3),
        **{f"white_{k}": v for k, v in ws.items()}, **{f"black_{k}": v for k, v in bs.items()},
        "material_white_final": _tour_material_white(env.board), "status": status,
    }
    decisions = [rec.csv_row(termination, result) for rec in ctl.records]

    headers = ctl.pgn_headers()
    headers.update({
        "Event": cfg.event, "Site": f"{platform.system()} {platform.release()}",
        "Round": spec["index"], "Opening": op["name"], "OpeningId": op["id"], "ECO": op["eco"],
        "OpeningMoves": op["moves"], "OpeningPlies": len(moves), "SeedBase": spec["seed"],
        "MaxAgentPlies": cfg.max_plies, "Workers": workers,
        "ExcessPolicy": "se aplica la jugada y se registra time_exceeded",
        "TruncationPolicy": "limite de plies -> truncada, resultado oficial *",
        "Mode": "torneo agente-agente",
    })
    game_pgn = env.to_pgn(headers)
    node, k = game_pgn, 0
    by_ply = {r.ply: r for r in ctl.records}
    while node.variations:
        node = node.variation(0)
        k += 1
        if k == len(moves):
            node.comment = "fin de la apertura; desde aqui juegan los agentes"
        rec = by_ply.get(k)
        if rec is not None:
            info = (f"d={rec.completed_depth}" if rec.simulations == 0
                    else f"sims={rec.simulations}")
            flag = " EXCESO" if rec.time_exceeded else ""
            node.comment = f"[%emt {rec.elapsed_s:.3f}] {info}{flag}"
    pgn_text = str(game_pgn) + "\n"
    return {"game": game, "decisions": decisions, "pgn": pgn_text}


def _tour_worker(args):
    """Funcion de los procesos del pool (debe vivir a nivel de modulo para Windows)."""
    spec, cfg, workers = args
    return spec["index"], play_game(spec, cfg, workers)


# --------------------------------------------------------------------- torneo completo
def run_games(schedule: list, cfg: TournamentConfig, workers: int = 1, progress=None) -> list:
    """Juega las partidas del calendario (en serie o con `workers` procesos), en orden."""
    jobs = [(spec, cfg, workers) for spec in schedule]
    results = {}
    done = 0
    t0 = time.perf_counter()

    def _report(idx, res):
        nonlocal done
        done += 1
        if progress is not None:
            g = res["game"]
            progress(f"[{done:>3}/{len(jobs)}] {g['game_id']} {g['white_agent']} - "
                     f"{g['black_agent']} ({g['opening_id']}): {g['termination']} "
                     f"{g['official_result']} en {g['agent_plies']} plies, {g['duration_s']:.1f} s "
                     f"(acumulado {time.perf_counter() - t0:.0f} s)")

    use_pool = workers > 1 and _tour_worker.__module__ != "__main__"
    if workers > 1 and not use_pool and progress is not None:
        progress("Aviso: el codigo vive en __main__ (cuaderno); se juega en serie.")
    if use_pool:
        import multiprocessing
        with multiprocessing.get_context("spawn").Pool(workers) as pool:
            for idx, res in pool.imap_unordered(_tour_worker, jobs, chunksize=1):
                results[idx] = res
                _report(idx, res)
    else:
        for job in jobs:
            idx, res = _tour_worker(job)
            results[idx] = res
            _report(idx, res)
    return [results[spec["index"]] for spec in schedule]


def _tour_write_csv(path: str, fields, rows) -> None:
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(fields), lineterminator="\n")
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row.get(k, "") for k in fields})


def environment_info() -> dict:
    """Versiones y hardware (la reproducibilidad incluye el entorno, no solo semillas)."""
    import importlib.metadata as md
    info = {"python": sys.version.split()[0], "python_chess": chess.__version__,
            "platform": platform.platform(), "processor": platform.processor(),
            "machine": platform.machine(), "logical_cpus": os.cpu_count()}
    for pkg in ("numpy", "matplotlib"):
        try:
            info[pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            info[pkg] = None
    return info


def run_tournament(cfg: TournamentConfig | None = None, *, schedule: list | None = None,
                   workers: int = 1, out_dir: str | None = None, prefix: str = "torneo",
                   progress=print, extra_config: dict | None = None) -> dict:
    """Juega el torneo y, si `out_dir`, escribe `<prefix>_decisiones.csv`,
    `<prefix>_partidas.csv`, `<prefix>.pgn` y `<prefix>_config.json`.

    Devuelve {"games", "decisions", "pgn", "config"}.
    """
    cfg = cfg or TournamentConfig()
    schedule = schedule if schedule is not None else round_robin_schedule(cfg)
    started = datetime.datetime.now().isoformat(timespec="seconds")
    t0, c0 = time.perf_counter(), time.process_time()
    results = run_games(schedule, cfg, workers=workers, progress=progress)
    wall = time.perf_counter() - t0
    games = [r["game"] for r in results]
    decisions = [row for r in results for row in r["decisions"]]
    labels = sorted({s["white"] for s in schedule} | {s["black"] for s in schedule},
                    key=lambda x: (AGENT_NAMES.index(x) if x in AGENT_NAMES else 99, x))
    config = {
        "description": cfg.event,
        "started": started, "finished": datetime.datetime.now().isoformat(timespec="seconds"),
        "duration_wall_s": round(wall, 1),
        "cpu_main_process_s": round(time.process_time() - c0, 1),
        "workers": workers,
        "n_games": len(games),
        "tournament": {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cfg).items()},
        "policies": {
            "points": "victoria 1, tablas 0.5, derrota 0; solo partidas terminadas",
            "truncation": (f"{cfg.max_plies} plies de los agentes tras la apertura -> truncada, "
                           "resultado oficial '*', fuera de los puntos; sin adjudicacion por E"),
            "time_excess": (f"tiempo real de choose > {cfg.time_limit_s} + {cfg.tolerance_s} s -> "
                            "time_exceeded=1; la jugada se aplica igual (politica uniforme)"),
            "seeds": f"partida k: semilla base = {cfg.seed_base} + {cfg.seed_stride}*k; jugada: base + ply",
            "draws": "reclamo automatico (board.outcome(claim_draw=True))",
            "openings": "mismas aperturas para todos los pares; cada apertura con ambos colores",
        },
        "agents": {lab: describe_agent(lab, cfg.agent_config(0)) for lab in labels},
        "openings": [dict(OPENINGS_BY_ID[o], fen=opening_board(o).fen()) for o in cfg.opening_ids],
        "games_seeds": {g["game_id"]: g["seed"] for g in schedule},
        "environment": environment_info(),
    }
    if extra_config:
        config.update(extra_config)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        _tour_write_csv(os.path.join(out_dir, f"{prefix}_decisiones.csv"), CSV_FIELDS, decisions)
        _tour_write_csv(os.path.join(out_dir, f"{prefix}_partidas.csv"), GAME_FIELDS, games)
        with open(os.path.join(out_dir, f"{prefix}.pgn"), "w", encoding="utf-8", newline="\n") as fh:
            fh.write("\n".join(r["pgn"] for r in results))
        with open(os.path.join(out_dir, f"{prefix}_config.json"), "w", encoding="utf-8") as fh:
            json.dump(config, fh, ensure_ascii=False, indent=2)
    return {"games": games, "decisions": decisions, "pgn": [r["pgn"] for r in results],
            "config": config}


# --------------------------------------------------------------------- lectura y tablas
_TOUR_INT_FIELDS = {"seed", "ply", "nodes", "leaves", "cutoffs", "completed_depth", "simulations",
                    "time_exceeded", "opening_plies", "plies", "agent_plies", "truncated",
                    "white_moves", "black_moves", "white_exceeded", "black_exceeded"}
_TOUR_FLOAT_FIELDS = {"elapsed_s", "duration_s", "white_time_s", "black_time_s",
                      "white_time_max_s", "black_time_max_s", "white_depth_mean",
                      "black_depth_mean", "white_sims_mean", "black_sims_mean",
                      "material_white_final", "white_points", "black_points"}


def read_typed_csv(path: str) -> list:
    """Lee un CSV del torneo convirtiendo las columnas numericas conocidas."""
    with open(path, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    for row in rows:
        for k, v in row.items():
            if v == "" or v is None:
                continue
            if k in _TOUR_INT_FIELDS:
                row[k] = int(float(v))
            elif k in _TOUR_FLOAT_FIELDS:
                row[k] = float(v)
    return rows


def load_tournament(out_dir: str, prefix: str = "torneo") -> dict:
    """Carga lo guardado por `run_tournament` (sin volver a jugar)."""
    out = {"games": read_typed_csv(os.path.join(out_dir, f"{prefix}_partidas.csv")),
           "decisions": read_typed_csv(os.path.join(out_dir, f"{prefix}_decisiones.csv"))}
    cfg_path = os.path.join(out_dir, f"{prefix}_config.json")
    out["config"] = json.load(open(cfg_path, encoding="utf-8")) if os.path.exists(cfg_path) else {}
    return out


def decision_mover(row: dict) -> str:
    """Agente que hizo la jugada de una fila de decisiones (ply impar = blancas).

    Vale porque todas las partidas empiezan en la posicion inicial (blancas mueven en
    el ply 1) y la apertura se cuenta en `ply`.
    """
    return row["white_agent"] if int(row["ply"]) % 2 == 1 else row["black_agent"]


def standings(games: list, agents=None) -> list:
    """Tabla del enunciado por agente (solo terminadas puntuan; truncadas aparte).

    Columnas: agente, partidas, terminadas, puntos, victorias, tablas, derrotas,
    truncadas, errores, puntos_por_terminada, cobertura (terminadas / partidas),
    jugadas, tiempo_medio_s (por decision), tiempo_max_s, excesos y prop_excesos.
    Ordenada por puntos por partida terminada (clasificacion PROVISIONAL: depende
    de la cobertura) y despues por puntos.
    """
    labels = list(agents) if agents is not None else sorted(
        {g["white_agent"] for g in games} | {g["black_agent"] for g in games},
        key=lambda x: (AGENT_NAMES.index(x) if x in AGENT_NAMES else 99, x))
    table = []
    for lab in labels:
        row = {"agente": lab, "partidas": 0, "terminadas": 0, "puntos": 0.0, "victorias": 0,
               "tablas": 0, "derrotas": 0, "truncadas": 0, "errores": 0, "jugadas": 0,
               "tiempo_total_s": 0.0, "tiempo_max_s": 0.0, "excesos": 0}
        for g in games:
            for side in ("white", "black"):
                if g[f"{side}_agent"] != lab:
                    continue
                row["partidas"] += 1
                row["jugadas"] += int(g[f"{side}_moves"])
                row["tiempo_total_s"] += float(g[f"{side}_time_s"])
                row["tiempo_max_s"] = max(row["tiempo_max_s"], float(g[f"{side}_time_max_s"]))
                row["excesos"] += int(g[f"{side}_exceeded"])
                if int(g["truncated"]):
                    row["truncadas"] += 1
                    continue
                if g["official_result"] not in ("1-0", "0-1", "1/2-1/2"):
                    row["errores"] += 1
                    continue
                pts = float(g[f"{side}_points"])
                row["terminadas"] += 1
                row["puntos"] += pts
                if pts == 1.0:
                    row["victorias"] += 1
                elif pts == 0.5:
                    row["tablas"] += 1
                else:
                    row["derrotas"] += 1
        n, fin, mv = row["partidas"], row["terminadas"], row["jugadas"]
        row["puntos_por_terminada"] = round(row["puntos"] / fin, 3) if fin else None
        row["cobertura"] = round(fin / n, 3) if n else None
        row["tiempo_medio_s"] = round(row["tiempo_total_s"] / mv, 4) if mv else None
        row["prop_excesos"] = round(row["excesos"] / mv, 4) if mv else None
        row["tiempo_total_s"] = round(row["tiempo_total_s"], 2)
        table.append(row)
    table.sort(key=lambda r: (-(r["puntos_por_terminada"] or 0.0), -r["puntos"]))
    return table


def standings_markdown(table: list) -> str:
    """La tabla del enunciado en Markdown (mas cobertura y excesos)."""
    lines = ["| Agente | Puntos / partidas terminadas | Victorias | Tablas | Derrotas | Truncadas "
             "| Tiempo medio | Cobertura | Excesos |",
             "|---|---|---|---|---|---|---|---|---|"]
    for r in table:
        ppt = f" ({r['puntos_por_terminada']:.2f})" if r["puntos_por_terminada"] is not None else ""
        lines.append(
            f"| {r['agente']} | {r['puntos']:g} / {r['terminadas']}{ppt} | {r['victorias']} | "
            f"{r['tablas']} | {r['derrotas']} | {r['truncadas']} | "
            f"{r['tiempo_medio_s']:.3f} s | {r['terminadas']}/{r['partidas']} | "
            f"{100 * (r['prop_excesos'] or 0):.1f} % |")
    return "\n".join(lines)


def pairwise_table(games: list, agents=None) -> dict:
    """{(A, B): {"puntos_A", "terminadas", "truncadas", "partidas"}} desde A."""
    labels = list(agents) if agents is not None else sorted(
        {g["white_agent"] for g in games} | {g["black_agent"] for g in games})
    out = {}
    for a in labels:
        for b in labels:
            if a == b:
                continue
            cell = {"puntos_A": 0.0, "terminadas": 0, "truncadas": 0, "partidas": 0}
            for g in games:
                if {g["white_agent"], g["black_agent"]} != {a, b}:
                    continue
                cell["partidas"] += 1
                if int(g["truncated"]) or g["official_result"] == "*":
                    cell["truncadas"] += 1
                    continue
                side = "white" if g["white_agent"] == a else "black"
                cell["terminadas"] += 1
                cell["puntos_A"] += float(g[f"{side}_points"])
            out[(a, b)] = cell
    return out


def score_interval(scores, level: float = 0.95, n_boot: int = 5000, seed: int = 0) -> dict:
    """Media de puntos (1/0.5/0) con intervalo bootstrap percentil reproducible.

    Con pocas partidas el intervalo es ancho: es la incertidumbre honesta del
    resultado. Tambien devuelve el error estandar normal.
    """
    scores = [float(s) for s in scores]
    n = len(scores)
    if n == 0:
        return {"n": 0, "mean": None, "low": None, "high": None, "se": None}
    mean = sum(scores) / n
    se = statistics.stdev(scores) / math.sqrt(n) if n > 1 else float("nan")
    rng = random.Random(seed)
    boots = sorted(sum(rng.choice(scores) for _ in range(n)) / n for _ in range(n_boot))
    lo = boots[int((1 - level) / 2 * n_boot)]
    hi = boots[min(n_boot - 1, int((1 + level) / 2 * n_boot))]
    return {"n": n, "mean": round(mean, 4), "low": round(lo, 4), "high": round(hi, 4),
            "se": round(se, 4) if n > 1 else None}


# --------------------------------------------------------------------- carga en paralelo
def _tour_probe(args):
    """Profundidad completada del mejorado y simulaciones de MCTS en unas posiciones."""
    fens, time_limit_s = args
    out = []
    for fen in fens:
        board = chess.Board(fen)
        d = AlphaBetaImprovedAgent().choose(board.copy(stack=True), time_limit_s=time_limit_s)
        m = MCTSAgent().choose(board.copy(stack=True), time_limit_s=time_limit_s, seed=1)
        out.append((d.stats.completed_depth, d.stats.nodes, m.stats.simulations))
    return out


def parallel_load_check(workers: int = 4, fens=None, time_limit_s: float = TOURNAMENT_BUDGET_S,
                        repeats: int = 2) -> dict:
    """Compara en serie vs con `workers` procesos simultaneos (carga del torneo).

    Mide, en las mismas posiciones, la profundidad media completada por el
    mejorado, sus nodos y las simulaciones de MCTS por decision. Si caen poco, el
    paralelismo no cambia de forma grave la fuerza de los agentes por plazo.
    """
    fens = fens or [opening_board(o).fen() for o in OPENINGS[:8]]
    serial = []
    for _ in range(repeats):
        serial.extend(_tour_probe((fens, time_limit_s)))
    parallel = []
    if workers > 1:
        import multiprocessing
        with multiprocessing.get_context("spawn").Pool(workers) as pool:
            for chunk in pool.map(_tour_probe, [(fens, time_limit_s)] * (workers * repeats)):
                parallel.extend(chunk)

    def _summ(rows):
        if not rows:
            return {}
        return {"n": len(rows), "depth_mean": round(statistics.mean(r[0] for r in rows), 3),
                "nodes_mean": round(statistics.mean(r[1] for r in rows), 1),
                "sims_mean": round(statistics.mean(r[2] for r in rows), 1),
                "depth_dist": {d: sum(1 for r in rows if r[0] == d)
                               for d in sorted({r[0] for r in rows})}}

    return {"workers": workers, "time_limit_s": time_limit_s, "positions": len(fens),
            "serial": _summ(serial), "parallel": _summ(parallel)}


In [ ]:
# ===== chess_ai/experiments.py =====
"""Experimentos de la seccion 9: ahorro de busqueda, torneo, ablacion, MCTS y graficas.

Las funciones `run_*` ejecutan (las pesadas se corren con tools/correr_experimentos.py);
las funciones `*_load` / `*_summary` / `fig_*` leen los CSV guardados en `resultados/`
para que el cuaderno muestre los resultados SIN volver a jugar el torneo.

Advertencia de definiciones: «nodos» de alfa-beta = llamadas recursivas sobre una
posicion (S-02); «nodos» de MCTS = nodos creados en el arbol (uno por simulacion
como maximo). No son comparables entre si; solo se comparan dentro de una familia.
"""
from __future__ import annotations

import csv
import os
import statistics

import chess


EXP_RESULTS_DIR = os.path.join(_tune_project_root(), "resultados")
# Paleta categorica fija (orden estable: el color sigue a la entidad).
EXP_COLORS = ("#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948")
EXP_INK, EXP_MUTED = "#0b0b0b", "#52514e"

# --------------------------------------------------------------------- Experimento 1
EXP1_POSITIONS = [
    {"id": "P1_captura", "character": "captura",
     "description": "Escocesa tras 3.d4: negras al turno con exd4 y Cxd4 disponibles",
     "fen": "r1bqkbnr/pppp1ppp/2n5/4p3/3PP3/5N2/PPP2PPP/RNBQKB1R b KQkq - 0 3"},
    {"id": "P2_jaque", "character": "jaque",
     "description": "Centro 1.e4 e5 2.d4 exd4 3.Dxd4 Cc6 4.De3 Ab4+: blancas en jaque",
     "fen": "r1bqk1nr/pppp1ppp/2n5/8/1b2P3/4Q3/PPP2PPP/RNB1KBNR w KQkq - 3 5"},
    {"id": "P3_defensa", "character": "defensa",
     "description": "Caballo d4 atacado por e5 y cebo en h5 (MINIMAX_DEFENSE_FEN, pregunta 1)",
     "fen": MINIMAX_DEFENSE_FEN},
    {"id": "P4_tranquila", "character": "tranquila",
     "description": "Sistema Londres tras 6...Ad6: desarrollo sin amenazas inmediatas",
     "fen": "r1bqk2r/pp3ppp/2nbpn2/2pp4/3P1B2/2P1PN2/PP1N1PPP/R2QKB1R w KQkq - 3 7"},
    {"id": "P5_final", "character": "final",
     "description": "Final de torres con peones (3 contra 3)",
     "fen": "8/5pk1/6p1/p7/P2R4/6P1/2r2PK1/8 w - - 0 40"},
]
EXP1_ALGORITHMS = (("Minimax", "minimax", False), ("Alfa-beta natural", "alphabeta", False),
                   ("Alfa-beta ordenado", "alphabeta", True))
EXP1_FIELDS = ["position_id", "character", "turn", "depth", "algorithm", "value", "move",
               "nodes", "leaves", "cutoffs", "time_s", "sort_time_s", "search_time_s",
               "nodes_vs_minimax", "time_vs_minimax"]


def _exp_write_csv(path, rows, fields=None) -> None:
    if not path or not rows:
        return
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    fields = fields or list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields, lineterminator="\n")
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in fields})


def run_exp1(depths=(1, 2, 3), positions=None, evaluator=DEFAULT_EVALUATOR,
             out_path: str | None = None) -> list:
    """Minimax / alfa-beta natural / alfa-beta ordenado con E fija a cada profundidad.

    Comprueba (assert) que los tres valores coinciden y que el tablero queda intacto.
    `time_s` incluye el tiempo de ordenar; `search_time_s` = time_s - sort_time_s.
    """
    rows = []
    for pos in positions or EXP1_POSITIONS:
        board = chess.Board(pos["fen"])
        assert board.is_valid(), (pos["id"], board.status())
        fen0 = board.fen()
        for depth in depths:
            base = None
            for label, algo, ordering in EXP1_ALGORITHMS:
                r = search_root(board, algo, depth, evaluator, ordering)
                assert board.fen() == fen0 and not board.move_stack, "tablero modificado"
                if base is None:
                    base = r
                assert abs(r.value - base.value) < 1e-6, (pos["id"], depth, label, r.value, base.value)
                rows.append({
                    "position_id": pos["id"], "character": pos["character"],
                    "turn": "blancas" if board.turn else "negras", "depth": depth,
                    "algorithm": label, "value": round(r.value, 2),
                    "move": r.move.uci() if r.move else "", "nodes": r.stats.nodes,
                    "leaves": r.stats.leaves, "cutoffs": r.stats.cutoffs,
                    "time_s": round(r.stats.elapsed_s, 5), "sort_time_s": round(r.sort_time, 5),
                    "search_time_s": round(r.stats.elapsed_s - r.sort_time, 5),
                    "nodes_vs_minimax": round(r.stats.nodes / base.stats.nodes, 4),
                    "time_vs_minimax": round(r.stats.elapsed_s / base.stats.elapsed_s, 4)
                    if base.stats.elapsed_s > 0 else ""})
    _exp_write_csv(out_path, rows, EXP1_FIELDS)
    return rows


def exp1_load(path: str | None = None) -> list:
    path = path or os.path.join(EXP_RESULTS_DIR, "exp1_ahorro.csv")
    rows = read_typed_csv(path)
    for r in rows:
        for k in ("depth", "nodes", "leaves", "cutoffs"):
            r[k] = int(r[k])
        for k in ("value", "time_s", "sort_time_s", "search_time_s", "nodes_vs_minimax"):
            r[k] = float(r[k])
    return rows


def exp1_summary(rows) -> list:
    """Totales sobre las 5 posiciones por profundidad y algoritmo (+ razon frente a minimax)."""
    out = []
    for depth in sorted({r["depth"] for r in rows}):
        base_nodes = sum(r["nodes"] for r in rows if r["depth"] == depth and r["algorithm"] == "Minimax")
        base_time = sum(r["time_s"] for r in rows if r["depth"] == depth and r["algorithm"] == "Minimax")
        for label, _, _ in EXP1_ALGORITHMS:
            sel = [r for r in rows if r["depth"] == depth and r["algorithm"] == label]
            if not sel:
                continue
            nodes = sum(r["nodes"] for r in sel)
            t = sum(r["time_s"] for r in sel)
            out.append({"depth": depth, "algorithm": label, "nodes": nodes,
                        "leaves": sum(r["leaves"] for r in sel),
                        "cutoffs": sum(r["cutoffs"] for r in sel), "time_s": round(t, 4),
                        "sort_time_s": round(sum(r["sort_time_s"] for r in sel), 4),
                        "nodes_vs_minimax": round(nodes / base_nodes, 4) if base_nodes else None,
                        "time_vs_minimax": round(t / base_time, 4) if base_time else None})
    return out


# ---------------------------------------------------------- profundidad contra plazo
def _exp_iterative_depth(board: chess.Board, time_limit_s: float, ordering: bool,
                         evaluator=DEFAULT_EVALUATOR, max_depth: int = 8) -> tuple:
    """Profundizacion iterativa identica a la del mejorado, con o sin ordenacion.

    Con `ordering=False` no hay PV previa, MVV-LVA ni killers (orden natural): es la
    ablacion «ordenacion si/no» con plazo. Devuelve (profundidad completada, nodos).
    """
    deadline = Deadline(time_limit_s)
    ctx = SearchContext(evaluator, deadline=deadline, ordering=ordering)
    completed = 0
    for depth in range(1, max_depth + 1):
        if deadline.expired():
            break
        try:
            value = _search_alphabeta(board, depth, 0, -float("inf"), float("inf"), ctx)
        except SearchTimeout:
            break
        completed = depth
        ctx.prev_pv = list(ctx.pv[0])
        if is_mate_score(value):
            break
    return completed, ctx.stats.nodes


def depth_time_positions() -> list:
    """Posiciones de la curva profundidad-plazo: aperturas del torneo + las del Exp. 1."""
    out = [(o, opening_board(o).fen()) for o in TOURNAMENT_OPENING_IDS[:4]]
    out += [(p["id"], p["fen"]) for p in EXP1_POSITIONS]
    return out


def run_depth_vs_time(budgets=(0.05, 0.1, 0.2, 0.5, 1.0), positions=None,
                      out_path: str | None = None, n_positions: int | None = None) -> list:
    """Profundidad completada por el mejorado (con ordenacion) y por la misma
    profundizacion sin ordenacion, para varios plazos y posiciones."""
    positions = positions or depth_time_positions()
    if n_positions:
        positions = positions[:n_positions]
    rows = []
    for pid, fen in positions:
        for budget in budgets:
            board = chess.Board(fen)
            d = AlphaBetaImprovedAgent().choose(board, time_limit_s=budget)
            rows.append({"position_id": pid, "budget_s": budget, "ordering": 1,
                         "completed_depth": d.stats.completed_depth, "nodes": d.stats.nodes,
                         "elapsed_s": round(d.stats.elapsed_s, 4)})
            depth, nodes = _exp_iterative_depth(chess.Board(fen), budget, ordering=False)
            rows.append({"position_id": pid, "budget_s": budget, "ordering": 0,
                         "completed_depth": depth, "nodes": nodes, "elapsed_s": budget})
    _exp_write_csv(out_path, rows)
    return rows


def depth_time_load(path: str | None = None) -> list:
    rows = read_typed_csv(path or os.path.join(EXP_RESULTS_DIR, "exp_profundidad_tiempo.csv"))
    for r in rows:
        r["budget_s"] = float(r["budget_s"])
        for k in ("ordering", "completed_depth", "nodes"):
            r[k] = int(r[k])
    return rows


def depth_time_summary(rows) -> list:
    out = []
    for budget in sorted({r["budget_s"] for r in rows}):
        for ordering in (1, 0):
            sel = [r["completed_depth"] for r in rows
                   if r["budget_s"] == budget and r["ordering"] == ordering]
            if sel:
                out.append({"budget_s": budget, "ordering": ordering, "n": len(sel),
                            "depth_mean": round(statistics.mean(sel), 3),
                            "depth_min": min(sel), "depth_max": max(sel)})
    return out


# --------------------------------------------------------------------- Experimento 2
def tournament_analysis(data: dict) -> dict:
    """Resumen del torneo guardado: clasificacion, cruces, terminaciones, excesos,
    profundidad del mejorado y simulaciones de MCTS por jugada."""
    games, decisions = data["games"], data["decisions"]
    table = standings(games)
    terminations = {}
    for g in games:
        terminations[g["termination"]] = terminations.get(g["termination"], 0) + 1
    by_agent = {}
    for row in decisions:
        by_agent.setdefault(decision_mover(row), []).append(row)
    per_agent = {}
    for lab, rows in by_agent.items():
        times = [r["elapsed_s"] for r in rows]
        per_agent[lab] = {
            "jugadas": len(rows), "tiempo_medio_s": round(statistics.mean(times), 4),
            "tiempo_desv_s": round(statistics.pstdev(times), 4), "tiempo_max_s": round(max(times), 4),
            "excesos": sum(r["time_exceeded"] for r in rows),
            "prop_excesos": round(sum(r["time_exceeded"] for r in rows) / len(rows), 4),
            "nodos_medios": round(statistics.mean(r["nodes"] for r in rows), 1)}
    imp = [r["completed_depth"] for r in by_agent.get("Alfa-beta mejorado", [])]
    sims = [r["simulations"] for r in by_agent.get("MCTS", [])]
    out = {"standings": table, "pairwise": pairwise_table(games), "terminations": terminations,
           "n_games": len(games), "truncated": sum(int(g["truncated"]) for g in games),
           "per_agent": per_agent}
    if imp:
        out["improved_depth"] = {"mean": round(statistics.mean(imp), 3),
                                 "dist": {d: imp.count(d) for d in sorted(set(imp))}}
    if sims:
        s = sorted(sims)
        out["mcts_simulations"] = {"mean": round(statistics.mean(s), 1),
                                   "std": round(statistics.pstdev(s), 1), "min": s[0],
                                   "p10": s[len(s) // 10], "median": s[len(s) // 2],
                                   "p90": s[(9 * len(s)) // 10], "max": s[-1]}
    return out


def pairwise_markdown(pairwise: dict, agents) -> str:
    """Matriz de cruces: puntos de la fila contra la columna / terminadas (truncadas)."""
    head = "| Fila contra columna | " + " | ".join(agents) + " |"
    lines = [head, "|---" * (len(agents) + 1) + "|"]
    for a in agents:
        cells = []
        for b in agents:
            if a == b:
                cells.append("—")
                continue
            c = pairwise[(a, b)]
            cells.append(f"{c['puntos_A']:g}/{c['terminadas']} ({c['truncadas']}t)")
        lines.append(f"| {a} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


# --------------------------------------------------------------------- Experimento 3
ABLATION_OPENING_IDS = tuple(o["id"] for o in OPENINGS)     # las 16 aperturas
ABLATION_RIVAL = "Minimax"   # en el piloto empato ~50 % con el mejorado: rival informativo


def ablation_config(rapido: bool = False) -> TournamentConfig:
    if rapido:
        return TournamentConfig(agents=(ABLATION_E0_LABEL, ABLATION_ESTAR_LABEL, "Voraz"),
                                opening_ids=ABLATION_OPENING_IDS[:1], time_limit_s=0.05,
                                max_plies=16, seed_base=500_000, game_prefix="X",
                                event="Taller 3 IA UTP - ablacion RAPIDA de prueba")
    return TournamentConfig(agents=(ABLATION_E0_LABEL, ABLATION_ESTAR_LABEL, ABLATION_RIVAL),
                            opening_ids=ABLATION_OPENING_IDS, seed_base=500_000,
                            game_prefix="X", event="Taller 3 IA UTP - ablacion E0 vs E* (Experimento 3)")


def run_ablation(out_dir: str | None = None, workers: int = 1, rapido: bool = False,
                 rival: str = ABLATION_RIVAL, progress=print) -> dict:
    """E0 contra E* (mismo agente mejorado, mismo plazo, mismas aperturas, colores
    intercambiados) y ambos contra un rival comun. Escribe exp3_ablacion_*."""
    cfg = ablation_config(rapido)
    if not rapido:
        cfg.agents = (ABLATION_E0_LABEL, ABLATION_ESTAR_LABEL, rival)
    r = cfg.agents[2]
    schedule = paired_schedule(cfg, [(ABLATION_E0_LABEL, ABLATION_ESTAR_LABEL),
                                     (ABLATION_E0_LABEL, r), (ABLATION_ESTAR_LABEL, r)])
    res = run_tournament(cfg, schedule=schedule, workers=workers, out_dir=out_dir,
                         prefix="exp3_ablacion", progress=progress)
    if out_dir:
        ablation_summary(res["games"], out_path=os.path.join(out_dir, "exp3_ablacion_resumen.csv"))
    return res


def _exp_side_scores(games, label, opponent):
    """Puntos de `label` contra `opponent` en terminadas + material final en truncadas."""
    scores, material, n, trunc = [], [], 0, 0
    for g in games:
        if {g["white_agent"], g["black_agent"]} != {label, opponent}:
            continue
        n += 1
        side = "white" if g["white_agent"] == label else "black"
        if int(g["truncated"]) or g["official_result"] == "*":
            trunc += 1
            m = float(g["material_white_final"])
            material.append(m if side == "white" else -m)
            continue
        scores.append(float(g[f"{side}_points"]))
    return scores, material, n, trunc


ABLATION_SUMMARY_FIELDS = ["comparacion", "agente", "rival", "partidas", "terminadas", "truncadas",
                           "victorias", "tablas", "derrotas", "puntos", "puntuacion_media",
                           "ic95_bajo", "ic95_alto", "material_medio_truncadas_no_oficial"]


def ablation_summary(games, out_path: str | None = None) -> list:
    """Puntuacion con IC 95 % bootstrap (solo terminadas) y, aparte y rotulado como no
    oficial, el balance medio de material al truncar."""
    labels = sorted({g["white_agent"] for g in games} | {g["black_agent"] for g in games})
    rival = [x for x in labels if x not in (ABLATION_E0_LABEL, ABLATION_ESTAR_LABEL)]
    comps = [("E* contra E0 (directo)", ABLATION_ESTAR_LABEL, ABLATION_E0_LABEL)]
    for r in rival:
        comps += [(f"E0 contra {r}", ABLATION_E0_LABEL, r), (f"E* contra {r}", ABLATION_ESTAR_LABEL, r)]
    rows = []
    for name, a, b in comps:
        scores, material, n, trunc = _exp_side_scores(games, a, b)
        if n == 0:
            continue
        ci = score_interval(scores)
        rows.append({"comparacion": name, "agente": a, "rival": b, "partidas": n,
                     "terminadas": len(scores), "truncadas": trunc,
                     "victorias": scores.count(1.0), "tablas": scores.count(0.5),
                     "derrotas": scores.count(0.0), "puntos": sum(scores),
                     "puntuacion_media": ci["mean"], "ic95_bajo": ci["low"], "ic95_alto": ci["high"],
                     "material_medio_truncadas_no_oficial":
                         round(statistics.mean(material), 2) if material else ""})
    _exp_write_csv(out_path, rows, ABLATION_SUMMARY_FIELDS)
    return rows


def ablation_load(out_dir: str | None = None) -> dict:
    out_dir = out_dir or EXP_RESULTS_DIR
    data = load_tournament(out_dir, prefix="exp3_ablacion")
    path = os.path.join(out_dir, "exp3_ablacion_resumen.csv")
    data["summary"] = read_typed_csv(path) if os.path.exists(path) else ablation_summary(data["games"])
    return data


def ablation_depths(decisions) -> dict:
    """Profundidad media completada por etiqueta (E0 y E* cuestan lo mismo de evaluar)."""
    out = {}
    for row in decisions:
        out.setdefault(decision_mover(row), []).append(row["completed_depth"])
    return {k: round(statistics.mean(v), 3) for k, v in out.items()}


def ordering_ablation(exp1_rows, depth_rows) -> list:
    """Ablacion de la ordenacion: profundidad fija (Exp. 1) y con plazo (curva)."""
    out = []
    for depth in sorted({r["depth"] for r in exp1_rows}):
        nat = [r for r in exp1_rows if r["depth"] == depth and r["algorithm"] == "Alfa-beta natural"]
        ordd = [r for r in exp1_rows if r["depth"] == depth and r["algorithm"] == "Alfa-beta ordenado"]
        n_nat, n_ord = sum(r["nodes"] for r in nat), sum(r["nodes"] for r in ordd)
        t_nat, t_ord = sum(r["time_s"] for r in nat), sum(r["time_s"] for r in ordd)
        out.append({"modo": "profundidad fija", "parametro": f"d={depth}",
                    "sin_ordenacion": n_nat, "con_ordenacion": n_ord, "medida": "nodos (5 posiciones)",
                    "razon_con_sin": round(n_ord / n_nat, 4) if n_nat else None,
                    "tiempo_sin_s": round(t_nat, 4), "tiempo_con_s": round(t_ord, 4),
                    "tiempo_ordenar_s": round(sum(r["sort_time_s"] for r in ordd), 4)})
    for s in depth_time_summary(depth_rows):
        if s["ordering"] != 1:
            continue
        nat = [x for x in depth_time_summary(depth_rows)
               if x["budget_s"] == s["budget_s"] and x["ordering"] == 0][0]
        out.append({"modo": "plazo", "parametro": f"{s['budget_s']} s",
                    "sin_ordenacion": nat["depth_mean"], "con_ordenacion": s["depth_mean"],
                    "medida": f"profundidad media completada ({s['n']} posiciones)",
                    "razon_con_sin": round(s["depth_mean"] / nat["depth_mean"], 4)
                    if nat["depth_mean"] else None})
    return out


# --------------------------------------------------------------------- Experimento 4
EXP4_CONFIGS = {"captures_L4": {}, "eval_L0": {"rollout_limit": 0},
                "uniform_L8": {"rollout_policy": "uniform", "rollout_limit": 8}}


def run_exp4(path: str | None = None, budgets=(50, 200, 800, 2000), seeds=range(5),
             only_default: bool = False) -> list:
    """Regenera el benchmark de tacticas (misma funcion que la etapa de MCTS)."""
    rows = []
    for label, kwargs in EXP4_CONFIGS.items():
        if only_default and label != "captures_L4":
            continue
        rows += mcts_tactics_benchmark(budgets=budgets, seeds=seeds, label=label, **kwargs)
    if path:
        mcts_rows_to_csv(rows, path)
    return rows


def exp4_load(path: str | None = None) -> list:
    path = path or os.path.join(EXP_RESULTS_DIR, "mcts_tacticas.csv")
    with open(path, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    for r in rows:
        for k in ("budget", "seed", "correct", "simulations", "nodes", "leaves", "max_depth"):
            r[k] = int(r[k])
        r["elapsed_s"] = float(r["elapsed_s"])
    return rows


def exp4_summary(rows) -> dict:
    """Tasa de acierto por presupuesto (con desviacion ENTRE semillas), tiempos y
    casos no monotonos por semilla."""
    by_budget = mcts_benchmark_summary(rows)
    for s in by_budget:
        sel = [r for r in rows if r["config"] == s["config"] and r["budget"] == s["budget"]]
        per_seed = []
        for seed in sorted({r["seed"] for r in sel}):
            hits = [r["correct"] for r in sel if r["seed"] == seed]
            per_seed.append(sum(hits) / len(hits))
        s["hit_rate_seed_std"] = round(statistics.pstdev(per_seed), 4)
        s["hit_rate_seed_min"] = min(per_seed)
        s["hit_rate_seed_max"] = max(per_seed)
        s["hit_rate"] = round(s["hit_rate"], 4)
        s["time_mean_s"] = round(s["time_mean_s"], 5)
        s["time_std_s"] = round(s["time_std_s"], 5)
    by_tactic = {}
    for r in rows:
        if r["config"] != "captures_L4":
            continue
        by_tactic.setdefault(r["tactic"], {}).setdefault(r["budget"], []).append(r["correct"])
    tactic_table = [{"tactic": t, **{f"b{b}": f"{sum(v)}/{len(v)}" for b, v in sorted(d.items())}}
                    for t, d in by_tactic.items()]
    return {"by_budget": by_budget, "by_tactic": tactic_table,
            "non_monotonic": mcts_non_monotonic_cases(rows)}


# --------------------------------------------------------------------- graficas
def _exp_plt():
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                                "axes.edgecolor": EXP_MUTED, "axes.labelcolor": EXP_INK,
                                "xtick.color": EXP_MUTED, "ytick.color": EXP_MUTED,
                                "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
                                "axes.axisbelow": True, "font.size": 9})
    return plt


def _exp_save(fig, path):
    if path:
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        fig.savefig(path, dpi=130, bbox_inches="tight", facecolor="white")
    return fig


def fig_exp1_nodes(rows, path: str | None = None):
    """Barras de nodos por algoritmo y posicion, un panel por profundidad (escala log)."""
    plt = _exp_plt()
    depths = sorted({r["depth"] for r in rows})
    positions = [p["id"] for p in EXP1_POSITIONS if any(r["position_id"] == p["id"] for r in rows)]
    fig, axes = plt.subplots(1, len(depths), figsize=(4.2 * len(depths), 3.6), sharey=False)
    axes = axes if len(depths) > 1 else [axes]
    width = 0.26
    for ax, depth in zip(axes, depths):
        for i, (label, _, _) in enumerate(EXP1_ALGORITHMS):
            vals = [next(r["nodes"] for r in rows if r["depth"] == depth and r["position_id"] == p
                         and r["algorithm"] == label) for p in positions]
            xs = [k + (i - 1) * width for k in range(len(positions))]
            ax.bar(xs, vals, width * 0.92, color=EXP_COLORS[i], label=label)
        ax.set_yscale("log")
        ax.set_title(f"Profundidad {depth}", color=EXP_INK)
        ax.set_xticks(range(len(positions)))
        ax.set_xticklabels([p.split("_", 1)[1] for p in positions], rotation=20)
        ax.grid(axis="x", visible=False)
    axes[0].set_ylabel("nodos visitados (escala log)")
    axes[0].legend(frameon=False, loc="upper left")
    fig.suptitle("Experimento 1: nodos de minimax y alfa-beta con la misma E0 (mismo valor)",
                 color=EXP_INK)
    fig.tight_layout()
    return _exp_save(fig, path)


def fig_depth_time(rows, path: str | None = None):
    """Profundidad completada contra plazo: mejorado (con ordenacion) vs sin ordenacion."""
    plt = _exp_plt()
    fig, ax = plt.subplots(figsize=(6.2, 3.8))
    for ordering, color, label in ((1, EXP_COLORS[0], "con ordenacion (agente mejorado)"),
                                   (0, EXP_COLORS[1], "sin ordenacion (orden natural)")):
        summ = [s for s in depth_time_summary(rows) if s["ordering"] == ordering]
        xs = [s["budget_s"] for s in summ]
        ax.fill_between(xs, [s["depth_min"] for s in summ], [s["depth_max"] for s in summ],
                        color=color, alpha=0.12, linewidth=0)
        ax.plot(xs, [s["depth_mean"] for s in summ], "-o", color=color, lw=2, ms=6, label=label)
    ax.set_xscale("log")
    ax.set_xticks(sorted({r["budget_s"] for r in rows}))
    ax.set_xticklabels([f"{b:g}" for b in sorted({r["budget_s"] for r in rows})])
    ax.set_xlabel("plazo por decision (s, escala log)")
    ax.set_ylabel("profundidad completada")
    ax.set_title("Profundizacion iterativa: media y rango sobre "
                 f"{len({r['position_id'] for r in rows})} posiciones", color=EXP_INK)
    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout()
    return _exp_save(fig, path)


def fig_mcts_budget(rows, path: str | None = None):
    """Tasa de acierto de MCTS por presupuesto (barras de error = desviacion entre
    semillas) y tiempo medio por decision."""
    plt = _exp_plt()
    summ = exp4_summary(rows)["by_budget"]
    configs = list(dict.fromkeys(s["config"] for s in summ))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.6))
    for i, cfg in enumerate(configs):
        sel = [s for s in summ if s["config"] == cfg]
        xs = [s["budget"] for s in sel]
        a1.errorbar(xs, [s["hit_rate"] for s in sel], yerr=[s["hit_rate_seed_std"] for s in sel],
                    fmt="-o", color=EXP_COLORS[i], lw=2, ms=6, capsize=3, label=cfg)
        a2.errorbar(xs, [s["time_mean_s"] for s in sel], yerr=[s["time_std_s"] for s in sel],
                    fmt="-o", color=EXP_COLORS[i], lw=2, ms=6, capsize=3, label=cfg)
    for ax in (a1, a2):
        ax.set_xscale("log")
        ax.set_xlabel("simulaciones por decision (escala log)")
    a1.set_ylim(0, 1.05)
    a1.set_ylabel("tasa de acierto (10 tacticas)")
    a1.set_title("Acierto tactico (± desv. entre 5 semillas)", color=EXP_INK)
    a2.set_yscale("log")
    a2.set_ylabel("tiempo por decision (s, log)")
    a2.set_title("Tiempo medio (± desv.)", color=EXP_INK)
    a1.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    return _exp_save(fig, path)


def fig_tournament(table, path: str | None = None):
    """Victorias, tablas, derrotas y truncadas por agente; puntos/terminadas a la derecha."""
    plt = _exp_plt()
    table = list(table)[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    names = [r["agente"] for r in table]
    left = [0] * len(table)
    parts = (("victorias", "victorias", EXP_COLORS[2]), ("tablas", "tablas", "#b9b8b2"),
             ("derrotas", "derrotas", EXP_COLORS[7]), ("truncadas", "truncadas (sin resultado)", "#e4e3df"))
    for key, label, color in parts:
        vals = [r[key] for r in table]
        ax.barh(names, vals, left=left, color=color, label=label, edgecolor="white", linewidth=1.5)
        left = [a + b for a, b in zip(left, vals)]
    for k, r in enumerate(table):
        ppt = r["puntos_por_terminada"]
        txt = f"{r['puntos']:g}/{r['terminadas']}" + (f" = {ppt:.2f}" if ppt is not None else "")
        ax.text(left[k] + 0.5, k, txt, va="center", color=EXP_INK)
    ax.set_xlim(0, max(left) * 1.3)
    ax.set_xlabel("partidas")
    ax.grid(axis="y", visible=False)
    ax.set_title("Experimento 2: resultados por agente (puntos / partidas terminadas)", color=EXP_INK)
    ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.45, -0.18))
    fig.tight_layout()
    return _exp_save(fig, path)


def fig_ablation(summary, ordering_rows, path: str | None = None):
    """Izquierda: puntuacion (IC 95 %) de E* y E0. Derecha: ahorro por la ordenacion."""
    plt = _exp_plt()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
    summary = list(summary)
    names = [s["comparacion"] for s in summary]
    for k, s in enumerate(summary):
        if s["puntuacion_media"] in ("", None):
            continue
        m, lo, hi = float(s["puntuacion_media"]), float(s["ic95_bajo"]), float(s["ic95_alto"])
        color = EXP_COLORS[1] if "E*" in s["agente"] else EXP_COLORS[0]
        a1.errorbar([m], [k], xerr=[[m - lo], [hi - m]], fmt="o", color=color, ms=8, capsize=4, lw=2)
        a1.text(1.02, k, f"{int(s['terminadas'])} term. / {int(s['truncadas'])} trunc.",
                va="center", color=EXP_MUTED, transform=a1.get_yaxis_transform())
    a1.axvline(0.5, color=EXP_MUTED, lw=1, ls="--")
    a1.set_yticks(range(len(names)))
    a1.set_yticklabels(names)
    a1.set_xlim(-0.03, 1.03)
    a1.set_xlabel("puntuacion media del agente (terminadas, IC 95 % bootstrap)")
    a1.set_title("Ablacion de la mejora: E0 vs E*", color=EXP_INK)
    a1.grid(axis="y", visible=False)
    fixed = [r for r in ordering_rows if r["modo"] == "profundidad fija"]
    timed = [r for r in ordering_rows if r["modo"] == "plazo"]
    xs = list(range(len(fixed)))
    a2.bar([x - 0.2 for x in xs], [r["sin_ordenacion"] for r in fixed], 0.38,
           color=EXP_COLORS[1], label="sin ordenacion")
    a2.bar([x + 0.2 for x in xs], [r["con_ordenacion"] for r in fixed], 0.38,
           color=EXP_COLORS[0], label="con ordenacion")
    a2.set_yscale("log")
    a2.set_xticks(xs)
    a2.set_xticklabels([r["parametro"] for r in fixed])
    a2.set_ylabel("nodos totales (5 posiciones, log)")
    a2.grid(axis="x", visible=False)
    sub = ", ".join(f"{r['parametro']}: {r['sin_ordenacion']:.2f}->{r['con_ordenacion']:.2f}"
                    for r in timed if r["parametro"] in ("0.2 s", "1.0 s"))
    a2.set_title("Ablacion de la ordenacion (profundidad fija)", color=EXP_INK)
    if sub:
        a2.text(0.02, -0.22, "Profundidad media con plazo " + sub, transform=a2.transAxes,
                color=EXP_MUTED)
    a2.legend(frameon=False, loc="upper left")
    fig.tight_layout()
    return _exp_save(fig, path)


def make_all_figures(out_dir: str | None = None) -> list:
    """Regenera todas las PNG desde los CSV que existan en `out_dir`."""
    plt = _exp_plt()
    out_dir = out_dir or EXP_RESULTS_DIR
    made = []

    def _p(name):
        return os.path.join(out_dir, name)

    exp1 = exp1_load(_p("exp1_ahorro.csv")) if os.path.exists(_p("exp1_ahorro.csv")) else None
    dt = (depth_time_load(_p("exp_profundidad_tiempo.csv"))
          if os.path.exists(_p("exp_profundidad_tiempo.csv")) else None)
    if exp1:
        plt.close(fig_exp1_nodes(exp1, _p("fig_exp1_nodos.png")))
        made.append(_p("fig_exp1_nodos.png"))
    if dt:
        plt.close(fig_depth_time(dt, _p("fig_profundidad_tiempo.png")))
        made.append(_p("fig_profundidad_tiempo.png"))
    mcts_path = _p("mcts_tacticas.csv")
    if not os.path.exists(mcts_path):
        mcts_path = os.path.join(EXP_RESULTS_DIR, "mcts_tacticas.csv")
    if os.path.exists(mcts_path):
        plt.close(fig_mcts_budget(exp4_load(mcts_path), _p("fig_mcts_presupuesto.png")))
        made.append(_p("fig_mcts_presupuesto.png"))
    if os.path.exists(_p("torneo_partidas.csv")):
        data = load_tournament(out_dir, "torneo")
        plt.close(fig_tournament(standings(data["games"]), _p("fig_torneo.png")))
        made.append(_p("fig_torneo.png"))
    if os.path.exists(_p("exp3_ablacion_partidas.csv")) and exp1 and dt:
        ab = ablation_load(out_dir)
        plt.close(fig_ablation(ab["summary"], ordering_ablation(exp1, dt), _p("fig_ablacion.png")))
        made.append(_p("fig_ablacion.png"))
    return made


In [ ]:
# ===== Experimento 1 EN VIVO: ahorro de busqueda con evaluacion fija (profundidades 1, 2 y 3) =====
# Se re-ejecuta completo aqui (unos segundos a un minuto). run_exp1 comprueba con assert que los
# tres algoritmos dan el MISMO valor y que el tablero queda intacto.
import os
import time

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

_t0 = time.perf_counter()
exp1_filas = run_exp1(depths=(1, 2, 3))
print(f"Experimento 1 re-ejecutado en {time.perf_counter() - _t0:.1f} s "
      f"({len(exp1_filas)} busquedas: 5 posiciones x 3 profundidades x 3 algoritmos).")

_lineas = ["| Posición | d | Algoritmo | Valor | Jugada | Nodos | Hojas | Podas | Tiempo (s) | Ordenar (s) | Nodos / minimax |",
           "|---|---:|---|---:|---|---:|---:|---:|---:|---:|---:|"]
for _r in exp1_filas:
    _lineas.append(f"| {_r['position_id']} | {_r['depth']} | {_r['algorithm']} | {_r['value']:+.1f} | "
                   f"{_r['move']} | {_r['nodes']} | {_r['leaves']} | {_r['cutoffs']} | "
                   f"{_r['time_s']:.3f} | {_r['sort_time_s']:.4f} | {_r['nodes_vs_minimax']:.3f} |")
display(Markdown("\n".join(_lineas)))

_lineas = ["| d | Algoritmo | Nodos (5 posiciones) | Hojas | Podas | Tiempo total (s) | de él, ordenar (s) | Nodos / minimax |",
           "|---:|---|---:|---:|---:|---:|---:|---:|"]
for _s in exp1_summary(exp1_filas):
    _lineas.append(f"| {_s['depth']} | {_s['algorithm']} | {_s['nodes']} | {_s['leaves']} | {_s['cutoffs']} | "
                   f"{_s['time_s']:.3f} | {_s['sort_time_s']:.4f} | {_s['nodes_vs_minimax']:.3f} |")
display(Markdown("\n".join(_lineas)))

# Valores y nodos son deterministas: deben coincidir con la corrida guardada (los tiempos no).
_ruta_exp1 = os.path.join("resultados", "exp1_ahorro.csv")
if os.path.exists(_ruta_exp1):
    _guardadas = {(r["position_id"], r["depth"], r["algorithm"]): r for r in exp1_load(_ruta_exp1)}
    _iguales = sum(1 for r in exp1_filas
                   if (k := (r["position_id"], r["depth"], r["algorithm"])) in _guardadas
                   and _guardadas[k]["nodes"] == r["nodes"] and abs(_guardadas[k]["value"] - r["value"]) < 1e-6)
    print(f"Contra resultados/exp1_ahorro.csv: {_iguales}/{len(exp1_filas)} filas con el mismo valor y "
          "los mismos nodos.")
    assert _iguales == len(exp1_filas)
else:
    print("AVISO: aun no existe resultados/exp1_ahorro.csv (lo escribe tools/correr_experimentos.py).")

fig_exp1_nodes(exp1_filas)
plt.show()


In [ ]:
# ===== Experimento 2: torneo todos contra todos (resultados guardados) + partida corta en vivo =====
# EJECUTAR_COMPLETO = True repite el torneo COMPLETO aqui mismo, con este mismo codigo. Son cientos
# de partidas con 0.2 s por jugada: en serie tarda horas. Por eso lo corrimos con
#     python tools/correr_experimentos.py
# (mismo paquete chess_ai, en paralelo con procesos) y aqui se CARGAN sus archivos de resultados/.
# Con False, ademas, se juega en vivo una partida corta para mostrar que el codigo corre.
import json
import os

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

EJECUTAR_COMPLETO = False

if EJECUTAR_COMPLETO:
    run_tournament(TournamentConfig(), out_dir="resultados")
else:
    _cfg_demo = TournamentConfig(agents=("Alfa-beta mejorado", "MCTS"), opening_ids=("A1_italiana",),
                                 max_plies=16, game_prefix="DEMO", seed_base=0)
    _spec = round_robin_schedule(_cfg_demo)[0]
    _demo = play_game(_spec, _cfg_demo)
    _g = _demo["game"]
    print(f"Partida de demostracion (en vivo): {_g['white_agent']} - {_g['black_agent']}, apertura "
          f"{_g['opening_id']}, {_cfg_demo.max_plies} plies de agentes como maximo, "
          f"{_cfg_demo.time_limit_s} s por jugada -> {_g['termination']} {_g['official_result']}")
    print(_demo["pgn"])
    print(f"{len(_demo['decisions'])} filas con las columnas CSV_FIELDS; ultimas tres:")
    for _row in _demo["decisions"][-3:]:
        print("  " + ", ".join(f"{k}={_row[k]}" for k in ("ply", "move_uci", "nodes", "cutoffs",
                                                         "completed_depth", "simulations",
                                                         "elapsed_s", "time_exceeded")))

_ruta_partidas = os.path.join("resultados", "torneo_partidas.csv")
if os.path.exists(_ruta_partidas):
    torneo = load_tournament("resultados", "torneo")
    torneo_analisis = tournament_analysis(torneo)
    _conf = torneo["config"]
    print(f"\nTorneo guardado: {torneo_analisis['n_games']} partidas, {len(torneo['decisions'])} decisiones; "
          f"truncadas: {torneo_analisis['truncated']}; terminaciones: {torneo_analisis['terminations']}")
    if _conf:
        print("Configuracion (resultados/torneo_config.json): " + json.dumps(
            {k: _conf[k] for k in list(_conf)[:12]}, ensure_ascii=False)[:900])
    display(Markdown("**Tabla del enunciado** (clasificación provisional: solo puntúan las partidas "
                     "terminadas; las truncadas van aparte y sin resultado)\n\n"
                     + standings_markdown(torneo_analisis["standings"])))
    _agentes = [r["agente"] for r in torneo_analisis["standings"]]
    display(Markdown("**Cruces** (puntos de la fila contra la columna / terminadas, t = truncadas)\n\n"
                     + pairwise_markdown(torneo_analisis["pairwise"], _agentes)))
    _lineas = ["| Agente | Jugadas | Tiempo medio (s) | Desv. (s) | Máximo (s) | Excesos | Nodos medios |",
               "|---|---:|---:|---:|---:|---:|---:|"]
    for _ag, _p in torneo_analisis["per_agent"].items():
        _lineas.append(f"| {_ag} | {_p['jugadas']} | {_p['tiempo_medio_s']:.4f} | {_p['tiempo_desv_s']:.4f} | "
                       f"{_p['tiempo_max_s']:.3f} | {_p['excesos']} ({100 * _p['prop_excesos']:.1f} %) | "
                       f"{_p['nodos_medios']:.0f} |")
    display(Markdown("**Tiempo y excesos por decisión** (los nodos de agentes distintos NO son "
                     "comparables: cada familia los define distinto)\n\n" + "\n".join(_lineas)))
    if "improved_depth" in torneo_analisis:
        print("Alfa-beta mejorado, profundidad completada por jugada:", torneo_analisis["improved_depth"])
    if "mcts_simulations" in torneo_analisis:
        print("MCTS, simulaciones por jugada con el plazo:", torneo_analisis["mcts_simulations"])
    fig_tournament(torneo_analisis["standings"])
    plt.show()
else:
    print("\nAVISO: aun no existe resultados/torneo_partidas.csv; ejecute tools/correr_experimentos.py "
          "y vuelva a construir el cuaderno.")

_ruta_dt = os.path.join("resultados", "exp_profundidad_tiempo.csv")
if os.path.exists(_ruta_dt):
    fig_depth_time(depth_time_load(_ruta_dt))       # curva de profundidad contra tiempo
    plt.show()


In [ ]:
# ===== Experimento 3: ablacion (E0 frente a E*, y ordenacion si/no) desde los resultados guardados =====
import os

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

if os.path.exists(os.path.join("resultados", "exp3_ablacion_partidas.csv")):
    ablacion = ablation_load("resultados")
    _lineas = ["| Comparación | Partidas | Terminadas | Truncadas | V | T | D | Puntuación media | IC 95 % | Material medio al truncar (no oficial) |",
               "|---|---:|---:|---:|---:|---:|---:|---:|---|---:|"]
    def _num(valor, formato):
        return "-" if valor in ("", None) else format(float(valor), formato)

    for _s in ablacion["summary"]:
        _ic = ("-" if _s["ic95_bajo"] in ("", None)
               else f"[{_num(_s['ic95_bajo'], '.3f')}, {_num(_s['ic95_alto'], '.3f')}]")
        _lineas.append(f"| {_s['comparacion']} | {_s['partidas']} | {_s['terminadas']} | {_s['truncadas']} | "
                       f"{_s['victorias']} | {_s['tablas']} | {_s['derrotas']} | "
                       f"{_num(_s['puntuacion_media'], '.3f')} | {_ic} | "
                       f"{_num(_s['material_medio_truncadas_no_oficial'], '+.2f')} |")
    display(Markdown("**Ablación en partidas: el mismo alfa-beta mejorado con $E_0$ y con $E^*$** "
                     "(mismo plazo, mismas aperturas, colores intercambiados)\n\n" + "\n".join(_lineas)))
    print("Profundidad media completada por etiqueta (E0 y E* cuestan lo mismo de evaluar):",
          ablation_depths(ablacion["decisions"]))
    _cfg_ab = ablacion["config"]
    if _cfg_ab:
        print(f"Configuracion: {_cfg_ab.get('description', '')}; partidas: {len(ablacion['games'])}")
else:
    ablacion = None
    print("AVISO: aun no existe resultados/exp3_ablacion_partidas.csv (lo escribe "
          "tools/correr_experimentos.py).")

# Ablacion de la ordenacion: profundidad fija (Exp. 1, en vivo arriba) y con plazo (curva guardada).
_ruta_dt = os.path.join("resultados", "exp_profundidad_tiempo.csv")
if os.path.exists(_ruta_dt):
    _orden = ordering_ablation(exp1_filas, depth_time_load(_ruta_dt))
    _lineas = ["| Modo | Parámetro | Medida | Sin ordenación | Con ordenación | Con / sin |",
               "|---|---|---|---:|---:|---:|"]
    for _o in _orden:
        _lineas.append(f"| {_o['modo']} | {_o['parametro']} | {_o['medida']} | {_o['sin_ordenacion']} | "
                       f"{_o['con_ordenacion']} | {_o['razon_con_sin']} |")
    display(Markdown("**Ablación de la ordenación** (el resto fijo)\n\n" + "\n".join(_lineas)))
    if ablacion is not None:
        fig_ablation(ablacion["summary"], _orden)
        plt.show()


In [ ]:
# ===== Experimento 4: MCTS por presupuesto de simulaciones y semilla (resultados guardados) =====
# El benchmark completo (3 configuraciones x 4 presupuestos x 5 semillas x 10 tacticas = 600 filas)
# tarda varios minutos; aqui se carga resultados/mcts_tacticas.csv y se REPITE en vivo una muestra
# para comprobar que, con simulaciones fijas, el resultado es exactamente reproducible.
import os

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

_ruta_mcts = os.path.join("resultados", "mcts_tacticas.csv")
if os.path.exists(_ruta_mcts):
    mcts_filas = exp4_load(_ruta_mcts)
    _res = exp4_summary(mcts_filas)
    _lineas = ["| Configuración | Simulaciones | Tasa de acierto | Desv. entre semillas | Mín.–máx. por semilla | Tiempo medio (s) | Desv. tiempo (s) |",
               "|---|---:|---:|---:|---|---:|---:|"]
    for _s in _res["by_budget"]:
        _lineas.append(f"| {_s['config']} | {_s['budget']} | {_s['hit_rate']:.2f} | {_s['hit_rate_seed_std']:.3f} | "
                       f"{_s['hit_rate_seed_min']:.1f}–{_s['hit_rate_seed_max']:.1f} | {_s['time_mean_s']:.3f} | "
                       f"{_s['time_std_s']:.3f} |")
    display(Markdown("\n".join(_lineas)))
    _cols = [k for k in _res["by_tactic"][0] if k != "tactic"]
    _lineas = ["| Táctica (captures_L4) | " + " | ".join(c[1:] + " sims" for c in _cols) + " |",
               "|---" * (len(_cols) + 1) + "|"]
    for _t in _res["by_tactic"]:
        _lineas.append(f"| {_t['tactic']} | " + " | ".join(_t[c] for c in _cols) + " |")
    display(Markdown("\n".join(_lineas)))
    print(f"Casos (tactica, semilla) no monotonos al crecer el presupuesto: {len(_res['non_monotonic'])}")

    # Muestra en vivo: 4 filas del CSV repetidas con el mismo codigo (jugada y simulaciones deben coincidir).
    _tac = {t["id"]: t for t in MCTS_TACTICS}
    _muestra = [r for r in mcts_filas if r["config"] == "captures_L4" and r["budget"] == 200
                and r["seed"] in (0, 1) and r["tactic"] in ("T3_mate1_b_backrank", "T7_avoid_b_scholar")]
    for _r in _muestra:
        _d = MCTSAgent(simulations=_r["budget"]).choose(chess.Board(_tac[_r["tactic"]]["fen"]),
                                                       time_limit_s=0, seed=_r["seed"])
        assert _d.move.uci() == _r["move"] and _d.stats.simulations == _r["simulations"]
    print(f"Repetidas en vivo {len(_muestra)} filas del CSV: misma jugada y mismas simulaciones.")
    fig_mcts_budget(mcts_filas)
    plt.show()
else:
    print("AVISO: falta resultados/mcts_tacticas.csv (benchmark del Experimento 4).")


### Tablas para completar
| Agente | Puntos / partidas terminadas | Victorias | Tablas | Derrotas | Truncadas | Tiempo medio |
|---|---|---|---|---|---|---|
| Aleatorio | | | | | | |
| Voraz | | | | | | |
| Minimax | | | | | | |
| Alfa-beta | | | | | | |
| Alfa-beta mejorado | | | | | | |
| MCTS | | | | | | |

Incluyan barras de nodos por algoritmo y posición, curvas de profundidad contra tiempo y tasa de éxito de MCTS por presupuesto. No comparen números de “nodos” de implementaciones con definiciones distintas sin explicarlo.


## 10. Cronograma propuesto
| Días | Hito verificable |
|---|---|
| 1–2 | PEAS, contratos, reglas y tablero gráfico funcionando |
| 3–4 | Aleatorio, voraz, evaluación y pruebas de perspectiva |
| 5–6 | Minimax y alfa-beta con equivalencia de valor |
| 7–8 | Iterative deepening, plazo e integración de modos |
| 9–10 | MCTS y mejora adicional |
| 11–12 | Integraciones de capítulos 3–4 y experimentos |
| 13–14 | Análisis, limpieza del cuaderno y sustentación |

No aumenten la profundidad hasta verificar legalidad, restauración del estado y tratamiento de terminales. Diseñen el torneo primero con dos agentes y una partida corta.


## 11. Rúbrica · 100 puntos
| Criterio | Puntos | Evidencia |
|---|---:|---|
| Formulación, PEAS y arquitectura | 10 | Modelo, contratos y justificación |
| Interfaz y reglas | 15 | Tres modos, controles, turnos y pruebas especiales |
| Seis agentes | 30 | Aleatorio 2, voraz 3, minimax 6, alfa-beta 6, mejorado 6, MCTS 7 |
| Integración de capítulos 3–4 | 10 | Torre, búsqueda informada/local e incertidumbre |
| Mejora adicional y ablación | 10 | Comparación controlada y explicación |
| Experimentos reproducibles | 15 | Torneo, métricas, semillas, PGN y CSV |
| Informe y sustentación individual | 10 | Interpretación y modificación de código en vivo |
| **Total** | **100** | |

Conversión sugerida a escala de 0 a 5: puntos/20, sujeta a las reglas de evaluación del docente. La existencia de una interfaz vistosa no reemplaza la implementación de búsqueda. Los resultados inventados no cuentan como experimentos.

### Uso de herramientas y autoría
Declaren librerías, fuentes y ayudas de IA utilizadas. Cada integrante debe explicar el código, ejecutar una prueba nueva y modificar una decisión del algoritmo durante la sustentación. No se exige ocultar las herramientas usadas; se exige comprender y verificar lo entregado.


### Autoevaluación con la rúbrica (evidencia y huecos)

| Criterio | Puntos | Evidencia en este cuaderno y el repositorio | Huecos honestos |
|---|---:|---|---|
| Formulación, PEAS y arquitectura | 10 | §2 PEAS, estado con historia, clasificación e invariantes con su prueba; §4 tabla de módulos; contrato del enunciado sin cambios | — |
| Interfaz y reglas | 15 | §3 y §6: `ChessApp` con los tres modos, selector por color con los 6 agentes y Humano, Jugar turno / Iniciar-Pausar / Nueva / Deshacer (política explícita), profundidad, plazo y semilla, registro por turno, PGN y CSV; §8 reglas especiales (enroque, al paso, promoción, repetición, mate y ahogado) | La interacción es por listas, no por clic en casillas (mejora opcional). Pausar actúa entre turnos, no dentro de una búsqueda (documentado). |
| Seis agentes | 30 | §5 A–D: aleatorio, voraz, minimax, alfa-beta, alfa-beta mejorado (ordenación + profundización con plazo) y MCTS-UCT; §8 pruebas 1–7 | — |
| Integración caps. 3–4 | 10 | §7: torre con BFS/DFS/UCS, A\* y voraz con admisibilidad verificada exhaustivamente; búsqueda local de pesos; AND-OR, expectiminimax y Kriegspiel | — |
| Mejora adicional y ablación | 10 | Ajuste de pesos (§5 D y §7C, ablación por peso y en prueba); §9 Exp. 3: E0 contra E\* en partidas y ordenación sí/no | La mejora en la función objetivo es pequeña y no significativa (lo decimos); el resultado en partidas depende del Exp. 3. |
| Experimentos reproducibles | 15 | §9: Exp. 1 en vivo (valores y nodos idénticos a los guardados); torneo con aperturas y semillas comunes, colores intercambiados, CSV, PGN y JSON de configuración; Exp. 4 con 5 semillas y 4 presupuestos | MCTS y el mejorado con plazo de reloj no se repiten jugada a jugada en otra máquina (por diseño, lo decimos). |
| Informe y sustentación individual | 10 | §12 respuestas con evidencia y conclusiones individuales; guías de estudio personales | Se demuestra en la sustentación: cada integrante debe ejecutar una prueba nueva y modificar el código en vivo. |

## 12. Sustentación y conclusiones
Respondan con evidencia de su implementación:
1. Muestre una posición donde el voraz falla y minimax encuentra una defensa.
2. Trace a mano una poda real de su registro. ¿Por qué es válida?
3. ¿Por qué minimax y alfa-beta podrían elegir jugadas distintas con el mismo valor?
4. Cambie la perspectiva de evaluación. ¿Qué errores aparecerían si solo cambia un signo?
5. ¿Qué retorna su programa si vence el plazo durante una iteración?
6. ¿Cómo evita que MCTS elija respuestas favorables al adversario equivocado?
7. ¿Qué conserva el historial que no contiene un FEN aislado?
8. ¿Qué parte del microproblema de torre deja de servir cuando existe oposición?
9. ¿Cómo sabe que los pesos ajustados no se sobreajustaron?
10. ¿Qué evidencia tiene para afirmar que un agente es mejor y bajo qué presupuesto?

**Conclusión individual:** describa una decisión de diseño que cambió después de observar un fallo. Incluya el caso que lo reveló y cómo comprobó la corrección.

## Lista de entrega
- Cuaderno ejecutado, con integrantes y conclusiones individuales.
- Módulos propios y README con instalación y ejecución.
- Dependencias con versiones del entorno probado.
- CSV de decisiones y resultados, PGN y configuración del torneo.
- Posiciones de entrenamiento/prueba separadas y análisis de ablación.
- Declaración de fuentes y ayudas utilizadas.

Los datos de salida deben provenir de sus ejecuciones. Los espacios vacíos de este enunciado son parte del trabajo por realizar.

### Referencias
- Russell y Norvig, *AIMA*, 4.ª edición, capítulos 2–5. [Índice oficial](https://aima.cs.berkeley.edu/contents.html).
- [python-chess: reglas y estados](https://python-chess.readthedocs.io/en/latest/core.html).
- [python-chess: SVG](https://python-chess.readthedocs.io/en/latest/svg.html).
- [python-chess: PGN](https://python-chess.readthedocs.io/en/latest/pgn.html).
- [ipywidgets: documentación](https://ipywidgets.readthedocs.io/en/stable/).


### Respuestas de sustentación (con evidencia de nuestra implementación)

**1. Una posición donde el voraz falla y minimax encuentra una defensa.** `MINIMAX_DEFENSE_FEN = 4r1k1/ppp2pp1/8/4p2p/3N4/8/PPP2PPP/3Q2K1 w`: el voraz (= minimax de profundidad 1) juega Dxh5 (+786,7) y no ve exd4. Minimax-2 calcula que Dxh5 exd4 vale +432,8 y elige Cf5 (+673,4), salvando el caballo (celda de demostración B). En `GREEDY_TRAP_FEN` el voraz juega Dxb7 y recibe ...Te1#; minimax-2 juega Db4, que cubre e1.

**2. Una poda real trazada a mano.** Está en §5 B, con el registro de `AlphaBetaAgent(depth=2, log_prunes=True)`. La raíz MAX ya tiene α = 673,4 (Cf5). En el nodo MIN de Cf3, la primera respuesta Rh8 vale 669,4 ≤ α, así que se podan 21 respuestas. Es válida porque un nodo MIN vale como máximo 669,4 y MAX ya tiene 673,4: ninguna respuesta no mirada puede hacer que MAX prefiera Cf3. El valor de la raíz no cambia (190 nodos frente a 757, mismo valor que minimax).

**3. ¿Por qué minimax y alfa-beta podrían elegir jugadas distintas con el mismo valor?** Guardamos el *primer* hijo que alcanza el mejor valor (`>` estricto). Si dos jugadas empatan, el orden de recorrido decide cuál sale; alfa-beta ordenado recorre en otro orden. Además, para las ramas podadas alfa-beta solo conoce cotas. La prueba 1 de §8 exige igualdad exacta del valor y, si la jugada difiere, que ambas alcancen ese valor.

**4. Cambiar la perspectiva de evaluación: errores si solo se cambia un signo.** E está desde blancas y cada agente la convierte con `perspective_sign` una sola vez, en la raíz (D-06). Si se cambia un signo solo en E (p. ej. «desde el jugador al turno») sin cambiar dónde se maximiza y minimiza, los nodos MIN minimizarían un valor que ya está invertido: negras elegirían lo bueno para blancas y los mates se respaldarían con el signo equivocado (un mate en contra parecería una victoria). Las pruebas que lo detectan: negras capturan la dama (`7k/7p/8/8/Q7/2n5/7P/4K3 b`, Cxa4), el mate en uno con el tablero reflejado y la antisimetría de E (§5).

**5. ¿Qué retorna si vence el plazo durante una iteración?** La jugada y el valor de la **última iteración completa**, con `completed_depth` igual a esa profundidad; la iteración parcial se descarta y los `finally` deshacen el tablero. Si ni la profundidad 1 terminó, la jugada de respaldo (legal), `score=None` y `completed_depth=0`. Nunca `None` en una posición no terminal (celda C y prueba 6).

**6. ¿Cómo evita MCTS elegir respuestas favorables al adversario equivocado?** Cada nodo acumula la recompensa desde el jugador que hizo la jugada que lleva a él (`_reward_for`: r o 1−r). Así UCT, en cada nodo, maximiza para el jugador al turno. Lo probamos con una variante con el error (todo desde blancas): con negras al turno, en T3 y T6, el agente correcto acierta y la variante falla con las mismas semillas; y recorremos el árbol para verificar el dueño de cada nodo (prueba 7, M-07, M-E2).

**7. ¿Qué conserva el historial que no contiene un FEN aislado?** Las posiciones anteriores: sin ellas no se detecta la repetición triple (D-08). Tras Cf3 Cf6 Cg1 Cg8 dos veces, la pila detecta la repetición y el FEN solo no. Por eso los agentes reciben `board.copy(stack=True)` y el controlador compara `version`, no el FEN.

**8. ¿Qué parte del microproblema de torre deja de servir con oposición?** La solución como **secuencia fija** (con rival hace falta un plan contingente: AND-OR o minimax), la prueba objetivo «llegar a una casilla» (el mate es una propiedad de **todas** las respuestas), el costo como suma de pasos (se optimiza el valor minimax) y el modelo de transición determinista de un solo agente. Ejemplo: 1.Rf6 Rh7 2.Dg7# no cubre Rh5 ni a4, y el verificador la rechaza (§7 D).

**9. ¿Cómo sabemos que los pesos no se sobreajustaron?** Separamos entrenamiento y prueba por partida y por FEN, y elegimos método y pesos solo con entrenamiento. La mejora aparece en prueba (−1,3 %), pero es pequeña y no significativa (diferencia pareada −0,00028 ± 0,00056 en 240 partidas). La ablación muestra sobreajuste leve de wk y wa: devolverlos a su valor inicial *mejora* la prueba. No cambiamos los pesos por eso, porque sería elegir con prueba. Riesgo mayor: cambio de distribución (posiciones de juego débil), que mide la ablación en partidas (Exp. 3) (§7C, T-09).

**10. ¿Qué evidencia tenemos para afirmar que un agente es mejor, y bajo qué presupuesto?** Solo la del torneo y la ablación, con el presupuesto de **0,2 s por decisión**, las mismas aperturas para todos los pares, colores intercambiados y puntuación únicamente sobre partidas terminadas (las truncadas van aparte); ver §9, Experimentos 2 y 3. La fuerza a profundidad fija no se mide con esto (eso es el Exp. 1), y el resultado puede cambiar con otro plazo u otro equipo.



---

### Conclusiones individuales (BORRADORES: cada integrante debe revisarlas y hacerlas suyas)

**Santiago Guevara: la prueba de perspectiva de MCTS que pasaba por suerte (M-E2).** Al principio validábamos la perspectiva de MCTS con tácticas de «evitar mate», como T7, esperando que una variante con el error (guardar todo desde blancas) fallara. Con el rollout por defecto la variante errónea eligió d8h4, una de las 8 defensas válidas de 28, y la prueba falló; en una exploración previa incluso «acertaba» un mate en dos que el agente correcto no veía. Aprendí que un acierto aislado no demuestra que el código esté bien. Cambiamos la decisión: la prueba usa tácticas con solución única (T3, mate Ta1#, y T6, dama colgada, ambas con negras al turno), donde la variante errónea falla con las tres semillas, y añadimos un recorrido estructural del árbol que verifica que cada nodo pertenece al jugador que movió. La corrección se comprobó con esa prueba (prueba 7 de §8 y `tests/test_mcts.py`).

**Januar Mena: los terminales antes que E también en las posiciones de prueba (S-16 y S-17).** Para probar que con negras al turno el agente elige lo bueno para negras, usamos `7k/8/8/8/Q7/2n5/8/4K3 b`: negras capturan la dama. Minimax elegía Cxa4, pero con valor 0 a toda profundidad: tras la captura queda rey y caballo contra rey, material insuficiente, y la búsqueda declara tablas antes de consultar E, como manda nuestra regla D-07. El programa tenía razón y la posición estaba mal escogida. Después, la expectativa de valor (> 500) también era incorrecta: confundí lo que gana la jugada con el valor de la posición resultante, y además la prueba dependía del reloj. La decisión que cambió: las posiciones de prueba se verifican primero con el propio programa (añadimos peones en h7 y h2), las pruebas de comportamiento se ejecutan sin plazo (`time_limit_s=0`) y el umbral se justificó (> 200, Cxa4 vale +336,5). Se comprobó con 3 corridas seguidas y 4 en paralelo con la máquina cargada.

### Lista de entrega: estado y dónde está cada cosa

| | Punto | Dónde |
|---|---|---|
| ✅ | Cuaderno ejecutado, con integrantes y conclusiones individuales | este archivo (portada, §2 y §12); se ejecuta en un kernel nuevo con `tools/ejecutar_cuaderno.py` |
| ✅ | Módulos propios y README con instalación y ejecución | `chess_ai/` (un módulo por responsabilidad), `tests/`, `README.md` |
| ✅ | Dependencias con versiones del entorno probado | `requirements.txt` (Python 3.13.7, Windows 11); la celda de §1 las compara con el kernel |
| ✅ | CSV de decisiones y resultados, PGN y configuración del torneo | `resultados/torneo_decisiones.csv`, `torneo_partidas.csv`, `torneo.pgn`, `torneo_config.json`; ablación `exp3_ablacion_*`; Exp. 1 `exp1_ahorro.csv`; Exp. 4 `mcts_tacticas.csv` |
| ✅ | Posiciones de entrenamiento/prueba separadas y análisis de ablación | `resultados/tuning_posiciones.csv` (columna `split`), `pesos_ajustados.json`, `tuning_ablacion.csv`, `tuning_historial.csv`; §7C y §9 Exp. 3 |
| ✅ | Declaración de fuentes y ayudas utilizadas | portada del cuaderno y README |
| ⬜ | Cada integrante explica, ejecuta una prueba nueva y modifica el código en vivo | se demuestra en la sustentación |